In [ ]:
# =============================================================================
# CELL A0: Notebook overview and usage
#
#   The Labeling Problem in Hallucination Detection Benchmarks:
#                  An Empirical Evaluation
#
#
#   Reproducibility Notebook
#
# =============================================================================
#
# This notebook reproduces all computed tables and figures reported in the paper.
# It has two modes:
#
#   RELEASE ANALYSIS MODE (default)
#     Reproduces all paper tables and figures from the released master CSV
#     and the released audit CSVs under audits/.
#     Does not call any APIs, does not load any large models.
#     No GPU required. No API keys required.
#     Estimated runtime: under 10 minutes on CPU.
#
#   LABEL REGENERATION MODE (optional)
#     Recomputes all automatic labels from scratch:
#       - ROUGE-L and BERTScore baselines
#       - NLI entailment baseline (T5-11B, ~30 GB VRAM)
#       - OpenAI API judges (GPT-5-mini, GPT-5-nano, GPT-5.4)
#       - Anthropic API judge (Claude Opus 4.7)
#       - Local open-weight judges (Gemma-2-9B, Qwen2.5-7B, Llama-3-8B)
#       - METEOR / BARTScore-CNN baselines
#       - The criterion x structure factorial prompt experiment
#       - The elaboration intervention (core-answer extraction + re-judging)
#     Requires GPU, API keys, and significant compute time and cost.
#     See CELL A1 for flags to enable individual components.
#
#   NOTE: The generated answers (answer_text) were produced outside this
#   notebook; see the paper for the generation settings. The master CSV
#   contains the canonical answers, which are not regenerated here.
#
#   This repository can be run locally (a requirements.txt is provided) or
#   in Colab. Section B was originally run in Colab; a local run needs the
#   same API keys and, for the GPU-heavy cells, a local or cloud GPU.
#
# =============================================================================
# QUICK START — reproduce paper results only
# =============================================================================
#
#   Step 1.  Run the notebook from the repository root, or set REPRO_ROOT.
#
#   Step 2.  Confirm data/human_judge_release_master.csv is present.
#
#   Step 3.  Run cells in order:
#              CELL A1  — execution flags (all regeneration flags are False by default)
#              CELL A2  — config, paths, validate master CSV
#              CELL A3  — initialize temp_results.csv sandbox
#              CELL C0  — release master integrity check (recomputes 106
#                         values from raw columns and checks them against
#                         the manuscript; not a paper table itself)
#              CELL C1  — inter-annotator agreement table (tab:iaa)
#              CELL C2  — prompt-ablation tables (tab:results_prompt, tab:results_full)
#              CELL C3  — lexical/NLI baseline table (tab:results_lexical)
#              CELL C3b — METEOR/BARTScore baseline table (tab:results_new_metrics)
#              CELL C4  — pairwise kappa matrix (fig:kappa_matrix, fig:kappa_matrix_per_generator)
#              CELL C5  — bootstrap CIs + McNemar tests (tab:prompt_comparisons)
#              CELL C6  — error-direction space figure (fig:error_direction_space)
#              CELL C7  — per-dataset prompt-ablation breakdown (audit CSVs only)
#              CELL C8  — GPT-5-mini per-dataset kappa table (tab:per_dataset_gpt5mini)
#              CELL C9  — criterion x structure factorial tables (tab:factorial_agreement, tab:factorial_contrasts)
#              CELL C10 — qualitative FP -> TN examples (tab:elaboration_examples)
#              CELL C11 — answer-length reduction (Appendix E / Section 5.1 numbers)
#
#   After installing requirements.txt, no API keys and no GPU are needed
#   for the above.
#
#   Faster alternative — command-line scripts, no notebook required:
#     python scripts/00_validate_inputs.py
#     python scripts/20_run_release_analyses.py
#     python scripts/31_verify_paper_tables.py
#   The last command must end with "Blocking findings: 0". This reproduces
#   every paper table and figure produced by CELL C1-C11 (tab:results_prompt,
#   tab:results_lexical, tab:results_new_metrics, tab:iaa,
#   tab:prompt_comparisons, fig:error_direction_space,
#   tab:per_dataset_gpt5mini, tab:results_full, tab:factorial_agreement,
#   tab:factorial_contrasts, tab:elaboration_examples, and the paper's
#   kappa-matrix figures); it does not reproduce CELL C0's integrity check
#   (see src/ and scripts/ for what is and is not ported from the notebook).
#
# =============================================================================
# FULL RUN — regenerate all labels from scratch
# =============================================================================
#
#   Prerequisites before running B cells:
#
#   1. GPU runtime
#      Required for: B2 (BERTScore), B3 (NLI), B6 (local judges), B9 (BARTScore)
#      Colab:  Runtime -> Change runtime type -> GPU (A100 recommended, 40+ GB VRAM)
#      Local:  any CUDA-capable GPU meeting the same VRAM requirement
#
#   2. API keys
#      Required:
#        OPENAI_API_KEY      — B4, B7, B8, B10, B10b (OpenAI judges/extraction)
#        ANTHROPIC_API_KEY   — B5, B7 (Anthropic judge)
#        HF_TOKEN            — B6 (gated models: Llama, Gemma)
#      Colab:  add as Colab Secrets ( icon in the left sidebar)
#      Local:  set as environment variables
#      Do not hard-code keys into the notebook.
#
#   3. HuggingFace model access
#      Accept terms of use for gated models before running B6:
#        meta-llama/Meta-Llama-3-8B-Instruct  — https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
#        google/gemma-2-9b-it                 — https://huggingface.co/google/gemma-2-9b-it
#
#   4. Execution flags in CELL A1
#      Set the following:
#        RUN_LABEL_REGENERATION = True
#        ALLOW_EXPENSIVE_RUNS   = True
#      Then enable individual sub-flags as needed:
#        RUN_ROUGE_LABELING              = True   # B2
#        RUN_BERTSCORE_LABELING          = True   # B2
#        RUN_NLI_LABELING                = True   # B3
#        RUN_OPENAI_LABELING             = True   # B4
#        RUN_ANTHROPIC_LABELING          = True   # B5
#        RUN_LOCAL_LABELING              = True   # B6
#        RUN_SIX_BRANCH_SANITY           = True   # B7
#        RUN_FACTORIAL_PROMPT_EXPERIMENT = True   # B8
#        RUN_METEOR_LABELING             = True   # B9
#        RUN_BARTSCORE_LABELING          = True   # B9
#        RUN_B10                         = False  # B10  (off by default; set True to re-run)
#        RUN_B10B                        = False  # B10b (off by default; calls the OpenAI API)
#
#   Run B cells in this order:
#     B1 -> (install lexical deps) -> B2 -> B3 -> B4
#        -> (install anthropic) -> B5 -> B6 -> B7 -> B8 -> B9 -> B10 -> B10b
#   Then run C cells as above.
#
#   Approximate time and cost per B cell (indicative, varies by provider):
#     B2  ROUGE-L / BERTScore      ~10 min on GPU,    no API cost
#     B3  NLI (T5-11B)             ~20 min on A100,   no API cost
#     B4  OpenAI judges            ~2–4 hours,        ~$20–40 USD
#     B5  Anthropic judge          ~1–2 hours,        ~$15-30 USD
#     B6  Local judges             ~2–4 hours on A100, no API cost
#     B7  SIX-BRANCH sanity       ~15 min,           minimal API cost
#     B8  Factorial experiment     not separately benchmarked; OpenAI API cost only, no GPU
#     B9  METEOR / BARTScore       not separately benchmarked; GPU only, no API cost
#     B10  Core-answer extraction  not separately benchmarked; OpenAI API cost only, no GPU
#     B10b P1 original-vs-core     not separately benchmarked; OpenAI API cost only, no GPU
#
#   Note on reproducibility of regenerated labels:
#     API judge outputs (B4, B5, B7, B8, B10, B10b) are not guaranteed to be
#     perfectly reproducible across runs due to provider-side model updates
#     and sampling. Small label differences from the release master are
#     expected and documented in the audit CSVs.
#     Local judge outputs (B6) may also differ due to GPU
#     floating-point non-determinism under greedy decoding.
#     The release master CSV is never modified by any B cell.
#
# =============================================================================
# CELL STRUCTURE
# =============================================================================
#
#   A cells — Setup and initialization
#     A1   Execution flags and resource guards
#     A2   Config, paths, imports, release master validation
#     A3   Initialize temp_results.csv sandbox
#
#   B cells — Label regeneration (optional, expensive)
#     B1   Prompt definitions (P1, P2, P3)
#          (pip-install cell, run before B2: rouge-score, bert-score)
#     B2   ROUGE-L / BERTScore recomputation
#     B3   NLI entailment baseline recomputation (T5-11B)
#     B4   OpenAI judge labeling (GPT-5-mini, GPT-5-nano, GPT-5.4)
#          (pip-install cell, run before B5: anthropic)
#     B5   Anthropic judge labeling (Claude Opus 4.7)
#     B6   Local open-weight judge labeling (Gemma, Qwen, Llama)
#     B7   SIX-BRANCH sanity check (diagnostic only)
#     B8   Criterion x structure factorial prompt experiment (feeds tab:factorial_agreement, tab:factorial_contrasts)
#     B9   METEOR / BARTScore-CNN baselines (feeds tab:results_new_metrics)
#     B10  Core-only answer extraction for the elaboration intervention
#     B10b GPT-5-mini P1, original vs. core-only (feeds tab:elaboration_examples, Appendix E)
#
#   C cells — Analysis and output generation
#     C0   Release master integrity check (106 recomputed checks; not a
#          paper table)
#     C1   Inter-annotator agreement table (tab:iaa)
#     C2   Prompt-ablation tables (tab:results_prompt, tab:results_full)
#     C3   Lexical/NLI baseline table (tab:results_lexical)
#     C3b  METEOR/BARTScore baseline table (tab:results_new_metrics)
#     C4   Pairwise inter-method kappa matrix (fig:kappa_matrix, fig:kappa_matrix_per_generator)
#     C5   Bootstrap CIs + McNemar tests (tab:prompt_comparisons)
#     C6   Error-direction space figure (fig:error_direction_space)
#     C7   Per-dataset prompt-ablation breakdown (audit CSVs only)
#     C8   GPT-5-mini per-dataset kappa table (tab:per_dataset_gpt5mini)
#     C9   Criterion x structure factorial tables (tab:factorial_agreement, tab:factorial_contrasts)
#     C10  Writes the example CSV and displays the examples behind tab:elaboration_examples
#     C11  Answer-length reduction for the elaboration intervention
#          (Appendix E / Section 5.1 numbers; no table of its own)
#
# =============================================================================
# RELEASED ASSETS
# =============================================================================
#
#   human_judge_release_master.csv
#     Analysis-ready master CSV with all 900 question-answer pairs,
#     human annotations (annotator1_label, annotator2_label), and all
#     automatic labels. This is the single source of truth for all
#     release analyses, together with the released audit CSVs (audits/)
#     read by CELL C9, C10 and C11. It is never modified by this notebook.
#
#   human_judge_release_data_dictionary.csv
#     Column-by-column description of all fields in the master CSV.
#
#   Judge prompts
#     The exact prompt texts (P1, P2, P3) used for all LLM judge
#     evaluations are defined in CELL B1 and reproduced in Appendix B
#     of the paper. The auxiliary prompts p1s and p2t (used only in the
#     B8 criterion x structure factorial experiment) are defined in
#     CELL B8. The prompts/ folder contains the text of every prompt
#     (p1, p1s, p2, p2t, p3) as a standalone file.
#
#   audits/
#     Audit CSVs produced by B and C cells documenting label-level
#     reproducibility checks against the release master.
#
# =============================================================================
# PYTHON REQUIREMENTS
# =============================================================================
#
#   This repository ships a requirements.txt. Install it locally with:
#     pip install -r requirements.txt
#   Its core section covers release analysis only (A + C cells):
#     pandas, numpy, scikit-learn, matplotlib, seaborn, scipy
#   These are also present in the standard Colab environment by default.
#
#   Label regeneration (B cells) needs the packages listed in
#   requirements.txt's commented-out optional section — install only what
#   the sub-flags you enable actually need:
#     rouge-score, bert-score      B2
#     torch, transformers          B3, B6, B9 (standard in a Colab GPU runtime)
#     nltk                         B9 (METEOR)
#     openai                       B4, B7, B8, B10, B10b
#     anthropic                    B5, B7
#     huggingface_hub              B6, B7
#     pydantic                     B10
#     tqdm                         most Section B cells
#
# =============================================================================

print("=" * 70)
print("  Hallucination Labeling — Reproducibility Notebook")
print("=" * 70)
print()
print("Default mode: RELEASE ANALYSIS")
print("  → Reproduces all paper tables and figures from released master CSV")
print("    and the released audit CSVs under audits/.")
print("  → After installing requirements.txt, no API keys and no GPU are needed.")
print()
print("To regenerate labels from scratch:")
print("  1. GPU runtime (Colab: Runtime menu; locally: any CUDA-capable GPU)")
print("  2. Set OPENAI_API_KEY, ANTHROPIC_API_KEY, HF_TOKEN (Colab Secrets or env vars)")
print("  3. Accept HuggingFace terms for Llama and Gemma")
print("  4. Set in CELL A1:")
print("       RUN_LABEL_REGENERATION = True")
print("       ALLOW_EXPENSIVE_RUNS   = True")
print()
print("See README.md and the comments in this cell for full instructions and cost estimates.")
print("=" * 70)


In [ ]:
# =============================================================================
# CELL A1: Execution mode + safety/resource guards
#
# Purpose:
#   Control whether the notebook runs only release analyses or also regenerates
#   labels. Expensive/API/GPU-heavy cells must be explicitly enabled.
#
# Default mode:
#   - Reproduce paper tables/figures from release master CSV.
#   - Do NOT regenerate labels.
#   - Do NOT call APIs.
#   - Do NOT load large local models.
# =============================================================================

import importlib
from pathlib import Path

# =============================================================================
# Main execution mode
# =============================================================================
# Default: reproduce paper results from existing release master.
RUN_RELEASE_ANALYSIS = True

# Set True only if intentionally recomputing labels.
RUN_LABEL_REGENERATION = False

# Extra safety switch for expensive cells.
# Expensive cells should require BOTH:
#   RUN_LABEL_REGENERATION=True
#   ALLOW_EXPENSIVE_RUNS=True
ALLOW_EXPENSIVE_RUNS = False

# Optional diagnostic, not needed for ordinary release analysis.
RUN_SIX_BRANCH_SANITY = False

# Appendix-only factorial prompt experiment (B8). Already run; the tables are
# rebuilt from stored labels in C9.
RUN_FACTORIAL_PROMPT_EXPERIMENT = False

# Set True only to re-run B10 / B10b; B10b calls the OpenAI API.
RUN_B10 = False
RUN_B10B = False

# =============================================================================
# Label-regeneration sub-flags
# =============================================================================
# Keep all False unless intentionally rebuilding label columns.

RUN_ROUGE_LABELING = False
RUN_BERTSCORE_LABELING = False
RUN_NLI_LABELING = False


RUN_METEOR_LABELING = False
RUN_BARTSCORE_LABELING = False

RUN_OPENAI_LABELING = False
RUN_ANTHROPIC_LABELING = False
RUN_LOCAL_LABELING = False
FORCE_REINIT_TEMP = False

# =============================================================================
# How to re-run Section B (B10 / B10b)
# =============================================================================
#
#   1. Set OPENAI_API_KEY (required by both B10 and B10b); set
#      ANTHROPIC_API_KEY and HF_TOKEN too if other B cells are being
#      re-run in the same pass.
#   2. Set RUN_B10 = True and/or RUN_B10B = True above.
#   3. Use a NEW CORE_RUN_ID (set it as a global before running B10) rather
#      than reusing an existing run directory.
#   4. Outputs are written under audits/, never under data/: B10 writes to
#      audits/elaboration_core_only/<CORE_RUN_ID>/, and B10b writes to a
#      p1_original_vs_core_b4matched/ subfolder of that same run directory.
#   5. B10b additionally requires a human-reviewed
#      human_validation_template_validated.csv to already exist in that run
#      directory; no cell regenerates this file automatically.
#
# =============================================================================

# =============================================================================
# Resource guard helpers
# =============================================================================

def require_release_analysis_enabled():
    """Guard for paper-analysis cells that read the release master."""
    if not RUN_RELEASE_ANALYSIS:
        raise RuntimeError(
            "Release analysis is disabled.\n"
            "Set RUN_RELEASE_ANALYSIS=True to run paper analysis cells."
        )


def require_label_regeneration_enabled():
    """Guard for cells that recompute labels."""
    if not RUN_LABEL_REGENERATION:
        raise RuntimeError(
            "Label regeneration is disabled.\n"
            "This cell is only needed if you want to recompute labels.\n"
            "Set RUN_LABEL_REGENERATION=True to continue."
        )


def require_expensive_runs_allowed(reason="This cell may be expensive."):
    """Guard for API/GPU/memory-heavy cells."""
    if not ALLOW_EXPENSIVE_RUNS:
        raise RuntimeError(
            reason + "\n"
            "Set ALLOW_EXPENSIVE_RUNS=True only after confirming that you "
            "want to run this computation."
        )


def require_flag(flag_name, flag_value):
    """Require a specific RUN_* flag."""
    if not flag_value:
        raise RuntimeError(
            f"{flag_name}=False.\n"
            f"Set {flag_name}=True only if you intentionally want to run this cell."
        )


def require_package(import_name, pip_name=None):
    """
    Check whether a Python package is importable.

    Does not auto-install packages, because dependencies should be installed
    explicitly from requirements.txt.
    """
    pip_name = pip_name or import_name

    spec = importlib.util.find_spec(import_name)
    if spec is None:
        raise ImportError(
            f"Missing package '{import_name}'. Install dependencies with:\n"
            f"    pip install -r requirements.txt\n"
            f"or install this package explicitly with:\n"
            f"    pip install {pip_name}"
        )


def require_cuda(min_total_gb=None):
    """
    Require CUDA GPU and optionally a minimum total GPU memory amount.

    Returns:
        total GPU memory in GB
    """
    require_package("torch")

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU is not available.\n"
            "This cell is expected to be very slow or infeasible on CPU.\n"
            "Use a GPU runtime before running it."
        )

    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / (1024 ** 3)

    print(f"✓ CUDA available: {torch.cuda.get_device_name(0)}")
    print(f"  Total GPU memory: {total_gb:.1f} GB")

    if min_total_gb is not None and total_gb < min_total_gb:
        raise RuntimeError(
            f"GPU memory may be insufficient: {total_gb:.1f} GB available, "
            f"but this cell requests at least {min_total_gb:.1f} GB.\n"
            "Use a larger GPU runtime, reduce batch size, or use quantization."
        )

    return total_gb


def print_resource_warning(task_name, gpu_required=True, min_gpu_gb=None):
    """Print a clear warning before expensive computations."""
    print("=" * 80)
    print(f"RESOURCE WARNING: {task_name}")
    print("=" * 80)

    if gpu_required:
        print("This cell is intended to run on GPU.")
        if min_gpu_gb is not None:
            print(f"Recommended minimum GPU memory: {min_gpu_gb:.1f} GB.")
        print("Running on CPU may be extremely slow or may not finish.")
    else:
        print("This cell can run on CPU, but may still be slow.")

    print("Run this only if you are intentionally regenerating labels.")
    print("=" * 80)


# =============================================================================
# Backend-specific guards for later cells
# =============================================================================

def guard_rouge_labeling():
    require_label_regeneration_enabled()
    require_flag("RUN_ROUGE_LABELING", RUN_ROUGE_LABELING)


def guard_bertscore_labeling():
    require_label_regeneration_enabled()
    require_flag("RUN_BERTSCORE_LABELING", RUN_BERTSCORE_LABELING)
    require_expensive_runs_allowed(
        "BERTScore recomputation can be slow and memory-intensive."
    )
    print_resource_warning(
        "BERTScore recomputation",
        gpu_required=True,
        min_gpu_gb=12,
    )
    require_cuda(min_total_gb=12)


def guard_nli_labeling():
    require_label_regeneration_enabled()
    require_flag("RUN_NLI_LABELING", RUN_NLI_LABELING)
    require_expensive_runs_allowed(
        "NLI recomputation loads a large entailment model and may require substantial GPU memory."
    )
    print_resource_warning(
        "NLI recomputation",
        gpu_required=True,
        min_gpu_gb=24,
    )
    require_cuda(min_total_gb=24)


def guard_openai_labeling():
    require_label_regeneration_enabled()
    require_flag("RUN_OPENAI_LABELING", RUN_OPENAI_LABELING)
    require_expensive_runs_allowed(
        "OpenAI labeling may incur API costs."
    )


def guard_anthropic_labeling():
    require_label_regeneration_enabled()
    require_flag("RUN_ANTHROPIC_LABELING", RUN_ANTHROPIC_LABELING)
    require_expensive_runs_allowed(
        "Anthropic labeling may incur API costs."
    )


def guard_local_labeling(min_gpu_gb=24):
    require_label_regeneration_enabled()
    require_flag("RUN_LOCAL_LABELING", RUN_LOCAL_LABELING)
    require_expensive_runs_allowed(
        "Local judge recomputation loads 7B/9B models and may take a long time."
    )
    print_resource_warning(
        "Local judge recomputation",
        gpu_required=True,
        min_gpu_gb=min_gpu_gb,
    )
    require_cuda(min_total_gb=min_gpu_gb)


def guard_six_branch_sanity():
    if not RUN_SIX_BRANCH_SANITY:
        raise RuntimeError(
            "Six-branch sanity check is disabled.\n"
            "Set RUN_SIX_BRANCH_SANITY=True to run this diagnostic."
        )


# =============================================================================
# Status print
# =============================================================================

print("=" * 80)
print("EXECUTION MODE")
print("=" * 80)
print("RUN_RELEASE_ANALYSIS:         ", RUN_RELEASE_ANALYSIS)
print("RUN_LABEL_REGENERATION:       ", RUN_LABEL_REGENERATION)
print("ALLOW_EXPENSIVE_RUNS:         ", ALLOW_EXPENSIVE_RUNS)
print("RUN_SIX_BRANCH_SANITY:        ", RUN_SIX_BRANCH_SANITY)
print("RUN_FACTORIAL_PROMPT_EXPERIMENT:", RUN_FACTORIAL_PROMPT_EXPERIMENT)
print("RUN_B10:                      ", RUN_B10)
print("RUN_B10B:                     ", RUN_B10B)
print("FORCE_REINIT_TEMP:            ", FORCE_REINIT_TEMP)
print()
print("Label regeneration flags:")
print("  RUN_ROUGE_LABELING:         ", RUN_ROUGE_LABELING)
print("  RUN_BERTSCORE_LABELING:     ", RUN_BERTSCORE_LABELING)
print("  RUN_NLI_LABELING:           ", RUN_NLI_LABELING)
print("  RUN_OPENAI_LABELING:        ", RUN_OPENAI_LABELING)
print("  RUN_ANTHROPIC_LABELING:     ", RUN_ANTHROPIC_LABELING)
print("  RUN_LOCAL_LABELING:         ", RUN_LOCAL_LABELING)
print("=" * 80)

if RUN_RELEASE_ANALYSIS and not RUN_LABEL_REGENERATION:
    print("✓ Release-analysis mode: paper results can be reproduced from release master without API keys.")
elif RUN_LABEL_REGENERATION:
    print("⚠ Label-regeneration mode enabled. Expensive/API/GPU cells may run if their sub-flags are also enabled.")

In [ ]:
# =============================================================================
# CELL A2: Config, paths, imports + release master access check
#
# Assumes CELL A1 has defined:
#   RUN_RELEASE_ANALYSIS
#   RUN_LABEL_REGENERATION
#
# Purpose:
#
#   - Define repository-relative paths
#   - Define shared paper constants
#   - Validate release master if RUN_RELEASE_ANALYSIS=True
#
# This cell does NOT require API keys and does NOT regenerate labels.
# =============================================================================

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score,
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")


def get_secret(name, required=False):
    """Read a secret from environment variables, then Colab userdata if available."""
    value = os.environ.get(name, "")
    if value and str(value).strip():
        return str(value).strip()

    try:
        from google.colab import userdata
        value = getattr(userdata, "get")(name)
    except Exception:
        value = None

    if value and str(value).strip():
        return str(value).strip()

    if required:
        raise RuntimeError(
            f"{name} is not set. Provide it as an environment variable "
            "or, when running in Colab, as a Colab secret."
        )
    return ""

# =============================================================================
# Paths
# =============================================================================

# =============================================================================
# USER CONFIGURATION — repository root
# =============================================================================
# Repository root. Run the notebook from the repository root, or set
# REPRO_ROOT explicitly in the environment.
PROJECT_ROOT = Path(os.environ.get("REPRO_ROOT", ".")).resolve()

PAPER_DIR = PROJECT_ROOT
DATA_DIR = PAPER_DIR / "data"
FIG_DIR = PAPER_DIR / "figures"
AUDIT_DIR = PAPER_DIR / "audits"
TABLE_DIR = PAPER_DIR / "tables"

for d in [DATA_DIR, FIG_DIR, AUDIT_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Source of truth for all release analyses
MASTER_PATH = DATA_DIR / "human_judge_release_master.csv"

# =============================================================================
# Shared constants
# =============================================================================

TARGET_MODELS = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

TARGET_DATASETS = ["triviaqa", "hotpotqa", "truthfulqa"]

PROMPTS = ["p1", "p2", "p3"]

HUMAN_COL = "annotator1_label"

GEN_DISPLAY = {
    "llama3-8b": "L",
    "gemma-2-9b": "G",
    "mistral-7b": "M",
}

DATASET_DISPLAY = {
    "triviaqa": "TriviaQA",
    "hotpotqa": "HotpotQA",
    "truthfulqa": "TruthfulQA",
}

# Release-master judge prefixes
JUDGES_RELEASE = [
    ("gpt_5_mini",        "GPT-5-mini",          "api"),
    ("gpt_5_nano",        "GPT-5-nano",          "api"),
    ("gpt_5_4",           "GPT-5.4",             "api"),
    ("claude_opus_4_7",   "Claude Opus 4.7",     "api"),
    ("gemma_2_9b_local",  "Gemma-2-9B local",    "local"),
    ("qwen2_5_7b_local",  "Qwen2.5-7B local",    "local"),
    ("llama_3_8b_local",  "Llama-3-8B local",    "local"),
]

LLM_LABEL_COLS = [
    f"label_{prefix}_{p}"
    for prefix, _, _ in JUDGES_RELEASE
    for p in PROMPTS
]

LEXICAL_REQUIRED_COLS = [
    "rougeL",
    "bertscore_f1",
    "rougeL_label_oracle",
    "rougeL_label_cv",
    "bertscore_label_oracle",
    "bertscore_label_cv",
]

LEXICAL_OPTIONAL_COLS = [
    "rougeL_tau_oracle",
    "rougeL_tau_cv_mean",
    "rougeL_tau_cv_min",
    "rougeL_tau_cv_max",
    "rougeL_cv_fold",
    "rougeL_cv_kappa_mean_fold",
    "rougeL_cv_kappa_oof",
    "bertscore_tau_oracle",
    "bertscore_tau_cv_mean",
    "bertscore_tau_cv_min",
    "bertscore_tau_cv_max",
    "bertscore_cv_fold",
    "bertscore_cv_kappa_mean_fold",
    "bertscore_cv_kappa_oof",
]

NLI_REQUIRED_COLS = [
    "nli_entail_prob",
    "nli_label_strict",
    "nli_label_best_tau",
    "nli_tau_strict",
    "nli_tau_best",
]

NLI_OPTIONAL_COLS = [
    "nli_label_best",  # legacy alias, okay if present
]

REQUIRED_CORE_COLS = [
    "item_id",
    "qid",
    "qid_order",
    "dataset",
    "model",
    "question",
    "ref_str",
    "answer_text",
    "annotator1_label",
    "annotator2_label",
]

# =============================================================================
# Helper functions
# =============================================================================

def check_binary_complete(df, col, errors, allow_minus_one=False, context=""):
    """Check label values."""
    allowed = {-1, 0, 1} if allow_minus_one else {0, 1}
    vals = set(pd.to_numeric(df[col], errors="coerce").dropna().unique().tolist())

    if not vals.issubset(allowed):
        errors.append(
            f"Invalid values in {col}"
            + (f" ({context})" if context else "")
            + f": {sorted(vals)}"
        )

    valid = int(df[col].isin([0, 1]).sum())
    invalid = len(df) - valid

    return valid, invalid


def print_numeric_summary(df, col, errors):
    numeric = pd.to_numeric(df[col], errors="coerce")
    missing = int(numeric.isna().sum())

    print(
        f"  {col:<30} "
        f"min={numeric.min():.4f} "
        f"mean={numeric.mean():.4f} "
        f"max={numeric.max():.4f} "
        f"missing={missing}"
    )

    if missing > 0:
        errors.append(f"Missing/non-numeric values in {col}: {missing}")


# =============================================================================
# Release master access check
# =============================================================================

print("=" * 80)
print("CONFIG + RELEASE MASTER ACCESS CHECK")
print("=" * 80)

print("PAPER_DIR: ", PAPER_DIR)
print("DATA_DIR:  ", DATA_DIR)
print("FIG_DIR:   ", FIG_DIR)
print("AUDIT_DIR: ", AUDIT_DIR)
print("TABLE_DIR: ", TABLE_DIR)
print("MASTER:    ", MASTER_PATH)

errors = []

# If CELL A1 was not run, default to release-analysis behavior.
if "RUN_RELEASE_ANALYSIS" not in globals():
    RUN_RELEASE_ANALYSIS = True

if "RUN_LABEL_REGENERATION" not in globals():
    RUN_LABEL_REGENERATION = False

master = None

if not RUN_RELEASE_ANALYSIS:
    print("\nRUN_RELEASE_ANALYSIS=False — skipping release master validation.")
    print("Paths and constants are still defined for label-regeneration cells.")

else:
    if not MASTER_PATH.exists():
        errors.append(f"Release master CSV not found: {MASTER_PATH}")

        print("\n" + "-" * 80)
        print("The release master CSV could not be found. Diagnosis:")
        print("-" * 80)

        if not PAPER_DIR.exists():
            print(f"  Project root does not exist:  {PAPER_DIR}")
            print("\n  REPRO_ROOT is unset or points nowhere. Set it to the folder")
            print("  that CONTAINS data/ — not to data/ itself:")
            print("      import os; os.environ['REPRO_ROOT'] = '/path/to/project'")

        elif not any(DATA_DIR.glob("*.csv")):
            print(f"  data/ exists but contains no CSV files: {DATA_DIR}")
            print("\n  REPRO_ROOT may be one level too deep. It must point to the")
            print("  folder CONTAINING data/, not to data/ itself.")

        else:
            print(f"  data/ folder exists:        {DATA_DIR}")
            print(f"  But this file is missing:   {MASTER_PATH.name}")
            print("\n  CSV files present in data/:")
            for f in sorted(DATA_DIR.glob("*.csv")):
                print(f"      {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
            print("\n  If the release master is here under a different name, rename")
            print(f"  it to {MASTER_PATH.name} rather than repointing MASTER_PATH.")

        print("-" * 80)
    else:
        master = pd.read_csv(MASTER_PATH)

        print(f"\n✓ Loaded release master:")
        print(f"  Path:  {MASTER_PATH}")
        print(f"  Shape: {master.shape}")

        # ---------------------------------------------------------------------
        # Required core columns
        # ---------------------------------------------------------------------
        missing_core = [c for c in REQUIRED_CORE_COLS if c not in master.columns]

        if missing_core:
            errors.append(f"Missing required core columns: {missing_core}")
        else:
            print("\n✓ Required core columns found.")

        # ---------------------------------------------------------------------
        # Basic shape checks
        # ---------------------------------------------------------------------
        if len(master) != 900:
            errors.append(f"Expected 900 QA-pairs, got {len(master)}")

        if "qid" in master.columns and master["qid"].nunique() != 300:
            errors.append(f"Expected 300 unique qids, got {master['qid'].nunique()}")

        if "item_id" in master.columns and master["item_id"].duplicated().any():
            errors.append("Duplicate item_id values found.")

        if {"qid", "model"}.issubset(master.columns):
            if master.duplicated(["qid", "model"]).any():
                errors.append("Duplicate qid × model rows found.")

            rows_per_qid = master.groupby("qid").size()
            if not (rows_per_qid == 3).all():
                bad = rows_per_qid[rows_per_qid != 3]
                errors.append(f"Some qids do not have exactly three rows:\n{bad}")

            models_per_qid = master.groupby("qid")["model"].nunique()
            if not (models_per_qid == 3).all():
                bad = models_per_qid[models_per_qid != 3]
                errors.append(f"Some qids do not have all three generator models:\n{bad}")

        # ---------------------------------------------------------------------
        # Model checks
        # ---------------------------------------------------------------------
        if "model" in master.columns:
            found_models = set(master["model"].unique())
            expected_models = set(TARGET_MODELS)

            if found_models != expected_models:
                errors.append(
                    f"Unexpected generator models. "
                    f"Expected {sorted(expected_models)}, found {sorted(found_models)}"
                )

            print("\nRows by generator model:")
            print(master["model"].value_counts().reindex(TARGET_MODELS))

            for m in TARGET_MODELS:
                n = int((master["model"] == m).sum())
                if n != 300:
                    errors.append(f"Expected 300 rows for {m}, got {n}")

        # ---------------------------------------------------------------------
        # Dataset checks
        # ---------------------------------------------------------------------
        if "dataset" in master.columns:
            found_datasets = set(master["dataset"].unique())
            expected_datasets = set(TARGET_DATASETS)

            if found_datasets != expected_datasets:
                errors.append(
                    f"Unexpected datasets. "
                    f"Expected {sorted(expected_datasets)}, found {sorted(found_datasets)}"
                )

            print("\nRows by dataset:")
            print(master["dataset"].value_counts().reindex(TARGET_DATASETS))

        if {"dataset", "model"}.issubset(master.columns):
            print("\nDataset × model:")
            print(pd.crosstab(master["dataset"], master["model"]))

        if {"qid", "dataset"}.issubset(master.columns):
            dataset_per_qid = master.groupby("qid")["dataset"].nunique()
            if not (dataset_per_qid == 1).all():
                bad = dataset_per_qid[dataset_per_qid != 1]
                errors.append(f"Some qids map to multiple datasets:\n{bad}")

        # ---------------------------------------------------------------------
        # Human label checks
        # ---------------------------------------------------------------------
        if "annotator1_label" in master.columns:
            if not master["annotator1_label"].isin([0, 1]).all():
                bad_n = int((~master["annotator1_label"].isin([0, 1])).sum())
                errors.append(f"Invalid annotator1_label values found: {bad_n}")

            print("\nAnnotator 1 label distribution:")
            print(master["annotator1_label"].value_counts().sort_index())

        if "annotator2_label" in master.columns:
            if not master["annotator2_label"].isin([-1, 0, 1]).all():
                bad_n = int((~master["annotator2_label"].isin([-1, 0, 1])).sum())
                errors.append(f"Invalid annotator2_label values found: {bad_n}")

            print("\nAnnotator 2 label distribution:")
            print(master["annotator2_label"].value_counts().sort_index())

            overlap = master[master["annotator2_label"].isin([0, 1])]
            if len(overlap) > 0:
                agreement = (
                    overlap["annotator1_label"] == overlap["annotator2_label"]
                ).mean()

                kappa = cohen_kappa_score(
                    overlap["annotator1_label"],
                    overlap["annotator2_label"],
                    labels=[0, 1],
                )

                print("\nAnnotator overlap:")
                print(f"  n_overlap:      {len(overlap)}")
                print(f"  raw agreement:  {agreement:.3f}")
                print(f"  Cohen kappa:    {kappa:.3f}")

        # ---------------------------------------------------------------------
        # Label-column naming guardrail
        # ---------------------------------------------------------------------
        forbidden_cols = [
            c for c in master.columns
            if "P2B" in c
            or "P3B" in c
            or c.endswith("_P1")
            or c.endswith("_P2")
            or c.endswith("_P3")
        ]

        if forbidden_cols:
            errors.append(
                "Found old/internal prompt-style column names in release master: "
                f"{forbidden_cols}"
            )

        label_cols = [c for c in master.columns if c.startswith("label_")]
        print(f"\nAutomatic judge label columns found: {len(label_cols)}")

        # ---------------------------------------------------------------------
        # Required LLM judge columns
        # ---------------------------------------------------------------------
        missing_llm = [c for c in LLM_LABEL_COLS if c not in master.columns]

        if missing_llm:
            errors.append(f"Missing required LLM judge label columns: {missing_llm}")
        else:
            print("\n✓ Required LLM judge label columns found.")

            for c in LLM_LABEL_COLS:
                valid, invalid = check_binary_complete(
                    master,
                    c,
                    errors,
                    allow_minus_one=False,
                    context="LLM judge labels",
                )
                print(f"  {c:<35} valid={valid}/{len(master)} invalid={invalid}")

                if valid != len(master):
                    errors.append(
                        f"Invalid/refusal values in {c}: valid={valid}/{len(master)}"
                    )

        # ---------------------------------------------------------------------
        # Lexical baseline checks: ROUGE-L and BERTScore
        # ---------------------------------------------------------------------
        missing_lexical_required = [
            c for c in LEXICAL_REQUIRED_COLS
            if c not in master.columns
        ]

        if missing_lexical_required:
            errors.append(
                f"Missing required lexical baseline columns: {missing_lexical_required}"
            )
        else:
            print("\n✓ Required lexical baseline columns found.")

            for c in [
                "rougeL_label_oracle",
                "rougeL_label_cv",
                "bertscore_label_oracle",
                "bertscore_label_cv",
            ]:
                valid, invalid = check_binary_complete(
                    master,
                    c,
                    errors,
                    allow_minus_one=False,
                    context="lexical labels",
                )
                print(f"  {c:<30} valid={valid}/{len(master)} invalid={invalid}")

                if valid != len(master):
                    errors.append(
                        f"Invalid lexical baseline labels in {c}: "
                        f"valid={valid}/{len(master)}"
                    )

            for c in ["rougeL", "bertscore_f1"]:
                print_numeric_summary(master, c, errors)

        missing_lexical_optional = [
            c for c in LEXICAL_OPTIONAL_COLS
            if c not in master.columns
        ]

        if missing_lexical_optional:
            print("\nOptional lexical diagnostic columns missing:")
            for c in missing_lexical_optional:
                print(f"  – {c}")
        else:
            print("\n✓ Optional lexical diagnostic columns found.")

        # ---------------------------------------------------------------------
        # NLI baseline checks
        # ---------------------------------------------------------------------
        missing_nli_required = [
            c for c in NLI_REQUIRED_COLS
            if c not in master.columns
        ]

        if missing_nli_required:
            errors.append(
                f"Missing required NLI baseline columns: {missing_nli_required}"
            )
        else:
            print("\n✓ Required NLI baseline columns found.")

            for c in ["nli_label_strict", "nli_label_best_tau"]:
                valid, invalid = check_binary_complete(
                    master,
                    c,
                    errors,
                    allow_minus_one=False,
                    context="NLI labels",
                )
                print(f"  {c:<30} valid={valid}/{len(master)} invalid={invalid}")

                if valid != len(master):
                    errors.append(
                        f"Invalid NLI baseline labels in {c}: "
                        f"valid={valid}/{len(master)}"
                    )

            for c in ["nli_entail_prob", "nli_tau_strict", "nli_tau_best"]:
                print_numeric_summary(master, c, errors)

        for c in NLI_OPTIONAL_COLS:
            if c in master.columns:
                print(f"\n✓ Optional legacy NLI column found: {c}")
            else:
                print(f"\nOptional legacy NLI column missing: {c}")

# =============================================================================
# Final summary
# =============================================================================

print("\n" + "=" * 80)

if errors:
    print(f"✗ {len(errors)} BLOCKING ERROR(S):")
    for e in errors:
        print(f"  • {e}")
    print("\nFix these before running release-analysis cells.")
    print("=" * 80)
    raise RuntimeError("Release master access check failed.")
else:
    if RUN_RELEASE_ANALYSIS:
        print("✓ Release master checks passed — ready to run analysis cells.")
    else:
        print("✓ Paths/constants initialized. Release master validation was skipped.")
    print("=" * 80)

In [ ]:
# =============================================================================
# CELL A3: Initialize temp_results.csv from locked release master
#
# Purpose:
#   Create a sandbox CSV for any regenerated or recomputed labels/results.
#   The locked release master is never overwritten.
#
# Rule:
#   MASTER_PATH is read-only.
#   TEMP_RESULTS_PATH is the only row-level CSV that regeneration/recompute
#   cells are allowed to modify.
# =============================================================================

from pathlib import Path
import pandas as pd

# Store temp results outside data/ to avoid confusing them with release data.
TEMP_RESULTS_DIR = AUDIT_DIR / "temp_results"
TEMP_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TEMP_RESULTS_PATH = TEMP_RESULTS_DIR / "temp_results.csv"

if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
    raise RuntimeError(
        "Safety error: TEMP_RESULTS_PATH points to MASTER_PATH. "
        "Refusing to continue."
    )

master = pd.read_csv(MASTER_PATH)

# Do not clobber an existing sandbox: a rerun of this cell after B4/B5/B6
# would otherwise discard hours of regenerated labels.
if TEMP_RESULTS_PATH.exists() and not FORCE_REINIT_TEMP:
    temp = pd.read_csv(TEMP_RESULTS_PATH)

    print("temp_results.csv already exists; leaving it untouched.")
    print("  Path:   ", TEMP_RESULTS_PATH)
    print("  Rows:   ", len(temp))
    print("  Columns:", len(temp.columns))
    print("\n  Set FORCE_REINIT_TEMP=True in CELL A1 to discard it and")
    print("  start again from the release master.")

else:
    # Start temp_results as a full sandbox copy of the release master.
    # This keeps the schema simple: temp has the same column names as master.
    temp = master.copy(deep=True)

    temp["_temp_source_master"] = MASTER_PATH.name
    temp["_temp_initialized_at"] = pd.Timestamp.now(tz="UTC").isoformat()

    temp.to_csv(TEMP_RESULTS_PATH, index=False)

    print("Initialized temp_results.csv from the release master.")
    print("  Source master:", MASTER_PATH)
    print("  Temp results: ", TEMP_RESULTS_PATH)
    print("  Rows:         ", len(temp))
    print("  Columns:      ", len(temp.columns))

# Basic safety checks
assert MASTER_PATH.exists(), f"Missing master: {MASTER_PATH}"
assert TEMP_RESULTS_PATH.exists(), f"Missing temp results: {TEMP_RESULTS_PATH}"
assert len(temp) == len(master), "Temp row count differs from master."
assert temp["item_id"].is_unique, "item_id is not unique in temp_results."

print("Safety check passed.")

**SECTION B — Optional label regeneration**

Section B is frozen. Its outputs are already in the released master. Do not use Run All; run Sections A and C only. Cells B10 and B10b are skipped unless RUN_B10 / RUN_B10B is set to True in A1; B10b calls the OpenAI API only when RUN_B10B is True and OPENAI_API_KEY is set.

In [ ]:
# =============================================================================
# CELL B1: Judge prompt definitions
#
# Purpose:
#   Define the three paper-facing judge prompts used in all main experiments.
#
# Paper-facing prompt names:
#   P1  reference-faithfulness prompt
#   P2  factual-correctness prompt
#   P3  extended factual-correctness prompt
#
# These are the prompts reproduced in Appendix B. They are run on all seven
# judges over all 900 items.
#
# Appendix-only prompt variants (p1s, p2t) are defined separately with the
# experiment that uses them; they are not part of the main run.
#
# Used by:
#   B4, B5, B6 (judge labeling), B7 (prompt-following check)
# =============================================================================

PROMPT_P1 = """You are a hallucination auditor for open-domain QA.

Your task: decide if a model answer contains a hallucination by comparing
it against the reference answer.

DEFINITION
A hallucination is any claim that is not supported by, or contradicted by,
the reference answer.

Use the reference answer as ground truth. Do not use external knowledge.

OUTPUT FORMAT (STRICT JSON ONLY)
Return ONLY valid JSON with exactly these two fields:
{"label": 0 or 1, "confidence": <float between 0 and 1>}

Rules:
- Do not include markdown, code fences, or extra text.
- Do not include a reasoning field.
- confidence must be a numeric value (not a string)."""




PROMPT_P2 = """You are a hallucination auditor for open-domain QA.

Your task: decide if a model answer contains a hallucination.

DEFINITION
A hallucination is a claim that is EXPLICITLY CONTRADICTED by the reference
answer, internally self-contradictory, or factually false at the level of a
specific verifiable detail (such as a name, date, number, place, entity, or
other concrete factual attribute).

CRITICAL RULES
1. "Absent from reference" is NOT the same as "contradicted by reference".
   Additional correct details not in the reference are NOT hallucinations.
2. Wrong-but-vague is NOT a hallucination.
   Vague, incomplete, or non-committal answers are incorrect, not
   hallucinations, unless they explicitly contradict the reference or include
   a specific verifiable false detail.
3. Focus on factual error, not mere non-overlap.
   Do not label an answer as hallucination only because it contains content
   that does not appear in the reference.

DECISION TREE
Q1. Does the answer match the reference or a clear paraphrase?
    YES -> Q2. Does it add any claim EXPLICITLY CONTRADICTED by the reference or
              factually false at the level of a specific verifiable detail?
              NO  -> LABEL=0
              YES -> LABEL=1
    NO  -> Q3. Does the core answer conflict with the reference (different
              entity/value/fact)?
              YES -> LABEL=1
              NO  -> Q4. Is the answer a refusal or expression of uncertainty?
                        YES -> LABEL=0
                        NO  -> Q5. Does it include a specific verifiable detail
                                  that is factually false?
                                  YES -> LABEL=1
                                  NO  -> LABEL=0

OUTPUT FORMAT (STRICT JSON ONLY)
Return ONLY valid JSON with exactly these two fields:
{"label": 0 or 1, "confidence": <float between 0 and 1>}

Rules:
- Do not include markdown, code fences, or extra text.
- Do not include a reasoning field.
- confidence must be a numeric value (not a string)."""

PROMPT_P3 = """You are a hallucination auditor for open-domain QA.

Your task: decide if a model answer contains a hallucination.

DEFINITION
A hallucination is a claim that is EXPLICITLY CONTRADICTED by the reference
answer, internally self-contradictory, or factually false at the level of a
specific verifiable detail (such as a name, date, number, place, entity, or
other concrete factual attribute).

CRITICAL RULES
1. "Absent from reference" is NOT the same as "contradicted by reference".
   Additional correct details not in the reference are NOT hallucinations.
2. Wrong-but-vague is NOT a hallucination.
   Vague or incomplete answers are incorrect, not hallucinations, unless
   they explicitly contradict the reference or include a specific verifiable
   false detail.
3. Core mismatch = hallucination.
   If the core answer conflicts with the reference (different entity/value/fact),
   label=1 even without an additional false specific detail.

SPECIAL CASE -- non-committal references:
If the reference is an explicit refusal or non-answer (e.g. "I have no comment",
"Unknown", "Cannot be determined"), then any confident specific claim from the
model is a hallucination. If the reference explains a dependency ("It depends
on X because..."), evaluate normally.

DECISION TREE
Q1. Does the answer match the reference or a clear paraphrase?
    YES -> Q2. Does it add any claim EXPLICITLY CONTRADICTED by the reference or
              factually false at the level of a specific verifiable detail?
              NO  -> LABEL=0
              YES -> LABEL=1
    NO  -> Q3. Is the reference non-committal (refusal / unknown / cannot be
              determined)?
              YES -> Is the model answer a confident specific claim?
                    YES -> LABEL=1
                    NO  -> LABEL=0
              NO  -> Q4. Does the core answer conflict with the reference?
                        YES -> LABEL=1
                        NO  -> Q5. Is the answer a refusal/uncertainty?
                                  YES -> LABEL=0
                                  NO  -> Q6. Does it include a specific
                                            verifiable detail that is factually
                                            false?
                                            YES -> LABEL=1
                                            NO  -> LABEL=0

OUTPUT FORMAT (STRICT JSON ONLY)
Return ONLY valid JSON with exactly these two fields:
{"label": 0 or 1, "confidence": <float between 0 and 1>}

Rules:
- Do not include markdown, code fences, or extra text.
- Do not include a reasoning field.
- confidence must be a numeric value (not a string)."""



PROMPTS = {
    "P1": PROMPT_P1,
    "P2": PROMPT_P2,
    "P3": PROMPT_P3,
}

In [ ]:
# =============================================================================
# Optional: install the lexical baseline dependencies if they are not
# already available. Run this before CELL B2. Skip if rouge-score and
# bert-score are already installed.
# =============================================================================

# %pip install rouge-score --quiet
# %pip install bert-score --quiet

In [ ]:
# =============================================================================
# CELL B2: Recompute ROUGE-L / BERTScore baselines into temp_results + audit
#
# Purpose:
#   Optionally recompute lexical baseline scores from ref_str and answer_text,
#   write the recomputed scores/labels/threshold metadata into temp_results.csv,
#   and compare the resulting temp columns against the locked release master.
#
# Computation notes:
#   - ref_str may contain multiple references separated by "|".
#     Scores are computed per reference and the maximum is taken.
#   - ROUGE-L uses use_stemmer=True (matches original generation).
#   - BERTScore uses roberta-large without baseline rescaling (matches original).
#
# Execution:
#   - If RUN_LABEL_REGENERATION=False, this cell is skipped.
#   - ROUGE-L runs only if RUN_ROUGE_LABELING=True.
#   - BERTScore runs only if RUN_BERTSCORE_LABELING=True.
#
# Safety:
#   - MASTER_PATH is read-only.
#   - This cell writes only to TEMP_RESULTS_PATH and audits/.
#
# Outputs:
#   - audits/temp_results/temp_results.csv
#   - audits/lexical_recompute/lexical_recompute_audit.csv
#   - audits/lexical_recompute/lexical_score_diff_audit.csv
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL B2: ROUGE-L / BERTScore recomputation"

if not RUN_LABEL_REGENERATION:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=False.")
    print("No lexical scores or labels were recomputed.")
    print("temp_results.csv was not modified by this cell.")

elif not (RUN_ROUGE_LABELING or RUN_BERTSCORE_LABELING):
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=True, but both lexical sub-flags are False:")
    print(f"  RUN_ROUGE_LABELING     = {RUN_ROUGE_LABELING}")
    print(f"  RUN_BERTSCORE_LABELING = {RUN_BERTSCORE_LABELING}")
    print("No lexical scores or labels were recomputed.")
    print("temp_results.csv was not modified by this cell.")

else:
    # =========================================================================
    # Paths
    # =========================================================================
    LEX_AUDIT_DIR = AUDIT_DIR / "lexical_recompute"
    LEX_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_AUDIT_CSV = LEX_AUDIT_DIR / "lexical_recompute_audit.csv"
    OUT_SCORE_AUDIT_CSV = LEX_AUDIT_DIR / "lexical_score_diff_audit.csv"

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError(
            "Safety error: TEMP_RESULTS_PATH points to MASTER_PATH. "
            "Refusing to continue."
        )

    if not TEMP_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"temp_results.csv not found. Run CELL A3 first: {TEMP_RESULTS_PATH}"
        )

    # =========================================================================
    # Config
    # =========================================================================

    HUMAN_COL = "annotator1_label"

    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    RNG_SEED = 42
    N_SPLITS = 3
    THRESHOLD_STEPS = 101

    # ROUGE-L: stemmer must match original generation.
    ROUGE_USE_STEMMER = True

    # BERTScore: configuration must match original generation.
    BERTSCORE_MODEL_TYPE = globals().get("BERTSCORE_MODEL_TYPE", "roberta-large")
    BERTSCORE_LANG = globals().get("BERTSCORE_LANG", "en")
    BERTSCORE_RESCALE_WITH_BASELINE = globals().get(
        "BERTSCORE_RESCALE_WITH_BASELINE", False
    )
    BERTSCORE_BATCH_SIZE = globals().get("BERTSCORE_BATCH_SIZE", 16)

    LEXICAL_SPECS = []

    if RUN_ROUGE_LABELING:
        LEXICAL_SPECS.append(
            {
                "metric": "ROUGE-L",
                "score_col": "rougeL",
                "label_oracle_col": "rougeL_label_oracle",
                "label_cv_col": "rougeL_label_cv",
                "tau_oracle_col": "rougeL_tau_oracle",
                "tau_cv_mean_col": "rougeL_tau_cv_mean",
                "tau_cv_min_col": "rougeL_tau_cv_min",
                "tau_cv_max_col": "rougeL_tau_cv_max",
                "cv_fold_col": "rougeL_cv_fold",
                "cv_kappa_mean_fold_col": "rougeL_cv_kappa_mean_fold",
                "cv_kappa_oof_col": "rougeL_cv_kappa_oof",
            }
        )

    if RUN_BERTSCORE_LABELING:
        LEXICAL_SPECS.append(
            {
                "metric": "BERTScore",
                "score_col": "bertscore_f1",
                "label_oracle_col": "bertscore_label_oracle",
                "label_cv_col": "bertscore_label_cv",
                "tau_oracle_col": "bertscore_tau_oracle",
                "tau_cv_mean_col": "bertscore_tau_cv_mean",
                "tau_cv_min_col": "bertscore_tau_cv_min",
                "tau_cv_max_col": "bertscore_tau_cv_max",
                "cv_fold_col": "bertscore_cv_fold",
                "cv_kappa_mean_fold_col": "bertscore_cv_kappa_mean_fold",
                "cv_kappa_oof_col": "bertscore_cv_kappa_oof",
            }
        )

    # =========================================================================
    # Guards
    # =========================================================================

    if RUN_ROUGE_LABELING:
        guard_rouge_labeling()
        require_package("rouge_score", "rouge-score")

    if RUN_BERTSCORE_LABELING:
        guard_bertscore_labeling()
        require_package("bert_score", "bert-score")

    # =========================================================================
    # Helpers: thresholding
    # =========================================================================

    def safe_kappa(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)
        try:
            k = cohen_kappa_score(y_true, y_pred, labels=[0, 1])
            return float(k) if np.isfinite(k) else np.nan
        except Exception:
            return np.nan


    def labels_at_threshold(scores, tau):
        """score < tau -> hallucination label 1; NaN scores receive -1."""
        scores = np.asarray(scores, dtype=float)
        pred = np.full(len(scores), -1, dtype=int)
        valid = ~np.isnan(scores)
        if np.isfinite(tau):
            pred[valid] = (scores[valid] < tau).astype(int)
        return pred


    def sweep_threshold_oracle(scores, human_labels, steps=THRESHOLD_STEPS):
        """
        Oracle threshold: select tau on the full generator subset and evaluate
        on the same subset. Returns (best_kappa, best_tau).
        """
        scores = np.asarray(scores, dtype=float)
        human_labels = np.asarray(human_labels, dtype=int)
        valid = ~np.isnan(scores) & np.isin(human_labels, [0, 1])
        if int(valid.sum()) < 10:
            return np.nan, np.nan
        best_k, best_tau = -np.inf, np.nan
        for tau in np.linspace(0, 1, steps):
            pred = (scores < tau).astype(int)
            k = safe_kappa(human_labels[valid], pred[valid])
            if np.isfinite(k) and k > best_k:
                best_k, best_tau = k, tau
        return float(best_k), float(best_tau)


    def cv_threshold_oof(
        scores,
        human_labels,
        n_splits=N_SPLITS,
        steps=THRESHOLD_STEPS,
        random_state=RNG_SEED,
    ):
        """
        3-fold out-of-fold threshold selection.

        For each fold:
          1. Select tau on the two training folds.
          2. Apply selected tau to the held-out fold only.
          3. Store held-out predictions.

        Returns:
          full_oof_pred       : length-N array of OOF predictions (-1 for invalid)
          full_fold_id        : length-N array of fold assignments (-1 for invalid)
          fold_taus           : list of selected tau per fold
          fold_kappas         : list of held-out kappa per fold
          mean_tau            : mean of fold_taus
          mean_fold_kappa     : mean of fold_kappas
          oof_kappa           : kappa on concatenated OOF predictions
        """
        scores = np.asarray(scores, dtype=float)
        human_labels = np.asarray(human_labels, dtype=int)
        valid = ~np.isnan(scores) & np.isin(human_labels, [0, 1])
        valid_idx = np.where(valid)[0]

        full_oof_pred = np.full(len(scores), -1, dtype=int)
        full_fold_id = np.full(len(scores), -1, dtype=int)

        if len(valid_idx) < n_splits * 10:
            return full_oof_pred, full_fold_id, [], [], np.nan, np.nan, np.nan

        s_valid = scores[valid]
        y_valid = human_labels[valid]

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        oof_valid_pred = np.full(len(s_valid), -1, dtype=int)
        fold_taus, fold_kappas = [], []

        for fold_id, (train_idx, test_idx) in enumerate(kf.split(s_valid)):
            s_train, s_test = s_valid[train_idx], s_valid[test_idx]
            y_train, y_test = y_valid[train_idx], y_valid[test_idx]

            best_k_train, best_tau = -np.inf, np.nan
            for tau in np.linspace(0, 1, steps):
                pred_train = (s_train < tau).astype(int)
                k = safe_kappa(y_train, pred_train)
                if np.isfinite(k) and k > best_k_train:
                    best_k_train, best_tau = k, tau

            if not np.isfinite(best_tau):
                best_tau = 0.5

            pred_test = (s_test < best_tau).astype(int)
            oof_valid_pred[test_idx] = pred_test
            full_fold_id[valid_idx[test_idx]] = fold_id
            fold_taus.append(float(best_tau))
            fold_kappas.append(float(safe_kappa(y_test, pred_test)))

        full_oof_pred[valid_idx] = oof_valid_pred
        oof_kappa = safe_kappa(human_labels[valid], full_oof_pred[valid])

        return (
            full_oof_pred,
            full_fold_id,
            fold_taus,
            fold_kappas,
            float(np.nanmean(fold_taus)),
            float(np.nanmean(fold_kappas)),
            float(oof_kappa),
        )

    # =========================================================================
    # Helpers: score computation
    # =========================================================================

    def _parse_refs(ref_str):
        """Split a ref_str field on '|' and return a list of non-empty strings."""
        refs = [r.strip() for r in str(ref_str or "").split("|") if r.strip()]
        return refs if refs else [" "]


    def recompute_rouge_l_scores(ref_strs, answers):
        """
        Recompute ROUGE-L F1 from ref_str and answer_text.

        ref_str may contain multiple references separated by '|'.
        The maximum ROUGE-L score across all references is taken per item.
        use_stemmer=True matches original generation.
        """
        from rouge_score import rouge_scorer

        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=ROUGE_USE_STEMMER)
        scores = []

        for ref_str, ans in zip(ref_strs, answers):
            refs = _parse_refs(ref_str)
            best = max(
                scorer.score(ref, str(ans))["rougeL"].fmeasure
                for ref in refs
            )
            scores.append(float(best))

        return np.asarray(scores, dtype=float)


    def recompute_bertscore_f1_scores(ref_strs, answers):
        """
        Recompute BERTScore F1 from ref_str and answer_text.

        ref_str may contain multiple references separated by '|'.
        All (answer, reference) pairs are scored in one batched call;
        the maximum F1 across references is taken per item.

        Raw values can differ from the release master if the package/model/device
        versions or BERTScore configuration differ from the original run.
        """
        from bert_score import score as bert_score

        all_cands, all_refs, ref_counts = [], [], []

        for ref_str, ans in zip(ref_strs, answers):
            refs = _parse_refs(ref_str)
            all_cands.extend([str(ans)] * len(refs))
            all_refs.extend(refs)
            ref_counts.append(len(refs))

        _, _, f1 = bert_score(
            cands=all_cands,
            refs=all_refs,
            lang=BERTSCORE_LANG,
            model_type=BERTSCORE_MODEL_TYPE,
            rescale_with_baseline=BERTSCORE_RESCALE_WITH_BASELINE,
            batch_size=BERTSCORE_BATCH_SIZE,
            verbose=True,
        )

        f1_np = f1.detach().cpu().numpy().astype(float)

        best_scores = []
        pos = 0
        for count in ref_counts:
            best_scores.append(float(f1_np[pos : pos + count].max()))
            pos += count

        return np.asarray(best_scores, dtype=float)

    # =========================================================================
    # Helpers: audit utilities
    # =========================================================================

    def metric_summary(y_true, y_pred):
        mask = np.isin(y_true, [0, 1]) & np.isin(y_pred, [0, 1])
        yh = np.asarray(y_true)[mask].astype(int)
        yp = np.asarray(y_pred)[mask].astype(int)
        tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()
        err = fp + fn
        return {
            "n_valid": int(mask.sum()),
            "kappa": safe_kappa(yh, yp),
            "bias": float(yp.mean() - yh.mean()),
            "fp": int(fp),
            "fn": int(fn),
            "err": int(err),
            "fp_pct": float(100.0 * fp / max(err, 1)),
        }


    def get_unique_value(df, col):
        vals = pd.to_numeric(df[col], errors="coerce").dropna().round(12).unique()
        if len(vals) != 1:
            raise ValueError(f"Expected one unique value in {col}, found {vals}")
        return float(vals[0])


    def score_diff_summary(master_df, temp_df, col):
        m = pd.to_numeric(master_df[col], errors="coerce").to_numpy()
        t = pd.to_numeric(temp_df[col], errors="coerce").to_numpy()
        valid = np.isfinite(m) & np.isfinite(t)
        diff = np.abs(m[valid] - t[valid])
        if len(diff) == 0:
            return {"score_col": col, "n_valid": 0,
                    "max_absdiff": np.nan, "mean_absdiff": np.nan,
                    "n_absdiff_gt_1e_12": np.nan, "n_absdiff_gt_1e_9": np.nan,
                    "n_absdiff_gt_1e_6": np.nan, "n_absdiff_gt_1e_4": np.nan}
        return {
            "score_col": col,
            "n_valid": int(valid.sum()),
            "max_absdiff": float(diff.max()),
            "mean_absdiff": float(diff.mean()),
            "n_absdiff_gt_1e_12": int((diff > 1e-12).sum()),
            "n_absdiff_gt_1e_9": int((diff > 1e-9).sum()),
            "n_absdiff_gt_1e_6": int((diff > 1e-6).sum()),
            "n_absdiff_gt_1e_4": int((diff > 1e-4).sum()),
        }


    def require_same_row_identity(master_df, temp_df):
        identity_cols = ["item_id", "qid", "dataset", "model"]
        missing_master = [c for c in identity_cols if c not in master_df.columns]
        missing_temp = [c for c in identity_cols if c not in temp_df.columns]
        if missing_master or missing_temp:
            raise KeyError(
                f"Missing identity columns. "
                f"master missing={missing_master}, temp missing={missing_temp}"
            )
        if len(master_df) != len(temp_df):
            raise ValueError(
                f"Row-count mismatch: master={len(master_df)}, temp={len(temp_df)}"
            )
        for col in identity_cols:
            if not master_df[col].astype(str).equals(temp_df[col].astype(str)):
                raise ValueError(
                    f"Row identity mismatch in column '{col}'. "
                    "Reinitialize temp_results.csv from master before continuing."
                )

    # =========================================================================
    # Load and validate master / temp
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)
    temp = pd.read_csv(TEMP_RESULTS_PATH)

    require_same_row_identity(master, temp)

    required_cols = [
        "item_id", "qid", "dataset", "model",
        "ref_str", "answer_text", HUMAN_COL,
    ]
    for spec in LEXICAL_SPECS:
        required_cols.extend([
            spec["score_col"],
            spec["label_oracle_col"],
            spec["label_cv_col"],
            spec["tau_oracle_col"],
            spec["tau_cv_mean_col"],
            spec["tau_cv_min_col"],
            spec["tau_cv_max_col"],
            spec["cv_fold_col"],
            spec["cv_kappa_mean_fold_col"],
            spec["cv_kappa_oof_col"],
        ])

    missing = [c for c in required_cols if c not in master.columns]
    if missing:
        raise KeyError(f"Missing required columns from release master: {missing}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows in release master, got {len(master)}")

    if not master[HUMAN_COL].isin([0, 1]).all():
        bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
        raise ValueError(f"Invalid human labels in {HUMAN_COL}: {bad_n}")

    for model_name in GEN_ORDER:
        n = int((master["model"] == model_name).sum())
        if n != 300:
            raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

    print("=" * 80)
    print("LEXICAL BASELINE FULL RECOMPUTE INTO TEMP_RESULTS + AUDIT")
    print("=" * 80)
    print("Release master: ", MASTER_PATH)
    print("Temp results:   ", TEMP_RESULTS_PATH)
    print("Audit CSV:      ", OUT_AUDIT_CSV)
    print("Score audit CSV:", OUT_SCORE_AUDIT_CSV)
    print()
    print("Computation settings:")
    print(f"  ROUGE_USE_STEMMER              = {ROUGE_USE_STEMMER}")
    print(f"  BERTSCORE_MODEL_TYPE           = {BERTSCORE_MODEL_TYPE}")
    print(f"  BERTSCORE_RESCALE_WITH_BASELINE= {BERTSCORE_RESCALE_WITH_BASELINE}")
    print(f"  Multi-reference handling       = max score across '|'-split refs")
    print()
    print("Execution flags:")
    print(f"  RUN_ROUGE_LABELING     = {RUN_ROUGE_LABELING}")
    print(f"  RUN_BERTSCORE_LABELING = {RUN_BERTSCORE_LABELING}")
    print()
    print("This cell writes only to temp_results.csv and audits/.")

    # =========================================================================
    # Raw score recomputation from text
    # =========================================================================

    recomputed_score_cols = []

    print("\n" + "=" * 80)
    print("RAW SCORE RECOMPUTATION FROM ref_str + answer_text")
    print("=" * 80)

    if RUN_ROUGE_LABELING:
        print("ROUGE-L: recomputing raw rougeL scores from text.")
        print(f"  use_stemmer={ROUGE_USE_STEMMER}, multi-ref max over '|'-split refs.")
        temp["rougeL"] = recompute_rouge_l_scores(
            ref_strs=master["ref_str"],
            answers=master["answer_text"],
        )
        recomputed_score_cols.append("rougeL")
    else:
        print("ROUGE-L: skipped (RUN_ROUGE_LABELING=False).")

    if RUN_BERTSCORE_LABELING:
        print("BERTScore: recomputing raw bertscore_f1 scores from text.")
        print(f"  model_type={BERTSCORE_MODEL_TYPE}, multi-ref max over '|'-split refs.")
        temp["bertscore_f1"] = recompute_bertscore_f1_scores(
            ref_strs=master["ref_str"],
            answers=master["answer_text"],
        )
        recomputed_score_cols.append("bertscore_f1")
    else:
        print("BERTScore: skipped (RUN_BERTSCORE_LABELING=False).")

    score_audit = pd.DataFrame([
        score_diff_summary(master, temp, col)
        for col in recomputed_score_cols
    ])
    score_audit.to_csv(OUT_SCORE_AUDIT_CSV, index=False)

    print("\nRaw score difference audit against release master:")
    if len(score_audit) > 0:
        display(score_audit)
    else:
        print("No raw score columns were recomputed.")

    # =========================================================================
    # Per-generator label + threshold recomputation
    # =========================================================================

    audit_rows = []
    blocking_errors = []

    for model_name in GEN_ORDER:
        mask_g = master["model"] == model_name

        df_master_g = master.loc[mask_g].copy().reset_index(drop=True)
        df_temp_g = temp.loc[mask_g].copy().reset_index(drop=True)

        h = df_master_g[HUMAN_COL].astype(int).to_numpy()

        print("\n" + "-" * 80)
        print(model_name)
        print("-" * 80)

        for spec in LEXICAL_SPECS:
            metric = spec["metric"]
            score_col = spec["score_col"]

            scores = pd.to_numeric(df_temp_g[score_col], errors="coerce").to_numpy()

            if np.isnan(scores).any():
                n_missing = int(np.isnan(scores).sum())
                blocking_errors.append(
                    f"{model_name}/{metric}: {n_missing} missing recomputed scores"
                )

            # -----------------------------------------------------------------
            # Oracle threshold
            # -----------------------------------------------------------------
            k_oracle_temp, tau_oracle_temp = sweep_threshold_oracle(scores, h)
            label_oracle_temp = labels_at_threshold(scores, tau_oracle_temp)

            label_oracle_master = df_master_g[spec["label_oracle_col"]].astype(int).to_numpy()
            tau_oracle_master = get_unique_value(df_master_g, spec["tau_oracle_col"])

            oracle_label_mismatch = int(np.sum(label_oracle_temp != label_oracle_master))
            oracle_tau_absdiff = abs(tau_oracle_temp - tau_oracle_master)

            # -----------------------------------------------------------------
            # CV threshold
            # -----------------------------------------------------------------
            (
                label_cv_temp,
                fold_temp,
                fold_taus,
                fold_kappas,
                tau_cv_mean_temp,
                k_cv_mean_temp,
                k_cv_oof_temp,
            ) = cv_threshold_oof(scores, h)

            label_cv_master = df_master_g[spec["label_cv_col"]].astype(int).to_numpy()
            fold_master = df_master_g[spec["cv_fold_col"]].astype(int).to_numpy()

            tau_cv_mean_master = get_unique_value(df_master_g, spec["tau_cv_mean_col"])
            tau_cv_min_master = get_unique_value(df_master_g, spec["tau_cv_min_col"])
            tau_cv_max_master = get_unique_value(df_master_g, spec["tau_cv_max_col"])
            k_cv_mean_master = get_unique_value(df_master_g, spec["cv_kappa_mean_fold_col"])
            k_cv_oof_master = get_unique_value(df_master_g, spec["cv_kappa_oof_col"])

            tau_cv_min_temp = min(fold_taus) if fold_taus else np.nan
            tau_cv_max_temp = max(fold_taus) if fold_taus else np.nan

            cv_label_mismatch = int(np.sum(label_cv_temp != label_cv_master))
            cv_fold_mismatch = int(np.sum(fold_temp != fold_master))

            # -----------------------------------------------------------------
            # Write recomputed values to temp_results
            # -----------------------------------------------------------------
            temp.loc[mask_g, spec["label_oracle_col"]] = label_oracle_temp
            temp.loc[mask_g, spec["label_cv_col"]] = label_cv_temp
            temp.loc[mask_g, spec["tau_oracle_col"]] = tau_oracle_temp
            temp.loc[mask_g, spec["tau_cv_mean_col"]] = tau_cv_mean_temp
            temp.loc[mask_g, spec["tau_cv_min_col"]] = tau_cv_min_temp
            temp.loc[mask_g, spec["tau_cv_max_col"]] = tau_cv_max_temp
            temp.loc[mask_g, spec["cv_fold_col"]] = fold_temp
            temp.loc[mask_g, spec["cv_kappa_mean_fold_col"]] = k_cv_mean_temp
            temp.loc[mask_g, spec["cv_kappa_oof_col"]] = k_cv_oof_temp

            # -----------------------------------------------------------------
            # Audit summaries
            # -----------------------------------------------------------------
            oracle_metrics_master = metric_summary(h, label_oracle_master)
            oracle_metrics_temp = metric_summary(h, label_oracle_temp)
            cv_metrics_master = metric_summary(h, label_cv_master)
            cv_metrics_temp = metric_summary(h, label_cv_temp)

            audit_rows.append({
                "model": model_name,
                "metric": metric,
                "score_col": score_col,

                "oracle_tau_master": tau_oracle_master,
                "oracle_tau_temp": tau_oracle_temp,
                "oracle_tau_absdiff": oracle_tau_absdiff,
                "oracle_label_mismatch": oracle_label_mismatch,
                "oracle_kappa_master": oracle_metrics_master["kappa"],
                "oracle_kappa_temp": oracle_metrics_temp["kappa"],
                "oracle_fp_master": oracle_metrics_master["fp"],
                "oracle_fp_temp": oracle_metrics_temp["fp"],
                "oracle_fn_master": oracle_metrics_master["fn"],
                "oracle_fn_temp": oracle_metrics_temp["fn"],
                "oracle_fp_pct_master": oracle_metrics_master["fp_pct"],
                "oracle_fp_pct_temp": oracle_metrics_temp["fp_pct"],

                "cv_tau_mean_master": tau_cv_mean_master,
                "cv_tau_mean_temp": tau_cv_mean_temp,
                "cv_tau_mean_absdiff": abs(tau_cv_mean_temp - tau_cv_mean_master),
                "cv_tau_min_master": tau_cv_min_master,
                "cv_tau_min_temp": tau_cv_min_temp,
                "cv_tau_min_absdiff": abs(tau_cv_min_temp - tau_cv_min_master),
                "cv_tau_max_master": tau_cv_max_master,
                "cv_tau_max_temp": tau_cv_max_temp,
                "cv_tau_max_absdiff": abs(tau_cv_max_temp - tau_cv_max_master),

                "cv_kappa_mean_fold_master": k_cv_mean_master,
                "cv_kappa_mean_fold_temp": k_cv_mean_temp,
                "cv_kappa_mean_fold_absdiff": abs(k_cv_mean_temp - k_cv_mean_master),
                "cv_kappa_oof_master": k_cv_oof_master,
                "cv_kappa_oof_temp": k_cv_oof_temp,
                "cv_kappa_oof_absdiff": abs(k_cv_oof_temp - k_cv_oof_master),

                "cv_label_mismatch": cv_label_mismatch,
                "cv_fold_mismatch": cv_fold_mismatch,

                "cv_fp_master": cv_metrics_master["fp"],
                "cv_fp_temp": cv_metrics_temp["fp"],
                "cv_fn_master": cv_metrics_master["fn"],
                "cv_fn_temp": cv_metrics_temp["fn"],
                "cv_fp_pct_master": cv_metrics_master["fp_pct"],
                "cv_fp_pct_temp": cv_metrics_temp["fp_pct"],
            })

            print(
                f"  {metric:<10} oracle: "
                f"τ master={tau_oracle_master:.4f}, temp={tau_oracle_temp:.4f}, "
                f"label mismatches={oracle_label_mismatch}"
            )
            print(
                f"  {metric:<10} CV:     "
                f"τ mean master={tau_cv_mean_master:.4f}, temp={tau_cv_mean_temp:.4f}, "
                f"label mismatches={cv_label_mismatch}, fold mismatches={cv_fold_mismatch}"
            )

            if oracle_label_mismatch != 0:
                blocking_errors.append(
                    f"{model_name}/{metric}: oracle label mismatches={oracle_label_mismatch}"
                )
            if cv_label_mismatch != 0:
                blocking_errors.append(
                    f"{model_name}/{metric}: CV label mismatches={cv_label_mismatch}"
                )

    # =========================================================================
    # Save temp_results and audit CSVs
    # =========================================================================

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError("Refusing to write temp results to MASTER_PATH.")

    temp.to_csv(TEMP_RESULTS_PATH, index=False)

    audit = pd.DataFrame(audit_rows)
    audit.to_csv(OUT_AUDIT_CSV, index=False)

    print("\n" + "=" * 80)
    print("LEXICAL RECOMPUTE AUDIT SUMMARY")
    print("=" * 80)

    display_cols = [
        "model", "metric",
        "oracle_tau_master", "oracle_tau_temp", "oracle_label_mismatch",
        "cv_tau_mean_master", "cv_tau_mean_temp", "cv_label_mismatch",
        "cv_fold_mismatch",
        "oracle_kappa_master", "oracle_kappa_temp",
        "cv_kappa_oof_master", "cv_kappa_oof_temp",
    ]
    display(audit[display_cols])

    print("\nSaved temp_results.csv:")
    print(f"  {TEMP_RESULTS_PATH}")
    print("\nSaved audit CSVs:")
    print(f"  {OUT_AUDIT_CSV}")
    print(f"  {OUT_SCORE_AUDIT_CSV}")

    raw_score_diffs = score_audit[
        score_audit["n_absdiff_gt_1e_6"].fillna(0).astype(int) > 0
    ]
    if len(raw_score_diffs) > 0:
        print(
            "\nNOTE: Some recomputed raw metric scores differ from the release master.\n"
            "This does not automatically invalidate the paper results.\n"
            "The blocking reproduction criterion is label-level agreement.\n"
            "If label mismatches occur, investigate before changing any paper result."
        )

    if blocking_errors:
        print("\nBlocking audit errors:")
        for e in blocking_errors:
            print(f"  • {e}")
        raise AssertionError(
            "Lexical baseline recomputation audit failed. "
            "Recomputed labels in temp_results.csv do not match the locked "
            "release-master labels. Do not modify paper results until this is understood."
        )

    print("\n✓ Lexical baseline recomputation audit passed.")
    print("  Recomputed labels in temp_results.csv match release-master labels.")

In [ ]:
# =============================================================================
# CELL B3: Recompute NLI entailment baseline into temp_results + audit
#
# Purpose:
#   Optionally recompute the TRUE/NLI entailment baseline from ref_str and
#   answer_text, write the recomputed probabilities/labels/threshold metadata
#   into temp_results.csv, and compare temp_results against the locked release
#   master.
#
# Operationalization:
#   premise    = reference
#   hypothesis = model answer
#
# NLI columns:
#   nli_entail_prob        — max entailment probability across reference variants
#   nli_label_strict       — strict label at tau=0.5
#   nli_label_best_tau     — oracle-thresholded diagnostic label
#   nli_tau_strict         — constant column = 0.5
#   nli_tau_best           — oracle tau that maximizes kappa within generator
#
# Execution:
#   - If RUN_LABEL_REGENERATION=False, this cell is skipped.
#   - If RUN_NLI_LABELING=False, this cell is skipped.
#   - If enabled, this cell requires ALLOW_EXPENSIVE_RUNS=True and CUDA.
#
# Safety:
#   - MASTER_PATH is read-only.
#   - This cell writes only to TEMP_RESULTS_PATH and audits/.
#   - No old per-generator result CSVs are read or written.
#   - No load_results(...), save_results(...), or human_label fallback.
#
# Outputs:
#   - audits/temp_results/temp_results.csv
#   - audits/nli_recompute/nli_recompute_audit.csv
#   - audits/nli_recompute/nli_score_diff_audit.csv
# =============================================================================

import gc
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL B3: NLI recomputation"

if not RUN_LABEL_REGENERATION:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=False.")
    print("No NLI probabilities or labels were recomputed.")
    print("temp_results.csv was not modified by this cell.")

elif not RUN_NLI_LABELING:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=True, but RUN_NLI_LABELING=False.")
    print("No NLI probabilities or labels were recomputed.")
    print("temp_results.csv was not modified by this cell.")

else:
    # =========================================================================
    # Guards
    # =========================================================================

    guard_nli_labeling()

    require_package("torch", "torch")
    require_package("transformers", "transformers")
    require_package("sentencepiece", "sentencepiece")

    import torch
    from transformers import AutoTokenizer, T5ForConditionalGeneration

    # =========================================================================
    # Paths
    # =========================================================================

    NLI_AUDIT_DIR = AUDIT_DIR / "nli_recompute"
    NLI_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_AUDIT_CSV = NLI_AUDIT_DIR / "nli_recompute_audit.csv"
    OUT_SCORE_AUDIT_CSV = NLI_AUDIT_DIR / "nli_score_diff_audit.csv"

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError(
            "Safety error: TEMP_RESULTS_PATH points to MASTER_PATH. "
            "Refusing to continue."
        )

    if not TEMP_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"temp_results.csv not found. Run CELL A3 first: {TEMP_RESULTS_PATH}"
        )

    # =========================================================================
    # Config
    # =========================================================================

    HUMAN_COL = "annotator1_label"

    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    NLI_MODEL_ID = globals().get(
        "NLI_MODEL_ID",
        "google/t5_xxl_true_nli_mixture",
    )

    NLI_BATCH_SIZE = globals().get("NLI_BATCH_SIZE", 8)
    NLI_MAX_LENGTH = globals().get("NLI_MAX_LENGTH", 2048)
    NLI_TAU_STRICT = globals().get("NLI_TAU_STRICT", 0.5)
    NLI_THRESHOLD_STEPS = globals().get("NLI_THRESHOLD_STEPS", 101)

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    NLI_REQUIRED_COLS = [
        "nli_entail_prob",
        "nli_label_strict",
        "nli_label_best_tau",
        "nli_tau_strict",
        "nli_tau_best",
    ]

    # =========================================================================
    # Helpers
    # =========================================================================

    def require_same_row_identity(master_df, temp_df):
        identity_cols = ["item_id", "qid", "dataset", "model"]

        missing_master = [c for c in identity_cols if c not in master_df.columns]
        missing_temp = [c for c in identity_cols if c not in temp_df.columns]

        if missing_master or missing_temp:
            raise KeyError(
                f"Missing identity columns. "
                f"master missing={missing_master}, temp missing={missing_temp}"
            )

        if len(master_df) != len(temp_df):
            raise ValueError(
                f"Row-count mismatch: master={len(master_df)}, temp={len(temp_df)}"
            )

        for col in identity_cols:
            if not master_df[col].astype(str).equals(temp_df[col].astype(str)):
                raise ValueError(
                    f"Row identity mismatch in column {col}. "
                    "Reinitialize temp_results.csv from master before continuing."
                )


    def safe_kappa(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)

        try:
            k = cohen_kappa_score(y_true, y_pred, labels=[0, 1])
            return float(k) if np.isfinite(k) else np.nan
        except Exception:
            return np.nan


    def metric_summary(y_true, y_pred):
        mask = np.isin(y_true, [0, 1]) & np.isin(y_pred, [0, 1])

        yh = np.asarray(y_true)[mask].astype(int)
        yp = np.asarray(y_pred)[mask].astype(int)

        tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()

        err = fp + fn
        fp_pct = 100.0 * fp / max(err, 1)

        return {
            "n_valid": int(mask.sum()),
            "kappa": safe_kappa(yh, yp),
            "bias": float(yp.mean() - yh.mean()),
            "fp": int(fp),
            "fn": int(fn),
            "err": int(err),
            "fp_pct": float(fp_pct),
        }


    def get_unique_value(df, col, decimals=12):
        vals = (
            pd.to_numeric(df[col], errors="coerce")
            .dropna()
            .round(decimals)
            .unique()
        )

        if len(vals) != 1:
            raise ValueError(f"Expected one unique value in {col}, found {vals}")

        return float(vals[0])


    def format_nli_input(premise, hypothesis):
        return f"premise: {premise} hypothesis: {hypothesis}"


    def split_reference_variants(ref_str):
        refs = [
            r.strip()
            for r in str(ref_str or "").split("|")
            if r.strip()
        ]

        if not refs:
            refs = [" "]

        return refs


    def labels_at_threshold(entail_probs, tau):
        """
        Direction:
          entail_prob < tau -> hallucination label 1
          entail_prob >= tau -> non-hallucination label 0
        """
        probs = np.asarray(entail_probs, dtype=float)
        pred = np.full(len(probs), -1, dtype=int)

        valid = np.isfinite(probs)

        if np.isfinite(tau):
            pred[valid] = (probs[valid] < tau).astype(int)

        return pred


    def sweep_threshold_oracle(entail_probs, human_labels, steps=NLI_THRESHOLD_STEPS):
        """
        Oracle threshold selected on the full generator subset.

        Direction:
          entail_prob < tau -> hallucination label 1
        """
        probs = np.asarray(entail_probs, dtype=float)
        labels = np.asarray(human_labels, dtype=int)

        valid = np.isfinite(probs) & np.isin(labels, [0, 1])

        if int(valid.sum()) < 10:
            return np.nan, NLI_TAU_STRICT

        best_k = -np.inf
        best_tau = NLI_TAU_STRICT

        for tau in np.linspace(0, 1, steps):
            pred = (probs < tau).astype(int)
            k = safe_kappa(labels[valid], pred[valid])

            if np.isfinite(k) and k > best_k:
                best_k = k
                best_tau = tau

        return float(best_k), float(best_tau)


    def score_diff_summary(master_df, temp_df, col):
        master_scores = pd.to_numeric(master_df[col], errors="coerce").to_numpy()
        temp_scores = pd.to_numeric(temp_df[col], errors="coerce").to_numpy()

        valid = np.isfinite(master_scores) & np.isfinite(temp_scores)
        diff = np.abs(master_scores[valid] - temp_scores[valid])

        if len(diff) == 0:
            return {
                "score_col": col,
                "n_valid": 0,
                "max_absdiff": np.nan,
                "mean_absdiff": np.nan,
                "n_absdiff_gt_1e_12": np.nan,
                "n_absdiff_gt_1e_9": np.nan,
                "n_absdiff_gt_1e_6": np.nan,
                "n_absdiff_gt_1e_4": np.nan,
            }

        return {
            "score_col": col,
            "n_valid": int(valid.sum()),
            "max_absdiff": float(diff.max()),
            "mean_absdiff": float(diff.mean()),
            "n_absdiff_gt_1e_12": int((diff > 1e-12).sum()),
            "n_absdiff_gt_1e_9": int((diff > 1e-9).sum()),
            "n_absdiff_gt_1e_6": int((diff > 1e-6).sum()),
            "n_absdiff_gt_1e_4": int((diff > 1e-4).sum()),
        }


    @torch.no_grad()
    def nli_entailment_batch(
        tokenizer,
        model,
        premises,
        hypotheses,
        batch_size=NLI_BATCH_SIZE,
    ):
        """
        Return entailment probability for each (premise, hypothesis) pair.

        High entailment probability means:
          reference entails answer -> non-hallucination under y_faith.
        """
        all_probs = []

        inputs = [
            format_nli_input(p, h)
            for p, h in zip(premises, hypotheses)
        ]

        token_1 = tokenizer.encode("1", add_special_tokens=False)[0]
        token_0 = tokenizer.encode("0", add_special_tokens=False)[0]

        if hasattr(model, "hf_device_map"):
            first_device = next(iter(model.hf_device_map.values()))
            if first_device in ["cpu", "disk"]:
                input_device = torch.device("cpu")
            else:
                input_device = torch.device(f"cuda:{first_device}" if isinstance(first_device, int) else first_device)
        else:
            input_device = next(model.parameters()).device

        for i in range(0, len(inputs), batch_size):
            batch = inputs[i:i + batch_size]

            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=NLI_MAX_LENGTH,
                return_tensors="pt",
            )

            enc = {k: v.to(input_device) for k, v in enc.items()}

            decoder_ids = torch.full(
                (len(batch), 1),
                tokenizer.pad_token_id,
                dtype=torch.long,
                device=input_device,
            )

            out = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                decoder_input_ids=decoder_ids,
            )

            logits = out.logits[:, 0, :]

            lgt_1 = logits[:, token_1]
            lgt_0 = logits[:, token_0]

            pair_logits = torch.stack([lgt_1, lgt_0], dim=1)
            probs = torch.softmax(pair_logits, dim=1)[:, 0]

            all_probs.extend(probs.float().cpu().numpy().tolist())

        return all_probs


    def recompute_nli_entailment_probs(master_df, tokenizer, model):
        """
        Recompute row-level NLI entailment probabilities.

        Multi-reference aggregation:
          ref_str may contain variants separated by '|'.
          The row score is max entailment probability across variants.
        """
        pair_premises = []
        pair_hypotheses = []
        pair_row_idx = []

        for row_pos, row in master_df.reset_index(drop=True).iterrows():
            refs = split_reference_variants(row["ref_str"])
            ans = str(row["answer_text"] or "")

            for ref in refs:
                pair_premises.append(ref)
                pair_hypotheses.append(ans)
                pair_row_idx.append(row_pos)

        print(f"Constructed {len(pair_premises)} NLI premise/hypothesis pairs.")
        print(f"Rows: {len(master_df)}")

        pair_probs = nli_entailment_batch(
            tokenizer=tokenizer,
            model=model,
            premises=pair_premises,
            hypotheses=pair_hypotheses,
            batch_size=NLI_BATCH_SIZE,
        )

        row_to_probs = {}

        for row_pos, prob in zip(pair_row_idx, pair_probs):
            row_to_probs.setdefault(row_pos, []).append(prob)

        nli_prob = []

        for row_pos in range(len(master_df)):
            probs = row_to_probs.get(row_pos, [np.nan])
            nli_prob.append(float(np.nanmax(probs)))

        return np.asarray(nli_prob, dtype=float)

    # =========================================================================
    # Load and validate master/temp
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)
    temp = pd.read_csv(TEMP_RESULTS_PATH)

    require_same_row_identity(master, temp)

    required_cols = [
        "item_id",
        "qid",
        "dataset",
        "model",
        "ref_str",
        "answer_text",
        HUMAN_COL,
    ] + NLI_REQUIRED_COLS

    missing = [c for c in required_cols if c not in master.columns]

    if missing:
        raise KeyError(f"Missing required columns from release master: {missing}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows in release master, got {len(master)}")

    if not master[HUMAN_COL].isin([0, 1]).all():
        bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
        raise ValueError(f"Invalid human labels in {HUMAN_COL}: {bad_n}")

    for model_name in GEN_ORDER:
        n = int((master["model"] == model_name).sum())
        if n != 300:
            raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

    print("=" * 80)
    print("NLI FULL RECOMPUTE INTO TEMP_RESULTS + AUDIT")
    print("=" * 80)
    print("Release master: ", MASTER_PATH)
    print("Temp results:   ", TEMP_RESULTS_PATH)
    print("Audit CSV:      ", OUT_AUDIT_CSV)
    print("Score audit CSV:", OUT_SCORE_AUDIT_CSV)
    print()
    print("NLI configuration:")
    print(f"  model_id        = {NLI_MODEL_ID}")
    print(f"  device          = {DEVICE}")
    print(f"  dtype           = {DTYPE}")
    print(f"  batch_size      = {NLI_BATCH_SIZE}")
    print(f"  max_length      = {NLI_MAX_LENGTH}")
    print(f"  strict tau      = {NLI_TAU_STRICT}")
    print()
    print("This cell writes only to temp_results.csv and audits/.")

    # =========================================================================
    # Load model and recompute raw NLI entailment probabilities
    # =========================================================================

    print("\n" + "=" * 80)
    print("LOADING NLI MODEL")
    print("=" * 80)

    tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_ID)

    model = T5ForConditionalGeneration.from_pretrained(
        NLI_MODEL_ID,
        torch_dtype=DTYPE,
        device_map="auto",
    )

    model.eval()

    if torch.cuda.is_available():
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
        print(f"VRAM allocated after load: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    else:
        print("Model loaded on CPU.")

    print("\n" + "=" * 80)
    print("RAW NLI SCORE RECOMPUTATION FROM ref_str + answer_text")
    print("=" * 80)

    temp["nli_entail_prob"] = recompute_nli_entailment_probs(
        master_df=master,
        tokenizer=tokenizer,
        model=model,
    )

    score_audit = pd.DataFrame([
        score_diff_summary(master, temp, "nli_entail_prob")
    ])
    score_audit.to_csv(OUT_SCORE_AUDIT_CSV, index=False)

    print("\nRaw NLI score difference audit against release master:")
    display(score_audit)

    # =========================================================================
    # Recompute labels/threshold metadata from recomputed NLI probabilities
    # =========================================================================

    audit_rows = []
    blocking_errors = []

    for model_name in GEN_ORDER:
        mask_g = master["model"] == model_name

        df_master_g = master.loc[mask_g].copy().reset_index(drop=True)
        df_temp_g = temp.loc[mask_g].copy().reset_index(drop=True)

        h = df_master_g[HUMAN_COL].astype(int).to_numpy()
        probs = pd.to_numeric(df_temp_g["nli_entail_prob"], errors="coerce").to_numpy()

        if np.isnan(probs).any():
            n_missing = int(np.isnan(probs).sum())
            blocking_errors.append(f"{model_name}/NLI: missing recomputed probabilities={n_missing}")

        # ---------------------------------------------------------------------
        # Strict NLI, tau = 0.5
        # ---------------------------------------------------------------------
        strict_labels_temp = labels_at_threshold(probs, NLI_TAU_STRICT)

        strict_labels_master = df_master_g["nli_label_strict"].astype(int).to_numpy()
        tau_strict_master = get_unique_value(df_master_g, "nli_tau_strict")

        strict_label_mismatch = int(np.sum(strict_labels_temp != strict_labels_master))
        strict_tau_absdiff = abs(NLI_TAU_STRICT - tau_strict_master)

        # ---------------------------------------------------------------------
        # Oracle best-tau NLI
        # ---------------------------------------------------------------------
        k_best_temp, tau_best_temp = sweep_threshold_oracle(probs, h)
        best_labels_temp = labels_at_threshold(probs, tau_best_temp)

        best_labels_master = df_master_g["nli_label_best_tau"].astype(int).to_numpy()
        tau_best_master = get_unique_value(df_master_g, "nli_tau_best")

        best_label_mismatch = int(np.sum(best_labels_temp != best_labels_master))
        best_tau_absdiff = abs(tau_best_temp - tau_best_master)

        # ---------------------------------------------------------------------
        # Write recomputed values to temp_results using paper column names
        # ---------------------------------------------------------------------
        temp.loc[mask_g, "nli_label_strict"] = strict_labels_temp
        temp.loc[mask_g, "nli_label_best_tau"] = best_labels_temp
        temp.loc[mask_g, "nli_tau_strict"] = NLI_TAU_STRICT
        temp.loc[mask_g, "nli_tau_best"] = tau_best_temp

        if "nli_label_best" in temp.columns:
            temp.loc[mask_g, "nli_label_best"] = best_labels_temp

        # ---------------------------------------------------------------------
        # Summaries
        # ---------------------------------------------------------------------
        strict_metrics_master = metric_summary(h, strict_labels_master)
        strict_metrics_temp = metric_summary(h, strict_labels_temp)

        best_metrics_master = metric_summary(h, best_labels_master)
        best_metrics_temp = metric_summary(h, best_labels_temp)

        row = {
            "model": model_name,
            "metric": "NLI",

            "strict_tau_master": tau_strict_master,
            "strict_tau_temp": NLI_TAU_STRICT,
            "strict_tau_absdiff": strict_tau_absdiff,
            "strict_label_mismatch": strict_label_mismatch,
            "strict_kappa_master": strict_metrics_master["kappa"],
            "strict_kappa_temp": strict_metrics_temp["kappa"],
            "strict_fp_master": strict_metrics_master["fp"],
            "strict_fp_temp": strict_metrics_temp["fp"],
            "strict_fn_master": strict_metrics_master["fn"],
            "strict_fn_temp": strict_metrics_temp["fn"],
            "strict_fp_pct_master": strict_metrics_master["fp_pct"],
            "strict_fp_pct_temp": strict_metrics_temp["fp_pct"],

            "best_tau_master": tau_best_master,
            "best_tau_temp": tau_best_temp,
            "best_tau_absdiff": best_tau_absdiff,
            "best_label_mismatch": best_label_mismatch,
            "best_kappa_master": best_metrics_master["kappa"],
            "best_kappa_temp": best_metrics_temp["kappa"],
            "best_fp_master": best_metrics_master["fp"],
            "best_fp_temp": best_metrics_temp["fp"],
            "best_fn_master": best_metrics_master["fn"],
            "best_fn_temp": best_metrics_temp["fn"],
            "best_fp_pct_master": best_metrics_master["fp_pct"],
            "best_fp_pct_temp": best_metrics_temp["fp_pct"],
        }

        audit_rows.append(row)

        print("\n" + "-" * 80)
        print(model_name)
        print("-" * 80)
        print(
            f"NLI strict: "
            f"tau master={tau_strict_master:.4f}, temp={NLI_TAU_STRICT:.4f}, "
            f"label mismatches={strict_label_mismatch}"
        )
        print(
            f"NLI best-tau: "
            f"tau master={tau_best_master:.4f}, temp={tau_best_temp:.4f}, "
            f"label mismatches={best_label_mismatch}"
        )

        if strict_label_mismatch != 0:
            blocking_errors.append(
                f"{model_name}/NLI strict: label mismatches={strict_label_mismatch}"
            )

        if best_label_mismatch != 0:
            blocking_errors.append(
                f"{model_name}/NLI best-tau: label mismatches={best_label_mismatch}"
            )

    # =========================================================================
    # Save temp_results and audits
    # =========================================================================

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError("Refusing to write temp results to MASTER_PATH.")

    temp.to_csv(TEMP_RESULTS_PATH, index=False)

    audit = pd.DataFrame(audit_rows)
    audit.to_csv(OUT_AUDIT_CSV, index=False)

    print("\n" + "=" * 80)
    print("NLI RECOMPUTE AUDIT SUMMARY")
    print("=" * 80)

    display_cols = [
        "model",
        "strict_tau_master",
        "strict_tau_temp",
        "strict_label_mismatch",
        "strict_kappa_master",
        "strict_kappa_temp",
        "best_tau_master",
        "best_tau_temp",
        "best_label_mismatch",
        "best_kappa_master",
        "best_kappa_temp",
    ]

    display(audit[display_cols])

    print("\nSaved temp_results.csv:")
    print(TEMP_RESULTS_PATH)

    print("\nSaved audit CSVs:")
    print(OUT_AUDIT_CSV)
    print(OUT_SCORE_AUDIT_CSV)

    raw_score_diffs = score_audit[
        score_audit["n_absdiff_gt_1e_6"].fillna(0).astype(int) > 0
    ]

    if len(raw_score_diffs) > 0:
        print("\nNOTE: Recomputed NLI entailment probabilities differ from the release master.")
        print("This can happen if model/package/device settings differ from the original run.")
        print("The blocking reproduction criterion here is label-level agreement.")
        print("If label mismatches occur, investigate before changing any paper result.")

    if blocking_errors:
        print("\nBlocking audit errors:")
        for e in blocking_errors:
            print("  •", e)

        raise AssertionError(
            "NLI recomputation audit failed. "
            "Recomputed NLI labels in temp_results.csv do not match the locked "
            "release-master labels. Do not modify paper results until this is understood."
        )

    # =========================================================================
    # Unload model
    # =========================================================================

    del model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\n✓ NLI recomputation audit passed.")
    print("  Recomputed NLI labels in temp_results.csv match release-master labels.")
    print("NLI model unloaded.")

In [ ]:
# =============================================================================
# CELL B4: OpenAI judge labeling into temp_results + audit
#
# Purpose:
#   Optionally rerun OpenAI LLM judges for prompts P1/P2/P3, write labels and
#   confidences into temp_results.csv, and compare the regenerated labels against
#   the locked release master.
#
# Execution:
#   - If RUN_LABEL_REGENERATION=False, this cell is skipped.
#   - If RUN_OPENAI_LABELING=False, this cell is skipped.
#   - If enabled, this cell requires:
#       RUN_LABEL_REGENERATION=True
#       ALLOW_EXPENSIVE_RUNS=True
#       RUN_OPENAI_LABELING=True
#       OPENAI_API_KEY in environment
#
# Safety:
#   - MASTER_PATH is read-only.
#   - This cell writes only to TEMP_RESULTS_PATH and audits/.
#   - No old per-generator result CSVs are read or written.
#
# Outputs:
#   - audits/temp_results/temp_results.csv
#   - audits/openai_recompute/openai_recompute_audit.csv
#   - audits/openai_recompute/openai_parse_failures.csv
# =============================================================================

import os
import re
import time
import json
import numpy as np
import pandas as pd
from openai import OpenAI

from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL B4: OpenAI judge labeling"

if not RUN_LABEL_REGENERATION:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=False.")
    print("No OpenAI judge labels were regenerated.")
    print("temp_results.csv was not modified by this cell.")

elif not RUN_OPENAI_LABELING:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=True, but RUN_OPENAI_LABELING=False.")
    print("No OpenAI judge labels were regenerated.")
    print("temp_results.csv was not modified by this cell.")

else:
    # =========================================================================
    # Guards
    # =========================================================================

    guard_openai_labeling()
    require_package("openai", "openai")

    openai_key = get_secret("OPENAI_API_KEY", required=True)
    client = OpenAI(api_key=openai_key)


    # =========================================================================
    # Paths
    # =========================================================================

    OPENAI_AUDIT_DIR = AUDIT_DIR / "openai_recompute"
    OPENAI_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_AUDIT_CSV = OPENAI_AUDIT_DIR / "openai_recompute_audit.csv"
    OUT_PARSE_FAILURES_CSV = OPENAI_AUDIT_DIR / "openai_parse_failures.csv"

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError(
            "Safety error: TEMP_RESULTS_PATH points to MASTER_PATH. "
            "Refusing to continue."
        )

    if not TEMP_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"temp_results.csv not found. Run CELL A3 first: {TEMP_RESULTS_PATH}"
        )

    # =========================================================================
    # Config
    # =========================================================================

    HUMAN_COL = "annotator1_label"

    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    # These are release-master prefixes and API model IDs.
    # Override OPENAI_JUDGE_SPECS before this cell if needed.
    OPENAI_JUDGE_SPECS = globals().get(
        "OPENAI_JUDGE_SPECS",
        [
            ("gpt_5_mini", "GPT-5-mini", "gpt-5-mini"),
            ("gpt_5_nano", "GPT-5-nano", "gpt-5-nano"),
            ("gpt_5_4",    "GPT-5.4",    "gpt-5.4"),
        ],
    )

    # Prompt keys are notebook prompt keys. Output columns use lowercase p1/p2/p3.
    OPENAI_PROMPT_KEYS = globals().get(
        "OPENAI_PROMPT_KEYS",
        ["P1", "P2", "P3"],
    )

    OPENAI_MAX_WORKERS = globals().get("OPENAI_MAX_WORKERS", 4)
    OPENAI_MAX_ATTEMPTS = globals().get("OPENAI_MAX_ATTEMPTS", 2)
    OPENAI_MAX_OUTPUT_TOKENS = globals().get("OPENAI_MAX_OUTPUT_TOKENS", 2500)
    OPENAI_REASONING_EFFORT = globals().get("OPENAI_REASONING_EFFORT", "low")

    # Default True because this is a regeneration cell.
    # If False, rows with already-valid temp labels are skipped.
    OPENAI_FORCE_RECOMPUTE = globals().get("OPENAI_FORCE_RECOMPUTE", True)

    # Save temp_results after each generator × judge × prompt block.
    OPENAI_CHECKPOINT_EVERY_BLOCK = globals().get(
        "OPENAI_CHECKPOINT_EVERY_BLOCK",
        True,
    )

    # =========================================================================
    # Helpers
    # =========================================================================

    def require_same_row_identity(master_df, temp_df):
        identity_cols = ["item_id", "qid", "dataset", "model"]

        missing_master = [c for c in identity_cols if c not in master_df.columns]
        missing_temp = [c for c in identity_cols if c not in temp_df.columns]

        if missing_master or missing_temp:
            raise KeyError(
                f"Missing identity columns. "
                f"master missing={missing_master}, temp missing={missing_temp}"
            )

        if len(master_df) != len(temp_df):
            raise ValueError(
                f"Row-count mismatch: master={len(master_df)}, temp={len(temp_df)}"
            )

        for col in identity_cols:
            if not master_df[col].astype(str).equals(temp_df[col].astype(str)):
                raise ValueError(
                    f"Row identity mismatch in column {col}. "
                    "Reinitialize temp_results.csv from master before continuing."
                )


    def resolve_prompt_dict():
        """
        Find the prompt dictionary defined in CELL B1.

        Accepted globals:
          - PROMPT_DEFS
          - PROMPT_TEXTS
          - JUDGE_PROMPTS
          - PROMPTS, if it is a dict

        The old A2 constant PROMPTS=["p1","p2","p3"] is not accepted here.
        """
        candidate_names = [
            "PROMPT_DEFS",
            "PROMPT_TEXTS",
            "JUDGE_PROMPTS",
            "PROMPTS",
        ]

        for name in candidate_names:
            obj = globals().get(name, None)
            if isinstance(obj, dict):
                return obj

        raise RuntimeError(
            "No prompt dictionary found. Run CELL B1 first.\n"
            "Expected one of these globals to be a dict: "
            "PROMPT_DEFS, PROMPT_TEXTS, JUDGE_PROMPTS, PROMPTS.\n"
            "Prompt keys must include P1, P2, P3 or p1, p2, p3."
        )


    def get_prompt_text(prompt_dict, prompt_key):
        candidates = [
            prompt_key,
            prompt_key.upper(),
            prompt_key.lower(),
        ]

        for key in candidates:
            if key in prompt_dict:
                return str(prompt_dict[key])

        raise KeyError(
            f"Prompt key {prompt_key!r} not found in prompt dictionary. "
            f"Available keys: {list(prompt_dict.keys())}"
        )


    def output_prompt_suffix(prompt_key):
        """
        Convert notebook prompt key P1/P2/P3 to release-master suffix p1/p2/p3.
        """
        p = str(prompt_key).lower()

        if p not in {"p1", "p2", "p3"}:
            raise ValueError(
                f"Invalid prompt key {prompt_key!r}. "
                "Only P1/P2/P3 are allowed in this reproduction notebook."
            )

        return p


    def parse_json_label(text):
        """
        Parse judge output.

        Expected JSON:
          {"label": 0 or 1, "confidence": float in [0,1]}

        Returns:
          (label, confidence, raw_text, parse_error)
        """
        if not isinstance(text, str):
            return -1, -1.0, "", "non-string response"

        raw_text = text
        s = text.strip()

        s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s*```$", "", s)

        try:
            obj = json.loads(s)
            label = int(obj.get("label", -1))
            conf_raw = obj.get("confidence", -1)

            if isinstance(conf_raw, str):
                m = re.match(r"([\d.]+)", conf_raw.strip())
                conf_raw = float(m.group(1)) if m else -1.0

            conf = float(conf_raw)

            if conf > 1.0:
                conf = conf / 100.0

            if label in (0, 1) and 0.0 <= conf <= 1.0:
                return label, conf, raw_text, ""

        except Exception:
            pass

        for match in re.finditer(r"\{[^{}]+\}", s):
            try:
                obj = json.loads(match.group())
                label = int(obj.get("label", -1))
                conf_raw = obj.get("confidence", -1)

                if isinstance(conf_raw, str):
                    m = re.match(r"([\d.]+)", conf_raw.strip())
                    conf_raw = float(m.group(1)) if m else -1.0

                conf = float(conf_raw)

                if conf > 1.0:
                    conf = conf / 100.0

                if label in (0, 1) and 0.0 <= conf <= 1.0:
                    return label, conf, raw_text, ""

            except Exception:
                continue

        return -1, -1.0, raw_text, "could not parse valid JSON label/confidence"


    def build_user_message(question, answer, reference):
        refs = [
            r.strip()
            for r in str(reference or "").split("|")
            if r.strip()
        ]

        refs_text = "\n".join(f"- {r}" for r in refs) if refs else "- "

        return (
            f"QUESTION: {question}\n"
            f"MODEL_ANSWER: {answer}\n"
            f"REFERENCE_ANSWERS (any one is acceptable):\n{refs_text}"
        )

    def create_openai_response(model_id, prompt_text, user_message):
        try:
            return client.responses.create(
                model=model_id,
                reasoning={"effort": OPENAI_REASONING_EFFORT},
                input=prompt_text + "\n\n" + user_message,
                max_output_tokens=OPENAI_MAX_OUTPUT_TOKENS,
            )
        except Exception as e:
            msg = str(e).lower()
            if "reasoning" not in msg and "unsupported" not in msg and "unknown parameter" not in msg:
                raise
            return client.responses.create(
                model=model_id,
                input=prompt_text + "\n\n" + user_message,
                max_output_tokens=OPENAI_MAX_OUTPUT_TOKENS,
        )



    def run_openai_judge_one(row, prompt_text, model_id):
        """
        Return one row-level judge result.

        Output:
          {
            label, confidence, raw_text, parse_error
          }
        """
        user_message = build_user_message(
            question=row["question"],
            answer=row["answer_text"],
            reference=row["ref_str"],
        )

        last_error = ""

        for attempt in range(OPENAI_MAX_ATTEMPTS):
            try:
                resp = create_openai_response(
                    model_id=model_id,
                    prompt_text=prompt_text,
                    user_message=user_message,
                )

                text = getattr(resp, "output_text", "") or ""
                label, conf, raw_text, parse_error = parse_json_label(text)

                if label in (0, 1):
                    return {
                        "label": label,
                        "confidence": conf,
                        "raw_text": raw_text,
                        "parse_error": "",
                    }

                last_error = parse_error or "invalid label"

                time.sleep(1.0 + attempt)

            except Exception as e:
                last_error = repr(e)
                wait = 2.0 * (attempt + 1)

                if hasattr(e, "status_code") and getattr(e, "status_code") == 429:
                    wait = max(wait, 10.0)

                time.sleep(wait)

        return {
            "label": -1,
            "confidence": -1.0,
            "raw_text": "",
            "parse_error": last_error or "failed after max attempts",
        }


    def safe_kappa(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)

        try:
            k = cohen_kappa_score(y_true, y_pred, labels=[0, 1])
            return float(k) if np.isfinite(k) else np.nan
        except Exception:
            return np.nan


    def metric_summary(y_true, y_pred):
        mask = np.isin(y_true, [0, 1]) & np.isin(y_pred, [0, 1])

        yh = np.asarray(y_true)[mask].astype(int)
        yp = np.asarray(y_pred)[mask].astype(int)

        if len(yh) == 0:
            return {
                "n_valid": 0,
                "n_total": int(len(y_pred)),
                "kappa": np.nan,
                "bias": np.nan,
                "fp": np.nan,
                "fn": np.nan,
                "err": np.nan,
                "fp_pct": np.nan,
                "n_fail": int(len(y_pred)),
            }

        tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()

        err = fp + fn
        fp_pct = 100.0 * fp / max(err, 1)

        return {
            "n_valid": int(mask.sum()),
            "n_total": int(len(y_pred)),
            "kappa": safe_kappa(yh, yp),
            "bias": float(yp.mean() - yh.mean()),
            "fp": int(fp),
            "fn": int(fn),
            "err": int(err),
            "fp_pct": float(fp_pct),
            "n_fail": int(len(y_pred) - mask.sum()),
        }


    # =========================================================================
    # Load and validate master/temp
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)
    temp = pd.read_csv(TEMP_RESULTS_PATH)

    require_same_row_identity(master, temp)

    prompt_dict = resolve_prompt_dict()

    required_base_cols = [
        "item_id",
        "qid",
        "dataset",
        "model",
        "question",
        "ref_str",
        "answer_text",
        HUMAN_COL,
    ]

    missing_base = [c for c in required_base_cols if c not in master.columns]

    if missing_base:
        raise KeyError(f"Missing required columns from release master: {missing_base}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows in release master, got {len(master)}")

    if not master[HUMAN_COL].isin([0, 1]).all():
        bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
        raise ValueError(f"Invalid human labels in {HUMAN_COL}: {bad_n}")

    for model_name in GEN_ORDER:
        n = int((master["model"] == model_name).sum())
        if n != 300:
            raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

    # Validate output columns in master.
    required_label_cols = []

    for prefix, _, _ in OPENAI_JUDGE_SPECS:
        for prompt_key in OPENAI_PROMPT_KEYS:
            p = output_prompt_suffix(prompt_key)
            required_label_cols.append(f"label_{prefix}_{p}")

    missing_label_cols = [c for c in required_label_cols if c not in master.columns]

    if missing_label_cols:
        raise KeyError(
            "Missing required OpenAI label columns from release master: "
            f"{missing_label_cols}"
        )

    # Ensure temp has all required label/conf columns.
    for label_col in required_label_cols:
        if label_col not in temp.columns:
            temp[label_col] = -1

        conf_col = label_col.replace("label_", "conf_", 1)
        if conf_col not in temp.columns:
            temp[conf_col] = -1.0

    print("=" * 80)
    print("OPENAI JUDGE LABEL REGENERATION INTO TEMP_RESULTS + AUDIT")
    print("=" * 80)
    print("Release master: ", MASTER_PATH)
    print("Temp results:   ", TEMP_RESULTS_PATH)
    print("Audit CSV:      ", OUT_AUDIT_CSV)
    print("Parse failures: ", OUT_PARSE_FAILURES_CSV)
    print()
    print("OpenAI judge specs:")
    for prefix, display_name, model_id in OPENAI_JUDGE_SPECS:
        print(f"  {prefix:<15} {display_name:<15} model_id={model_id}")
    print()
    print("Prompt keys:", OPENAI_PROMPT_KEYS)
    print("Max workers:", OPENAI_MAX_WORKERS)
    print("Max attempts:", OPENAI_MAX_ATTEMPTS)
    print("Force recompute:", OPENAI_FORCE_RECOMPUTE)
    print()
    print("This cell writes only to temp_results.csv and audits/.")

    # =========================================================================
    # Run labeling
    # =========================================================================

    audit_rows = []
    parse_failure_rows = []
    blocking_errors = []

    for prefix, display_name, api_model_id in OPENAI_JUDGE_SPECS:
        for prompt_key in OPENAI_PROMPT_KEYS:
            prompt_suffix = output_prompt_suffix(prompt_key)
            prompt_text = get_prompt_text(prompt_dict, prompt_key)

            label_col = f"label_{prefix}_{prompt_suffix}"
            conf_col = f"conf_{prefix}_{prompt_suffix}"

            print("\n" + "=" * 80)
            print(f"{display_name} / {prompt_key} -> {label_col}")
            print("=" * 80)

            for gen_model in GEN_ORDER:
                mask_g = master["model"] == gen_model
                row_indices = master.index[mask_g].tolist()

                print("\n" + "-" * 80)
                print(f"{display_name} / {prompt_key} / {gen_model}")
                print("-" * 80)

                if OPENAI_FORCE_RECOMPUTE:
                    target_indices = row_indices
                else:
                    current = pd.to_numeric(
                        temp.loc[row_indices, label_col],
                        errors="coerce",
                    )
                    target_indices = [
                        idx for idx, val in zip(row_indices, current)
                        if val not in [0, 1]
                    ]

                if not target_indices:
                    print("All rows already have valid temp labels; skipping this block.")
                else:
                    print(f"Running {len(target_indices)} rows with {OPENAI_MAX_WORKERS} workers.")

                    results = {}

                    def _task(idx):
                        row = master.loc[idx]
                        result = run_openai_judge_one(
                            row=row,
                            prompt_text=prompt_text,
                            model_id=api_model_id,
                        )
                        return idx, result

                    with ThreadPoolExecutor(max_workers=OPENAI_MAX_WORKERS) as executor:
                        futures = {
                            executor.submit(_task, idx): idx
                            for idx in target_indices
                        }

                        with tqdm(
                            total=len(target_indices),
                            desc=f"{prefix} {prompt_key} {gen_model}",
                        ) as pbar:
                            for future in as_completed(futures):
                                idx, result = future.result()
                                results[idx] = result
                                pbar.update(1)

                    for idx in target_indices:
                        result = results[idx]

                        temp.loc[idx, label_col] = int(result["label"])
                        temp.loc[idx, conf_col] = float(result["confidence"])

                        if int(result["label"]) not in [0, 1]:
                            parse_failure_rows.append(
                                {
                                    "item_id": master.loc[idx, "item_id"],
                                    "qid": master.loc[idx, "qid"],
                                    "dataset": master.loc[idx, "dataset"],
                                    "generator_model": master.loc[idx, "model"],
                                    "judge_prefix": prefix,
                                    "judge_display": display_name,
                                    "api_model_id": api_model_id,
                                    "prompt": prompt_suffix,
                                    "label_col": label_col,
                                    "parse_error": result["parse_error"],
                                    "raw_text": result["raw_text"],
                                }
                            )

                    if OPENAI_CHECKPOINT_EVERY_BLOCK:
                        if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
                            raise RuntimeError("Refusing to write temp results to MASTER_PATH.")
                        temp.to_csv(TEMP_RESULTS_PATH, index=False)
                        print(f"Checkpoint saved: {TEMP_RESULTS_PATH}")

                # -------------------------------------------------------------
                # Audit temp labels against release master for this block
                # -------------------------------------------------------------
                h = master.loc[row_indices, HUMAN_COL].astype(int).to_numpy()
                master_labels = master.loc[row_indices, label_col].astype(int).to_numpy()
                temp_labels = pd.to_numeric(
                    temp.loc[row_indices, label_col],
                    errors="coerce",
                ).fillna(-1).astype(int).to_numpy()

                n_parse_fail = int(np.sum(~np.isin(temp_labels, [0, 1])))
                n_mismatch = int(np.sum(temp_labels != master_labels))

                master_metrics = metric_summary(h, master_labels)
                temp_metrics = metric_summary(h, temp_labels)

                audit_rows.append(
                    {
                        "judge_prefix": prefix,
                        "judge_display": display_name,
                        "api_model_id": api_model_id,
                        "prompt": prompt_suffix,
                        "label_col": label_col,
                        "generator_model": gen_model,

                        "n_total": int(len(row_indices)),
                        "n_parse_fail": n_parse_fail,
                        "n_label_mismatch": n_mismatch,

                        "kappa_master": master_metrics["kappa"],
                        "kappa_temp": temp_metrics["kappa"],
                        "bias_master": master_metrics["bias"],
                        "bias_temp": temp_metrics["bias"],
                        "fp_master": master_metrics["fp"],
                        "fp_temp": temp_metrics["fp"],
                        "fn_master": master_metrics["fn"],
                        "fn_temp": temp_metrics["fn"],
                        "err_master": master_metrics["err"],
                        "err_temp": temp_metrics["err"],
                        "fp_pct_master": master_metrics["fp_pct"],
                        "fp_pct_temp": temp_metrics["fp_pct"],
                    }
                )

                print(
                    f"Audit: parse_fail={n_parse_fail}, "
                    f"label_mismatch={n_mismatch}, "
                    f"kappa master={master_metrics['kappa']:.3f}, "
                    f"temp={temp_metrics['kappa']:.3f}"
                )

                if n_parse_fail != 0:
                    blocking_errors.append(
                        f"{display_name}/{prompt_key}/{gen_model}: "
                        f"parse failures={n_parse_fail}"
                    )

                if n_mismatch > 0:
                    print(
                        f"  NOTE: {n_mismatch}/{len(row_indices)} label(s) differ from "
                        f"release master. Expected: API model outputs are not guaranteed "
                        f"to be reproducible across calls due to server-side sampling "
                        f"and model version updates."
                    )

    # =========================================================================
    # Save outputs
    # =========================================================================

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError("Refusing to write temp results to MASTER_PATH.")

    temp.to_csv(TEMP_RESULTS_PATH, index=False)

    audit = pd.DataFrame(audit_rows)
    audit.to_csv(OUT_AUDIT_CSV, index=False)

    parse_failures = pd.DataFrame(parse_failure_rows)
    parse_failures.to_csv(OUT_PARSE_FAILURES_CSV, index=False)

    print("\n" + "=" * 80)
    print("OPENAI RECOMPUTE AUDIT SUMMARY")
    print("=" * 80)

    display_cols = [
        "judge_prefix",
        "prompt",
        "generator_model",
        "n_parse_fail",
        "n_label_mismatch",
        "kappa_master",
        "kappa_temp",
        "fp_master",
        "fp_temp",
        "fn_master",
        "fn_temp",
    ]

    display(audit[display_cols])

    print("\nSaved temp_results.csv:")
    print(TEMP_RESULTS_PATH)

    print("\nSaved audit CSVs:")
    print(OUT_AUDIT_CSV)
    print(OUT_PARSE_FAILURES_CSV)

    if blocking_errors:
        print("\nBlocking parse errors:")
        for e in blocking_errors:
            print("  •", e)

        raise AssertionError(
            "OpenAI judge labeling produced unparseable outputs. "
            "See openai_parse_failures.csv. Every item must yield a valid "
            "label before temp_results.csv is used to rebuild the master."
        )

    print("\n✓ All OpenAI judge outputs parsed successfully.")
    print("  Label differences from the release master are expected under the")
    print("  corrected p2/p3 prompts and are reported in the audit CSV.")


In [ ]:
# =========================================================================
# Optional: install the Anthropic SDK if it is not already available.
# Run this before CELL B5. Skip if `anthropic` is already installed.
# =========================================================================

#%pip install anthropic --quiet

In [ ]:
# =============================================================================
# CELL B5: Anthropic judge labeling into temp_results + audit
#
# Purpose:
#   Optionally rerun Anthropic/Claude judge labels for prompts P1/P2/P3, write
#   labels and confidences into temp_results.csv, and compare regenerated labels
#   against the locked release master.
#
# Execution:
#   - If RUN_LABEL_REGENERATION=False, this cell is skipped.
#   - If RUN_ANTHROPIC_LABELING=False, this cell is skipped.
#   - If enabled, this cell requires:
#       RUN_LABEL_REGENERATION=True
#       ALLOW_EXPENSIVE_RUNS=True
#       RUN_ANTHROPIC_LABELING=True
#       ANTHROPIC_API_KEY in Colab Secrets (key icon in left sidebar)
#
# Safety:
#   - MASTER_PATH is read-only.
#   - This cell writes only to TEMP_RESULTS_PATH and audits/.
#   - No old per-generator result CSVs are read or written.
#
# Outputs:
#   - audits/temp_results/temp_results.csv
#   - audits/anthropic_recompute/anthropic_recompute_audit.csv
#   - audits/anthropic_recompute/anthropic_parse_failures.csv
# =============================================================================

import os
import re
import time
import json
import numpy as np
import pandas as pd


from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL B5: Anthropic judge labeling"

if not RUN_LABEL_REGENERATION:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=False.")
    print("No Anthropic judge labels were regenerated.")
    print("temp_results.csv was not modified by this cell.")

elif not RUN_ANTHROPIC_LABELING:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=True, but RUN_ANTHROPIC_LABELING=False.")
    print("No Anthropic judge labels were regenerated.")
    print("temp_results.csv was not modified by this cell.")

else:
    # =========================================================================
    # Guards
    # =========================================================================

    guard_anthropic_labeling()
    require_package("anthropic", "anthropic")
    import anthropic

    anthropic_key = get_secret("ANTHROPIC_API_KEY", required=True)
    client = anthropic.Anthropic(api_key=anthropic_key)

    # =========================================================================
    # Paths
    # =========================================================================

    ANTHROPIC_AUDIT_DIR = AUDIT_DIR / "anthropic_recompute"
    ANTHROPIC_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_AUDIT_CSV = ANTHROPIC_AUDIT_DIR / "anthropic_recompute_audit.csv"
    OUT_PARSE_FAILURES_CSV = ANTHROPIC_AUDIT_DIR / "anthropic_parse_failures.csv"

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError(
            "Safety error: TEMP_RESULTS_PATH points to MASTER_PATH. "
            "Refusing to continue."
        )

    if not TEMP_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"temp_results.csv not found. Run CELL A3 first: {TEMP_RESULTS_PATH}"
        )

    # =========================================================================
    # Config
    #
    # For a test run use one generator and one prompt:
    #   GEN_ORDER = ["llama3-8b"]
    #   ANTHROPIC_PROMPT_KEYS = ["P1"]
    #
    # For the full run restore:
    #   GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]
    #   ANTHROPIC_PROMPT_KEYS = ["P1", "P2", "P3"]
    # =========================================================================

    HUMAN_COL = "annotator1_label"

    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    ANTHROPIC_JUDGE_SPECS = [
        ("claude_opus_4_7", "Claude Opus 4.7", "claude-opus-4-7"),
    ]

    ANTHROPIC_PROMPT_KEYS = ["P1", "P2", "P3"]


    ANTHROPIC_MAX_WORKERS = 2
    ANTHROPIC_MAX_ATTEMPTS = 1
    ANTHROPIC_MAX_TOKENS = 2500
    ANTHROPIC_FORCE_RECOMPUTE = True
    ANTHROPIC_USE_PROMPT_CACHE = True
    ANTHROPIC_CHECKPOINT_EVERY_BLOCK = True

    # =========================================================================
    # Helpers
    # =========================================================================

    def require_same_row_identity(master_df, temp_df):
        identity_cols = ["item_id", "qid", "dataset", "model"]

        missing_master = [c for c in identity_cols if c not in master_df.columns]
        missing_temp = [c for c in identity_cols if c not in temp_df.columns]

        if missing_master or missing_temp:
            raise KeyError(
                f"Missing identity columns. "
                f"master missing={missing_master}, temp missing={missing_temp}"
            )

        if len(master_df) != len(temp_df):
            raise ValueError(
                f"Row-count mismatch: master={len(master_df)}, temp={len(temp_df)}"
            )

        for col in identity_cols:
            if not master_df[col].astype(str).equals(temp_df[col].astype(str)):
                raise ValueError(
                    f"Row identity mismatch in column {col}. "
                    "Reinitialize temp_results.csv from master before continuing."
                )


    def resolve_prompt_dict():
        """
        Find the prompt dictionary defined in CELL B1.

        Accepted globals:
          - PROMPT_DEFS
          - PROMPT_TEXTS
          - JUDGE_PROMPTS
          - PROMPTS, if it is a dict
        """
        candidate_names = [
            "PROMPT_DEFS",
            "PROMPT_TEXTS",
            "JUDGE_PROMPTS",
            "PROMPTS",
        ]

        for name in candidate_names:
            obj = globals().get(name, None)
            if isinstance(obj, dict):
                return obj

        raise RuntimeError(
            "No prompt dictionary found. Run CELL B1 first.\n"
            "Expected one of these globals to be a dict: "
            "PROMPT_DEFS, PROMPT_TEXTS, JUDGE_PROMPTS, PROMPTS.\n"
            "Prompt keys must include P1, P2, P3 or p1, p2, p3."
        )


    def get_prompt_text(prompt_dict, prompt_key):
        for key in [prompt_key, prompt_key.upper(), prompt_key.lower()]:
            if key in prompt_dict:
                return str(prompt_dict[key])

        raise KeyError(
            f"Prompt key {prompt_key!r} not found in prompt dictionary. "
            f"Available keys: {list(prompt_dict.keys())}"
        )


    def output_prompt_suffix(prompt_key):
        """Convert notebook prompt key P1/P2/P3 to release-master suffix p1/p2/p3."""
        p = str(prompt_key).lower()

        if p not in {"p1", "p2", "p3"}:
            raise ValueError(
                f"Invalid prompt key {prompt_key!r}. "
                "Only P1/P2/P3 are allowed in this reproduction notebook."
            )

        return p


    def parse_json_label(text):
        """
        Parse judge output.

        Expected JSON:
          {"label": 0 or 1, "confidence": float in [0,1]}

        Returns:
          (label, confidence, raw_text, parse_error)
        """
        if not isinstance(text, str):
            return -1, -1.0, "", "non-string response"

        raw_text = text
        s = text.strip()

        s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s*```$", "", s)

        try:
            obj = json.loads(s)
            label = int(obj.get("label", -1))
            conf_raw = obj.get("confidence", -1)

            if isinstance(conf_raw, str):
                m = re.match(r"([\d.]+)", conf_raw.strip())
                conf_raw = float(m.group(1)) if m else -1.0

            conf = float(conf_raw)

            if conf > 1.0:
                conf = conf / 100.0

            if label in (0, 1) and 0.0 <= conf <= 1.0:
                return label, conf, raw_text, ""

        except Exception:
            pass

        for match in re.finditer(r"\{[^{}]+\}", s):
            try:
                obj = json.loads(match.group())
                label = int(obj.get("label", -1))
                conf_raw = obj.get("confidence", -1)

                if isinstance(conf_raw, str):
                    m = re.match(r"([\d.]+)", conf_raw.strip())
                    conf_raw = float(m.group(1)) if m else -1.0

                conf = float(conf_raw)

                if conf > 1.0:
                    conf = conf / 100.0

                if label in (0, 1) and 0.0 <= conf <= 1.0:
                    return label, conf, raw_text, ""

            except Exception:
                continue

        return -1, -1.0, raw_text, "could not parse valid JSON label/confidence"


    def anthropic_text_from_response(resp):
        """Extract text from an Anthropic message response."""
        parts = []

        try:
            for block in resp.content or []:
                txt = getattr(block, "text", None)
                if txt:
                    parts.append(txt)
        except Exception:
            pass

        return "\n".join(parts).strip()


    def build_user_message(question, answer, reference):
        refs = [
            r.strip()
            for r in str(reference or "").split("|")
            if r.strip()
        ]

        refs_text = "\n".join(f"- {r}" for r in refs) if refs else "- "

        return (
            f"QUESTION: {question}\n"
            f"MODEL_ANSWER: {answer}\n"
            f"REFERENCE_ANSWERS (any one is acceptable):\n{refs_text}"
        )


    def create_anthropic_message(model_id, prompt_text, user_message):
        """
        Create an Anthropic message with optional prompt caching.

        If the model rejects cache_control, retries without caching.
        """
        if ANTHROPIC_USE_PROMPT_CACHE:
            try:
                return client.messages.create(
                    model=model_id,
                    max_tokens=ANTHROPIC_MAX_TOKENS,
                    system=[
                        {
                            "type": "text",
                            "text": prompt_text,
                            "cache_control": {"type": "ephemeral"},
                        }
                    ],
                    messages=[
                        {
                            "role": "user",
                            "content": user_message,
                        }
                    ],
                )
            except Exception as e:
                msg = str(e).lower()

                if (
                    "cache_control" not in msg
                    and "prompt caching" not in msg
                    and "unsupported" not in msg
                    and "unknown" not in msg
                ):
                    raise

        return client.messages.create(
            model=model_id,
            max_tokens=ANTHROPIC_MAX_TOKENS,
            system=prompt_text,
            messages=[
                {
                    "role": "user",
                    "content": user_message,
                }
            ],
        )


    def run_anthropic_judge_one(row, prompt_text, model_id):
        """Return one row-level Anthropic judge result."""
        user_message = build_user_message(
            question=row["question"],
            answer=row["answer_text"],
            reference=row["ref_str"],
        )

        last_error = ""

        for attempt in range(ANTHROPIC_MAX_ATTEMPTS):
            try:
                resp = create_anthropic_message(
                    model_id=model_id,
                    prompt_text=prompt_text,
                    user_message=user_message,
                )

                text = anthropic_text_from_response(resp)
                label, conf, raw_text, parse_error = parse_json_label(text)

                if label in (0, 1):
                    return {
                        "label": label,
                        "confidence": conf,
                        "raw_text": raw_text,
                        "parse_error": "",
                    }

                last_error = parse_error or "invalid label"
                time.sleep(1.0 + attempt)

            except anthropic.RateLimitError as e:
                retry_after = None

                if hasattr(e, "response") and e.response is not None:
                    retry_after = e.response.headers.get("retry-after")

                wait = float(retry_after) + 1.0 if retry_after else min(180.0, 15.0 * (2 ** attempt))
                last_error = repr(e)
                time.sleep(wait)

            except anthropic.APIStatusError as e:
                last_error = repr(e)

                if 500 <= int(getattr(e, "status_code", 0)) < 600:
                    time.sleep(2.0 * (attempt + 1))
                else:
                    break

            except Exception as e:
                last_error = repr(e)
                time.sleep(2.0 * (attempt + 1))

        return {
            "label": -1,
            "confidence": -1.0,
            "raw_text": "",
            "parse_error": last_error or "failed after max attempts",
        }


    def safe_kappa(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)

        try:
            k = cohen_kappa_score(y_true, y_pred, labels=[0, 1])
            return float(k) if np.isfinite(k) else np.nan
        except Exception:
            return np.nan


    def metric_summary(y_true, y_pred):
        mask = np.isin(y_true, [0, 1]) & np.isin(y_pred, [0, 1])

        yh = np.asarray(y_true)[mask].astype(int)
        yp = np.asarray(y_pred)[mask].astype(int)

        if len(yh) == 0:
            return {
                "n_valid": 0,
                "n_total": int(len(y_pred)),
                "kappa": np.nan,
                "bias": np.nan,
                "fp": np.nan,
                "fn": np.nan,
                "err": np.nan,
                "fp_pct": np.nan,
                "n_fail": int(len(y_pred)),
            }

        tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()
        err = fp + fn

        return {
            "n_valid": int(mask.sum()),
            "n_total": int(len(y_pred)),
            "kappa": safe_kappa(yh, yp),
            "bias": float(yp.mean() - yh.mean()),
            "fp": int(fp),
            "fn": int(fn),
            "err": int(err),
            "fp_pct": float(100.0 * fp / max(err, 1)),
            "n_fail": int(len(y_pred) - mask.sum()),
        }

    # =========================================================================
    # Load and validate master/temp
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)
    temp = pd.read_csv(TEMP_RESULTS_PATH)

    require_same_row_identity(master, temp)

    prompt_dict = resolve_prompt_dict()

    required_base_cols = [
        "item_id", "qid", "dataset", "model",
        "question", "ref_str", "answer_text", HUMAN_COL,
    ]

    missing_base = [c for c in required_base_cols if c not in master.columns]

    if missing_base:
        raise KeyError(f"Missing required columns from release master: {missing_base}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows in release master, got {len(master)}")

    if not master[HUMAN_COL].isin([0, 1]).all():
        bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
        raise ValueError(f"Invalid human labels in {HUMAN_COL}: {bad_n}")

    for model_name in GEN_ORDER:
        n = int((master["model"] == model_name).sum())
        if n != 300:
            raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

    required_label_cols = []

    for prefix, _, _ in ANTHROPIC_JUDGE_SPECS:
        for prompt_key in ANTHROPIC_PROMPT_KEYS:
            p = output_prompt_suffix(prompt_key)
            required_label_cols.append(f"label_{prefix}_{p}")

    missing_label_cols = [c for c in required_label_cols if c not in master.columns]

    if missing_label_cols:
        raise KeyError(
            "Missing required Anthropic label columns from release master: "
            f"{missing_label_cols}"
        )

    for label_col in required_label_cols:
        if label_col not in temp.columns:
            temp[label_col] = -1

        conf_col = label_col.replace("label_", "conf_", 1)
        if conf_col not in temp.columns:
            temp[conf_col] = -1.0

    print("=" * 80)
    print("ANTHROPIC JUDGE LABEL REGENERATION INTO TEMP_RESULTS + AUDIT")
    print("=" * 80)
    print("Release master: ", MASTER_PATH)
    print("Temp results:   ", TEMP_RESULTS_PATH)
    print("Audit CSV:      ", OUT_AUDIT_CSV)
    print("Parse failures: ", OUT_PARSE_FAILURES_CSV)
    print()
    print("Anthropic judge specs:")
    for prefix, display_name, model_id in ANTHROPIC_JUDGE_SPECS:
        print(f"  {prefix:<20} {display_name:<20} model_id={model_id}")
    print()
    print("Prompt keys:", ANTHROPIC_PROMPT_KEYS)
    print("Generators: ", GEN_ORDER)
    print("Max workers:", ANTHROPIC_MAX_WORKERS)
    print("Max attempts:", ANTHROPIC_MAX_ATTEMPTS)
    print("Force recompute:", ANTHROPIC_FORCE_RECOMPUTE)
    print("Use prompt cache:", ANTHROPIC_USE_PROMPT_CACHE)
    print()
    print("This cell writes only to temp_results.csv and audits/.")

    # =========================================================================
    # Run labeling
    # =========================================================================

    audit_rows = []
    parse_failure_rows = []
    blocking_errors = []

    for prefix, display_name, api_model_id in ANTHROPIC_JUDGE_SPECS:
        for prompt_key in ANTHROPIC_PROMPT_KEYS:
            prompt_suffix = output_prompt_suffix(prompt_key)
            prompt_text = get_prompt_text(prompt_dict, prompt_key)

            label_col = f"label_{prefix}_{prompt_suffix}"
            conf_col = f"conf_{prefix}_{prompt_suffix}"

            print("\n" + "=" * 80)
            print(f"{display_name} / {prompt_key} -> {label_col}")
            print("=" * 80)

            for gen_model in GEN_ORDER:
                mask_g = master["model"] == gen_model
                row_indices = master.index[mask_g].tolist()

                print("\n" + "-" * 80)
                print(f"{display_name} / {prompt_key} / {gen_model}")
                print("-" * 80)

                if ANTHROPIC_FORCE_RECOMPUTE:
                    target_indices = row_indices
                else:
                    current = pd.to_numeric(
                        temp.loc[row_indices, label_col],
                        errors="coerce",
                    )
                    target_indices = [
                        idx for idx, val in zip(row_indices, current)
                        if val not in [0, 1]
                    ]

                if not target_indices:
                    print("All rows already have valid temp labels; skipping this block.")
                else:
                    print(f"Running {len(target_indices)} rows with {ANTHROPIC_MAX_WORKERS} workers.")

                    results = {}

                    def _task(idx):
                        row = master.loc[idx]
                        result = run_anthropic_judge_one(
                            row=row,
                            prompt_text=prompt_text,
                            model_id=api_model_id,
                        )
                        return idx, result

                    with ThreadPoolExecutor(max_workers=ANTHROPIC_MAX_WORKERS) as executor:
                        futures = {
                            executor.submit(_task, idx): idx
                            for idx in target_indices
                        }

                        with tqdm(
                            total=len(target_indices),
                            desc=f"{prefix} {prompt_key} {gen_model}",
                        ) as pbar:
                            for future in as_completed(futures):
                                idx = futures[future]

                                try:
                                    idx, result = future.result()
                                except Exception as e:
                                    result = {
                                        "label": -1,
                                        "confidence": -1.0,
                                        "raw_text": "",
                                        "parse_error": repr(e),
                                    }

                                results[idx] = result
                                pbar.update(1)

                    for idx in target_indices:
                        result = results[idx]

                        temp.loc[idx, label_col] = int(result["label"])
                        temp.loc[idx, conf_col] = float(result["confidence"])

                        if int(result["label"]) not in [0, 1]:
                            parse_failure_rows.append(
                                {
                                    "item_id": master.loc[idx, "item_id"],
                                    "qid": master.loc[idx, "qid"],
                                    "dataset": master.loc[idx, "dataset"],
                                    "generator_model": master.loc[idx, "model"],
                                    "judge_prefix": prefix,
                                    "judge_display": display_name,
                                    "api_model_id": api_model_id,
                                    "prompt": prompt_suffix,
                                    "label_col": label_col,
                                    "parse_error": result["parse_error"],
                                    "raw_text": result["raw_text"],
                                }
                            )

                    if ANTHROPIC_CHECKPOINT_EVERY_BLOCK:
                        if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
                            raise RuntimeError("Refusing to write temp results to MASTER_PATH.")
                        temp.to_csv(TEMP_RESULTS_PATH, index=False)
                        print(f"Checkpoint saved: {TEMP_RESULTS_PATH}")

                # -------------------------------------------------------------
                # Audit temp labels against release master for this block
                # -------------------------------------------------------------
                h = master.loc[row_indices, HUMAN_COL].astype(int).to_numpy()
                master_labels = master.loc[row_indices, label_col].astype(int).to_numpy()
                temp_labels = pd.to_numeric(
                    temp.loc[row_indices, label_col],
                    errors="coerce",
                ).fillna(-1).astype(int).to_numpy()

                n_parse_fail = int(np.sum(~np.isin(temp_labels, [0, 1])))
                n_mismatch = int(np.sum(temp_labels != master_labels))

                master_metrics = metric_summary(h, master_labels)
                temp_metrics = metric_summary(h, temp_labels)

                audit_rows.append(
                    {
                        "judge_prefix": prefix,
                        "judge_display": display_name,
                        "api_model_id": api_model_id,
                        "prompt": prompt_suffix,
                        "label_col": label_col,
                        "generator_model": gen_model,
                        "n_total": int(len(row_indices)),
                        "n_parse_fail": n_parse_fail,
                        "n_label_mismatch": n_mismatch,
                        "kappa_master": master_metrics["kappa"],
                        "kappa_temp": temp_metrics["kappa"],
                        "bias_master": master_metrics["bias"],
                        "bias_temp": temp_metrics["bias"],
                        "fp_master": master_metrics["fp"],
                        "fp_temp": temp_metrics["fp"],
                        "fn_master": master_metrics["fn"],
                        "fn_temp": temp_metrics["fn"],
                        "err_master": master_metrics["err"],
                        "err_temp": temp_metrics["err"],
                        "fp_pct_master": master_metrics["fp_pct"],
                        "fp_pct_temp": temp_metrics["fp_pct"],
                    }
                )

                print(
                    f"Audit: parse_fail={n_parse_fail}, "
                    f"label_mismatch={n_mismatch}, "
                    f"kappa master={master_metrics['kappa']:.3f}, "
                    f"temp={temp_metrics['kappa']:.3f}"
                )

                if n_parse_fail != 0:
                    blocking_errors.append(
                        f"{display_name}/{prompt_key}/{gen_model}: "
                        f"parse failures={n_parse_fail}"
                    )

                if n_mismatch > 0:
                    print(
                        f"  NOTE: {n_mismatch}/{len(row_indices)} label(s) differ from "
                        f"release master. Expected: API model outputs are not guaranteed "
                        f"to be reproducible across calls due to server-side sampling "
                        f"and model version updates."
                    )

    # =========================================================================
    # Save outputs
    # =========================================================================

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError("Refusing to write temp results to MASTER_PATH.")

    temp.to_csv(TEMP_RESULTS_PATH, index=False)

    audit = pd.DataFrame(audit_rows)
    audit.to_csv(OUT_AUDIT_CSV, index=False)

    parse_failures = pd.DataFrame(parse_failure_rows)
    parse_failures.to_csv(OUT_PARSE_FAILURES_CSV, index=False)

    print("\n" + "=" * 80)
    print("ANTHROPIC RECOMPUTE AUDIT SUMMARY")
    print("=" * 80)

    display_cols = [
        "judge_prefix",
        "prompt",
        "generator_model",
        "n_parse_fail",
        "n_label_mismatch",
        "kappa_master",
        "kappa_temp",
        "fp_master",
        "fp_temp",
        "fn_master",
        "fn_temp",
    ]

    display(audit[display_cols])

    print("\nSaved temp_results.csv:")
    print(f"  {TEMP_RESULTS_PATH}")

    print("\nSaved audit CSVs:")
    print(f"  {OUT_AUDIT_CSV}")
    print(f"  {OUT_PARSE_FAILURES_CSV}")

    if blocking_errors:
        print("\nBlocking audit errors (parse failures):")
        for e in blocking_errors:
            print(f"  • {e}")

        raise AssertionError(
            "Anthropic judge recomputation audit failed: parse failures detected. "
            "All Anthropic judge responses must produce valid JSON labels. "
            "Do not modify paper results until this is understood."
        )

    print("\n✓ Anthropic judge recomputation audit passed.")
    print("  No parse failures detected.")
    print("  NOTE: Anthropic API outputs are not guaranteed to be reproducible")
    print("  across calls. Label differences from the release master are expected")
    print("  and recorded in the audit CSV.")

In [ ]:
# =============================================================================
# CELL B6: Local judge labeling into temp_results + audit
#
# Purpose:
#   Optionally rerun local open-weight judge models for prompts P1/P2/P3, write
#   labels and confidences into temp_results.csv, and compare regenerated labels
#   against the locked release master.
#
# Local judges:
#   - Gemma-2-9B-it
#   - Qwen2.5-7B-Instruct
#   - Llama-3-8B-Instruct
#
# Execution:
#   - If RUN_LABEL_REGENERATION=False, this cell is skipped.
#   - If RUN_LOCAL_LABELING=False, this cell is skipped.
#   - If enabled, this cell requires:
#       RUN_LABEL_REGENERATION=True
#       ALLOW_EXPENSIVE_RUNS=True
#       RUN_LOCAL_LABELING=True
#       CUDA GPU
#
# Safety:
#   - MASTER_PATH is read-only.
#   - This cell writes only to TEMP_RESULTS_PATH and audits/.
#   - No old per-generator result CSVs are read or written.
#
#
# Outputs:
#   - audits/temp_results/temp_results.csv
#   - audits/local_recompute/local_recompute_audit.csv
#   - audits/local_recompute/local_parse_failures.csv
# =============================================================================

import os
import re
import gc
import json
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL B6: Local judge labeling"

if not RUN_LABEL_REGENERATION:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=False.")
    print("No local judge labels were regenerated.")
    print("temp_results.csv was not modified by this cell.")

elif not RUN_LOCAL_LABELING:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=True, but RUN_LOCAL_LABELING=False.")
    print("No local judge labels were regenerated.")
    print("temp_results.csv was not modified by this cell.")

else:
    # =========================================================================
    # Guards
    # =========================================================================

    guard_local_labeling(min_gpu_gb=24)

    require_package("torch", "torch")
    require_package("transformers", "transformers")
    require_package("accelerate", "accelerate")

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # Optional Hugging Face login.
    # Use this only if needed for gated models. Do not hard-code tokens.
    try:
        hf_token = get_secret("HF_TOKEN", required=False)
        if hf_token:
            require_package("huggingface_hub", "huggingface_hub")
            from huggingface_hub import login
            login(token=hf_token)
            print("✓ Logged in to Hugging Face.")
        else:
            print("HF_TOKEN is empty — skipping HF login.")
            print("This is fine if all models are public.")
    except Exception as e:
        print(f"Could not load HF_TOKEN: {e}")
        print("Proceeding without HF login — gated models will fail at download.")

    # =========================================================================
    # Paths
    # =========================================================================

    LOCAL_AUDIT_DIR = AUDIT_DIR / "local_recompute"
    LOCAL_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_AUDIT_CSV = LOCAL_AUDIT_DIR / "local_recompute_audit.csv"
    OUT_PARSE_FAILURES_CSV = LOCAL_AUDIT_DIR / "local_parse_failures.csv"

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError(
            "Safety error: TEMP_RESULTS_PATH points to MASTER_PATH. "
            "Refusing to continue."
        )

    if not TEMP_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"temp_results.csv not found. Run CELL A3 first: {TEMP_RESULTS_PATH}"
        )

    # =========================================================================
    # Config
    # =========================================================================

    HUMAN_COL = "annotator1_label"

    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    LOCAL_JUDGE_SPECS = globals().get(
        "LOCAL_JUDGE_SPECS",
        [
            (
                "gemma_2_9b_local",
                "Gemma-2-9B local",
                "google/gemma-2-9b-it",
            ),
            (
                "qwen2_5_7b_local",
                "Qwen2.5-7B local",
                "Qwen/Qwen2.5-7B-Instruct",
            ),
            (
                "llama_3_8b_local",
                "Llama-3-8B local",
                "meta-llama/Meta-Llama-3-8B-Instruct",
            ),
        ],
    )

    LOCAL_PROMPT_KEYS = ["P1", "P2", "P3"]

    LOCAL_MAX_NEW_TOKENS = globals().get("LOCAL_MAX_NEW_TOKENS", 2500)
    LOCAL_TEMPERATURE = globals().get("LOCAL_TEMPERATURE", 0.0)
    LOCAL_DO_SAMPLE = globals().get("LOCAL_DO_SAMPLE", False)

    LOCAL_FORCE_RECOMPUTE = globals().get(
        "LOCAL_FORCE_RECOMPUTE",
        True,
    )

    LOCAL_CHECKPOINT_EVERY_BLOCK = globals().get(
        "LOCAL_CHECKPOINT_EVERY_BLOCK",
        True,
    )

    # =========================================================================
    # Helpers
    # =========================================================================

    def require_same_row_identity(master_df, temp_df):
        identity_cols = ["item_id", "qid", "dataset", "model"]

        missing_master = [c for c in identity_cols if c not in master_df.columns]
        missing_temp = [c for c in identity_cols if c not in temp_df.columns]

        if missing_master or missing_temp:
            raise KeyError(
                f"Missing identity columns. "
                f"master missing={missing_master}, temp missing={missing_temp}"
            )

        if len(master_df) != len(temp_df):
            raise ValueError(
                f"Row-count mismatch: master={len(master_df)}, temp={len(temp_df)}"
            )

        for col in identity_cols:
            if not master_df[col].astype(str).equals(temp_df[col].astype(str)):
                raise ValueError(
                    f"Row identity mismatch in column {col}. "
                    "Reinitialize temp_results.csv from master before continuing."
                )


    def resolve_prompt_dict():
        """
        Find the prompt dictionary defined in CELL B1.

        Accepted:
          - PROMPT_DEFS / PROMPT_TEXTS / JUDGE_PROMPTS / PROMPTS if dict
          - separate global strings P1, P2, P3
        """
        candidate_names = [
            "PROMPT_DEFS",
            "PROMPT_TEXTS",
            "JUDGE_PROMPTS",
            "PROMPTS",
        ]

        for name in candidate_names:
            obj = globals().get(name, None)
            if isinstance(obj, dict):
                return obj

        prompt_vars = {}

        for key in ["P1", "P2", "P3"]:
            val = globals().get(key, None)
            if isinstance(val, str) and val.strip():
                prompt_vars[key] = val

        if prompt_vars:
            return prompt_vars

        raise RuntimeError(
            "No prompt dictionary found. Run CELL B1 first.\n"
            "Expected prompt keys P1, P2, P3 in a dict, or global strings "
            "P1, P2, P3."
        )


    def get_prompt_text(prompt_dict, prompt_key):
        candidates = [
            prompt_key,
            prompt_key.upper(),
            prompt_key.lower(),
        ]

        for key in candidates:
            if key in prompt_dict:
                return str(prompt_dict[key])

        raise KeyError(
            f"Prompt key {prompt_key!r} not found in prompt dictionary. "
            f"Available keys: {list(prompt_dict.keys())}"
        )


    def output_prompt_suffix(prompt_key):
        """
        Convert notebook prompt key P1/P2/P3 to release-master suffix p1/p2/p3.
        """
        p = str(prompt_key).lower()

        if p not in {"p1", "p2", "p3"}:
            raise ValueError(
                f"Invalid prompt key {prompt_key!r}. "
                "Only P1/P2/P3 are allowed in this reproduction notebook."
            )

        return p


    def parse_json_label(text):
        """
        Parse judge output.

        Expected JSON:
          {"label": 0 or 1, "confidence": float in [0,1]}

        Returns:
          (label, confidence, raw_text, parse_error)
        """
        if not isinstance(text, str):
            return -1, -1.0, "", "non-string response"

        raw_text = text
        s = text.strip()

        s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s*```$", "", s)

        try:
            obj = json.loads(s)
            label = int(obj.get("label", -1))
            conf_raw = obj.get("confidence", -1)

            if isinstance(conf_raw, str):
                m = re.match(r"([\d.]+)", conf_raw.strip())
                conf_raw = float(m.group(1)) if m else -1.0

            conf = float(conf_raw)

            if conf > 1.0:
                conf = conf / 100.0

            if label in (0, 1) and 0.0 <= conf <= 1.0:
                return label, conf, raw_text, ""

        except Exception:
            pass

        for match in re.finditer(r"\{[^{}]+\}", s):
            try:
                obj = json.loads(match.group())
                label = int(obj.get("label", -1))
                conf_raw = obj.get("confidence", -1)

                if isinstance(conf_raw, str):
                    m = re.match(r"([\d.]+)", conf_raw.strip())
                    conf_raw = float(m.group(1)) if m else -1.0

                conf = float(conf_raw)

                if conf > 1.0:
                    conf = conf / 100.0

                if label in (0, 1) and 0.0 <= conf <= 1.0:
                    return label, conf, raw_text, ""

            except Exception:
                continue

        return -1, -1.0, raw_text, "could not parse valid JSON label/confidence"


    def build_user_message(question, answer, reference):
        refs = [
            r.strip()
            for r in str(reference or "").split("|")
            if r.strip()
        ]

        refs_text = "\n".join(f"- {r}" for r in refs) if refs else "- "

        return (
            f"QUESTION: {question}\n"
            f"MODEL_ANSWER: {answer}\n"
            f"REFERENCE_ANSWERS (any one is acceptable):\n{refs_text}"
        )



    def build_chat_messages(prompt_text, row):
        user_message = build_user_message(
            question=row["question"],
            answer=row["answer_text"],
            reference=row["ref_str"],
        )

        return [
            {
            "role": "user",
            "content": prompt_text + "\n\n" + user_message,
            }
        ]


    def generate_local_response(tokenizer, model, messages):
        """
        Generate one local judge response with greedy decoding by default.
        """
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            encoded = tokenizer.apply_chat_template(
                messages,
                return_tensors="pt",
                add_generation_prompt=True,
                return_dict=True,
            )
        else:
            text = (
                messages[0]["content"].strip()
                + "\n\n"
                + messages[1]["content"].strip()
                + "\n\nReturn only JSON."
            )
            encoded = tokenizer(
                text,
                return_tensors="pt",
                padding=False,
                truncation=True,
            )

        input_ids = encoded["input_ids"].to(model.device)
        attention_mask = encoded.get("attention_mask", None)

        if attention_mask is not None:
            attention_mask = attention_mask.to(model.device)

        generation_kwargs = {
            "input_ids": input_ids,
            "max_new_tokens": LOCAL_MAX_NEW_TOKENS,
            "do_sample": LOCAL_DO_SAMPLE,
            "pad_token_id": tokenizer.pad_token_id,
        }

        if attention_mask is not None:
            generation_kwargs["attention_mask"] = attention_mask

        if LOCAL_DO_SAMPLE:
            generation_kwargs["temperature"] = LOCAL_TEMPERATURE

        with torch.no_grad():
            out = model.generate(**generation_kwargs)

        response = tokenizer.decode(
            out[0][input_ids.shape[1]:],
            skip_special_tokens=True,
        )

        return response


    def run_local_judge_one(row, prompt_text, tokenizer, model):
        """
        Return one row-level local judge result.

        Output:
          {
            label, confidence, raw_text, parse_error
          }
        """
        try:
            messages = build_chat_messages(prompt_text, row)
            response = generate_local_response(tokenizer, model, messages)
            label, conf, raw_text, parse_error = parse_json_label(response)

            return {
                "label": label,
                "confidence": conf,
                "raw_text": raw_text,
                "parse_error": parse_error,
            }

        except Exception as e:
            return {
                "label": -1,
                "confidence": -1.0,
                "raw_text": "",
                "parse_error": repr(e),
            }


    def safe_kappa(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)

        try:
            k = cohen_kappa_score(y_true, y_pred, labels=[0, 1])
            return float(k) if np.isfinite(k) else np.nan
        except Exception:
            return np.nan


    def metric_summary(y_true, y_pred):
        mask = np.isin(y_true, [0, 1]) & np.isin(y_pred, [0, 1])

        yh = np.asarray(y_true)[mask].astype(int)
        yp = np.asarray(y_pred)[mask].astype(int)

        if len(yh) == 0:
            return {
                "n_valid": 0,
                "n_total": int(len(y_pred)),
                "kappa": np.nan,
                "bias": np.nan,
                "fp": np.nan,
                "fn": np.nan,
                "err": np.nan,
                "fp_pct": np.nan,
                "n_fail": int(len(y_pred)),
            }

        tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()

        err = fp + fn
        fp_pct = 100.0 * fp / max(err, 1)

        return {
            "n_valid": int(mask.sum()),
            "n_total": int(len(y_pred)),
            "kappa": safe_kappa(yh, yp),
            "bias": float(yp.mean() - yh.mean()),
            "fp": int(fp),
            "fn": int(fn),
            "err": int(err),
            "fp_pct": float(fp_pct),
            "n_fail": int(len(y_pred) - mask.sum()),
        }

    # =========================================================================
    # Load and validate master/temp
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)
    temp = pd.read_csv(TEMP_RESULTS_PATH)

    require_same_row_identity(master, temp)

    prompt_dict = resolve_prompt_dict()

    required_base_cols = [
        "item_id",
        "qid",
        "dataset",
        "model",
        "question",
        "ref_str",
        "answer_text",
        HUMAN_COL,
    ]

    missing_base = [c for c in required_base_cols if c not in master.columns]

    if missing_base:
        raise KeyError(f"Missing required columns from release master: {missing_base}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows in release master, got {len(master)}")

    if not master[HUMAN_COL].isin([0, 1]).all():
        bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
        raise ValueError(f"Invalid human labels in {HUMAN_COL}: {bad_n}")

    for model_name in GEN_ORDER:
        n = int((master["model"] == model_name).sum())
        if n != 300:
            raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

    required_label_cols = []

    for prefix, _, _ in LOCAL_JUDGE_SPECS:
        for prompt_key in LOCAL_PROMPT_KEYS:
            p = output_prompt_suffix(prompt_key)
            required_label_cols.append(f"label_{prefix}_{p}")

    missing_label_cols = [c for c in required_label_cols if c not in master.columns]

    if missing_label_cols:
        raise KeyError(
            "Missing required local judge label columns from release master: "
            f"{missing_label_cols}"
        )

    for label_col in required_label_cols:
        if label_col not in temp.columns:
            temp[label_col] = -1

        conf_col = label_col.replace("label_", "conf_", 1)
        if conf_col not in temp.columns:
            temp[conf_col] = -1.0

    print("=" * 80)
    print("LOCAL JUDGE LABEL REGENERATION INTO TEMP_RESULTS + AUDIT")
    print("=" * 80)
    print("Release master: ", MASTER_PATH)
    print("Temp results:   ", TEMP_RESULTS_PATH)
    print("Audit CSV:      ", OUT_AUDIT_CSV)
    print("Parse failures: ", OUT_PARSE_FAILURES_CSV)
    print()
    print("Local judge specs:")
    for prefix, display_name, hf_model_id in LOCAL_JUDGE_SPECS:
        print(f"  {prefix:<22} {display_name:<22} hf_model_id={hf_model_id}")
    print()
    print("Prompt keys:", LOCAL_PROMPT_KEYS)
    print("Max new tokens:", LOCAL_MAX_NEW_TOKENS)
    print("Force recompute:", LOCAL_FORCE_RECOMPUTE)
    print()
    print("This cell writes only to temp_results.csv and audits/.")

    # =========================================================================
    # Run local labeling
    # =========================================================================

    audit_rows = []
    parse_failure_rows = []
    blocking_errors = []

    for prefix, display_name, hf_model_id in LOCAL_JUDGE_SPECS:
        print("\n" + "=" * 80)
        print(f"LOADING LOCAL JUDGE: {display_name}")
        print("=" * 80)
        print(hf_model_id)

        tokenizer = AutoTokenizer.from_pretrained(hf_model_id)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            hf_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )

        model.eval()

        if torch.cuda.is_available():
            print(f"CUDA device: {torch.cuda.get_device_name(0)}")
            print(f"VRAM allocated after load: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

        for prompt_key in LOCAL_PROMPT_KEYS:
            prompt_suffix = output_prompt_suffix(prompt_key)
            prompt_text = get_prompt_text(prompt_dict, prompt_key)

            label_col = f"label_{prefix}_{prompt_suffix}"
            conf_col = f"conf_{prefix}_{prompt_suffix}"

            print("\n" + "=" * 80)
            print(f"{display_name} / {prompt_key} -> {label_col}")
            print("=" * 80)

            for gen_model in GEN_ORDER:
                mask_g = master["model"] == gen_model
                row_indices = master.index[mask_g].tolist()

                is_self_eval = (
                    (prefix == "gemma_2_9b_local" and gen_model == "gemma-2-9b")
                    or (prefix == "llama_3_8b_local" and gen_model == "llama3-8b")
                )

                tag = " [SELF-EVAL]" if is_self_eval else ""

                print("\n" + "-" * 80)
                print(f"{display_name} / {prompt_key} / {gen_model}{tag}")
                print("-" * 80)

                if LOCAL_FORCE_RECOMPUTE:
                    target_indices = row_indices
                else:
                    current = pd.to_numeric(
                        temp.loc[row_indices, label_col],
                        errors="coerce",
                    )
                    target_indices = [
                        idx for idx, val in zip(row_indices, current)
                        if val not in [0, 1]
                    ]

                if not target_indices:
                    print("All rows already have valid temp labels; skipping this block.")
                else:
                    labels = []
                    confs = []

                    for idx in tqdm(
                        target_indices,
                        desc=f"{prefix} {prompt_key} {gen_model}",
                    ):
                        result = run_local_judge_one(
                            row=master.loc[idx],
                            prompt_text=prompt_text,
                            tokenizer=tokenizer,
                            model=model,
                        )

                        label = int(result["label"])
                        conf = float(result["confidence"])

                        temp.loc[idx, label_col] = label
                        temp.loc[idx, conf_col] = conf

                        labels.append(label)
                        confs.append(conf)

                        if label not in [0, 1]:
                            parse_failure_rows.append(
                                {
                                    "item_id": master.loc[idx, "item_id"],
                                    "qid": master.loc[idx, "qid"],
                                    "dataset": master.loc[idx, "dataset"],
                                    "generator_model": master.loc[idx, "model"],
                                    "judge_prefix": prefix,
                                    "judge_display": display_name,
                                    "hf_model_id": hf_model_id,
                                    "prompt": prompt_suffix,
                                    "label_col": label_col,
                                    "is_self_eval": is_self_eval,
                                    "parse_error": result["parse_error"],
                                    "raw_text": result["raw_text"],
                                }
                            )

                    n_fail_block = int(sum(1 for x in labels if x not in [0, 1]))

                    if n_fail_block:
                        print(f"  parse_fail={n_fail_block}/{len(labels)}")

                    if LOCAL_CHECKPOINT_EVERY_BLOCK:
                        if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
                            raise RuntimeError("Refusing to write temp results to MASTER_PATH.")
                        temp.to_csv(TEMP_RESULTS_PATH, index=False)
                        print(f"Checkpoint saved: {TEMP_RESULTS_PATH}")

                # -------------------------------------------------------------
                # Audit temp labels against release master for this block
                # -------------------------------------------------------------
                h = master.loc[row_indices, HUMAN_COL].astype(int).to_numpy()
                master_labels = master.loc[row_indices, label_col].astype(int).to_numpy()
                temp_labels = pd.to_numeric(
                    temp.loc[row_indices, label_col],
                    errors="coerce",
                ).fillna(-1).astype(int).to_numpy()

                n_parse_fail = int(np.sum(~np.isin(temp_labels, [0, 1])))
                n_mismatch = int(np.sum(temp_labels != master_labels))

                master_metrics = metric_summary(h, master_labels)
                temp_metrics = metric_summary(h, temp_labels)

                audit_rows.append(
                    {
                        "judge_prefix": prefix,
                        "judge_display": display_name,
                        "hf_model_id": hf_model_id,
                        "prompt": prompt_suffix,
                        "label_col": label_col,
                        "generator_model": gen_model,
                        "is_self_eval": is_self_eval,

                        "n_total": int(len(row_indices)),
                        "n_parse_fail": n_parse_fail,
                        "n_label_mismatch": n_mismatch,

                        "kappa_master": master_metrics["kappa"],
                        "kappa_temp": temp_metrics["kappa"],
                        "bias_master": master_metrics["bias"],
                        "bias_temp": temp_metrics["bias"],
                        "fp_master": master_metrics["fp"],
                        "fp_temp": temp_metrics["fp"],
                        "fn_master": master_metrics["fn"],
                        "fn_temp": temp_metrics["fn"],
                        "err_master": master_metrics["err"],
                        "err_temp": temp_metrics["err"],
                        "fp_pct_master": master_metrics["fp_pct"],
                        "fp_pct_temp": temp_metrics["fp_pct"],
                    }
                )

                print(
                    f"Audit: parse_fail={n_parse_fail}, "
                    f"label_mismatch={n_mismatch}, "
                    f"kappa master={master_metrics['kappa']:.3f}, "
                    f"temp={temp_metrics['kappa']:.3f}"
                )

                if n_parse_fail != 0:
                    blocking_errors.append(
                        f"{display_name}/{prompt_key}/{gen_model}: "
                        f"parse failures={n_parse_fail}"
                    )

                if n_mismatch > 0:
                    print(
                        f"  NOTE: {n_mismatch}/{len(row_indices)} label(s) differ from "
                        f"release master. Expected due to GPU floating-point "
                        f"non-determinism under greedy decoding."
                    )

        del model, tokenizer
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"\nUnloaded local judge: {display_name}")

    # =========================================================================
    # Save outputs
    # =========================================================================

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError("Refusing to write temp results to MASTER_PATH.")

    temp.to_csv(TEMP_RESULTS_PATH, index=False)

    audit = pd.DataFrame(audit_rows)
    audit.to_csv(OUT_AUDIT_CSV, index=False)

    parse_failures = pd.DataFrame(parse_failure_rows)
    parse_failures.to_csv(OUT_PARSE_FAILURES_CSV, index=False)

    print("\n" + "=" * 80)
    print("LOCAL JUDGE RECOMPUTE AUDIT SUMMARY")
    print("=" * 80)

    display_cols = [
        "judge_prefix",
        "prompt",
        "generator_model",
        "is_self_eval",
        "n_parse_fail",
        "n_label_mismatch",
        "kappa_master",
        "kappa_temp",
        "fp_master",
        "fp_temp",
        "fn_master",
        "fn_temp",
    ]

    display(audit[display_cols])

    print("\nSaved temp_results.csv:")
    print(TEMP_RESULTS_PATH)

    print("\nSaved audit CSVs:")
    print(OUT_AUDIT_CSV)
    print(OUT_PARSE_FAILURES_CSV)

    if blocking_errors:
        print("\nBlocking audit errors (parse failures):")
        for e in blocking_errors:
            print("  •", e)

        raise AssertionError(
            "Local judge recomputation audit failed: parse failures detected. "
            "All local judge responses must produce valid JSON labels. "
            "Do not modify paper results until this is understood."
        )

    print("\n✓ Local judge recomputation audit passed.")
    print("  No parse failures detected.")
    print("  NOTE: Local LLM inference is not perfectly reproducible across runs.")
    print("  Small label differences from the release master are expected due to")
    print("  GPU floating-point non-determinism even under greedy decoding.")
    print("  Mismatch counts are recorded in the audit CSV for transparency.")

In [ ]:
# =============================================================================
# CELL B7: Six-branch prompt-following check for the p2 prompt
#
# Purpose:
#   Lightweight diagnostic only.
#   Tests whether each judge follows the core factual-correctness branches:
#     1. exact correct answer -> 0
#     2. correct answer + true elaboration absent from reference -> 0
#     3. correct core answer + false added specific detail -> 1
#     4. wrong answer + fabricated specific claim -> 1
#     5. core answer mismatch -> 1
#     6. refusal / uncertainty -> 0
#
#   Branch 3 is the canonical divergence case of Section 3.1: yfact is
#   answer-level, so a correct core does not override a false added detail.
#
# Important:
#   - This is NOT a validation set.
#   - This should NOT be reported as benchmark performance.
#   - Full performance is evaluated on the 900-item benchmark.
#
# Execution:
#   - If RUN_SIX_BRANCH_SANITY=False, this cell is skipped.
#   - If enabled, this cell requires ALLOW_EXPENSIVE_RUNS=True because it may
#     call APIs and/or load local models.
#
# Safety:
#   - Does not read or write MASTER_PATH.
#   - Does not read or write temp_results.csv.
#   - Writes only to audits/sanity_check/ and tables/
#
# Outputs:
#   audits/sanity_check/six_branch_sanity_check.csv
#   audits/sanity_check/six_branch_sanity_check.json
#   audits/sanity_check/six_branch_sanity_check_summary.csv
#   tables/table_six_branch_sanity_check.tex   (tab:sanity_check)
# =============================================================================

import os
import re
import gc
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL B7: Six-branch prompt-following check"

if not RUN_SIX_BRANCH_SANITY:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_SIX_BRANCH_SANITY=False.")
    print("No API calls, local model inference, or sanity-check files were produced.")

else:
    if not ALLOW_EXPENSIVE_RUNS:
        raise RuntimeError(
            "RUN_SIX_BRANCH_SANITY=True, but ALLOW_EXPENSIVE_RUNS=False.\n"
            "This diagnostic may call APIs and/or load local models.\n"
            "Set ALLOW_EXPENSIVE_RUNS=True only if you intentionally want to run it."
        )

    # =========================================================================
    # Paths
    # =========================================================================
    SANITY_DIR = AUDIT_DIR / "sanity_check"
    SANITY_DIR.mkdir(parents=True, exist_ok=True)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)

    OUT_CSV = SANITY_DIR / "six_branch_sanity_check.csv"
    OUT_JSON = SANITY_DIR / "six_branch_sanity_check.json"
    OUT_SUMMARY_CSV = SANITY_DIR / "six_branch_sanity_check_summary.csv"
    OUT_TEX = TABLE_DIR / "table_six_branch_sanity_check.tex"

    # =========================================================================
    # Config
    # =========================================================================

    SANITY_PROMPT_KEY = "P2"

    SANITY_RUN_OPENAI = globals().get("SANITY_RUN_OPENAI", True)
    SANITY_RUN_ANTHROPIC = globals().get("SANITY_RUN_ANTHROPIC", True)
    SANITY_RUN_LOCAL = globals().get("SANITY_RUN_LOCAL", True)

    SANITY_OPENAI_MAX_ATTEMPTS = globals().get("SANITY_OPENAI_MAX_ATTEMPTS", 2)
    SANITY_ANTHROPIC_MAX_ATTEMPTS = globals().get("SANITY_ANTHROPIC_MAX_ATTEMPTS", 4)
    SANITY_MAX_OUTPUT_TOKENS = globals().get("SANITY_MAX_OUTPUT_TOKENS", 1000)
    SANITY_LOCAL_MAX_NEW_TOKENS = globals().get("SANITY_LOCAL_MAX_NEW_TOKENS", 512)

    SANITY_OPENAI_JUDGES = globals().get(
        "SANITY_OPENAI_JUDGES",
        [
            ("gpt_5_mini", "GPT-5-mini", "gpt-5-mini"),
            ("gpt_5_nano", "GPT-5-nano", "gpt-5-nano"),
            ("gpt_5_4",    "GPT-5.4",    "gpt-5.4"),
        ],
    )

    SANITY_ANTHROPIC_JUDGES = globals().get(
        "SANITY_ANTHROPIC_JUDGES",
        [
            ("claude_opus_4_7", "Claude Opus 4.7", "claude-opus-4-7"),
        ],
    )

    SANITY_LOCAL_JUDGES = globals().get(
        "SANITY_LOCAL_JUDGES",
        [
            ("gemma_2_9b_local", "Gemma-2-9B local", "google/gemma-2-9b-it"),
            ("qwen2_5_7b_local", "Qwen2.5-7B local", "Qwen/Qwen2.5-7B-Instruct"),
            ("llama_3_8b_local", "Llama-3-8B local", "meta-llama/Meta-Llama-3-8B-Instruct"),
        ],
    )

    # =========================================================================
    # Sanity cases
    # =========================================================================

    SANITY_CASES = [
        {
            "branch": "1: Correct, no extras",
            "question": "What is the capital of France?",
            "answer": "Paris.",
            "reference": "Paris",
            "expected": 0,
            "criterion": "Exact correct answer should not be hallucination.",
        },
        {
            "branch": "2: Correct + true elaboration absent from reference",
            "question": "What is the capital of France?",
            "answer": "Paris, the largest city in France and home to the Eiffel Tower.",
            "reference": "Paris",
            "expected": 0,
            "criterion": (
                "Correct elaboration absent from the reference should not be "
                "hallucination under factual correctness."
            ),
        },
        {
            "branch": "3: Correct core + false added detail",
            "question": "What is the capital of France?",
            "answer": "Paris, which was founded in 1887.",
            "reference": "Paris",
            "expected": 1,
            "criterion": (
                "A correct core answer does not override a false verifiable "
                "detail added elsewhere in the response."
            ),
        },
        {
            "branch": "4: Wrong + fabricated specifics",
            "question": "What is the capital of France?",
            "answer": "Lyon, which became the capital in 1987 after a referendum.",
            "reference": "Paris",
            "expected": 1,
            "criterion": (
                "Wrong core answer plus fabricated specific detail should be "
                "hallucination."
            ),
        },
        {
            "branch": "5: Core mismatch only",
            "question": "What is the capital of France?",
            "answer": "The capital of France is Lyon.",
            "reference": "Paris",
            "expected": 1,
            "criterion": (
                "A core answer that conflicts with the reference should be "
                "hallucination even without an invented date or number."
            ),
        },
        {
            "branch": "6: Refusal / uncertainty",
            "question": "What is the capital of France?",
            "answer": "I'm not sure about this one.",
            "reference": "Paris",
            "expected": 0,
            "criterion": (
                "A non-committal answer may be unhelpful, but it does not assert "
                "a contradictory or fabricated specific claim."
            ),
        },
      ]
    # =========================================================================
    # Helpers
    # =========================================================================

    def resolve_prompt_dict():
        """
        Find the prompt dictionary defined in CELL B1.

        Accepted:
          - PROMPT_DEFS / PROMPT_TEXTS / JUDGE_PROMPTS / PROMPTS if dict
          - separate global string PROMPT_P2
          - separate global string P2
        """
        candidate_names = [
            "PROMPT_DEFS",
            "PROMPT_TEXTS",
            "JUDGE_PROMPTS",
            "PROMPTS",
        ]

        for name in candidate_names:
            obj = globals().get(name, None)
            if isinstance(obj, dict):
                return obj

        if isinstance(globals().get("PROMPT_P2", None), str):
            return {"P2": globals()["PROMPT_P2"]}

        if isinstance(globals().get("P2", None), str):
            return {"P2": globals()["P2"]}

        raise RuntimeError(
            "No p2 prompt found. Run CELL B1 first.\n"
            "Expected a prompt dictionary with P2/p2, or a global string PROMPT_P2 or P2."
        )


    def get_prompt_text(prompt_dict, prompt_key):
        candidates = [
            prompt_key,
            prompt_key.upper(),
            prompt_key.lower(),
        ]

        for key in candidates:
            if key in prompt_dict:
                return str(prompt_dict[key])

        raise KeyError(
            f"Prompt key {prompt_key!r} not found in prompt dictionary. "
            f"Available keys: {list(prompt_dict.keys())}"
        )


    def parse_json_label(text):
        """
        Parse judge output.

        Expected JSON:
          {"label": 0 or 1, "confidence": float in [0,1]}

        Returns:
          (label, confidence, raw_text, parse_error)
        """
        if not isinstance(text, str):
            return -1, -1.0, "", "non-string response"

        raw_text = text
        s = text.strip()

        s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s*```$", "", s)

        try:
            obj = json.loads(s)
            label = int(obj.get("label", -1))
            conf_raw = obj.get("confidence", -1)

            if isinstance(conf_raw, str):
                m = re.match(r"([\d.]+)", conf_raw.strip())
                conf_raw = float(m.group(1)) if m else -1.0

            conf = float(conf_raw)

            if conf > 1.0:
                conf = conf / 100.0

            if label in (0, 1) and 0.0 <= conf <= 1.0:
                return label, conf, raw_text, ""

        except Exception:
            pass

        for match in re.finditer(r"\{[^{}]+\}", s):
            try:
                obj = json.loads(match.group())
                label = int(obj.get("label", -1))
                conf_raw = obj.get("confidence", -1)

                if isinstance(conf_raw, str):
                    m = re.match(r"([\d.]+)", conf_raw.strip())
                    conf_raw = float(m.group(1)) if m else -1.0

                conf = float(conf_raw)

                if conf > 1.0:
                    conf = conf / 100.0

                if label in (0, 1) and 0.0 <= conf <= 1.0:
                    return label, conf, raw_text, ""

            except Exception:
                continue

        return -1, -1.0, raw_text, "could not parse valid JSON label/confidence"


    def build_user_message(question, answer, reference):
        refs = [
            r.strip()
            for r in str(reference or "").split("|")
            if r.strip()
        ]

        refs_text = "\n".join(f"- {r}" for r in refs) if refs else "- "

        return (
            f"QUESTION: {question}\n"
            f"MODEL_ANSWER: {answer}\n"
            f"REFERENCE_ANSWERS (any one is acceptable):\n{refs_text}"
        )


    def make_record(
        judge_prefix,
        judge_display,
        judge_type,
        model_id,
        case,
        label,
        confidence,
        raw_response=None,
        error=None,
    ):
        expected = int(case["expected"])
        label_int = int(label)

        return {
            "judge_prefix": judge_prefix,
            "judge": judge_display,
            "judge_type": judge_type,
            "model_id": model_id,
            "prompt": "p2",
            "branch": case["branch"],
            "question": case["question"],
            "answer": case["answer"],
            "reference": case["reference"],
            "criterion": case["criterion"],
            "expected": expected,
            "label": label_int,
            "confidence": float(confidence),
            "correct": bool(label_int == expected),
            "raw_response": raw_response,
            "error": error,
        }


    def anthropic_text_from_response(resp):
        parts = []

        try:
            for block in resp.content or []:
                txt = getattr(block, "text", None)
                if txt:
                    parts.append(txt)
        except Exception:
            pass

        return "\n".join(parts).strip()


    def latex_escape(text):
        text = str(text)
        replacements = {
            "\\": r"\textbackslash{}",
            "&": r"\&",
            "%": r"\%",
            "$": r"\$",
            "#": r"\#",
            "_": r"\_",
            "{": r"\{",
            "}": r"\}",
            "~": r"\textasciitilde{}",
            "^": r"\textasciicircum{}",
        }

        for old, new in replacements.items():
            text = text.replace(old, new)

        return text

    # =========================================================================
    # Resolve p2 prompt
    # =========================================================================

    prompt_dict = resolve_prompt_dict()
    p2_prompt = get_prompt_text(prompt_dict, SANITY_PROMPT_KEY)

    if "P2B" in p2_prompt or "P3B" in p2_prompt:
        raise ValueError("Old internal prompt name P2B/P3B found in p2 prompt text.")

    print("=" * 80)
    print("SIX-BRANCH SANITY CHECK: p2 FACTUAL-CORRECTNESS PROMPT")
    print("=" * 80)
    print("This is a lightweight diagnostic only, not a benchmark.")
    print("Outputs:")
    print("  ", OUT_CSV)
    print("  ", OUT_JSON)
    print("  ", OUT_SUMMARY_CSV)
    print("  ", OUT_TEX)
    print()
    print("Enabled judge families:")
    print(f"  SANITY_RUN_OPENAI    = {SANITY_RUN_OPENAI}")
    print(f"  SANITY_RUN_ANTHROPIC = {SANITY_RUN_ANTHROPIC}")
    print(f"  SANITY_RUN_LOCAL     = {SANITY_RUN_LOCAL}")

    all_records = []

    # =========================================================================
    # OpenAI sanity checks
    # =========================================================================

    if SANITY_RUN_OPENAI:
        require_package("openai", "openai")

        openai_key = get_secret("OPENAI_API_KEY", required=True)

        from openai import OpenAI

        openai_client = OpenAI(api_key=openai_key)

        def run_openai_case(api_model_id, case):
            user_message = build_user_message(
                question=case["question"],
                answer=case["answer"],
                reference=case["reference"],
            )

            last_error = ""

            for attempt in range(SANITY_OPENAI_MAX_ATTEMPTS):
                try:
                    try:
                        resp = openai_client.responses.create(
                            model=api_model_id,
                            reasoning={"effort": "low"},
                            input=[
                                {"role": "system", "content": p2_prompt},
                                {"role": "user", "content": user_message},
                            ],
                            max_output_tokens=SANITY_MAX_OUTPUT_TOKENS,
                        )
                    except Exception as e:
                        msg = str(e).lower()
                        if (
                            "reasoning" not in msg
                            and "unsupported" not in msg
                            and "unknown parameter" not in msg
                        ):
                            raise

                        resp = openai_client.responses.create(
                            model=api_model_id,
                            input=[
                                {"role": "system", "content": p2_prompt},
                                {"role": "user", "content": user_message},
                            ],
                            max_output_tokens=SANITY_MAX_OUTPUT_TOKENS,
                        )

                    text = getattr(resp, "output_text", "") or ""
                    label, conf, raw_text, parse_error = parse_json_label(text)

                    if label in [0, 1]:
                        return label, conf, raw_text, ""

                    last_error = parse_error or "invalid label"
                    time.sleep(1.0 + attempt)

                except Exception as e:
                    last_error = repr(e)
                    wait = 2.0 * (attempt + 1)

                    if hasattr(e, "status_code") and getattr(e, "status_code") == 429:
                        wait = max(wait, 10.0)

                    time.sleep(wait)

            return -1, -1.0, "", last_error or "failed after max attempts"

        print("\n" + "=" * 80)
        print("OPENAI SANITY CHECKS")
        print("=" * 80)

        for judge_prefix, judge_display, api_model_id in SANITY_OPENAI_JUDGES:
            print(f"\n{judge_display} ({api_model_id})")

            score = 0

            for case in SANITY_CASES:
                label, conf, raw_text, error = run_openai_case(api_model_id, case)
                ok = label == int(case["expected"])
                score += int(ok)

                mark = "✓" if ok else "✗"

                print(
                    f"  [{mark}] {case['branch']}: "
                    f"expected={case['expected']} got={label} conf={conf:.2f}"
                )

                all_records.append(
                    make_record(
                        judge_prefix=judge_prefix,
                        judge_display=judge_display,
                        judge_type="api",
                        model_id=api_model_id,
                        case=case,
                        label=label,
                        confidence=conf,
                        raw_response=raw_text,
                        error=error,
                    )
                )

            print(f"  Score: {score}/{len(SANITY_CASES)}")

    else:
        print("\nOpenAI sanity checks skipped: SANITY_RUN_OPENAI=False.")

    # =========================================================================
    # Anthropic sanity checks
    # =========================================================================

    if SANITY_RUN_ANTHROPIC:
        require_package("anthropic", "anthropic")

        anthropic_key = get_secret("ANTHROPIC_API_KEY", required=True)

        import anthropic

        anthropic_client = anthropic.Anthropic(api_key=anthropic_key)

        def run_anthropic_case(api_model_id, case):
            user_message = build_user_message(
                question=case["question"],
                answer=case["answer"],
                reference=case["reference"],
            )

            last_error = ""

            for attempt in range(SANITY_ANTHROPIC_MAX_ATTEMPTS):
                try:
                    try:
                        resp = anthropic_client.messages.create(
                            model=api_model_id,
                            max_tokens=SANITY_MAX_OUTPUT_TOKENS,
                            system=[
                                {
                                    "type": "text",
                                    "text": p2_prompt,
                                    "cache_control": {"type": "ephemeral"},
                                }
                            ],
                            messages=[
                                {
                                    "role": "user",
                                    "content": user_message,
                                }
                            ],
                        )
                    except Exception as e:
                        msg = str(e).lower()

                        if (
                            "cache_control" not in msg
                            and "prompt caching" not in msg
                            and "unsupported" not in msg
                            and "unknown" not in msg
                        ):
                            raise

                        resp = anthropic_client.messages.create(
                            model=api_model_id,
                            max_tokens=SANITY_MAX_OUTPUT_TOKENS,
                            system=p2_prompt,
                            messages=[
                                {
                                    "role": "user",
                                    "content": user_message,
                                }
                            ],
                        )

                    text = anthropic_text_from_response(resp)
                    label, conf, raw_text, parse_error = parse_json_label(text)

                    if label in [0, 1]:
                        return label, conf, raw_text, ""

                    last_error = parse_error or "invalid label"
                    time.sleep(1.0 + attempt)

                except anthropic.RateLimitError as e:
                    retry_after = None

                    if hasattr(e, "response") and e.response is not None:
                        retry_after = e.response.headers.get("retry-after")

                    if retry_after:
                        wait = float(retry_after) + 1.0
                    else:
                        wait = min(180.0, 15.0 * (2 ** attempt))

                    last_error = repr(e)
                    time.sleep(wait)

                except anthropic.APIStatusError as e:
                    last_error = repr(e)

                    if 500 <= int(getattr(e, "status_code", 0)) < 600:
                        time.sleep(2.0 * (attempt + 1))
                    else:
                        break

                except Exception as e:
                    last_error = repr(e)
                    time.sleep(2.0 * (attempt + 1))

            return -1, -1.0, "", last_error or "failed after max attempts"

        print("\n" + "=" * 80)
        print("ANTHROPIC SANITY CHECKS")
        print("=" * 80)

        for judge_prefix, judge_display, api_model_id in SANITY_ANTHROPIC_JUDGES:
            print(f"\n{judge_display} ({api_model_id})")

            score = 0

            for case in SANITY_CASES:
                label, conf, raw_text, error = run_anthropic_case(api_model_id, case)
                ok = label == int(case["expected"])
                score += int(ok)

                mark = "✓" if ok else "✗"

                print(
                    f"  [{mark}] {case['branch']}: "
                    f"expected={case['expected']} got={label} conf={conf:.2f}"
                )

                all_records.append(
                    make_record(
                        judge_prefix=judge_prefix,
                        judge_display=judge_display,
                        judge_type="api",
                        model_id=api_model_id,
                        case=case,
                        label=label,
                        confidence=conf,
                        raw_response=raw_text,
                        error=error,
                    )
                )

            print(f"  Score: {score}/{len(SANITY_CASES)}")

    else:
        print("\nAnthropic sanity checks skipped: SANITY_RUN_ANTHROPIC=False.")

    # =========================================================================
    # Local sanity checks
    # =========================================================================

    if SANITY_RUN_LOCAL:
        require_package("torch", "torch")
        require_package("transformers", "transformers")
        require_package("accelerate", "accelerate")

        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        if not torch.cuda.is_available():
            print("\nNo GPU available — local sanity checks skipped.")
            print("Switch to a GPU runtime to test local judges.")

            for judge_prefix, judge_display, hf_model_id in SANITY_LOCAL_JUDGES:
                for case in SANITY_CASES:
                    all_records.append(
                        make_record(
                            judge_prefix=judge_prefix,
                            judge_display=judge_display,
                            judge_type="local",
                            model_id=hf_model_id,
                            case=case,
                            label=-1,
                            confidence=-1.0,
                            raw_response=None,
                            error="skipped_no_gpu",
                        )
                    )
        else:
            hf_token = get_secret("HF_TOKEN", required=False)
            if hf_token:
                require_package("huggingface_hub", "huggingface_hub")
                from huggingface_hub import login
                login(token=hf_token)

            print("\n" + "=" * 80)
            print("LOCAL SANITY CHECKS")
            print("=" * 80)
            print("CUDA:", torch.cuda.get_device_name(0))

            def run_local_case(tokenizer, model_local, case):
                user_message = build_user_message(
                    question=case["question"],
                    answer=case["answer"],
                    reference=case["reference"],
                )

                messages = [
                    {
                        "role": "user",
                        "content": p2_prompt + "\n\n" + user_message,
                    },
                ]

                try:
                    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
                        encoded = tokenizer.apply_chat_template(
                            messages,
                            return_tensors="pt",
                            add_generation_prompt=True,
                            return_dict=True,
                        )
                    else:
                        text = (
                            p2_prompt.strip()
                            + "\n\n"
                            + user_message.strip()
                            + "\n\nReturn only JSON."
                        )
                        encoded = tokenizer(text, return_tensors="pt", truncation=True)

                    input_ids = encoded["input_ids"].to(model_local.device)
                    attention_mask = encoded.get("attention_mask", None)

                    if attention_mask is not None:
                        attention_mask = attention_mask.to(model_local.device)

                    kwargs = {
                        "input_ids": input_ids,
                        "max_new_tokens": SANITY_LOCAL_MAX_NEW_TOKENS,
                        "do_sample": False,
                        "pad_token_id": tokenizer.pad_token_id,
                    }

                    if attention_mask is not None:
                        kwargs["attention_mask"] = attention_mask

                    if tokenizer.eos_token_id is not None:
                        kwargs["eos_token_id"] = tokenizer.eos_token_id

                    with torch.no_grad():
                        out = model_local.generate(**kwargs)

                    response = tokenizer.decode(
                        out[0][input_ids.shape[1]:],
                        skip_special_tokens=True,
                    ).strip()

                    label, conf, raw_text, parse_error = parse_json_label(response)

                    return label, conf, raw_text, parse_error

                except Exception as e:
                    return -1, -1.0, "", repr(e)

            for judge_prefix, judge_display, hf_model_id in SANITY_LOCAL_JUDGES:
                print(f"\n{judge_display} ({hf_model_id})")

                try:
                    tokenizer = AutoTokenizer.from_pretrained(hf_model_id)

                    if tokenizer.pad_token is None:
                        tokenizer.pad_token = tokenizer.eos_token

                    model_local = AutoModelForCausalLM.from_pretrained(
                        hf_model_id,
                        torch_dtype=torch.bfloat16,
                        device_map="auto",
                    )

                    model_local.eval()

                    print(f"  Loaded. VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

                except Exception as e:
                    print(f"  Failed to load local judge: {e}")

                    for case in SANITY_CASES:
                        all_records.append(
                            make_record(
                                judge_prefix=judge_prefix,
                                judge_display=judge_display,
                                judge_type="local",
                                model_id=hf_model_id,
                                case=case,
                                label=-1,
                                confidence=-1.0,
                                raw_response=None,
                                error=f"load_failed: {e}",
                            )
                        )

                    continue

                score = 0

                for case in SANITY_CASES:
                    label, conf, raw_text, error = run_local_case(
                        tokenizer=tokenizer,
                        model_local=model_local,
                        case=case,
                    )

                    ok = label == int(case["expected"])
                    score += int(ok)

                    mark = "✓" if ok else "✗"

                    print(
                        f"  [{mark}] {case['branch']}: "
                        f"expected={case['expected']} got={label} conf={conf:.2f}"
                    )

                    if not ok and raw_text:
                        print(f"      RAW: {repr(raw_text[:250])}")

                    all_records.append(
                        make_record(
                            judge_prefix=judge_prefix,
                            judge_display=judge_display,
                            judge_type="local",
                            model_id=hf_model_id,
                            case=case,
                            label=label,
                            confidence=conf,
                            raw_response=raw_text,
                            error=error,
                        )
                    )

                print(f"  Score: {score}/{len(SANITY_CASES)}")

                del model_local, tokenizer
                gc.collect()
                torch.cuda.empty_cache()

    else:
        print("\nLocal sanity checks skipped: SANITY_RUN_LOCAL=False.")

    # =========================================================================
    # Summaries and save
    # =========================================================================

    df_sanity = pd.DataFrame(all_records)

    if len(df_sanity) == 0:
        raise RuntimeError(
            "No sanity-check records were produced. "
            "Enable at least one of SANITY_RUN_OPENAI, SANITY_RUN_ANTHROPIC, SANITY_RUN_LOCAL."
        )

    summary = (
        df_sanity[df_sanity["label"].isin([0, 1])]
        .groupby(["judge_prefix", "judge", "judge_type", "model_id"], as_index=False)
        .agg(
            score=("correct", "sum"),
            max_score=("correct", "count"),
            mean_confidence=("confidence", "mean"),
        )
    )

    skipped_or_failed = (
        df_sanity[~df_sanity["label"].isin([0, 1])]
        .groupby(["judge_prefix", "judge", "judge_type", "model_id"], as_index=False)
        .size()
        .rename(columns={"size": "n_invalid_or_skipped"})
    )

    summary = summary.merge(
        skipped_or_failed,
        on=["judge_prefix", "judge", "judge_type", "model_id"],
        how="outer",
    )

    summary["score"] = summary["score"].fillna(0).astype(int)
    summary["max_score"] = summary["max_score"].fillna(0).astype(int)
    summary["n_invalid_or_skipped"] = summary["n_invalid_or_skipped"].fillna(0).astype(int)

    summary["score_text"] = (
        summary["score"].astype(str)
        + "/"
        + summary["max_score"].astype(str)
    )

    df_sanity.to_csv(OUT_CSV, index=False)
    summary.to_csv(OUT_SUMMARY_CSV, index=False)

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_records, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 80)
    print("SIX-BRANCH SANITY CHECK SUMMARY")
    print("=" * 80)
    display(summary[[
        "judge",
        "judge_type",
        "score_text",
        "mean_confidence",
        "n_invalid_or_skipped",
    ]])

    # =========================================================================
    # LaTeX summary table
    # =========================================================================
    # -------------------------------------------------------------------------
    # Judge order and display names for the paper table.
    # Fixed rather than derived, so the table row order matches the paper and a
    # missing judge is caught instead of silently dropped.
    # -------------------------------------------------------------------------

    JUDGE_ORDER = [
        ("gpt_5_mini",       "GPT-5-mini"),
        ("gpt_5_nano",       "GPT-5-nano"),
        ("gpt_5_4",          "GPT-5.4"),
        ("claude_opus_4_7",  "Claude Opus 4.7"),
        ("gemma_2_9b_local", "Gemma-2-9B"),
        ("llama_3_8b_local", "Llama-3-8B"),
        ("qwen2_5_7b_local", "Qwen2.5-7B"),
    ]

    TYPE_DISPLAY = {"api": "API", "local": "Local"}

    summary_by_prefix = summary.set_index("judge_prefix")

    missing_judges = [
        prefix for prefix, _ in JUDGE_ORDER
        if prefix not in summary_by_prefix.index
    ]

    if missing_judges:
        print(f"\nWARNING: no results for {missing_judges}; omitted from the table.")

    latex = r"""\begin{table}[t]
  \centering
  \caption{Six-branch prompt-following check for the p2 factual-correctness prompt. The check tests basic prompt-following behavior on simple diagnostic cases and is not used to estimate benchmark performance.}
  \label{tab:six_branch_sanity}
  \small
  \begin{tabular}{llr}
    \toprule
    Judge & Type & Score \\
    \midrule
"""

    n_rows = 0

    for prefix, display_name in JUDGE_ORDER:
        if prefix not in summary_by_prefix.index:
            continue

        row = summary_by_prefix.loc[prefix]
        judge_type = TYPE_DISPLAY.get(row["judge_type"], row["judge_type"])

        latex += (
            f"    {latex_escape(display_name)} & "
            f"{latex_escape(judge_type)} & "
            f"{latex_escape(row['score_text'])} \\\\\n"
        )

        n_rows += 1

    latex += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

    with open(OUT_TEX, "w", encoding="utf-8") as f:
        f.write(latex)

    # Structural check against the paper table.
    if n_rows != len(JUDGE_ORDER):
        print(
            f"\nNOTE: table has {n_rows} rows; the paper table has "
            f"{len(JUDGE_ORDER)}. Run all judge families for the final version."
        )

    if latex.count("\\\\\n") != n_rows + 1:
        raise AssertionError("Table 5: row count does not match the header plus data rows.")

    print("\n" + "=" * 80)
    print(f"Table 5: {OUT_TEX.name}  ({n_rows} judges)")
    print("=" * 80)
    print(latex)




In [ ]:
# =============================================================================
# CELL B8: Criterion x structure factorial prompt experiment
#
# Purpose:
#   Test whether p1's weaker agreement is explained by its terse formulation
#   rather than by its reference-faithfulness criterion. Four prompts form a
#   2x2 design over criterion (faithfulness / factual correctness) and prompt
#   structure (terse / structured decision tree):
#
#                     terse          structured
#     faithfulness    p1             p1s
#     factual         p2t            p2
#
#   p1 and p2 are the paper prompts from CELL B1. p1s and p2t are defined
#   below and are used only in this experiment; they are not part of the main
#   judge run.
#
# Judge:
#   GPT-5-mini only. This is the paper's primary ablation judge, and
#   all other prompt comparisons in the paper use it.
#
# Design:
#   All prompts are evaluated on the same 900 items in a single interleaved
#   pass with a fixed shuffle seed, so that the criterion contrast is not
#   confounded with run-to-run variation.
#
# Execution:
#   - If RUN_LABEL_REGENERATION=False, this cell is skipped.
#   - If RUN_FACTORIAL_PROMPT_EXPERIMENT=False, this cell is skipped.
#   - If enabled, this cell also requires ALLOW_EXPENSIVE_RUNS=True and
#     OPENAI_API_KEY in the environment.
#
# Safety:
#   - MASTER_PATH is read-only.
#   - This cell does NOT write to temp_results.csv and does NOT touch any
#     paper label column. It writes only to its own run directory.
#
# Outputs:
#   audits/factorial_prompt_run/<RUN_ID>/labels.csv
#   audits/factorial_prompt_run/<RUN_ID>/parse_failures.csv
#   audits/factorial_prompt_run/<RUN_ID>/metadata.json
#
# Analysis and LaTeX tables are produced separately in CELL C9, which reads
# labels.csv and makes no API calls.
# =============================================================================

import json
import random
import hashlib

import numpy as np
import pandas as pd

from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# =============================================================================
# Appendix-only prompts
#
# p1s  faithfulness criterion, structured like p2
# p2t  factual-correctness criterion, terse like p1
#
# These are reproduced verbatim in the appendix. Do not edit them without
# rerunning the experiment: the reported numbers are tied to these texts.
# =============================================================================

PROMPT_P1S = """You are a hallucination auditor for open-domain QA.

Your task: decide if a model answer contains a hallucination by comparing
it against the reference answer.

DEFINITION
A hallucination is any claim that is not supported by the reference answer
or is contradicted by it.

CRITICAL RULES
1. Evaluate reference support, including clear paraphrases and claims
   entailed by the reference. Different wording alone does not establish
   that a claim is unsupported.
2. Evaluate the claims actually made by the answer.
   An incomplete or non-committal answer is not a hallucination merely
   because it fails to provide the requested information; assess whether
   it makes a claim that is not supported by the reference answer or is
   contradicted by it.
3. Use the reference answer as ground truth. Do not use external knowledge.

DECISION TREE
Q1. Does the answer match the reference or a clear paraphrase?
    YES -> Q2. Does it add any claim that is not supported by the reference
              answer or is contradicted by it?
              NO  -> LABEL=0
              YES -> LABEL=1
    NO  -> Q3. Does the core answer conflict with the reference (different
              entity/value/fact)?
              YES -> LABEL=1
              NO  -> Q4. Is the answer a refusal or expression of uncertainty
                        that makes no factual claim?
                        YES -> LABEL=0
                        NO  -> Q5. Does it include any claim that is not
                                  supported by the reference answer or is
                                  contradicted by it?
                                  YES -> LABEL=1
                                  NO  -> LABEL=0

OUTPUT FORMAT (STRICT JSON ONLY)
Return ONLY valid JSON with exactly these two fields:
{"label": 0 or 1, "confidence": <float between 0 and 1>}

Rules:
- Do not include markdown, code fences, or extra text.
- Do not include a reasoning field.
- confidence must be a numeric value (not a string)."""


PROMPT_P2T = """You are a hallucination auditor for open-domain QA.

Your task: decide if a model answer contains a hallucination by comparing
it against the reference answer.

DEFINITION
A hallucination is any claim that is contradicted by the reference answer,
or that is factually false in a specific verifiable detail.

Information absent from the reference is not a hallucination if it is true.

OUTPUT FORMAT (STRICT JSON ONLY)
Return ONLY valid JSON with exactly these two fields:
{"label": 0 or 1, "confidence": <float between 0 and 1>}

Rules:
- Do not include markdown, code fences, or extra text.
- Do not include a reasoning field.
- confidence must be a numeric value (not a string)."""

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL B8: Criterion x structure factorial prompt experiment"

RUN_FACTORIAL_PROMPT_EXPERIMENT = globals().get(
    "RUN_FACTORIAL_PROMPT_EXPERIMENT",
    False,
)

if not RUN_LABEL_REGENERATION:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=False.")
    print("No factorial prompt labels were generated.")

elif not RUN_FACTORIAL_PROMPT_EXPERIMENT:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_FACTORIAL_PROMPT_EXPERIMENT=False.")
    print("The appendix tables are built from the stored labels.csv in CELL C9.")

else:
    if not ALLOW_EXPENSIVE_RUNS:
        raise RuntimeError(
            "RUN_FACTORIAL_PROMPT_EXPERIMENT=True, but ALLOW_EXPENSIVE_RUNS=False.\n"
            "This experiment makes 3600 API calls."
        )

    require_package("openai", "openai")

    from openai import OpenAI

    openai_key = get_secret("OPENAI_API_KEY", required=True)
    client = OpenAI(api_key=openai_key)

    # =========================================================================
    # Locked run configuration
    #
    # A new experiment must always get a new RUN_ID. FORCE_RECOMPUTE=False so
    # that an interrupted run resumes from its own labels.csv.
    # =========================================================================

    FACTORIAL_RUN_ID = globals().get(
        "FACTORIAL_RUN_ID",
        "gpt5mini_factorial_v3",
    )

    RUN_DIR = AUDIT_DIR / "factorial_prompt_run" / FACTORIAL_RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    LABELS_PATH = RUN_DIR / "labels.csv"
    FAILURES_PATH = RUN_DIR / "parse_failures.csv"
    METADATA_PATH = RUN_DIR / "metadata.json"

    FORCE_RECOMPUTE = False
    SHUFFLE_SEED = 42

    JUDGE_PREFIX = "gpt_5_mini"
    JUDGE_DISPLAY = "GPT-5-mini"
    JUDGE_MODEL_ID = "gpt-5-mini"

    FACTORIAL_MAX_WORKERS = 4

    # =========================================================================
    # The 2x2 design
    #
    # p1 and p2 come from CELL B1 so that this experiment can never drift from
    # the prompts used in the paper.
    # =========================================================================

    def _b1_prompt(key):
        for name in ["PROMPT_DEFS", "PROMPT_TEXTS", "JUDGE_PROMPTS", "PROMPTS"]:
            obj = globals().get(name, None)
            if isinstance(obj, dict):
                for k in [key, key.upper(), key.lower()]:
                    if k in obj:
                        return str(obj[k])

        for k in [key.upper(), f"PROMPT_{key.upper()}"]:
            val = globals().get(k, None)
            if isinstance(val, str) and val.strip():
                return val

        raise RuntimeError(
            f"Prompt {key!r} not found. Run CELL B1 first."
        )

    FACTORIAL_PROMPTS = {
        "p1":  _b1_prompt("p1"),   # faithfulness, terse       (paper prompt)
        "p1s": PROMPT_P1S,         # faithfulness, structured  (appendix only)
        "p2t": PROMPT_P2T,         # factual, terse            (appendix only)
        "p2":  _b1_prompt("p2"),   # factual, structured       (paper prompt)
    }

    # =========================================================================
    # Metadata lock
    # =========================================================================

    def sha256_text(text):
        return hashlib.sha256(text.encode("utf-8")).hexdigest()

    metadata_now = {
        "run_id": FACTORIAL_RUN_ID,
        "cell": "B8",
        "master_path": str(MASTER_PATH),
        "judge": {
            "prefix": JUDGE_PREFIX,
            "display": JUDGE_DISPLAY,
            "model_id": JUDGE_MODEL_ID,
        },
        "prompt_sha256": {
            name: sha256_text(text)
            for name, text in FACTORIAL_PROMPTS.items()
        },
        "settings": {
            "shuffle_seed": SHUFFLE_SEED,
            "max_workers": FACTORIAL_MAX_WORKERS,
        },
    }

    if METADATA_PATH.exists():
        with open(METADATA_PATH, "r", encoding="utf-8") as f:
            metadata_old = json.load(f)

        if metadata_old != metadata_now:
            raise RuntimeError(
                "Existing run metadata does not match the current configuration.\n"
                "Set a new FACTORIAL_RUN_ID rather than mixing experiments.\n"
                "This most often means a prompt text changed."
            )
    else:
        with open(METADATA_PATH, "w", encoding="utf-8") as f:
            json.dump(metadata_now, f, indent=2, ensure_ascii=False)

    # =========================================================================
    # Load read-only master, initialise the experiment table
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)

    BASE_COLS = [
        "item_id",
        "qid",
        "dataset",
        "model",
        "question",
        "ref_str",
        "answer_text",
        "annotator1_label",
    ]

    missing = [c for c in BASE_COLS if c not in master.columns]
    if missing:
        raise KeyError(f"Missing master columns: {missing}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows, found {len(master)}")

    if LABELS_PATH.exists():
        labels = pd.read_csv(LABELS_PATH)

        for col in ["item_id", "qid", "dataset", "model"]:
            if not master[col].astype(str).equals(labels[col].astype(str)):
                raise ValueError(
                    f"Row identity mismatch in {col} between master and "
                    f"{LABELS_PATH.name}. Use a new FACTORIAL_RUN_ID."
                )
    else:
        labels = master[BASE_COLS].copy()

    for prompt_name in FACTORIAL_PROMPTS:
        for prefix, default in [
            ("label", -1),
            ("conf", -1.0),
            ("raw", ""),
            ("error", ""),
        ]:
            col = f"{prefix}_{prompt_name}"
            if col not in labels:
                labels[col] = default

    # =========================================================================
    # Interleaved run
    # =========================================================================

    jobs = []

    for prompt_name in FACTORIAL_PROMPTS:
        current = pd.to_numeric(labels[f"label_{prompt_name}"], errors="coerce")

        for idx in master.index:
            if FORCE_RECOMPUTE or current.loc[idx] not in [0, 1]:
                jobs.append((idx, prompt_name))

    random.Random(SHUFFLE_SEED).shuffle(jobs)

    print("=" * 80)
    print("CRITERION x STRUCTURE FACTORIAL PROMPT EXPERIMENT")
    print("=" * 80)
    print("Run ID:        ", FACTORIAL_RUN_ID)
    print("Release master:", MASTER_PATH)
    print("Judge:         ", f"{JUDGE_DISPLAY} ({JUDGE_MODEL_ID})")
    print("Output:        ", LABELS_PATH)
    print()
    print("Design:")
    print("                  terse       structured")
    print("  faithfulness    p1          p1s")
    print("  factual         p2t         p2")
    print()
    print("Prompt sha256:")
    for name, h in metadata_now["prompt_sha256"].items():
        print(f"  {name:<4} {h[:16]}…")
    print()
    print("Jobs remaining:", len(jobs))
    print()
    print("This cell does not write to temp_results.csv or the release master.")

    if not jobs:
        print("\nAll labels already present; nothing to run.")
    else:
        def run_job(job):
            idx, prompt_name = job

            result = run_openai_judge_one(
                row=master.loc[idx],
                prompt_text=FACTORIAL_PROMPTS[prompt_name],
                model_id=JUDGE_MODEL_ID,
            )

            return idx, prompt_name, result

        completed = 0

        with ThreadPoolExecutor(max_workers=FACTORIAL_MAX_WORKERS) as executor:
            futures = [executor.submit(run_job, job) for job in jobs]

            with tqdm(total=len(futures), desc=JUDGE_DISPLAY) as progress:
                for future in as_completed(futures):
                    idx, prompt_name, result = future.result()

                    labels.loc[idx, f"label_{prompt_name}"] = int(result["label"])
                    labels.loc[idx, f"conf_{prompt_name}"] = float(result["confidence"])
                    labels.loc[idx, f"raw_{prompt_name}"] = result["raw_text"]
                    labels.loc[idx, f"error_{prompt_name}"] = result["parse_error"]

                    completed += 1
                    progress.update(1)

                    if completed % 200 == 0:
                        labels.to_csv(LABELS_PATH, index=False)

        labels.to_csv(LABELS_PATH, index=False)

    # =========================================================================
    # Parse-failure audit
    # =========================================================================

    failure_rows = []

    for prompt_name in FACTORIAL_PROMPTS:
        bad = ~pd.to_numeric(
            labels[f"label_{prompt_name}"], errors="coerce"
        ).isin([0, 1])

        for idx in labels.index[bad]:
            failure_rows.append({
                "item_id": labels.loc[idx, "item_id"],
                "qid": labels.loc[idx, "qid"],
                "generator": labels.loc[idx, "model"],
                "prompt": prompt_name,
                "error": labels.loc[idx, f"error_{prompt_name}"],
                "raw_text": labels.loc[idx, f"raw_{prompt_name}"],
            })

    failures = pd.DataFrame(failure_rows)
    failures.to_csv(FAILURES_PATH, index=False)

    print("\n" + "=" * 80)
    print("FACTORIAL RUN COMPLETE")
    print("=" * 80)
    print("Labels:       ", LABELS_PATH)
    print("Parse failures:", FAILURES_PATH)
    print("Metadata:     ", METADATA_PATH)

    if len(failures):
        print(f"\nWARNING: {len(failures)} parse failures remain.")
        print("Rerun this cell to retry only the failed items.")
    else:
        print("\n✓ No parse failures. Run CELL C9 to build the appendix tables.")

In [ ]:
# =============================================================================
# CELL B9: METEOR / BARTScore baselines
#
#
# Purpose:
#   a reviewer asked whether the agreement patterns reported for ROUGE-L
#   and BERTScore generalize to other similarity metrics. This cell evaluates
#   METEOR and BARTScore under the same protocol so the comparison is
#   like-for-like:
#
#     - the same 900-item release master and human labels;
#     - the same multi-reference aggregation (maximum over '|'-split refs);
#     - the same oracle-tau and 3-fold out-of-fold CV-tau framework;
#     - metric-appropriate threshold candidate sets;
#     - the same per-generator reporting.
#
#   BARTScore is evaluated in both directions, because the two are not
#   interchangeable:
#     bartscore_r2h  mean log P(answer | reference)
#     bartscore_h2r  mean log P(reference | answer)
#
# Conventions retained from CELL B2:
#   - Lower score => hallucination: label = 1 when score < tau.
#   - RNG seed = 42, 3 CV folds.
#   - Bounded metrics use the fixed 101-point [0, 1] grid.
#   - Column naming mirrors rougeL/bertscore exactly, so each metric
#     contributes the same ten columns the master already uses.
#
# Difference required for BARTScore:
#   BARTScore is an unbounded mean token log-likelihood, usually negative, so
#   a [0, 1] grid is meaningless for it. Its candidates are the exact score
#   midpoints observed in the relevant training data plus the two outer
#   boundaries, which enumerates every binary split a continuous score can
#   induce. Because that candidate set is finer than the 101-point grid, the
#   cell also reports a matched 101-point quantile-grid oracle as a
#   sensitivity check, so the comparison cannot be dismissed as an artifact of
#   an unequal search.
#
# Outputs:
#   TEMP_RESULTS_PATH                                    (scores + labels)
#   audits/new_metrics/new_metric_audit.csv              (per-generator)
#   audits/new_metrics/new_metric_macro.csv              (macro averages)
#   audits/new_metrics/new_metric_run_metadata.csv       (provenance)
#   tables/table_new_metrics.tex                         (appendix table)
#
# Paper label:
#   tab:results_new_metrics
#
# Master promotion:

#   MASTER_PATH is read-only here, as in CELL B8. The METEOR and BARTScore
#   columns are already present in the released master; this cell regenerates
#   the appendix table from them.
#
# Required packages/resources:
#   pip install numpy pandas scikit-learn nltk torch transformers
#   python -m nltk.downloader wordnet omw-1.4
# =============================================================================

from __future__ import annotations

import platform

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.model_selection import KFold

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

CELL_NAME = "CELL B9: METEOR / BARTScore baselines"

# Flags come from the notebook configuration cell, defaulting to off, so that a
# top-to-bottom run does not silently start a GPU job or rewrite temp results.
# This matches CELL B8's RUN_FACTORIAL_PROMPT_EXPERIMENT.
RUN_LABEL_REGENERATION = globals().get("RUN_LABEL_REGENERATION", False)
RUN_METEOR_LABELING = globals().get("RUN_METEOR_LABELING", False)
RUN_BARTSCORE_LABELING = globals().get("RUN_BARTSCORE_LABELING", False)

if not RUN_LABEL_REGENERATION:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=False. temp_results.csv was not modified.")

elif not (RUN_METEOR_LABELING or RUN_BARTSCORE_LABELING):
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_LABEL_REGENERATION=True, but both sub-flags are False:")
    print(f"  RUN_METEOR_LABELING    = {RUN_METEOR_LABELING}")
    print(f"  RUN_BARTSCORE_LABELING = {RUN_BARTSCORE_LABELING}")
    print("temp_results.csv was not modified.")

else:
    # =========================================================================
    # Path consistency
    #
    # Every path comes from the notebook globals rather than being hard-coded,
    # so this cell writes into the same tree as CELL B8 and CELL C9. The guard
    # catches the case where the cell has been pasted from another notebook
    # whose configuration cell points somewhere else.
    # =========================================================================
    for name in ["MASTER_PATH", "TEMP_RESULTS_PATH", "AUDIT_DIR", "TABLE_DIR"]:
        if name not in globals():
            raise RuntimeError(
                f"{name} is not defined. Run the configuration cell first; "
                "this cell must not define its own paths."
            )

    # The notebook's convention is AUDIT_DIR = <project root>/audits, so the
    # root is defined by AUDIT_DIR rather than guessed from path depth. Nesting
    # below the root is fine: temp_results lives two levels down.


    def _under_root(path):
        resolved = path.resolve()
        return resolved == PROJECT_ROOT or PROJECT_ROOT in resolved.parents

    outside = {
        name: str(path.resolve())
        for name, path in [
            ("MASTER_PATH", MASTER_PATH),
            ("TEMP_RESULTS_PATH", TEMP_RESULTS_PATH),
            ("TABLE_DIR", TABLE_DIR),
        ]
        if not _under_root(path)
    }

    if outside:
        lines = "\n".join(f"    {k}: {v}" for k, v in outside.items())
        raise RuntimeError(
            f"These paths are outside the project root {PROJECT_ROOT}:\n"
            f"{lines}\n"
            "The factorial run (CELL B8/C9) and this cell must write into the "
            "same tree, or the release bundle will be incoherent."
        )


    NEW_AUDIT_DIR = AUDIT_DIR / "new_metrics"
    NEW_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_AUDIT_CSV = NEW_AUDIT_DIR / "new_metric_audit.csv"
    OUT_MACRO_CSV = NEW_AUDIT_DIR / "new_metric_macro.csv"
    OUT_METADATA_CSV = NEW_AUDIT_DIR / "new_metric_run_metadata.csv"
    OUT_TABLE_TEX = TABLE_DIR / "table_new_metrics.tex"

    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError(
            "Safety error: TEMP_RESULTS_PATH points to MASTER_PATH. Refusing."
        )
    if not TEMP_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"temp_results.csv not found. Run CELL A3 first: {TEMP_RESULTS_PATH}"
        )
    if not MASTER_PATH.exists():
        raise FileNotFoundError(f"MASTER_PATH not found: {MASTER_PATH}")

    # =========================================================================
    # Config
    # =========================================================================
    HUMAN_COL = "annotator1_label"
    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    GEN_DISPLAY = {"llama3-8b": "L", "gemma-2-9b": "G", "mistral-7b": "M"}

    RNG_SEED = 42
    N_SPLITS = 3
    THRESHOLD_STEPS = 101

    BART_CKPT = globals().get("BART_CKPT", "facebook/bart-large-cnn")
    BART_BATCH_SIZE = int(globals().get("BART_BATCH_SIZE", 8))
    BART_MAX_LEN = int(globals().get("BART_MAX_LEN", 1024))

    if BART_BATCH_SIZE < 1:
        raise ValueError("BART_BATCH_SIZE must be >= 1.")
    if BART_MAX_LEN < 2:
        raise ValueError("BART_MAX_LEN must be >= 2.")

    NEW_SPECS = []

    if RUN_METEOR_LABELING:
        NEW_SPECS.append({
            "metric": "METEOR",
            "display": "METEOR",
            "score_col": "meteor",
            "tau_range": (0.0, 1.0),
            "threshold_mode": "fixed_grid_101",
            "tau_fmt": "{:.2f}",
        })

    if RUN_BARTSCORE_LABELING:
        NEW_SPECS.extend([
            {
                "metric": "BARTScore-CNN (ref->hyp)",
                "display": r"BARTScore ref$\to$hyp",
                "score_col": "bartscore_r2h",
                "tau_range": None,
                "threshold_mode": "exact_midpoints",
                "tau_fmt": "{:.2f}",
            },
            {
                "metric": "BARTScore-CNN (hyp->ref)",
                "display": r"BARTScore hyp$\to$ref",
                "score_col": "bartscore_h2r",
                "tau_range": None,
                "threshold_mode": "exact_midpoints",
                "tau_fmt": "{:.2f}",
            },
        ])

    # Suffixes mirror the rougeL/bertscore columns already in the master, so
    # each metric contributes the same ten columns.
    for spec in NEW_SPECS:
        c = spec["score_col"]
        spec.update({
            "label_oracle_col": f"{c}_label_oracle",
            "label_cv_col": f"{c}_label_cv",
            "tau_oracle_col": f"{c}_tau_oracle",
            "tau_cv_mean_col": f"{c}_tau_cv_mean",
            "tau_cv_min_col": f"{c}_tau_cv_min",
            "tau_cv_max_col": f"{c}_tau_cv_max",
            "cv_fold_col": f"{c}_cv_fold",
            "cv_kappa_mean_fold_col": f"{c}_cv_kappa_mean_fold",
            "cv_kappa_oof_col": f"{c}_cv_kappa_oof",
        })

    NEW_SCORE_COLS = [spec["score_col"] for spec in NEW_SPECS]
    NEW_LABEL_COLS = [
        col
        for spec in NEW_SPECS
        for col in (spec["label_oracle_col"], spec["label_cv_col"])
    ]

    # =========================================================================
    # Helpers
    # =========================================================================
    def safe_kappa(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)

        if y_true.shape != y_pred.shape:
            raise ValueError(
                f"Kappa shape mismatch: y_true={y_true.shape}, y_pred={y_pred.shape}"
            )
        if y_true.size == 0:
            return np.nan

        try:
            kappa = cohen_kappa_score(y_true, y_pred, labels=[0, 1])
            return float(kappa) if np.isfinite(kappa) else np.nan
        except Exception:
            return np.nan


    def labels_at_threshold(scores, tau):
        scores = np.asarray(scores, dtype=float)
        pred = np.full(scores.shape, -1, dtype=int)

        valid = np.isfinite(scores)
        if np.isfinite(tau):
            pred[valid] = (scores[valid] < tau).astype(int)

        return pred


    def tau_grid(scores, tau_range, steps=THRESHOLD_STEPS):
        """Candidate thresholds.

        Bounded metrics keep CELL B2's fixed 101-point grid on [0, 1].

        Unbounded metrics get every exact binary decision boundary: one value
        below the minimum, midpoints between consecutive unique scores, and one
        value above the maximum. With `score < tau => label 1`, these cover
        every partition a continuous score can induce.
        """
        scores = np.asarray(scores, dtype=float)
        finite_scores = scores[np.isfinite(scores)]

        if finite_scores.size == 0:
            return np.array([], dtype=float)

        if tau_range is not None:
            lo, hi = map(float, tau_range)
            if not lo < hi:
                raise ValueError(f"Invalid tau_range: {tau_range}")
            return np.linspace(lo, hi, int(steps), dtype=float)

        unique_scores = np.unique(finite_scores)

        if unique_scores.size == 1:
            value = unique_scores[0]
            return np.array(
                [np.nextafter(value, -np.inf), np.nextafter(value, np.inf)],
                dtype=float,
            )

        midpoints = (
            unique_scores[:-1] + (unique_scores[1:] - unique_scores[:-1]) / 2.0
        )

        return np.concatenate([
            [np.nextafter(unique_scores[0], -np.inf)],
            midpoints,
            [np.nextafter(unique_scores[-1], np.inf)],
        ]).astype(float)


    def quantile_grid(scores, steps=THRESHOLD_STEPS):
        """101 equally spaced quantiles of the observed scores.

        Used only for the sensitivity check: it gives an unbounded metric the
        same number of candidates as the bounded 101-point grid, so the oracle
        comparison is not confounded by search-space size.
        """
        scores = np.asarray(scores, dtype=float)
        finite_scores = scores[np.isfinite(scores)]

        if finite_scores.size == 0:
            return np.array([], dtype=float)

        return np.unique(
            np.quantile(finite_scores, np.linspace(0.0, 1.0, int(steps)))
        )


    def best_tau_on(scores, human_labels, candidates):
        """Highest-kappa threshold among `candidates`, lowest tau wins ties."""
        best_kappa = -np.inf
        best_tau = np.nan

        for tau in candidates:
            pred = (scores < tau).astype(int)
            kappa = safe_kappa(human_labels, pred)

            if np.isfinite(kappa) and kappa > best_kappa:
                best_kappa = kappa
                best_tau = float(tau)

        if not np.isfinite(best_tau):
            return np.nan, np.nan

        return float(best_kappa), best_tau


    def sweep_threshold_oracle(scores, human_labels, tau_range):
        scores = np.asarray(scores, dtype=float)
        human_labels = np.asarray(human_labels, dtype=int)

        valid = np.isfinite(scores) & np.isin(human_labels, [0, 1])
        if int(valid.sum()) < 10:
            return np.nan, np.nan

        candidates = tau_grid(scores[valid], tau_range)

        if candidates.size == 0:
            return np.nan, np.nan

        return best_tau_on(scores[valid], human_labels[valid], candidates)


    def sweep_threshold_quantile(scores, human_labels):
        """Matched-budget oracle for the sensitivity check."""
        scores = np.asarray(scores, dtype=float)
        human_labels = np.asarray(human_labels, dtype=int)

        valid = np.isfinite(scores) & np.isin(human_labels, [0, 1])
        if int(valid.sum()) < 10:
            return np.nan, np.nan

        candidates = quantile_grid(scores[valid])

        if candidates.size == 0:
            return np.nan, np.nan

        return best_tau_on(scores[valid], human_labels[valid], candidates)


    def cv_threshold_oof(
        scores, human_labels, tau_range, n_splits=N_SPLITS, random_state=RNG_SEED
    ):
        scores = np.asarray(scores, dtype=float)
        human_labels = np.asarray(human_labels, dtype=int)

        valid = np.isfinite(scores) & np.isin(human_labels, [0, 1])
        valid_idx = np.flatnonzero(valid)

        full_oof_pred = np.full(scores.shape, -1, dtype=int)
        full_fold_id = np.full(scores.shape, -1, dtype=int)

        if valid_idx.size < n_splits * 10:
            return full_oof_pred, full_fold_id, [], [], np.nan, np.nan, np.nan

        s_valid = scores[valid]
        y_valid = human_labels[valid]

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        oof_valid_pred = np.full(s_valid.shape, -1, dtype=int)
        fold_taus = []
        fold_kappas = []

        for fold_id, (train_idx, test_idx) in enumerate(kf.split(s_valid)):
            s_train = s_valid[train_idx]
            y_train = y_valid[train_idx]

            candidates = tau_grid(s_train, tau_range)
            _, best_tau = best_tau_on(s_train, y_train, candidates)

            if not np.isfinite(best_tau):
                best_tau = float(np.nanmedian(s_train))

            test_pred = (s_valid[test_idx] < best_tau).astype(int)

            oof_valid_pred[test_idx] = test_pred
            full_fold_id[valid_idx[test_idx]] = fold_id

            fold_taus.append(best_tau)
            fold_kappas.append(float(safe_kappa(y_valid[test_idx], test_pred)))

        if np.any(oof_valid_pred < 0):
            raise RuntimeError("OOF prediction coverage is incomplete.")

        full_oof_pred[valid_idx] = oof_valid_pred

        return (
            full_oof_pred,
            full_fold_id,
            fold_taus,
            fold_kappas,
            float(np.nanmean(fold_taus)),
            float(np.nanmean(fold_kappas)),
            float(safe_kappa(y_valid, oof_valid_pred)),
        )


    def metric_summary(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        mask = np.isin(y_true, [0, 1]) & np.isin(y_pred, [0, 1])
        human = y_true[mask].astype(int)
        predicted = y_pred[mask].astype(int)

        if human.size == 0:
            return {
                "n_valid": 0, "kappa": np.nan, "bias": np.nan,
                "fp": 0, "fn": 0, "err": 0,
                "fpr": np.nan, "fnr": np.nan, "fp_pct": np.nan,
            }

        tn, fp, fn, tp = confusion_matrix(human, predicted, labels=[0, 1]).ravel()
        errors = fp + fn

        return {
            "n_valid": int(mask.sum()),
            "kappa": safe_kappa(human, predicted),
            "bias": float(predicted.mean() - human.mean()),
            "fp": int(fp),
            "fn": int(fn),
            "err": int(errors),
            "fpr": float(fp / (fp + tn)) if (fp + tn) else np.nan,
            "fnr": float(fn / (fn + tp)) if (fn + tp) else np.nan,
            "fp_pct": float(100.0 * fp / errors) if errors else 0.0,
        }


    def parse_refs(ref_value):
        if pd.isna(ref_value):
            raise ValueError("Missing reference encountered in ref_str.")

        refs = [r.strip() for r in str(ref_value).split("|") if r.strip()]

        if not refs:
            raise ValueError("Empty reference encountered in ref_str.")

        return refs


    def require_columns(df, required_cols, df_name):
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise KeyError(f"{df_name} is missing required columns: {missing}")


    def require_same_row_identity(master_df, temp_df):
        identity_cols = ["item_id", "qid", "dataset", "model"]

        require_columns(master_df, identity_cols, "master")
        require_columns(temp_df, identity_cols, "temp")

        if len(master_df) != len(temp_df):
            raise ValueError(
                f"Row-count mismatch: master={len(master_df)}, temp={len(temp_df)}"
            )

        for col in identity_cols:
            master_values = master_df[col].astype("string").fillna("<NA>")
            temp_values = temp_df[col].astype("string").fillna("<NA>")

            if not master_values.equals(temp_values):
                mismatch_idx = np.flatnonzero(
                    master_values.to_numpy() != temp_values.to_numpy()
                )
                first = int(mismatch_idx[0]) if mismatch_idx.size else "unknown"
                raise ValueError(
                    f"Row identity mismatch in '{col}' at row {first}. "
                    "Reinitialize temp_results.csv."
                )


    def fmt_signed(x, digits=3):
        return f"$+${abs(x):.{digits}f}" if x >= 0 else f"$-${abs(x):.{digits}f}"


    def fmt_tau(x, fmt="{:.2f}"):
        """LaTeX threshold: math minus for negatives, as BARTScore taus are."""
        text = fmt.format(abs(x))
        return f"$-${text}" if x < 0 else text

    # =========================================================================
    # METEOR
    # =========================================================================
    def compute_meteor(ref_strs, answers):
        """NLTK METEOR with one deterministic tokenizer.

        TreebankWordTokenizer needs no downloaded punkt model. METEOR's synonym
        matching still requires the NLTK WordNet resources.
        """
        try:
            from nltk.corpus import wordnet
            from nltk.tokenize import TreebankWordTokenizer
            from nltk.translate.meteor_score import meteor_score
        except ModuleNotFoundError as exc:
            raise RuntimeError(
                "METEOR requires NLTK. Install it with: pip install nltk"
            ) from exc

        try:
            wordnet.synsets("test")
        except LookupError:
            # Colab runtimes reset between sessions, so fetch on demand rather
            # than requiring a separate manual step.
            print("  WordNet not found; downloading NLTK resources.")
            import nltk
            nltk.download("wordnet", quiet=True)
            nltk.download("omw-1.4", quiet=True)

            try:
                wordnet.synsets("test")
            except LookupError as exc:
                raise RuntimeError(
                    "METEOR requires NLTK WordNet resources and the automatic "
                    "download failed. Run manually:\n"
                    "  python -m nltk.downloader wordnet omw-1.4"
                ) from exc

        tokenizer = TreebankWordTokenizer()

        def tokenize(text):
            return tokenizer.tokenize(str(text))

        scores = []

        for ref_value, answer in zip(ref_strs, answers):
            refs = parse_refs(ref_value)
            scores.append(
                float(meteor_score([tokenize(r) for r in refs], tokenize(answer)))
            )

        return np.asarray(scores, dtype=float), "NLTK TreebankWordTokenizer"

    # =========================================================================
    # BARTScore
    # =========================================================================
    def compute_bartscore(ref_strs, answers):
        """Return r2h = mean log P(answer | reference), h2r = the reverse.

        Higher (less negative) means better supported, so the downstream rule
        `score < tau => hallucination` stays consistent with the bounded
        metrics.
        """
        try:
            import torch
            import transformers
            from transformers import BartForConditionalGeneration, BartTokenizer
        except ModuleNotFoundError as exc:
            raise RuntimeError(
                "BARTScore requires torch and transformers. Install them with:\n"
                "  pip install torch transformers"
            ) from exc

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if device.type == "cpu":
            print("  WARNING: no GPU detected. BARTScore on CPU may be slow.")

        tokenizer = BartTokenizer.from_pretrained(BART_CKPT)
        model = BartForConditionalGeneration.from_pretrained(BART_CKPT)
        model = model.eval().to(device)

        model_limit = int(
            getattr(model.config, "max_position_embeddings", BART_MAX_LEN)
        )
        tokenizer_limit = getattr(tokenizer, "model_max_length", BART_MAX_LEN)

        if not isinstance(tokenizer_limit, int) or tokenizer_limit > 1_000_000:
            tokenizer_limit = BART_MAX_LEN

        effective_max_len = min(BART_MAX_LEN, model_limit, int(tokenizer_limit))

        if effective_max_len != BART_MAX_LEN:
            print(
                f"  BART_MAX_LEN adjusted from {BART_MAX_LEN} "
                f"to supported limit {effective_max_len}."
            )

        pad_id = model.config.pad_token_id
        if pad_id is None:
            raise RuntimeError("BART model has no pad_token_id.")

        loss_fn = torch.nn.NLLLoss(reduction="none", ignore_index=pad_id)

        def raw_token_lengths(texts):
            """Untruncated lengths, batched, for the truncation audit."""
            lengths = []

            for start in range(0, len(texts), 128):
                encoded = tokenizer(
                    texts[start:start + 128],
                    add_special_tokens=True,
                    truncation=False,
                    padding=False,
                )
                lengths.extend(len(ids) for ids in encoded["input_ids"])

            return np.asarray(lengths, dtype=int)

        def logprob(sources, targets):
            if len(sources) != len(targets):
                raise ValueError(
                    f"BARTScore source/target mismatch: "
                    f"{len(sources)} vs {len(targets)}"
                )

            values = []

            for start in range(0, len(sources), BART_BATCH_SIZE):
                source_batch = sources[start:start + BART_BATCH_SIZE]
                target_batch = targets[start:start + BART_BATCH_SIZE]

                encoded_source = tokenizer(
                    source_batch,
                    truncation=True,
                    padding=True,
                    max_length=effective_max_len,
                    return_tensors="pt",
                ).to(device)

                encoded_target = tokenizer(
                    text_target=target_batch,
                    truncation=True,
                    padding=True,
                    max_length=effective_max_len,
                    return_tensors="pt",
                ).to(device)

                target_ids = encoded_target["input_ids"]

                with torch.inference_mode():
                    logits = model(
                        input_ids=encoded_source["input_ids"],
                        attention_mask=encoded_source["attention_mask"],
                        labels=target_ids,
                    ).logits

                    log_probs = torch.log_softmax(logits, dim=-1)

                    token_nll = loss_fn(
                        log_probs.reshape(-1, log_probs.size(-1)),
                        target_ids.reshape(-1),
                    ).reshape(target_ids.shape)

                    token_counts = (target_ids != pad_id).sum(dim=1).clamp(min=1)
                    mean_log_prob = -token_nll.sum(dim=1) / token_counts

                values.extend(mean_log_prob.detach().cpu().tolist())

            return np.asarray(values, dtype=float)

        flat_refs = []
        flat_answers = []
        reference_counts = []

        for ref_value, answer in zip(ref_strs, answers):
            refs = parse_refs(ref_value)
            answer_text = str(answer)

            flat_refs.extend(refs)
            flat_answers.extend([answer_text] * len(refs))
            reference_counts.append(len(refs))

        ref_lengths = raw_token_lengths(flat_refs)
        answer_lengths = raw_token_lengths(flat_answers)

        truncation_stats = {
            "bart_effective_max_len": effective_max_len,
            "bart_flat_pairs": len(flat_refs),
            "bart_ref_over_limit": int((ref_lengths > effective_max_len).sum()),
            "bart_answer_over_limit": int(
                (answer_lengths > effective_max_len).sum()
            ),
            "bart_ref_max_tokens": int(ref_lengths.max()) if ref_lengths.size else 0,
            "bart_answer_max_tokens": (
                int(answer_lengths.max()) if answer_lengths.size else 0
            ),
        }

        print(
            "  BARTScore truncation audit: "
            f"refs over limit={truncation_stats['bart_ref_over_limit']}, "
            f"answers over limit={truncation_stats['bart_answer_over_limit']}, "
            f"max ref tokens={truncation_stats['bart_ref_max_tokens']}, "
            f"max answer tokens={truncation_stats['bart_answer_max_tokens']}, "
            f"limit={effective_max_len}"
        )

        logp_r2h = logprob(flat_refs, flat_answers)
        logp_h2r = logprob(flat_answers, flat_refs)

        r2h = []
        h2r = []
        position = 0

        for count in reference_counts:
            stop = position + count
            r2h.append(float(np.max(logp_r2h[position:stop])))
            h2r.append(float(np.max(logp_h2r[position:stop])))
            position = stop

        if position != len(flat_refs):
            raise RuntimeError(
                "Internal BARTScore aggregation error: reference counts do not "
                "cover the flattened arrays."
            )

        metadata = {
            "transformers_version": transformers.__version__,
            "torch_version": torch.__version__,
            "bart_checkpoint": BART_CKPT,
            "bart_device": str(device),
            "bart_batch_size": BART_BATCH_SIZE,
            "bartscore_aggregation": "max over '|'-split references, both directions",
            **truncation_stats,
        }

        return (
            np.asarray(r2h, dtype=float),
            np.asarray(h2r, dtype=float),
            metadata,
        )

    # =========================================================================
    # Load and validate
    # =========================================================================
    master = pd.read_csv(MASTER_PATH)
    temp = pd.read_csv(TEMP_RESULTS_PATH)

    require_columns(
        master,
        ["item_id", "qid", "dataset", "model", "ref_str", "answer_text", HUMAN_COL],
        "master",
    )
    require_same_row_identity(master, temp)

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows, got {len(master)}.")

    if master["ref_str"].isna().any():
        bad = master.index[master["ref_str"].isna()].tolist()[:10]
        raise ValueError(f"ref_str contains missing values at rows {bad}.")

    if master["answer_text"].isna().any():
        bad = master.index[master["answer_text"].isna()].tolist()[:10]
        raise ValueError(f"answer_text contains missing values at rows {bad}.")

    human_numeric = pd.to_numeric(master[HUMAN_COL], errors="coerce")
    if not human_numeric.isin([0, 1]).all():
        bad = master.index[~human_numeric.isin([0, 1])].tolist()[:10]
        raise ValueError(
            f"{HUMAN_COL} must contain only 0/1 labels; invalid rows: {bad}."
        )

    observed_models = set(master["model"].dropna().astype(str))
    missing_models = [m for m in GEN_ORDER if m not in observed_models]
    if missing_models:
        raise ValueError(
            f"Expected generator models missing from master: {missing_models}"
        )

    print("=" * 80)
    print("METEOR / BARTSCORE BASELINES")
    print("=" * 80)
    print("Project root:  ", PROJECT_ROOT)
    print("Release master:", MASTER_PATH)
    print("Temp results:  ", TEMP_RESULTS_PATH)
    print("Audit dir:     ", NEW_AUDIT_DIR)
    print("Table out:     ", OUT_TABLE_TEX)
    print()
    print(f"  RUN_METEOR_LABELING    = {RUN_METEOR_LABELING}")
    print(f"  RUN_BARTSCORE_LABELING = {RUN_BARTSCORE_LABELING}")
    print("  Multi-reference handling  = max score across '|'-split refs")
    print("  METEOR thresholding       = fixed 101-point [0, 1] grid")
    print("  BARTScore thresholding    = exact observed-score midpoints")
    print("  Sensitivity check         = matched 101-point quantile grid")
    print()
    print("MASTER_PATH is read-only here.")

    # =========================================================================
    # Raw score computation
    # =========================================================================
    print("\n" + "=" * 80)
    print("RAW SCORE COMPUTATION FROM ref_str + answer_text")
    print("=" * 80)

    run_metadata = {
        "cell_name": CELL_NAME,
        "project_root": str(PROJECT_ROOT),
        "master_path": str(MASTER_PATH),
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "sklearn_seed": RNG_SEED,
        "cv_splits": N_SPLITS,
        "bounded_threshold_steps": THRESHOLD_STEPS,
        "multi_reference_aggregation": "maximum",
        "label_rule": "score < tau => hallucination label 1",
    }

    if RUN_METEOR_LABELING:
        print("METEOR: computing.")
        meteor_scores, meteor_tokenizer = compute_meteor(
            master["ref_str"], master["answer_text"]
        )
        temp["meteor"] = meteor_scores

        run_metadata["meteor_tokenizer"] = meteor_tokenizer
        run_metadata["meteor_params"] = "NLTK defaults (alpha=0.9, beta=3, gamma=0.5)"

        try:
            import nltk
            run_metadata["nltk_version"] = nltk.__version__
        except Exception:
            run_metadata["nltk_version"] = "unknown"

        print(
            f"  range {np.nanmin(meteor_scores):.4f} .. "
            f"{np.nanmax(meteor_scores):.4f}"
        )

    if RUN_BARTSCORE_LABELING:
        print("BARTScore: computing both directions.")
        r2h, h2r, bart_metadata = compute_bartscore(
            master["ref_str"], master["answer_text"]
        )

        temp["bartscore_r2h"] = r2h
        temp["bartscore_h2r"] = h2r
        run_metadata.update(bart_metadata)

        print(f"  ref->hyp range {np.nanmin(r2h):.3f} .. {np.nanmax(r2h):.3f}")
        print(f"  hyp->ref range {np.nanmin(h2r):.3f} .. {np.nanmax(h2r):.3f}")

    # =========================================================================
    # Per-generator thresholds and labels
    # =========================================================================
    audit_rows = []

    for model_name in GEN_ORDER:
        generator_mask = master["model"].astype(str) == model_name
        generator_n = int(generator_mask.sum())

        if generator_n == 0:
            raise ValueError(f"No rows found for generator {model_name}.")

        human = (
            pd.to_numeric(master.loc[generator_mask, HUMAN_COL], errors="raise")
            .astype(int)
            .to_numpy()
        )

        print("\n" + "-" * 80)
        print(f"{model_name} (n={generator_n})")
        print("-" * 80)

        for spec in NEW_SPECS:
            score_col = spec["score_col"]

            if score_col not in temp.columns:
                raise KeyError(
                    f"Expected score column '{score_col}' is missing from temp."
                )

            scores = (
                pd.to_numeric(temp.loc[generator_mask, score_col], errors="coerce")
                .to_numpy(dtype=float)
            )

            if not np.isfinite(scores).all():
                invalid_count = int((~np.isfinite(scores)).sum())
                raise ValueError(
                    f"{score_col} contains {invalid_count} non-finite values "
                    f"for generator {model_name}."
                )

            tau_range = spec["tau_range"]

            oracle_kappa, oracle_tau = sweep_threshold_oracle(
                scores, human, tau_range
            )
            oracle_labels = labels_at_threshold(scores, oracle_tau)

            # Matched-budget oracle, reported but never used for labels.
            if tau_range is None:
                q_kappa, q_tau = sweep_threshold_quantile(scores, human)
            else:
                q_kappa, q_tau = oracle_kappa, oracle_tau

            (
                cv_labels, fold_ids, fold_taus, fold_kappas,
                cv_tau_mean, cv_kappa_mean_fold, cv_kappa_oof,
            ) = cv_threshold_oof(scores, human, tau_range)

            temp.loc[generator_mask, spec["label_oracle_col"]] = pd.array(
                oracle_labels, dtype="Int64"
            )
            temp.loc[generator_mask, spec["label_cv_col"]] = pd.array(
                cv_labels, dtype="Int64"
            )
            temp.loc[generator_mask, spec["cv_fold_col"]] = pd.array(
                fold_ids, dtype="Int64"
            )

            temp.loc[generator_mask, spec["tau_oracle_col"]] = oracle_tau
            temp.loc[generator_mask, spec["tau_cv_mean_col"]] = cv_tau_mean
            temp.loc[generator_mask, spec["tau_cv_min_col"]] = (
                min(fold_taus) if fold_taus else np.nan
            )
            temp.loc[generator_mask, spec["tau_cv_max_col"]] = (
                max(fold_taus) if fold_taus else np.nan
            )
            temp.loc[generator_mask, spec["cv_kappa_mean_fold_col"]] = (
                cv_kappa_mean_fold
            )
            temp.loc[generator_mask, spec["cv_kappa_oof_col"]] = cv_kappa_oof

            oracle_summary = metric_summary(human, oracle_labels)
            cv_summary = metric_summary(human, cv_labels)

            audit_rows.append({
                "model": model_name,
                "gen_display": GEN_DISPLAY[model_name],
                "n": generator_n,
                "metric": spec["metric"],
                "display": spec["display"],
                "score_col": score_col,
                "threshold_mode": spec["threshold_mode"],
                "score_min": float(np.min(scores)),
                "score_max": float(np.max(scores)),
                "tau_oracle": oracle_tau,
                "kappa_oracle": oracle_summary["kappa"],
                "bias_oracle": oracle_summary["bias"],
                "fp_oracle": oracle_summary["fp"],
                "fn_oracle": oracle_summary["fn"],
                "fpr_oracle": oracle_summary["fpr"],
                "fnr_oracle": oracle_summary["fnr"],
                "fp_pct_oracle": oracle_summary["fp_pct"],
                "tau_oracle_quantile_grid": q_tau,
                "kappa_oracle_quantile_grid": q_kappa,
                "tau_cv_mean": cv_tau_mean,
                "tau_cv_min": min(fold_taus) if fold_taus else np.nan,
                "tau_cv_max": max(fold_taus) if fold_taus else np.nan,
                "kappa_cv_mean_fold": cv_kappa_mean_fold,
                "kappa_cv_oof": cv_kappa_oof,
                "bias_cv": cv_summary["bias"],
                "fp_cv": cv_summary["fp"],
                "fn_cv": cv_summary["fn"],
                "fpr_cv": cv_summary["fpr"],
                "fnr_cv": cv_summary["fnr"],
                "fp_pct_cv": cv_summary["fp_pct"],
            })

            print(
                f"  {spec['metric']:<30} "
                f"oracle tau={oracle_tau:>9.4f} "
                f"kappa={oracle_summary['kappa']:.3f} "
                f"FP%={oracle_summary['fp_pct']:.1f} | "
                f"CV kappa={cv_kappa_oof:.3f}"
            )

    audit = pd.DataFrame(audit_rows)

    # =========================================================================
    # Label integrity
    #
    # temp.loc[mask, col] = pd.array(...) aligns positionally. Verify rather
    # than assume: a silent misalignment would corrupt every downstream number.
    # =========================================================================
    for spec in NEW_SPECS:
        for col in [spec["label_oracle_col"], spec["label_cv_col"]]:
            n_bad = int((~temp[col].isin([0, 1])).sum())
            if n_bad:
                raise AssertionError(
                    f"{col}: {n_bad} of {len(temp)} rows are not 0/1 after "
                    "assignment. Label columns are misaligned."
                )

        folds = temp[spec["cv_fold_col"]]
        if not folds.isin(range(N_SPLITS)).all():
            raise AssertionError(
                f"{spec['cv_fold_col']}: expected fold ids 0-{N_SPLITS - 1}, "
                f"found {sorted(folds.unique())}."
            )

    print("\n✓ All new label and fold columns are complete across 900 rows.")

    # =========================================================================
    # BARTScore search-budget sensitivity
    #
    # The exact-midpoint candidate set is finer than the bounded metrics'
    # 101-point grid. If the oracle values were an artifact of that, the
    # matched 101-point quantile grid would give materially lower kappa.
    # =========================================================================
    unbounded = audit[audit["threshold_mode"] == "exact_midpoints"]

    if len(unbounded):
        print("\n" + "=" * 80)
        print("SEARCH-BUDGET SENSITIVITY (unbounded metrics)")
        print("=" * 80)
        print("  Exact midpoints vs matched 101-point quantile grid:")

        for _, r in unbounded.iterrows():
            delta = r["kappa_oracle"] - r["kappa_oracle_quantile_grid"]
            print(
                f"    {r['metric']:<30} {r['gen_display']}  "
                f"midpoints={r['kappa_oracle']:.3f}  "
                f"quantile-101={r['kappa_oracle_quantile_grid']:.3f}  "
                f"diff={delta:+.3f}"
            )

        worst = float((
            unbounded["kappa_oracle"] - unbounded["kappa_oracle_quantile_grid"]
        ).max())
        print(f"\n  Largest advantage from the finer search: {worst:+.3f}")

    # =========================================================================
    # Macro averages and comparability with the existing metrics
    #
    # METEOR and both BARTScore directions fall in the
    # same broad agreement range as the metrics already in the paper. Compute
    # that comparison here rather than by hand: every *_label_oracle column in
    # temp_results is scored against the same human labels.
    # =========================================================================
    macro = (
        audit.groupby("metric")[
            ["kappa_oracle", "kappa_cv_oof", "fpr_oracle", "fnr_oracle",
             "fp_pct_oracle"]
        ]
        .mean()
        .round(3)
        .sort_values("kappa_oracle", ascending=False)
    )

    print("\n" + "=" * 80)
    print("MACRO-AVERAGED OVER GENERATORS (new metrics)")
    print("=" * 80)
    display(macro)

    def macro_kappa_for_column(label_col):
        """Macro-averaged kappa of a stored label column, by generator."""
        values = []

        for model_name in GEN_ORDER:
            mask = master["model"].astype(str) == model_name

            y = pd.to_numeric(
                master.loc[mask, HUMAN_COL], errors="coerce"
            ).fillna(-1).to_numpy(dtype=int)
            p = pd.to_numeric(
                temp.loc[mask, label_col], errors="coerce"
            ).fillna(-1).to_numpy(dtype=int)

            keep = np.isin(y, [0, 1]) & np.isin(p, [0, 1])
            if keep.sum() < 10:
                return np.nan

            values.append(safe_kappa(y[keep], p[keep]))

        return float(np.nanmean(values))

    oracle_cols = sorted(c for c in temp.columns if c.endswith("_label_oracle"))
    comparison_rows = []

    for col in oracle_cols:
        comparison_rows.append({
            "label_column": col,
            "is_new": col in NEW_LABEL_COLS,
            "macro_kappa_oracle": macro_kappa_for_column(col),
        })

    comparison = (
        pd.DataFrame(comparison_rows)
        .dropna(subset=["macro_kappa_oracle"])
        .sort_values("macro_kappa_oracle", ascending=False)
        .round(3)
    )

    print("\n" + "=" * 80)
    print("COMPARABILITY: ALL ORACLE-THRESHOLD LABEL COLUMNS")
    print("=" * 80)
    display(comparison)

    if len(comparison):
        existing = comparison[~comparison["is_new"]]["macro_kappa_oracle"]
        added = comparison[comparison["is_new"]]["macro_kappa_oracle"]

        if len(existing) and len(added):
            print(
                f"\n  Existing metrics span "
                f"{existing.min():.3f}--{existing.max():.3f}; "
                f"METEOR/BARTScore span {added.min():.3f}--{added.max():.3f}."
            )
            inside = (added.min() >= existing.min()) and (
                added.max() <= existing.max()
            )
            print(
                "  The added metrics fall inside the existing range."
                if inside else
                "  NOTE: the added metrics extend beyond the existing range; "
                "adjust the manuscript wording accordingly."
            )

    # =========================================================================
    # Appendix table
    # =========================================================================
    caption = (
        r"METEOR and BARTScore baselines against the same human "
        r"factual-correctness labels, using the multi-reference aggregation and "
        r"threshold protocol of Table~\ref{tab:results_lexical}. BARTScore is an "
        r"unbounded mean token log-likelihood, so its threshold candidates are "
        r"the exact observed score midpoints rather than a fixed grid. "
        r"ref$\to$hyp scores $\log P(\text{answer}\mid\text{reference})$ and "
        r"hyp$\to$ref the reverse. Bias is positive when the automated labeler "
        r"over-labels hallucinations relative to humans. FP\% is false positives "
        r"as a proportion of all errors. L=Llama-3-8B, G=Gemma-2-9B, "
        r"M=Mistral-7B."
    )

    latex = r"""\begin{table*}[t]
  \centering
  \caption{""" + caption + r"""}
  \label{tab:results_new_metrics}
  \scriptsize
  \setlength{\tabcolsep}{5pt}
  \renewcommand{\arraystretch}{0.95}
  \begin{tabular}{lllrrrrr}
    \toprule
    Metric & Gen & $\tau$ & $\kappa$ & Bias & FP & FN & FP\% \\
    \midrule
"""

    for block_idx, spec in enumerate(NEW_SPECS):
        if block_idx:
            latex += "    \\midrule\n"

        block = audit[audit["metric"] == spec["metric"]]
        fmt = spec["tau_fmt"]

        for mode in ["oracle", "cv"]:
            for model_name in GEN_ORDER:
                r = block[block["model"] == model_name].iloc[0]

                if mode == "oracle":
                    tau_cell = fmt_tau(r["tau_oracle"], fmt)
                    kappa, bias = r["kappa_oracle"], r["bias_oracle"]
                    fp, fn, fp_pct = r["fp_oracle"], r["fn_oracle"], r["fp_pct_oracle"]
                    label = f"{spec['display']} (oracle-$\\tau$)"
                else:
                    tau_cell = (
                        f"{fmt_tau(r['tau_cv_mean'], fmt)} "
                        f"[{fmt_tau(r['tau_cv_min'], fmt)}, "
                        f"{fmt_tau(r['tau_cv_max'], fmt)}]"
                    )
                    kappa, bias = r["kappa_cv_oof"], r["bias_cv"]
                    fp, fn, fp_pct = r["fp_cv"], r["fn_cv"], r["fp_pct_cv"]
                    label = f"{spec['display']} (CV-$\\tau$)"

                latex += (
                    f"    {label} & {GEN_DISPLAY[model_name]} & {tau_cell} & "
                    f"{kappa:.3f} & {fmt_signed(bias)} & "
                    f"{int(fp)} & {int(fn)} & {fp_pct:.1f} \\\\\n"
                )

    latex += r"""    \bottomrule
  \end{tabular}
\end{table*}
"""

    expected_rows = len(NEW_SPECS) * len(GEN_ORDER) * 2
    actual_rows = latex.count("\\\\\n") - 1  # minus the header row

    if actual_rows != expected_rows:
        raise AssertionError(
            f"Table: expected {expected_rows} data rows, built {actual_rows}."
        )

    with open(OUT_TABLE_TEX, "w", encoding="utf-8") as f:
        f.write(latex)

    # =========================================================================
    # Save
    # =========================================================================
    if TEMP_RESULTS_PATH.resolve() == MASTER_PATH.resolve():
        raise RuntimeError("Refusing to write temp results to MASTER_PATH.")

    temp_write_path = TEMP_RESULTS_PATH.with_suffix(
        TEMP_RESULTS_PATH.suffix + ".tmp"
    )
    temp.to_csv(temp_write_path, index=False)
    temp_write_path.replace(TEMP_RESULTS_PATH)

    audit.to_csv(OUT_AUDIT_CSV, index=False)
    macro.to_csv(OUT_MACRO_CSV)

    pd.DataFrame(
        [{"key": k, "value": v} for k, v in run_metadata.items()]
    ).to_csv(OUT_METADATA_CSV, index=False)

    print("\nSaved:")
    for path in [
        TEMP_RESULTS_PATH, OUT_AUDIT_CSV, OUT_MACRO_CSV,
        OUT_METADATA_CSV, OUT_TABLE_TEX,
    ]:
        print(" ", path)

    print("\n" + "=" * 80)
    print("NUMBERS FOR THE MANUSCRIPT")
    print("=" * 80)
    for metric, row in macro.iterrows():
        print(
            f"  {metric:<30} macro oracle kappa={row['kappa_oracle']:.3f}  "
            f"CV={row['kappa_cv_oof']:.3f}  FP%={row['fp_pct_oracle']:.1f}"
        )

    print("\n" + "=" * 80)
    print("Appendix table: " + OUT_TABLE_TEX.name)
    print("=" * 80)
    print(latex)

    print(
        "Audit note: an optimal threshold may legitimately lie outside the "
        "observed score range when the best prediction assigns every item to "
        "one class. Do not require tau to fall strictly inside the score range."
    )


In [ ]:
# =============================================================================
# CELL B10: Core-only answer extraction for the elaboration intervention
#
# Purpose:
#   For all items where BOTH human annotators assigned label 0, extract a
#   deletion-only "core answer" from the original generated answer.
#
# IMPORTANT:
#   - The extractor sees ONLY the question and original answer.
#   - It does NOT see the reference, human labels, or judge labels.
#   - It does NOT evaluate factual correctness.
#   - MASTER_PATH is read-only.
#
# Allowed semantic statuses:
#   CORE_EXTRACTED
#   ALREADY_CORE
#   NO_CLEAR_CORE_ANSWER
#
# Workflow:
#   1. Run a 30-item pilot.
#   2. Human-audit the pilot.
#   3. If prompt is accepted unchanged, set CORE_PILOT_N=None and rerun.
#   4. If prompt changes, use a NEW CORE_RUN_ID.
#
# Outputs:
#   audits/elaboration_core_only/<RUN_ID>/core_extractions.csv
#   audits/elaboration_core_only/<RUN_ID>/human_validation_template.csv
#   audits/elaboration_core_only/<RUN_ID>/core_extraction_prompt.txt
#   audits/elaboration_core_only/<RUN_ID>/run_config.json
# =============================================================================

import os
import re
import json
import time
import hashlib
from enum import Enum
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel
from openai import OpenAI
from IPython.display import display

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

if not RUN_B10:
    print("Skipped: RUN_B10 is False")

else:
    # =============================================================================
    # Config
    # =============================================================================

    CORE_MODEL_ID = globals().get("CORE_MODEL_ID", "gpt-5.6-sol")
    CORE_REASONING_EFFORT = globals().get("CORE_REASONING_EFFORT", "medium")

    CORE_RUN_ID = globals().get(
        "CORE_RUN_ID",
        "gpt56_core_extraction_v1",
    )

    # First run: 30-item pilot.
    # After human validation, set to None to run all 140.
    CORE_PILOT_N = None #globals().get("CORE_PILOT_N", 30)

    CORE_SHUFFLE_SEED = 42
    CORE_MAX_WORKERS = 2
    CORE_MAX_ATTEMPTS = 2
    CORE_MAX_OUTPUT_TOKENS = 2000

    EXPECTED_ELIGIBLE_N = 140

    ID_COL = "item_id"
    QUESTION_COL = "question"
    ANSWER_COL = "answer_text"
    HUMAN1_COL = "annotator1_label"
    HUMAN2_COL = "annotator2_label"


    # =============================================================================
    # Output paths
    # =============================================================================

    RUN_DIR = (
        AUDIT_DIR
        / "elaboration_core_only"
        / CORE_RUN_ID
    )
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    OUT_CSV = RUN_DIR / "core_extractions.csv"
    HUMAN_AUDIT_CSV = RUN_DIR / "human_validation_template.csv"
    PROMPT_TXT = RUN_DIR / "core_extraction_prompt.txt"
    CONFIG_JSON = RUN_DIR / "run_config.json"


    # =============================================================================
    # Extraction prompt
    # =============================================================================


    CORE_EXTRACTION_PROMPT = """
You are extracting the CORE ANSWER from an ORIGINAL_ANSWER to a QUESTION.

The CORE ANSWER is the complete part of the ORIGINAL_ANSWER that directly
answers the QUESTION, with elaboration removed.

Elaboration is content that adds information beyond the answer itself and
can be removed without removing or changing the complete answer-bearing
content.

The goal is to remove elaboration, not to reduce the answer to the shortest
possible word or phrase.

If a complete answer-bearing clause contains no elaboration, keep that
clause unchanged. Do not reduce it to only the name, number, phrase, or
other answer value it contains.

A short phrase or single word may be the CORE ANSWER when the answer is not
expressed within a complete answer-bearing clause in the ORIGINAL_ANSWER.

Elaboration may be removed between sentences or from within an
answer-bearing clause. Remove clearly separable elaboration while preserving
the underlying answer-bearing clause itself.

Remove discourse markers whose supporting context has been deleted when
they are not part of the answer itself.

Read the entire ORIGINAL_ANSWER before deciding. If later text corrects,
replaces, retracts, or materially qualifies an answer given earlier, do not
preserve the earlier answer in isolation.

The extracted text must be an exact subsequence of the ORIGINAL_ANSWER.
You may delete text. You may not add, rewrite, paraphrase, correct, reorder,
or re-inflect anything.

Choose exactly one status:

CORE_EXTRACTED
- The ORIGINAL_ANSWER contains a clear answer and some elaboration can be
  removed.
- Return the extracted CORE ANSWER in core_answer.

ALREADY_CORE
- The ORIGINAL_ANSWER already consists of the complete answer with no
  elaboration that should be removed.
- Do not shorten it merely because a shorter phrase could also answer the
  QUESTION.
- Return the ORIGINAL_ANSWER unchanged in core_answer.

NO_CLEAR_CORE_ANSWER
- The ORIGINAL_ANSWER does not contain a clear answer to the QUESTION that
  can be extracted under these rules.
- Do not invent or reconstruct an answer.
- Return the ORIGINAL_ANSWER unchanged in core_answer.


""".strip()
    # When uncertain whether removing text would alter the complete answer-bearing content, keep it.

    PROMPT_HASH = hashlib.sha256(
        CORE_EXTRACTION_PROMPT.encode("utf-8")
    ).hexdigest()


    # =============================================================================
    # Structured output
    # =============================================================================

    class CoreStatus(str, Enum):
        CORE_EXTRACTED = "CORE_EXTRACTED"
        ALREADY_CORE = "ALREADY_CORE"
        NO_CLEAR_CORE_ANSWER = "NO_CLEAR_CORE_ANSWER"


    class CoreExtraction(BaseModel):
        status: CoreStatus
        core_answer: str


    # =============================================================================
    # Mechanical validation
    # =============================================================================

    def tokenize_exact(text):
        """
    Words and punctuation, case-sensitive.

    Used only to verify that extraction introduced no new or reordered tokens.
    """
        return re.findall(
            r"\w+|[^\w\s]",
            str(text),
            flags=re.UNICODE,
        )


    def is_token_subsequence(candidate, original):
        """
    Candidate may delete tokens but may not add/change/reorder them.
    """
        candidate_tokens = tokenize_exact(candidate)
        original_tokens = tokenize_exact(original)

        if not candidate_tokens:
            return False

        j = 0

        for token in original_tokens:
            if (
                j < len(candidate_tokens)
                and token == candidate_tokens[j]
            ):
                j += 1

        return j == len(candidate_tokens)


    def validate_extraction(status, model_core, original):
        """
    Mechanical validation only.

    Semantic validity is checked later by humans.
    """

        # These statuses must leave the answer exactly unchanged.
        if status in {
            "ALREADY_CORE",
            "NO_CLEAR_CORE_ANSWER",
        }:
            return True, str(original), ""

        if status != "CORE_EXTRACTED":
            return False, str(original), "Unexpected status."

        candidate = str(model_core)
        original = str(original)

        candidate_tokens = tokenize_exact(candidate)
        original_tokens = tokenize_exact(original)

        if not candidate_tokens:
            return False, original, "Empty extracted core."

        if not is_token_subsequence(candidate, original):
            return (
                False,
                original,
                "Core is not an exact token-order subsequence.",
            )

        if len(candidate_tokens) >= len(original_tokens):
            return (
                False,
                original,
                "CORE_EXTRACTED did not remove anything.",
            )

        return True, candidate, ""


    # =============================================================================
    # Reproducibility guards
    # =============================================================================

    def get_openai_key():
        if "get_secret" in globals():
            return get_secret(
                "OPENAI_API_KEY",
                required=True,
            )

        key = os.environ.get(
            "OPENAI_API_KEY",
            "",
        ).strip()

        if not key:
            raise RuntimeError("OPENAI_API_KEY is not set.")

        return key


    run_config = {
        "run_id": CORE_RUN_ID,
        "model": CORE_MODEL_ID,
        "reasoning_effort": CORE_REASONING_EFFORT,
        "shuffle_seed": CORE_SHUFFLE_SEED,
        "prompt_sha256": PROMPT_HASH,
        "eligible_rule": (
            "annotator1_label == 0 AND "
            "annotator2_label == 0"
        ),
        "model_visible_fields": [
            QUESTION_COL,
            ANSWER_COL,
        ],
        "semantic_statuses": [
            s.value for s in CoreStatus
        ],
    }


    # Do not accidentally mix two prompt/model versions in one run directory.
    if CONFIG_JSON.exists():

        old_config = json.loads(
            CONFIG_JSON.read_text(encoding="utf-8")
        )

        for key in [
            "model",
            "reasoning_effort",
            "prompt_sha256",
            "eligible_rule",
        ]:
            if old_config.get(key) != run_config.get(key):
                raise RuntimeError(
                    f"{CORE_RUN_ID} already exists with a different "
                    f"{key}.\nUse a NEW CORE_RUN_ID."
                )

    else:

        CONFIG_JSON.write_text(
            json.dumps(
                run_config,
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

        PROMPT_TXT.write_text(
            CORE_EXTRACTION_PROMPT + "\n",
            encoding="utf-8",
        )


    # =============================================================================
    # Load eligible items
    # =============================================================================

    master = pd.read_csv(MASTER_PATH)

    required_cols = [
        ID_COL,
        "dataset",
        "model",
        QUESTION_COL,
        ANSWER_COL,
        HUMAN1_COL,
        HUMAN2_COL,
    ]

    missing = [
        c for c in required_cols
        if c not in master.columns
    ]

    if missing:
        raise KeyError(
            f"Missing columns: {missing}"
        )

    if not master[ID_COL].is_unique:
        raise ValueError("item_id is not unique.")


    eligible = master.loc[
        (master[HUMAN1_COL] == 0)
        & (master[HUMAN2_COL] == 0),
        required_cols,
    ].copy()


    if len(eligible) != EXPECTED_ELIGIBLE_N:
        raise ValueError(
            f"Expected {EXPECTED_ELIGIBLE_N} eligible rows, "
            f"found {len(eligible)}."
        )


    # Stable randomized order.
    eligible = eligible.sample(
        frac=1,
        random_state=CORE_SHUFFLE_SEED,
    ).reset_index(drop=True)


    # =============================================================================
    # One extraction call
    # =============================================================================

    def run_one(client, row):
        last_error = ""
        last_raw_core = ""
        last_raw_status = ""

        question = str(row[QUESTION_COL])
        original = str(row[ANSWER_COL])

        user_message = (
            f"QUESTION:\n{question}\n\n"
            f"ORIGINAL_ANSWER:\n{original}"
        )



        for attempt in range(
            1,
            CORE_MAX_ATTEMPTS + 1,
        ):

            try:

                response = client.responses.parse(
                    model=CORE_MODEL_ID,
                    reasoning={
                        "effort": CORE_REASONING_EFFORT
                    },
                    input=[
                        {
                            "role": "system",
                            "content": CORE_EXTRACTION_PROMPT,
                        },
                        {
                            "role": "user",
                            "content": user_message,
                        },
                    ],
                    text_format=CoreExtraction,
                    max_output_tokens=CORE_MAX_OUTPUT_TOKENS,
                    store=False,
                )

                parsed = response.output_parsed

                if parsed is None:
                    raise ValueError(
                        "Structured output was empty."
                    )

                status = (
                    parsed.status.value
                    if isinstance(parsed.status, CoreStatus)
                    else str(parsed.status)
                )

                raw_core = str(
                    parsed.core_answer
                )
                last_raw_status = status
                last_raw_core = raw_core

                valid, accepted_core, error = (
                    validate_extraction(
                        status,
                        raw_core,
                        original,
                    )
                )

                if valid:

                    return {
                        ID_COL: row[ID_COL],
                        "dataset": row["dataset"],
                        "model": row["model"],
                        QUESTION_COL: question,
                        "original_answer": original,

                        "extraction_status": status,
                        "model_status_raw": status,

                        # Preserve exactly what the model returned
                        # for audit purposes.
                        "model_core_answer_raw": raw_core,

                        # This is the version allowed to enter
                        # human validation.
                        "core_only_answer": accepted_core,

                        "auto_valid": True,
                        "pipeline_status": "OK",
                        "pipeline_error": "",

                        "attempts": attempt,

                        "extractor_model_requested":
                            CORE_MODEL_ID,

                        "extractor_model_returned":
                            getattr(
                                response,
                                "model",
                                CORE_MODEL_ID,
                            ),

                        "response_id":
                            getattr(
                                response,
                                "id",
                                "",
                            ),

                        "prompt_sha256":
                            PROMPT_HASH,
                    }

                last_error = error

            except Exception as e:
                last_error = repr(e)

            time.sleep(2 * attempt)


        # Do NOT convert a technical failure into
        # NO_CLEAR_CORE_ANSWER.
        return {
            ID_COL: row[ID_COL],
            "dataset": row["dataset"],
            "model": row["model"],
            QUESTION_COL: question,
            "original_answer": original,

            "extraction_status": "",
            "model_status_raw": last_raw_status,
            "model_core_answer_raw": last_raw_core,
            "core_only_answer": original,

            "auto_valid": False,
            "pipeline_status": "FAILED",
            "pipeline_error": last_error,

            "attempts": CORE_MAX_ATTEMPTS,

            "extractor_model_requested":
                CORE_MODEL_ID,

            "extractor_model_returned": "",
            "response_id": "",

            "prompt_sha256":
                PROMPT_HASH,
        }


    # =============================================================================
    # Resume existing run if present
    # =============================================================================

    if OUT_CSV.exists():

        completed = pd.read_csv(OUT_CSV)

        if completed[ID_COL].duplicated().any():
            raise ValueError(
                "Duplicate item_ids in extraction checkpoint."
            )

        completed_ids = set(
            completed.loc[
                completed["pipeline_status"] == "OK",
                ID_COL,
            ]
        )

    else:

        completed = pd.DataFrame()
        completed_ids = set()


    # Pilot = first N rows from the fixed randomized order.
    # Full run = all 140.
    if CORE_PILOT_N is None:

        target = eligible.copy()

    else:

        target = eligible.iloc[
            :CORE_PILOT_N
        ].copy()


    todo = target.loc[
        ~target[ID_COL].isin(completed_ids)
    ].copy()


    # =============================================================================
    # Run
    # =============================================================================

    print("=" * 80)
    print("CORE-ONLY EXTRACTION")
    print("=" * 80)

    print("Eligible rows:          ", len(eligible))
    print("Target this phase:      ", len(target))
    print("Already completed:      ", len(target) - len(todo))
    print("API calls remaining:    ", len(todo))
    print("Model:                  ", CORE_MODEL_ID)
    print("Reasoning:              ", CORE_REASONING_EFFORT)
    print("Prompt SHA256:          ", PROMPT_HASH)
    print("Output:                 ", OUT_CSV)


    if len(todo):

        client = OpenAI(
            api_key=get_openai_key()
        )

        existing_records = (
            completed.to_dict("records")
            if not completed.empty
            else []
        )

        new_records = []

        with ThreadPoolExecutor(
            max_workers=CORE_MAX_WORKERS
        ) as pool:

            futures = {
                pool.submit(
                    run_one,
                    client,
                    row,
                ): row[ID_COL]
                for _, row in todo.iterrows()
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Core extraction",
            ):

                result = future.result()

                new_records.append(
                    result
                )

                # 140 rows only: write checkpoint
                # after every completed request.
                checkpoint = pd.DataFrame(
                    existing_records
                    + new_records
                )

                checkpoint = (
                    checkpoint
                    .drop_duplicates(
                        ID_COL,
                        keep="last",
                    )
                )

                tmp_path = OUT_CSV.with_suffix(
                    ".csv.tmp"
                )

                checkpoint.to_csv(
                    tmp_path,
                    index=False,
                )

                tmp_path.replace(
                    OUT_CSV
                )


    # =============================================================================
    # Phase results
    # =============================================================================

    results = pd.read_csv(
        OUT_CSV
    )

    phase = results.loc[
        results[ID_COL].isin(
            target[ID_COL]
        )
    ].copy()


    # =============================================================================
    # Human-validation template
    # =============================================================================

    human_audit = phase[
        [
            ID_COL,
            "dataset",
            "model",
            QUESTION_COL,
            "original_answer",
            "extraction_status",
            "core_only_answer",
            "auto_valid",
            "pipeline_status",
        ]
    ].copy()


    # These are intentionally blank.
    # Human validators must NOT see reference answers,
    # human factuality labels, or p1 outcomes.
    human_audit["human_valid"] = ""
    human_audit["human_final_status"] = ""
    human_audit["human_notes"] = ""


    human_audit.to_csv(
        HUMAN_AUDIT_CSV,
        index=False,
    )


    # =============================================================================
    # Summary and safety checks
    # =============================================================================

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)


    print("\nExtraction statuses:")

    print(
        phase["extraction_status"]
        .fillna("")
        .replace("", "PIPELINE_FAILED")
        .value_counts(
            dropna=False
        )
    )


    print("\nPipeline statuses:")

    print(
        phase["pipeline_status"]
        .value_counts(
            dropna=False
        )
    )


    failed = phase.loc[
        phase["pipeline_status"] != "OK"
    ]


    if len(failed):

        print(
            f"\nWARNING: {len(failed)} rows failed."
        )

        print(
            "Do not continue to human validation "
            "until these rows have been rerun successfully."
        )

        display(
            failed[
                [
                    ID_COL,
                    QUESTION_COL,
                    "original_answer",
                    "pipeline_error",
                ]
            ]
        )

    else:

        # Strong mechanical check:
        # ALREADY_CORE and NO_CLEAR must be completely unchanged.
        unchanged = phase[
            "extraction_status"
        ].isin(
            [
                "ALREADY_CORE",
                "NO_CLEAR_CORE_ANSWER",
            ]
        )

        assert (
            phase.loc[
                unchanged,
                "core_only_answer",
            ].astype(str).values

            ==

            phase.loc[
                unchanged,
                "original_answer",
            ].astype(str).values
        ).all()


        print(
            "\nAll rows passed mechanical validation."
        )

        print(
            "Next step: blinded human validation."
        )

        print(
            "Human validation file:",
            HUMAN_AUDIT_CSV,
        )

In [ ]:
# =============================================================================
# CELL B10b: GPT-5-mini P1 — ORIGINAL vs CORE-ONLY
#
# Same paired experiment as before, but GPT-5-mini/P1 evaluation follows
# the B4 implementation:
#   - reasoning effort = low
#   - max_output_tokens = 2500
#   - B4 user-message format
#   - JSON label + confidence parsing
#   - 2 attempts
#   - 4 workers
# =============================================================================

import os
import re
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from tqdm import tqdm
from openai import OpenAI
from IPython.display import display

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

if not RUN_B10B:
    print("Skipped: RUN_B10B is False")

else:
    # =============================================================================
    # CONFIG
    # =============================================================================

    P1_MODEL = "gpt-5-mini"

    P1_PROMPT = r"""
You are a hallucination auditor for open-domain QA.

Your task: decide if a model answer contains a hallucination by comparing
it against the reference answer.

DEFINITION
A hallucination is any claim that is not supported by, or contradicted by,
the reference answer.

Use the reference answer as ground truth. Do not use external knowledge.

OUTPUT FORMAT (STRICT JSON ONLY)
Return ONLY valid JSON with exactly these two fields:
{"label": 0 or 1, "confidence": <float between 0 and 1>}

Rules:
- Do not include markdown, code fences, or extra text.
- Do not include a reasoning field.
- confidence must be a numeric value (not a string).
""".strip()


    MAX_WORKERS = 4
    MAX_ATTEMPTS = 2
    MAX_OUTPUT_TOKENS = 2500
    REASONING_EFFORT = "low"

    SHUFFLE_SEED = 42

    BOOTSTRAP_N = 10_000
    BOOTSTRAP_SEED = 42

    QUALITATIVE_N = 10
    QUALITATIVE_SEED = 42


    # =============================================================================
    # CORE EXTRACTION FILE
    #
    # Usually leave this as None.
    # If several core_extractions.csv files exist, set the exact path here.
    # =============================================================================

    CORE_CSV = None


    if CORE_CSV is None:

        matches = list(
            (AUDIT_DIR / "elaboration_core_only")
            .glob("*/human_validation_template_validated.csv")
        )

        if len(matches) == 0:
            raise FileNotFoundError(
                "No core_extractions.csv found under "
                f"{AUDIT_DIR / 'elaboration_core_only'}"
            )

        if len(matches) > 1:
            print("Found several core extraction files:")
            for p in matches:
                print("  ", p)

            raise RuntimeError(
                "Set CORE_CSV at the top of this cell to the file you want."
            )

        CORE_CSV = matches[0]

    else:
        CORE_CSV = Path(CORE_CSV)


    print("Core extraction:", CORE_CSV)


    # =============================================================================
    # OUTPUT PATHS
    # =============================================================================

    OUT_DIR = (
        CORE_CSV.parent
        / "p1_original_vs_core_b4matched"
    )

    OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    JOB_CSV = OUT_DIR / "p1_jobs.csv"
    PAIRED_CSV = OUT_DIR / "paired_results.csv"
    EXAMPLES_CSV = OUT_DIR / "fp_to_tn_random5.csv"


    # =============================================================================
    # LOAD + HUMAN-VALIDATION FILTER
    # =============================================================================

    df = pd.read_csv(CORE_CSV)

    required = [
        "item_id",
        "question",
        "original_answer",
        "core_only_answer",
        "human_valid",
    ]

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise KeyError(
            f"Missing columns: {missing}"
        )

    if df["item_id"].duplicated().any():
        raise ValueError(
            "Duplicate item_id values."
        )


    # ---------------------------------------------------------------------------
    # Parse human_valid robustly
    # ---------------------------------------------------------------------------

    human_valid_raw = (
        df["human_valid"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    human_valid_map = {
        "1": 1,
        "1.0": 1,
        "true": 1,
        "0": 0,
        "0.0": 0,
        "false": 0,
    }

    df["human_valid"] = human_valid_raw.map(
        human_valid_map
    )

    if df["human_valid"].isna().any():

        bad = df.loc[
            df["human_valid"].isna(),
            ["item_id", "human_valid"],
        ]

        display(bad)

        raise ValueError(
            "human_valid contains missing or invalid values. "
            "All rows must be validated as 0 or 1."
        )


    df["human_valid"] = (
        df["human_valid"].astype(int)
    )


    # ---------------------------------------------------------------------------
    # Exclude failed human validations
    # ---------------------------------------------------------------------------

    n_before = len(df)

    excluded = df[
        df["human_valid"] == 0
    ].copy()

    df = df[
        df["human_valid"] == 1
    ].copy()

    n_after = len(df)


    print("Human-validation filter:")
    print(f"  Before:   {n_before}")
    print(f"  Excluded: {len(excluded)}")
    print(f"  Included: {n_after}")

    if len(excluded):
        print("\nExcluded item_ids:")
        for item_id in excluded["item_id"]:
            print("  ", item_id)

    print("\nItems entering P1 analysis:", len(df))


    # =============================================================================
    # LOAD REFERENCES FROM MASTER
    # =============================================================================

    master = pd.read_csv(MASTER_PATH)

    REFERENCE_COL = "ref_str"

    df = df.merge(
        master[
            [
                "item_id",
                REFERENCE_COL,
            ]
        ],
        on="item_id",
        how="left",
        validate="one_to_one",
    )

    if df[REFERENCE_COL].isna().any():
        raise ValueError(
            "Missing reference answers after merge."
        )

    print("Reference column:", REFERENCE_COL)


    # =============================================================================
    # BUILD ORIGINAL + CORE JOBS
    # =============================================================================

    jobs = []

    for _, row in df.iterrows():

        common = {
            "item_id": row["item_id"],
            "question": str(row["question"]),
            "reference": str(row[REFERENCE_COL]),
        }

        jobs.append(
            {
                **common,
                "variant": "original",
                "answer": str(row["original_answer"]),
            }
        )

        jobs.append(
            {
                **common,
                "variant": "core",
                "answer": str(row["core_only_answer"]),
            }
        )


    jobs = pd.DataFrame(jobs)

    jobs["job_id"] = (
        jobs["item_id"].astype(str)
        + "::"
        + jobs["variant"]
    )

    jobs = jobs.sample(
        frac=1,
        random_state=SHUFFLE_SEED,
    ).reset_index(drop=True)

    print("Judge calls:", len(jobs))


    # =============================================================================
    # B4 USER-MESSAGE FORMAT
    # =============================================================================

    def build_user_message(question, answer, reference):

        refs = [
            r.strip()
            for r in str(reference or "").split("|")
            if r.strip()
        ]

        refs_text = (
            "\n".join(f"- {r}" for r in refs)
            if refs
            else "- "
        )

        return (
            f"QUESTION: {question}\n"
            f"MODEL_ANSWER: {answer}\n"
            f"REFERENCE_ANSWERS (any one is acceptable):\n"
            f"{refs_text}"
        )


    # =============================================================================
    # B4 JSON PARSER
    # =============================================================================

    def parse_json_label(text):

        if not isinstance(text, str):
            return -1, -1.0, "", "non-string response"

        raw_text = text
        s = text.strip()

        # tolerate accidental markdown fences
        s = re.sub(
            r"^```(?:json)?\s*",
            "",
            s,
            flags=re.IGNORECASE,
        )

        s = re.sub(
            r"\s*```$",
            "",
            s,
        )

        # Try complete response as JSON
        try:

            obj = json.loads(s)

            label = int(
                obj.get("label", -1)
            )

            conf_raw = obj.get(
                "confidence",
                -1,
            )

            if isinstance(conf_raw, str):
                m = re.match(
                    r"([\d.]+)",
                    conf_raw.strip(),
                )
                conf_raw = (
                    float(m.group(1))
                    if m
                    else -1.0
                )

            conf = float(conf_raw)

            if conf > 1.0:
                conf = conf / 100.0

            if (
                label in (0, 1)
                and 0.0 <= conf <= 1.0
            ):
                return (
                    label,
                    conf,
                    raw_text,
                    "",
                )

        except Exception:
            pass

        # Try finding JSON inside extra text
        for match in re.finditer(
            r"\{[^{}]+\}",
            s,
        ):

            try:

                obj = json.loads(
                    match.group()
                )

                label = int(
                    obj.get("label", -1)
                )

                conf_raw = obj.get(
                    "confidence",
                    -1,
                )

                if isinstance(conf_raw, str):
                    m = re.match(
                        r"([\d.]+)",
                        conf_raw.strip(),
                    )
                    conf_raw = (
                        float(m.group(1))
                        if m
                        else -1.0
                    )

                conf = float(conf_raw)

                if conf > 1.0:
                    conf = conf / 100.0

                if (
                    label in (0, 1)
                    and 0.0 <= conf <= 1.0
                ):
                    return (
                        label,
                        conf,
                        raw_text,
                        "",
                    )

            except Exception:
                continue

        return (
            -1,
            -1.0,
            raw_text,
            "could not parse valid JSON label/confidence",
        )


    # =============================================================================
    # OPENAI
    # =============================================================================

    client = OpenAI(
        api_key=os.environ["OPENAI_API_KEY"]
    )


    def create_response(user_message):
        """
    B4-style GPT-5-mini call.
    """

        try:

            return client.responses.create(
                model=P1_MODEL,
                reasoning={
                    "effort": REASONING_EFFORT
                },
                input=(
                    P1_PROMPT
                    + "\n\n"
                    + user_message
                ),
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )

        except Exception as e:

            msg = str(e).lower()

            # Same fallback as B4:
            # if reasoning parameter is unsupported,
            # retry without it.
            if (
                "reasoning" not in msg
                and "unsupported" not in msg
                and "unknown parameter" not in msg
            ):
                raise

            return client.responses.create(
                model=P1_MODEL,
                input=(
                    P1_PROMPT
                    + "\n\n"
                    + user_message
                ),
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )


    # =============================================================================
    # ONE JUDGE CALL
    # =============================================================================

    def judge_one(row):

        user_message = build_user_message(
            question=row["question"],
            answer=row["answer"],
            reference=row["reference"],
        )

        last_error = ""

        for attempt in range(MAX_ATTEMPTS):

            try:

                response = create_response(
                    user_message
                )

                raw = (
                    response.output_text
                    or ""
                ).strip()

                (
                    label,
                    confidence,
                    raw_text,
                    parse_error,
                ) = parse_json_label(raw)

                if label in (0, 1):

                    return {
                        "job_id": row["job_id"],
                        "item_id": row["item_id"],
                        "variant": row["variant"],
                        "p1_label": label,
                        "p1_confidence": confidence,
                        "raw_output": raw_text,
                        "response_id": response.id,
                        "model_returned": response.model,
                        "status": "OK",
                        "error": "",
                    }

                last_error = (
                    parse_error
                    or "invalid label"
                )

                time.sleep(
                    1.0 + attempt
                )

            except Exception as e:

                last_error = repr(e)

                wait = (
                    2.0
                    * (attempt + 1)
                )

                if (
                    hasattr(e, "status_code")
                    and getattr(e, "status_code") == 429
                ):
                    wait = max(
                        wait,
                        10.0,
                    )

                time.sleep(wait)

        return {
            "job_id": row["job_id"],
            "item_id": row["item_id"],
            "variant": row["variant"],
            "p1_label": -1,
            "p1_confidence": -1.0,
            "raw_output": "",
            "response_id": "",
            "model_returned": "",
            "status": "FAILED",
            "error": last_error,
        }


    # =============================================================================
    # RUN
    # =============================================================================

    results = []

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as pool:

        futures = [
            pool.submit(
                judge_one,
                row,
            )
            for _, row in jobs.iterrows()
        ]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="GPT-5-mini P1 original/core",
        ):

            results.append(
                future.result()
            )


    results = pd.DataFrame(results)

    results.to_csv(
        JOB_CSV,
        index=False,
    )


    # =============================================================================
    # CHECK FAILURES
    # =============================================================================

    failed = results[
        results["status"] != "OK"
    ]

    if len(failed):

        display(failed)

        raise RuntimeError(
            f"{len(failed)} judge calls failed."
        )


    print("✓ All judge calls completed.")


    # =============================================================================
    # PAIR ORIGINAL AND CORE
    # =============================================================================

    labels = (
        results
        .pivot(
            index="item_id",
            columns="variant",
            values="p1_label",
        )
        .reset_index()
        .rename(
            columns={
                "original": "p1_original",
                "core": "p1_core",
            }
        )
    )


    confidences = (
        results
        .pivot(
            index="item_id",
            columns="variant",
            values="p1_confidence",
        )
        .reset_index()
        .rename(
            columns={
                "original": "p1_original_confidence",
                "core": "p1_core_confidence",
            }
        )
    )


    paired = df[
        [
            "item_id",
            "question",
            "original_answer",
            "core_only_answer",
        ]
    ].merge(
        labels,
        on="item_id",
        how="left",
        validate="one_to_one",
    ).merge(
        confidences,
        on="item_id",
        how="left",
        validate="one_to_one",
    )


    paired["p1_original"] = (
        paired["p1_original"].astype(int)
    )

    paired["p1_core"] = (
        paired["p1_core"].astype(int)
    )


    # =============================================================================
    # TRANSITIONS
    #
    # Human label = 0 in this experiment:
    #
    # 1 -> 0 = FP -> TN
    # 0 -> 1 = TN -> FP
    # =============================================================================

    def transition(row):

        a = row["p1_original"]
        b = row["p1_core"]

        if a == 1 and b == 0:
            return "FP_TO_TN"

        if a == 0 and b == 1:
            return "TN_TO_FP"

        if a == 1 and b == 1:
            return "FP_STAYS_FP"

        return "TN_STAYS_TN"


    paired["transition"] = paired.apply(
        transition,
        axis=1,
    )

    paired.to_csv(
        PAIRED_CSV,
        index=False,
    )


    counts = (
        paired["transition"]
        .value_counts()
        .reindex(
            [
                "FP_TO_TN",
                "TN_TO_FP",
                "FP_STAYS_FP",
                "TN_STAYS_TN",
            ],
            fill_value=0,
        )
    )


    # =============================================================================
    # SUMMARY
    # =============================================================================

    n = len(paired)

    original_fp = int(
        paired["p1_original"].sum()
    )

    core_fp = int(
        paired["p1_core"].sum()
    )

    original_fpr = original_fp / n
    core_fpr = core_fp / n

    difference = (
        original_fpr
        - core_fpr
    )


    b = int(
        counts["FP_TO_TN"]
    )

    c = int(
        counts["TN_TO_FP"]
    )


    # Exact McNemar
    if b + c > 0:

        mcnemar_p = binomtest(
            b,
            b + c,
            p=0.5,
            alternative="two-sided",
        ).pvalue

    else:

        mcnemar_p = 1.0


    # =============================================================================
    # PAIRED BOOTSTRAP CI
    # =============================================================================

    rng = np.random.default_rng(
        BOOTSTRAP_SEED
    )

    orig = paired[
        "p1_original"
    ].to_numpy()

    core = paired[
        "p1_core"
    ].to_numpy()

    boot = []

    for _ in range(BOOTSTRAP_N):

        idx = rng.integers(
            0,
            n,
            n,
        )

        boot.append(
            orig[idx].mean()
            - core[idx].mean()
        )


    ci_low, ci_high = np.quantile(
        boot,
        [0.025, 0.975],
    )


    # =============================================================================
    # RESULTS
    # =============================================================================

    print("\n" + "=" * 70)
    print("ORIGINAL vs CORE-ONLY — GPT-5-mini P1")
    print("=" * 70)

    print(f"N: {n}")

    print(
        f"\nOriginal FP: "
        f"{original_fp}/{n} "
        f"({original_fpr:.3f})"
    )

    print(
        f"Core FP:     "
        f"{core_fp}/{n} "
        f"({core_fpr:.3f})"
    )

    print(
        f"\nFPR reduction: "
        f"{difference:+.3f}"
    )

    print(
        f"95% bootstrap CI: "
        f"[{ci_low:+.3f}, {ci_high:+.3f}]"
    )

    print("\nTransitions:")

    display(
        counts
        .rename("count")
        .to_frame()
    )

    print(
        f"\nExact McNemar p = "
        f"{mcnemar_p:.6g}"
    )


    # =============================================================================
    # RANDOM 5 FP -> TN EXAMPLES
    # =============================================================================

    flips = paired[
        paired["transition"] == "FP_TO_TN"
    ].copy()

    sample_n = min(
        QUALITATIVE_N,
        len(flips),
    )

    examples = (
        flips.sample(
            n=sample_n,
            random_state=QUALITATIVE_SEED,
        )
        if sample_n
        else flips
    )


    examples = examples[
        [
            "item_id",
            "question",
            "original_answer",
            "core_only_answer",
            "p1_original",
            "p1_core",
            "p1_original_confidence",
            "p1_core_confidence",
        ]
    ]


    examples.to_csv(
        EXAMPLES_CSV,
        index=False,
    )


    print(
        f"\nRandom FP -> TN examples "
        f"(n={sample_n}, seed={QUALITATIVE_SEED}):"
    )

    display(examples)


    print("\nSaved:")
    print("Jobs:    ", JOB_CSV)
    print("Paired:  ", PAIRED_CSV)
    print("Examples:", EXAMPLES_CSV)

**SECTION C — Paper analyses**

In [ ]:
# =============================================================================
# CELL C0: Release master integrity check
#
# Purpose:
#   Verify the release master end to end, from the raw columns rather than
#   from any stored summary. Everything here is recomputed: agreement metrics
#   are derived from the label columns, thresholds are re-applied to the score
#   columns, and the results are checked against the manuscript's own numbers.
#
#   This cell is the reason to trust the published file. It makes no API calls,
#   loads no models, and writes nothing. Run it before shipping, and again
#   after any change to the master.
#
# Checks:
#   A. Shape, row identity, no duplicate item_id
#   B. Human labels: primary complete 0/1, secondary uses -1 outside the
#      double-annotated subset
#   C. Inter-annotator agreement reproduces the manuscript
#   D. Every judge label column complete 0/1
#   E. Every metric family carries the full ten-column pattern
#   F. Thresholds re-apply: recomputing labels from scores and tau reproduces
#      the stored label columns
#   G. CV folds are a valid partition
#   H. Agreement reproduces the manuscript for judges, lexical metrics,
#      NLI and the new metrics
#   I. Macro-averaged oracle kappa places the new metrics inside the existing
#      range
#
# Expected values are stated inline. If the master changes legitimately, the
# expectations must be updated deliberately — that is the point.
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

CELL_NAME = "CELL C0: Release master integrity check"

# =============================================================================
# Expectations
# =============================================================================

N_ROWS = 900
N_IAA_SUBSET = 300

HUMAN_COL = "annotator1_label"
SECOND_HUMAN_COL = "annotator2_label"
SECOND_HUMAN_SENTINEL = -1

GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]
DATASETS = ["hotpotqa", "triviaqa", "truthfulqa"]

N_CV_FOLDS = 3

JUDGE_PREFIXES = [
    "gpt_5_mini", "gpt_5_nano", "gpt_5_4", "claude_opus_4_7",
    "gemma_2_9b_local", "qwen2_5_7b_local", "llama_3_8b_local",
]
PROMPTS = ["p1", "p2", "p3"]

# Metric families that must carry the full ten-column pattern.
METRIC_FAMILIES = [
    ("rougeL", "rougeL"),
    ("bertscore", "bertscore_f1"),
    ("meteor", "meteor"),
    ("bartscore_r2h", "bartscore_r2h"),
    ("bartscore_h2r", "bartscore_h2r"),
]

METRIC_SUFFIXES = [
    "label_oracle", "label_cv",
    "tau_oracle", "tau_cv_mean", "tau_cv_min", "tau_cv_max",
    "cv_fold", "cv_kappa_mean_fold", "cv_kappa_oof",
]

# Inter-annotator agreement, Appendix A.
EXPECTED_IAA = {
    "ALL": (0.808, 271, 300),
    "hotpotqa": (0.832, 96, 102),
    "triviaqa": (0.796, 90, 99),
    "truthfulqa": (0.690, 85, 99),
}

# Judge agreement, main results table. (judge, generator, prompt) -> kappa.
EXPECTED_JUDGE_KAPPA = {
    ("gpt_5_mini", "llama3-8b"): (0.530, 0.756, 0.744),
    ("gpt_5_mini", "gemma-2-9b"): (0.498, 0.770, 0.759),
    ("gpt_5_mini", "mistral-7b"): (0.194, 0.626, 0.646),
    ("gpt_5_nano", "llama3-8b"): (0.536, 0.655, 0.676),
    ("gpt_5_nano", "gemma-2-9b"): (0.592, 0.686, 0.641),
    ("gpt_5_nano", "mistral-7b"): (0.389, 0.601, 0.581),
    ("gpt_5_4", "llama3-8b"): (0.306, 0.779, 0.740),
    ("gpt_5_4", "gemma-2-9b"): (0.240, 0.766, 0.774),
    ("gpt_5_4", "mistral-7b"): (0.021, 0.619, 0.672),
    ("claude_opus_4_7", "llama3-8b"): (0.712, 0.764, 0.738),
    ("claude_opus_4_7", "gemma-2-9b"): (0.740, 0.764, 0.737),
    ("claude_opus_4_7", "mistral-7b"): (0.693, 0.706, 0.673),
    ("gemma_2_9b_local", "llama3-8b"): (0.567, 0.591, 0.557),
    ("gemma_2_9b_local", "gemma-2-9b"): (0.463, 0.623, 0.628),
    ("gemma_2_9b_local", "mistral-7b"): (0.378, 0.496, 0.455),
    ("qwen2_5_7b_local", "llama3-8b"): (0.313, 0.483, 0.472),
    ("qwen2_5_7b_local", "gemma-2-9b"): (0.376, 0.607, 0.559),
    ("qwen2_5_7b_local", "mistral-7b"): (0.213, 0.428, 0.452),
    ("llama_3_8b_local", "llama3-8b"): (0.370, 0.303, 0.255),
    ("llama_3_8b_local", "gemma-2-9b"): (0.227, 0.470, 0.384),
    ("llama_3_8b_local", "mistral-7b"): (0.278, 0.218, 0.185),
}

# Non-judge label columns, per generator, in GEN_ORDER.
EXPECTED_METRIC_KAPPA = {
    "rougeL_label_oracle": (0.403, 0.483, 0.412),
    "rougeL_label_cv": (0.353, 0.473, 0.385),
    "bertscore_label_oracle": (0.387, 0.486, 0.415),
    "bertscore_label_cv": (0.318, 0.486, 0.394),
    "nli_label_strict": (0.053, 0.025, 0.049),
    "nli_label_best_tau": (0.380, 0.217, 0.395),
    "meteor_label_oracle": (0.423, 0.380, 0.410),
    "meteor_label_cv": (0.354, 0.346, 0.384),
    "bartscore_r2h_label_oracle": (0.289, 0.402, 0.314),
    "bartscore_r2h_label_cv": (0.262, 0.336, 0.264),
    "bartscore_h2r_label_oracle": (0.379, 0.494, 0.360),
    "bartscore_h2r_label_cv": (0.371, 0.449, 0.322),
}

# Oracle-threshold columns, and which are the
# additions whose macro kappa must land inside the pre-existing range.
ORACLE_COLS_EXISTING = [
    "rougeL_label_oracle", "bertscore_label_oracle", "nli_label_best_tau",
]
ORACLE_COLS_ADDED = [
    "meteor_label_oracle",
    "bartscore_r2h_label_oracle",
    "bartscore_h2r_label_oracle",
]

TOL = 0.0006  # values are reported to three decimals

# =============================================================================
# Harness
# =============================================================================

failures = []
warnings = []
n_checks = 0


def check(name, condition, detail=""):
    global n_checks
    n_checks += 1

    if condition:
        return True

    failures.append(f"{name}: {detail}" if detail else name)
    return False


def close(a, b, tol=TOL):
    return abs(float(a) - float(b)) <= tol


def kappa_of(y, p):
    return float(cohen_kappa_score(np.asarray(y, int), np.asarray(p, int),
                                   labels=[0, 1]))


master_path = globals().get("MASTER_PATH", None)

if master_path is None:
    raise RuntimeError("MASTER_PATH is not defined. Run the configuration cell.")

df = pd.read_csv(master_path)

print("=" * 78)
print("RELEASE MASTER INTEGRITY CHECK")
print("=" * 78)
print("File: ", master_path)
print("Shape:", df.shape)

# -----------------------------------------------------------------------------
# A. Structure
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("A. Structure")
print("-" * 78)

check("A1 row count", len(df) == N_ROWS, f"found {len(df)}")
check("A2 item_id unique", df["item_id"].is_unique,
      f"{len(df) - df['item_id'].nunique()} duplicates")
check("A3 no duplicate columns", len(set(df.columns)) == len(df.columns))

gen_counts = df["model"].value_counts().to_dict()
check("A4 generators present", set(GEN_ORDER) <= set(gen_counts),
      f"found {sorted(gen_counts)}")
check("A5 generators balanced", all(gen_counts.get(g) == 300 for g in GEN_ORDER),
      f"{gen_counts}")

ds_counts = df["dataset"].value_counts().to_dict()
check("A6 datasets present", set(DATASETS) <= set(ds_counts),
      f"found {sorted(ds_counts)}")

print(f"  rows={len(df)}  cols={df.shape[1]}  "
      f"generators={ {g: gen_counts.get(g) for g in GEN_ORDER} }")
print(f"  datasets={ {d: ds_counts.get(d) for d in DATASETS} }")

# -----------------------------------------------------------------------------
# B. Human labels
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("B. Human labels")
print("-" * 78)

primary = pd.to_numeric(df[HUMAN_COL], errors="coerce")
check("B1 primary labels complete 0/1", bool(primary.isin([0, 1]).all()),
      f"{int((~primary.isin([0, 1])).sum())} invalid")

second = pd.to_numeric(df[SECOND_HUMAN_COL], errors="coerce")
valid_second = second.isin([0, 1])
sentinel_second = second == SECOND_HUMAN_SENTINEL

check("B2 secondary is 0/1 or sentinel",
      bool((valid_second | sentinel_second).all()),
      f"{int((~(valid_second | sentinel_second)).sum())} other values")
check("B3 double-annotated subset size", int(valid_second.sum()) == N_IAA_SUBSET,
      f"found {int(valid_second.sum())}")

print(f"  primary: {int(primary.sum())} positive of {len(primary)}")
print(f"  secondary: {int(valid_second.sum())} annotated, "
      f"{int(sentinel_second.sum())} sentinel ({SECOND_HUMAN_SENTINEL})")

# -----------------------------------------------------------------------------
# C. Inter-annotator agreement
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("C. Inter-annotator agreement (recomputed)")
print("-" * 78)

sub = df[valid_second]

for scope, (exp_kappa, exp_agree, exp_n) in EXPECTED_IAA.items():
    part = sub if scope == "ALL" else sub[sub["dataset"] == scope]

    a = part[HUMAN_COL].astype(int).to_numpy()
    b = part[SECOND_HUMAN_COL].astype(int).to_numpy()

    k = kappa_of(a, b)
    agree = int((a == b).sum())

    ok = (close(k, exp_kappa) and agree == exp_agree and len(part) == exp_n)
    check(f"C {scope}", ok,
          f"kappa={k:.3f} agree={agree}/{len(part)}, "
          f"expected {exp_kappa:.3f} {exp_agree}/{exp_n}")

    print(f"  {scope:<12} n={len(part):>3}  kappa={k:.3f}  "
          f"raw={100 * agree / len(part):.1f}% ({agree}/{len(part)})")

# -----------------------------------------------------------------------------
# D. Judge label columns
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("D. Judge label columns")
print("-" * 78)

judge_cols = [f"label_{p}_{pr}" for p in JUDGE_PREFIXES for pr in PROMPTS]
missing_judge = [c for c in judge_cols if c not in df.columns]

check("D1 all judge columns present", not missing_judge, f"missing {missing_judge}")

incomplete_judge = [
    c for c in judge_cols
    if c in df.columns
    and not pd.to_numeric(df[c], errors="coerce").isin([0, 1]).all()
]

check("D2 judge columns complete 0/1", not incomplete_judge,
      f"incomplete {incomplete_judge}")

print(f"  {len(judge_cols) - len(missing_judge)}/{len(judge_cols)} present, "
      f"{len(incomplete_judge)} incomplete")

# -----------------------------------------------------------------------------
# E. Metric column families
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("E. Metric column families")
print("-" * 78)

for family, score_col in METRIC_FAMILIES:
    expected = [score_col] + [f"{family}_{s}" for s in METRIC_SUFFIXES]
    missing = [c for c in expected if c not in df.columns]

    check(f"E {family} complete", not missing, f"missing {missing}")
    print(f"  {family:<16} {len(expected) - len(missing)}/{len(expected)} columns")

# -----------------------------------------------------------------------------
# F. Thresholds re-apply
#
# The strongest check in this cell: the stored labels are not taken on trust
# but regenerated from the stored scores and thresholds. A mismatch means the
# labels and scores in the published file do not describe the same decision.
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("F. Threshold re-application (labels regenerated from scores)")
print("-" * 78)

for family, score_col in METRIC_FAMILIES:
    if any(c not in df.columns for c in
           [score_col, f"{family}_label_oracle", f"{family}_tau_oracle"]):
        continue

    scores = pd.to_numeric(df[score_col], errors="coerce").to_numpy(float)

    for mode, tau_col in [("oracle", f"{family}_tau_oracle"),
                          ("cv", f"{family}_tau_cv_mean")]:
        label_col = f"{family}_label_{mode}"

        if label_col not in df.columns or tau_col not in df.columns:
            continue

        stored = pd.to_numeric(df[label_col], errors="coerce").to_numpy(int)
        tau = pd.to_numeric(df[tau_col], errors="coerce").to_numpy(float)

        recomputed = (scores < tau).astype(int)
        n_diff = int((stored != recomputed).sum())

        if mode == "oracle":
            check(f"F {family} oracle re-applies", n_diff == 0,
                  f"{n_diff}/{len(df)} labels differ")
            status = "OK" if n_diff == 0 else f"{n_diff} DIFFER"
        else:
            # CV labels come from per-fold thresholds, so the stored mean tau
            # is not expected to reproduce them exactly. Report, do not fail.
            if n_diff:
                warnings.append(
                    f"{family} CV labels differ from mean-tau re-application "
                    f"on {n_diff}/{len(df)} rows (expected: per-fold thresholds)"
                )
            status = "OK" if n_diff == 0 else f"{n_diff} differ (per-fold, expected)"

        print(f"  {family:<16} {mode:<7} {status}")

# -----------------------------------------------------------------------------
# G. CV folds
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("G. CV fold assignment")
print("-" * 78)

for family, _ in METRIC_FAMILIES:
    fold_col = f"{family}_cv_fold"

    if fold_col not in df.columns:
        continue

    folds = pd.to_numeric(df[fold_col], errors="coerce")

    check(f"G {family} folds valid", bool(folds.isin(range(N_CV_FOLDS)).all()),
          f"found {sorted(folds.dropna().unique())}")

    sizes = folds.value_counts().sort_index().to_dict()
    print(f"  {family:<16} {sizes}")

# -----------------------------------------------------------------------------
# H. Agreement reproduces the manuscript
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("H. Agreement vs the manuscript (recomputed from labels)")
print("-" * 78)

n_judge_ok = 0

for (prefix, gen), expected in EXPECTED_JUDGE_KAPPA.items():
    mask = df["model"] == gen
    y = df.loc[mask, HUMAN_COL].astype(int).to_numpy()

    for i, prompt in enumerate(PROMPTS):
        col = f"label_{prefix}_{prompt}"

        if col not in df.columns:
            continue

        k = kappa_of(y, df.loc[mask, col].astype(int).to_numpy())

        if check(f"H judge {prefix}/{gen}/{prompt}", close(k, expected[i]),
                 f"kappa={k:.3f}, expected {expected[i]:.3f}"):
            n_judge_ok += 1

print(f"  judges: {n_judge_ok}/{len(EXPECTED_JUDGE_KAPPA) * len(PROMPTS)} match")

n_metric_ok = 0

for col, expected in EXPECTED_METRIC_KAPPA.items():
    if col not in df.columns:
        check(f"H metric {col}", False, "column missing")
        continue

    row_ok = True

    for i, gen in enumerate(GEN_ORDER):
        mask = df["model"] == gen
        y = df.loc[mask, HUMAN_COL].astype(int).to_numpy()
        k = kappa_of(y, df.loc[mask, col].astype(int).to_numpy())

        if not close(k, expected[i]):
            row_ok = False
            check(f"H metric {col}/{gen}", False,
                  f"kappa={k:.3f}, expected {expected[i]:.3f}")

    if row_ok:
        n_metric_ok += 1
        n_checks += 1

print(f"  metrics: {n_metric_ok}/{len(EXPECTED_METRIC_KAPPA)} columns match")

# -----------------------------------------------------------------------------
# I. Macro-averaged comparability
#
# METEOR and both BARTScore directions land inside the
# agreement range already spanned by ROUGE-L, BERTScore and threshold-tuned
# NLI. Averaging happens before rounding, which is why the manuscript reports
# METEOR as 0.405 rather than 0.404.
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("I. Macro-averaged oracle agreement")
print("-" * 78)


def macro_kappa(col):
    values = []

    for gen in GEN_ORDER:
        mask = df["model"] == gen
        values.append(kappa_of(
            df.loc[mask, HUMAN_COL].astype(int).to_numpy(),
            df.loc[mask, col].astype(int).to_numpy(),
        ))

    return float(np.mean(values))


existing = {c: macro_kappa(c) for c in ORACLE_COLS_EXISTING if c in df.columns}
added = {c: macro_kappa(c) for c in ORACLE_COLS_ADDED if c in df.columns}

for col, value in sorted({**existing, **added}.items(),
                         key=lambda kv: -kv[1]):
    tag = "new     " if col in added else "existing"
    print(f"  {tag}  {col:<30} {value:.3f}")

if existing and added:
    lo_e, hi_e = min(existing.values()), max(existing.values())
    lo_a, hi_a = min(added.values()), max(added.values())

    print(f"\n  existing span {lo_e:.3f}--{hi_e:.3f}")
    print(f"  added span    {lo_a:.3f}--{hi_a:.3f}")

    check("I added metrics inside existing range",
          lo_a >= lo_e - TOL and hi_a <= hi_e + TOL,
          f"added {lo_a:.3f}--{hi_a:.3f} vs existing {lo_e:.3f}--{hi_e:.3f}")

# =============================================================================
# Verdict
# =============================================================================
print("\n" + "=" * 78)

if warnings:
    print("NOTES")
    print("=" * 78)
    for w in warnings:
        print("  •", w)
    print()

if failures:
    print(f"FAILED — {len(failures)} of {n_checks} checks")
    print("=" * 78)
    for f in failures:
        print("  ✗", f)
    raise AssertionError(
        f"{len(failures)} integrity checks failed; see the list above."
    )

print(f"PASSED — all {n_checks} checks")
print("=" * 78)
print("Every number was recomputed from the raw columns, not read from a")
print("stored summary. The file is consistent with the manuscript.")

In [ ]:
# =============================================================================
# CELL C1: Recompute inter-annotator agreement table from release master CSV
#
# Purpose:
#   Recompute independent inter-annotator agreement on the doubly annotated
#   subset: raw agreement, Cohen's kappa, and question-level bootstrap
#   confidence intervals, per dataset and overall. The three generator
#   responses belonging to each question are kept together in every resample.
#
# Source of truth:
#   MASTER_PATH = data/human_judge_release_master.csv
#
# Uses:
#   annotator1_label   complete primary label set (900 items, values 0/1)
#   annotator2_label   overlap subset only; items NOT double-annotated are
#                      encoded with the sentinel value -1, not left blank.
#                      Always select the overlap with .isin([0, 1]) rather
#                      than .notna(), which would admit all 900 rows.
#
# Outputs:
#   tables/table_iaa_recomputed.tex                      (tab:iaa)
#   audits/iaa/table_iaa_recomputed.tex                  (audit copy)
#   audits/iaa/inter_annotator_agreement_recomputed.csv
#
# Paper label:
#   tab:iaa
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

RELEASE_CSV = MASTER_PATH

OUT_DIR = AUDIT_DIR / "iaa"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUT_DIR / "inter_annotator_agreement_recomputed.csv"
OUT_TEX_AUDIT = OUT_DIR / "table_iaa_recomputed.tex"
OUT_TEX = TABLE_DIR / "table_iaa_recomputed.tex"

N_BOOT = 1000
BOOT_SEED = 42

# Sentinel used in the release master for items annotator 2 did not label.
A2_MISSING_SENTINEL = -1

DATASET_ORDER = ["triviaqa", "truthfulqa", "hotpotqa"]
DATASET_DISPLAY = {
    "triviaqa": "TriviaQA",
    "truthfulqa": "TruthfulQA",
    "hotpotqa": "HotpotQA",
    "overall": "Overall",
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def safe_kappa(y1, y2):
    """Compute Cohen's kappa, returning np.nan if undefined."""
    y1 = np.asarray(y1).astype(int)
    y2 = np.asarray(y2).astype(int)

    if len(y1) == 0:
        return np.nan

    # Kappa can become undefined in degenerate bootstrap samples.
    try:
        k = cohen_kappa_score(y1, y2, labels=[0, 1])
        return float(k)
    except Exception:
        return np.nan

def bootstrap_kappa_ci(df_block, n_boot=1000, seed=42):
    """Question-level bootstrap CI for Cohen's kappa.

    Questions are resampled with replacement. All three generator responses
    belonging to a sampled question are kept together.
    """
    rng = np.random.default_rng(seed)

    work = df_block[
        ["dataset", "qid", "annotator1_label", "annotator2_label"]
    ].reset_index(drop=True)

    y1 = work["annotator1_label"].astype(int).to_numpy()
    y2 = work["annotator2_label"].astype(int).to_numpy()

    # Use dataset and qid together as the question identifier. This prevents
    # accidental collisions if two datasets use the same raw qid value.
    question_rows = [
        np.asarray(idx, dtype=int)
        for idx in work.groupby(
            ["dataset", "qid"], sort=False, dropna=False
        ).indices.values()
    ]

    n_questions = len(question_rows)
    if n_questions == 0:
        return np.nan, np.nan, 0

    vals = []

    for _ in range(n_boot):
        # Draw questions rather than individual answer rows.
        sampled_questions = rng.integers(
            0, n_questions, size=n_questions
        )

        # If a question is sampled, include all of its generator responses.
        idx = np.concatenate(
            [question_rows[i] for i in sampled_questions]
        )

        k = safe_kappa(y1[idx], y2[idx])
        if np.isfinite(k):
            vals.append(k)

    vals = np.asarray(vals)

    if len(vals) == 0:
        return np.nan, np.nan, 0

    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi), int(len(vals))


def compute_iaa_block(df, name):
    """Compute n, disagreement count, agreement, kappa, and bootstrap CI."""
    y1 = df["annotator1_label"].astype(int).to_numpy()
    y2 = df["annotator2_label"].astype(int).to_numpy()

    n = len(df)
    agree_count = int((y1 == y2).sum())
    disag_count = int(n - agree_count)
    agreement = agree_count / n if n else np.nan
    kappa = safe_kappa(y1, y2)
    ci_low, ci_high, n_boot_valid = bootstrap_kappa_ci(
        df, n_boot=N_BOOT, seed=BOOT_SEED
    )

    return {
        "dataset": name,
        "n_questions": int(df[["dataset", "qid"]].drop_duplicates().shape[0]),
        "n": n,
        "disagreements": disag_count,
        "agreement": agreement,
        "kappa": kappa,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "n_boot_valid": n_boot_valid,
    }


# ── Load and validate ─────────────────────────────────────────────────────────
df = pd.read_csv(RELEASE_CSV)

required = [
    "item_id",
    "qid",
    "dataset",
    "model",
    "annotator1_label",
    "annotator2_label",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns from release CSV: {missing}")

# annotator2_label must contain only 0, 1, or the missing sentinel.
allowed_a2 = {0, 1, A2_MISSING_SENTINEL}
observed_a2 = set(pd.to_numeric(df["annotator2_label"], errors="coerce").dropna().astype(int))
unexpected_a2 = observed_a2 - allowed_a2
if unexpected_a2:
    raise ValueError(
        f"Unexpected annotator2_label values: {sorted(unexpected_a2)}. "
        f"Expected only {sorted(allowed_a2)} (sentinel {A2_MISSING_SENTINEL} = not annotated)."
    )

# Keep only doubly annotated rows.
# NOTE: .isin([0, 1]) is required here. The sentinel -1 is not NaN, so a
# .notna() filter would silently keep all 900 rows.
df2 = df[df["annotator2_label"].isin([0, 1])].copy()

# Keep only valid primary labels.
df2 = df2[df2["annotator1_label"].isin([0, 1])].copy()

# The overlap was sampled at the question level: 100 questions, each with one
# answer from each of the three generators. Validate that clustering by
# question is well defined before running the bootstrap.
if df2[["dataset", "qid"]].isna().any().any():
    raise ValueError("Missing dataset or qid values in the doubly annotated subset.")

question_groups = df2.groupby(["dataset", "qid"], dropna=False)
question_sizes = question_groups.size()
question_model_counts = question_groups["model"].nunique()

if len(question_sizes) != 100:
    raise AssertionError(
        f"Expected 100 doubly annotated questions, found {len(question_sizes)}."
    )

if not question_sizes.eq(3).all():
    bad_questions = question_sizes[question_sizes != 3]
    raise AssertionError(
        "Each doubly annotated question must have exactly three response rows. "
        f"Unexpected question sizes:\n{bad_questions}"
    )

if not question_model_counts.eq(3).all():
    bad_questions = question_model_counts[question_model_counts != 3]
    raise AssertionError(
        "Each doubly annotated question must contain three distinct generator "
        f"models. Unexpected model counts:\n{bad_questions}"
    )

print("=" * 100)
print("INTER-ANNOTATOR AGREEMENT INPUT")
print("=" * 100)
print("Release CSV:", RELEASE_CSV)
print("Full release shape:", df.shape)
print("Doubly annotated shape:", df2.shape)
print("Doubly annotated questions:", len(question_sizes))
print("Responses per question:", sorted(question_sizes.unique()))
print(f"Rows with annotator2_label == {A2_MISSING_SENTINEL} (not annotated):",
      int((df["annotator2_label"] == A2_MISSING_SENTINEL).sum()))
print()
print("Doubly annotated rows by dataset:")
print(df2["dataset"].value_counts())
print()
print("Doubly annotated rows by model:")
print(df2["model"].value_counts())
print()
print("Annotator 1 label distribution on overlap:")
print(df2["annotator1_label"].value_counts().sort_index())
print()
print("Annotator 2 label distribution on overlap:")
print(df2["annotator2_label"].value_counts().sort_index())

if len(df2) != 300:
    print()
    print(f"\u26a0 WARNING: expected 300 doubly annotated rows, found {len(df2)}.")

# ── Compute table ─────────────────────────────────────────────────────────────
rows = []

for ds in DATASET_ORDER:
    sub = df2[df2["dataset"] == ds].copy()
    if len(sub) == 0:
        print(f"\u26a0 WARNING: no rows for dataset {ds}")
        continue
    rows.append(compute_iaa_block(sub, ds))

rows.append(compute_iaa_block(df2, "overall"))

iaa = pd.DataFrame(rows)

# Regression guard for the frozen release master and the documented seed.
# This detects accidental changes to the bootstrap unit, seed, row filtering,
# or source data before a table with different numbers can be emitted.
EXPECTED_ROUNDED = {
    "triviaqa":  {"n": 99,  "disagreements": 9,  "agreement": 0.909,
                  "kappa": 0.796, "ci_low": 0.618, "ci_high": 0.935},
    "truthfulqa": {"n": 99,  "disagreements": 14, "agreement": 0.859,
                   "kappa": 0.690, "ci_low": 0.544, "ci_high": 0.816},
    "hotpotqa":   {"n": 102, "disagreements": 6,  "agreement": 0.941,
                   "kappa": 0.832, "ci_low": 0.642, "ci_high": 0.968},
    "overall":    {"n": 300, "disagreements": 29, "agreement": 0.903,
                   "kappa": 0.808, "ci_low": 0.729, "ci_high": 0.879},
}

for _, result in iaa.iterrows():
    name = result["dataset"]
    expected = EXPECTED_ROUNDED[name]

    for field in ["n", "disagreements"]:
        if int(result[field]) != expected[field]:
            raise AssertionError(
                f"{name}: expected {field}={expected[field]}, "
                f"got {int(result[field])}."
            )

    for field in ["agreement", "kappa", "ci_low", "ci_high"]:
        got = round(float(result[field]), 3)
        if got != expected[field]:
            raise AssertionError(
                f"{name}: expected rounded {field}={expected[field]:.3f}, "
                f"got {got:.3f}."
            )

if not iaa["n_boot_valid"].eq(N_BOOT).all():
    raise AssertionError(
        "At least one IAA interval used fewer valid bootstrap resamples than "
        f"the documented N_BOOT={N_BOOT}."
    )

print("\u2713 QUESTION-LEVEL BOOTSTRAP REGRESSION CHECK PASSED")

# Rounded display columns
iaa_display = iaa.copy()
iaa_display["Dataset"] = iaa_display["dataset"].map(DATASET_DISPLAY)
iaa_display["Agree."] = iaa_display["agreement"].round(3)
iaa_display["kappa_3dp"] = iaa_display["kappa"].round(3)
iaa_display["CI low"] = iaa_display["ci_low"].round(3)
iaa_display["CI high"] = iaa_display["ci_high"].round(3)

print("\n" + "=" * 100)
print("RECOMPUTED IAA TABLE")
print("=" * 100)
display_cols = ["Dataset", "n", "disagreements", "Agree.",
                "kappa_3dp", "CI low", "CI high"]
print(iaa_display[display_cols].to_string(index=False))

# ── Save CSV audit ────────────────────────────────────────────────────────────
iaa.to_csv(OUT_CSV, index=False)
print("\nSaved CSV audit:", OUT_CSV)

# ── Generate LaTeX ────────────────────────────────────────────────────────────
latex = r"""\begin{table}[t]
\centering
\caption{Independent inter-annotator agreement on the 300-item subset.
Confidence intervals are 95\% question-level bootstrap intervals for Cohen's
$\kappa$ with 1,000 resamples, keeping the three generator responses to each
question together.}
\label{tab:iaa}
\small
\begin{tabular}{lrrrrrr}
\toprule
Dataset & $n$ & Disagr. & Agree. & $\kappa$ & CI low & CI high \\
\midrule
"""

for _, r in iaa.iterrows():
    ds_name = DATASET_DISPLAY.get(r["dataset"], r["dataset"])
    latex += (
        f"{ds_name:<10} & "
        f"{int(r['n']):3d} & "
        f"{int(r['disagreements']):2d} & "
        f"{r['agreement']:.3f} & "
        f"{r['kappa']:.3f} & "
        f"{r['ci_low']:.3f} & "
        f"{r['ci_high']:.3f} \\\\\n"
    )

    if r["dataset"] == "hotpotqa":
        latex += r"\midrule" + "\n"

latex += r"""\bottomrule
\end{tabular}
\end{table}
"""

print("\n" + "=" * 100)
print("INTER-ANNOTATOR AGREEMENT TABLE")
print("=" * 100)
display(iaa)

with open(OUT_TEX, "w") as f:
    f.write(latex)

# Also save audit copy of the LaTeX
with open(OUT_TEX_AUDIT, "w") as f:
    f.write(latex)

print("\nSaved table files:")
print("  Audit:", OUT_TEX_AUDIT)
print("  Table:", OUT_TEX)

# ── Structural check on the emitted LaTeX ─────────────────────────────────────
# Guards against silently emitting a table that does not match the paper:
# the column count, the row count, and the absence of supplementary metrics.

text = OUT_TEX.read_text()

n_cols = text.count("&", text.index("Dataset"), text.index(r"\\", text.index("Dataset"))) + 1
if n_cols != 7:
    raise AssertionError(f"tab:iaa: expected 7 columns, header row has {n_cols}.")

n_rows = text.count("\\\\\n")
expected_rows = len(iaa) + 1  # data rows + header row
if n_rows != expected_rows:
    raise AssertionError(
        f"tab:iaa: expected {expected_rows} LaTeX rows "
        f"({len(iaa)} data + 1 header), found {n_rows}."
    )

for forbidden in [r"\alpha", "Krippendorff", "alpha"]:
    if forbidden in text:
        raise AssertionError(
            f"tab:iaa contains '{forbidden}'. The paper table reports Cohen's "
            "kappa only; supplementary agreement metrics do not belong here."
        )

print("\u2713 LATEX STRUCTURE CHECK PASSED")

# Echo the only IAA value that must also be updated outside the generated
# appendix table.
overall = iaa.loc[iaa["dataset"] == "overall"].iloc[0]
print()
print("MAIN-TEXT IAA UPDATE")
print(
    f"Cohen's kappa={overall['kappa']:.3f}, "
    f"95% CI: [{overall['ci_low']:.3f}, {overall['ci_high']:.3f}]"
)

# ── Echo generated LaTeX for visual inspection ────────────────────────────────
print("\n" + "=" * 100)
print(f"tab:iaa: {OUT_TEX.name}  ({n_rows} rows, {len(text):,} chars)")
print("=" * 100)
print(text)

In [ ]:
# =============================================================================
# CELL C2: Generate prompt-ablation LaTeX tables from release master CSV
#
# Purpose:
#   Recompute per-judge, per-generator, per-prompt agreement with the human
#   labels and emit the two paper tables: the compact main-text table and the
#   long-form appendix audit table.
#
# Source of truth:
#   MASTER_PATH = data/human_judge_release_master.csv
#
# Uses:
#   annotator1_label
#   label_<judge>_p1 / label_<judge>_p2 / label_<judge>_p3
#
# Outputs:
#   tables/table_results_prompt_wide.tex        (tab:results_prompt)
#   tables/table_results_full_appendix.tex      (tab:results_full)
#   audits/prompt_ablation/prompt_ablation_long.csv
#
# Paper labels:
#   tab:results_prompt
#   tab:results_full
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------

RELEASE_CSV = MASTER_PATH

OUT_DIR = TABLE_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_OUT_DIR = AUDIT_DIR / "prompt_ablation"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

WIDE_OUT = OUT_DIR / "table_results_prompt_wide.tex"
LONG_OUT = OUT_DIR / "table_results_full_appendix.tex"
LONG_AUDIT_CSV = AUDIT_OUT_DIR / "prompt_ablation_long.csv"

# -------------------------------------------------------------------------
# Config
# -------------------------------------------------------------------------

PROMPT_ORDER = ["p1", "p2", "p3"]

GEN_LABELS = {
    "llama3-8b": "L",
    "gemma-2-9b": "G",
    "mistral-7b": "M",
}

REFUSAL_MARKER = r"$^a$"

# Display name, release-master judge column prefix, generator label overrides.
# Display names are paper-facing: the "Local judges" group heading carries the
# local/API distinction, so it is not repeated on every row.
JUDGES = [
    ("GPT-5-mini",      "gpt_5_mini",        {}),
    ("GPT-5-nano",      "gpt_5_nano",        {}),
    ("GPT-5.4",         "gpt_5_4",           {}),
    ("Claude Opus 4.7", "claude_opus_4_7",   {}),

    ("Gemma-2-9B",      "gemma_2_9b_local",  {"gemma-2-9b": "G (self)"}),
    ("Qwen2.5-7B",      "qwen2_5_7b_local",  {}),
    ("Llama-3-8B",      "llama_3_8b_local",  {"llama3-8b": "L (self)"}),
]


def judge_group(clean_prefix):
    """
    Group a judge by its release-master column prefix.

    Grouping deliberately keys on the column prefix rather than the display
    name, so that paper-facing display names can be changed without silently
    changing the API/local grouping.
    """
    return "Local judges" if clean_prefix.endswith("_local") else "API judges"


def compute_row(df, human_col, pred_col):
    """
    Return kappa, bias, FP, FN, FP%, n_valid, n_total.

    Rows with invalid prediction labels outside {0,1} are excluded.
    This permits explicit refusal/missing values encoded as -1.
    """
    if pred_col not in df.columns:
        raise KeyError(f"Missing prediction column: {pred_col}")

    n_total = len(df)

    yh_all = pd.to_numeric(df[human_col], errors="coerce")
    yp_all = pd.to_numeric(df[pred_col], errors="coerce")

    mask = yh_all.isin([0, 1]) & yp_all.isin([0, 1])
    n_valid = int(mask.sum())

    if n_valid < 5:
        raise ValueError(
            f"Too few valid rows for {pred_col}: valid={n_valid}/{n_total}"
        )

    yh = yh_all.loc[mask].astype(int).to_numpy()
    yp = yp_all.loc[mask].astype(int).to_numpy()

    kappa = cohen_kappa_score(yh, yp, labels=[0, 1])
    bias = yp.mean() - yh.mean()

    tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()
    fp_pct = fp / max(fp + fn, 1) * 100.0

    return {
        "kappa": float(kappa),
        "bias": float(bias),
        "fp": int(fp),
        "fn": int(fn),
        "err": int(fp + fn),
        "fp_pct": float(fp_pct),
        "n_valid": int(n_valid),
        "n_total": int(n_total),
        "has_refusal": bool(n_valid < n_total),
    }


def fmt_bias(b):
    """Format signed bias for LaTeX."""
    if abs(b) < 0.0005:
        b = 0.0
    return f"$+${abs(b):.3f}" if b >= 0 else f"$-${abs(b):.3f}"


def fmt_kappa(k, bold=False):
    s = f"{k:.3f}"
    return f"\\textbf{{{s}}}" if bold else s


def fmt_fp_fn(fp, fn):
    return f"{int(fp)}/{int(fn)}"


def assert_no_internal_prompt_names_in_text(text):
    bad_terms = ["P2B", "P3B", "label_", "_P1", "_P2", "_P3"]
    found = [t for t in bad_terms if t in text]
    if found:
        raise ValueError(f"Internal prompt/source names leaked into LaTeX: {found}")


# =============================================================================
# Load and validate release master
# =============================================================================

master = pd.read_csv(RELEASE_CSV)

required_core = ["model", "annotator1_label"]
missing_core = [c for c in required_core if c not in master.columns]
if missing_core:
    raise KeyError(f"Missing required core columns: {missing_core}")

required_label_cols = [
    f"label_{clean_prefix}_{p}"
    for _, clean_prefix, _ in JUDGES
    for p in PROMPT_ORDER
]

missing_label_cols = [c for c in required_label_cols if c not in master.columns]
if missing_label_cols:
    raise KeyError(f"Missing required LLM judge columns: {missing_label_cols}")

if "TARGET_MODELS" not in globals():
    TARGET_MODELS = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

print("=" * 80)
print("PROMPT ABLATION INPUT")
print("=" * 80)
print("Release CSV:", RELEASE_CSV)
print("Shape:", master.shape)
print("\nRows by model:")
print(master["model"].value_counts().reindex(TARGET_MODELS))

for mn in TARGET_MODELS:
    n = int((master["model"] == mn).sum())
    if n != 300:
        raise ValueError(f"Expected 300 rows for {mn}, got {n}")

if not master["annotator1_label"].isin([0, 1]).all():
    bad_n = int((~master["annotator1_label"].isin([0, 1])).sum())
    raise ValueError(f"Invalid annotator1_label values: {bad_n}")

# Guardrail: release master should not expose old internal prompt names
forbidden_master_cols = [
    c for c in master.columns
    if "P2B" in c or "P3B" in c or c.endswith("_P1") or c.endswith("_P2") or c.endswith("_P3")
]
if forbidden_master_cols:
    raise ValueError(
        "Old/internal prompt columns found in release master: "
        f"{forbidden_master_cols}"
    )

# Guardrail: every local judge prefix must actually be groupable as local.
_local_prefixes = [p for _, p, _ in JUDGES if p.endswith("_local")]
if len(_local_prefixes) != 3:
    raise ValueError(
        f"Expected 3 local judge prefixes, found {len(_local_prefixes)}: "
        f"{_local_prefixes}"
    )

# =============================================================================
# Build long-form result rows
# =============================================================================

rows = []

for judge_name, clean_prefix, gen_override in JUDGES:
    for model_name in TARGET_MODELS:
        df_m = master[master["model"] == model_name].copy()
        gen_label = gen_override.get(model_name, GEN_LABELS[model_name])

        block_rows = []

        for p in PROMPT_ORDER:
            pred_col = f"label_{clean_prefix}_{p}"
            stats = compute_row(df_m, "annotator1_label", pred_col)

            row = {
                "judge": judge_name,
                "group": judge_group(clean_prefix),
                "clean_prefix": clean_prefix,
                "prompt": p,
                "model": model_name,
                "gen": gen_label,
                **stats,
            }

            block_rows.append(row)

        best_k = max(r["kappa"] for r in block_rows)

        for r in block_rows:
            r["best"] = abs(r["kappa"] - best_k) < 1e-12
            rows.append(r)

df_rows = pd.DataFrame(rows)

# Stable categorical ordering
judge_order = [j[0] for j in JUDGES]
df_rows["judge"] = pd.Categorical(df_rows["judge"], categories=judge_order, ordered=True)
df_rows["model"] = pd.Categorical(df_rows["model"], categories=TARGET_MODELS, ordered=True)
df_rows["prompt"] = pd.Categorical(df_rows["prompt"], categories=PROMPT_ORDER, ordered=True)
df_rows = df_rows.sort_values(["judge", "model", "prompt"]).reset_index(drop=True)

df_rows.to_csv(LONG_AUDIT_CSV, index=False)

print("\n" + "=" * 80)
print("PROMPT-ABLATION RESULTS TABLE")
print("=" * 80)
display(df_rows)

print("\n" + "=" * 80)
print("Prompt-ablation long-form rows")
print("=" * 80)
print("Rows:", len(df_rows))
print("Judges:", df_rows["judge"].nunique())
print("Prompts:", list(df_rows["prompt"].cat.categories))
print("Groups:", df_rows["group"].value_counts().to_dict())
print("Saved audit CSV:", LONG_AUDIT_CSV)

print("\nCells with invalid/refusal rows:")
refusal_cells = df_rows[df_rows["has_refusal"]]
if len(refusal_cells) == 0:
    print("  none")
else:
    for _, r in refusal_cells.iterrows():
        print(
            f"  {r['judge']:<22} {r['prompt']:<3} {r['gen']:<10} "
            f"n={r['n_valid']}/{r['n_total']}"
        )

# =============================================================================
# 1. Main-text wide table (tab:results_prompt)
# =============================================================================

has_any_refusal = bool(df_rows["has_refusal"].any())

wide_caption = (
    r"Prompt-ablation results against human labels for each judge--generator "
    r"pair. Bias is positive when the automated labeler over-labels "
    r"hallucinations relative to humans. FP/FN reports false positives and "
    r"false negatives; FP\% is false positives as a proportion of all errors. "
    r"Best $\kappa$ within each row is shown in bold; ties at the reported "
    r"precision are all marked. All values are computed against "
    r"\texttt{annotator1\_label}, the complete primary human label set for "
    r"the 900 question--answer pairs. Gen denotes the answer-generating model: "
    r"L=Llama-3-8B, G=Gemma-2-9B, and M=Mistral-7B."
)

if has_any_refusal:
    wide_caption += (
        r" $^a$ marks a cell with at least one invalid or refused judge output; "
        r"metrics for that cell use valid rows only."
    )

latex_wide = r"""\begin{table*}[t!]
  \centering
  \caption{""" + wide_caption + r"""}
  \label{tab:results_prompt}
  \scriptsize
  \setlength{\tabcolsep}{3pt}
  \renewcommand{\arraystretch}{0.92}
  \begin{tabular}{llrrrrrrrrrrrr}
    \toprule
    & & \multicolumn{4}{c}{p1} & \multicolumn{4}{c}{p2} & \multicolumn{4}{c}{p3} \\
    \cmidrule(lr){3-6}\cmidrule(lr){7-10}\cmidrule(lr){11-14}
    Judge & Gen
    & $\kappa$ & Bias & FP/FN & FP\%
    & $\kappa$ & Bias & FP/FN & FP\%
    & $\kappa$ & Bias & FP/FN & FP\% \\
    \midrule
"""

prev_group = None
prev_judge = None

for judge_name, clean_prefix, gen_override in JUDGES:
    group = judge_group(clean_prefix)

    if prev_group is None:
        latex_wide += f"    \\multicolumn{{14}}{{l}}{{\\textit{{{group}}}}} \\\\\n"
        latex_wide += "    \\midrule\n"
    elif group != prev_group:
        latex_wide += "    \\midrule\n"
        latex_wide += f"    \\multicolumn{{14}}{{l}}{{\\textit{{{group}}}}} \\\\\n"
        latex_wide += "    \\midrule\n"
    elif prev_judge is not None and judge_name != prev_judge:
        latex_wide += "    \\addlinespace[1pt]\n"

    prev_group = group
    prev_judge = judge_name

    for model_name in TARGET_MODELS:
        sub = df_rows[
            (df_rows["judge"] == judge_name) &
            (df_rows["model"] == model_name)
        ].copy()

        if sub.empty:
            raise ValueError(f"Missing rows for {judge_name} / {model_name}")

        gen_text = str(sub.iloc[0]["gen"])
        if sub["has_refusal"].any():
            gen_text += REFUSAL_MARKER

        cells = []

        for p in PROMPT_ORDER:
            rr = sub[sub["prompt"] == p]

            if rr.empty:
                raise ValueError(f"Missing prompt {p} for {judge_name} / {model_name}")

            r = rr.iloc[0]

            cells.extend([
                fmt_kappa(r["kappa"], bold=bool(r["best"])),
                fmt_bias(r["bias"]),
                fmt_fp_fn(r["fp"], r["fn"]),
                f"{r['fp_pct']:.1f}",
            ])

        latex_wide += (
            f"    {judge_name} & {gen_text} "
            f"& {cells[0]} & {cells[1]} & {cells[2]} & {cells[3]} "
            f"& {cells[4]} & {cells[5]} & {cells[6]} & {cells[7]} "
            f"& {cells[8]} & {cells[9]} & {cells[10]} & {cells[11]} \\\\\n"
        )

latex_wide += r"""    \bottomrule
  \end{tabular}
\end{table*}
"""

assert_no_internal_prompt_names_in_text(latex_wide)

with open(WIDE_OUT, "w") as f:
    f.write(latex_wide)

print("\nSaved main prompt-ablation table (tab:results_prompt):")
print(f"  {WIDE_OUT}")

# =============================================================================
# 2. Appendix long table (tab:results_full)
# =============================================================================

long_caption = (
    r"Full prompt-ablation results against human labels for each "
    r"judge--generator--prompt combination. Bias is positive when the "
    r"automated labeler over-labels hallucinations relative to humans. "
    r"Err=FP+FN, and FP\% is false positives as a proportion of Err. "
    r"Best $\kappa$ within each judge--generator block is shown in bold. "
    r"Generator abbreviations are L=Llama-3-8B, G=Gemma-2-9B, and "
    r"M=Mistral-7B."
)

if has_any_refusal:
    long_caption += (
        r" $^a$ marks a cell with at least one invalid or refused judge output; "
        r"metrics for that cell use valid rows only."
    )

latex_long = r"""\begin{table*}[p]
  \centering
  \caption{""" + long_caption + r"""}
  \label{tab:results_full}
  \scriptsize
  \setlength{\tabcolsep}{4pt}
  \renewcommand{\arraystretch}{0.92}
  \begin{tabular}{lllrrrrrr}
    \toprule
    Judge & Prompt & Gen & $\kappa$ & Bias & FP & FN & Err & FP\% \\
    \midrule
"""

prev_judge = None
prev_group = None

for _, r in df_rows.iterrows():
    group = r["group"]

    if prev_group is None:
        latex_long += f"    \\multicolumn{{9}}{{l}}{{\\textit{{{group}}}}} \\\\\n"
        latex_long += "    \\midrule\n"
    elif group != prev_group:
        latex_long += "    \\midrule\n"
        latex_long += f"    \\multicolumn{{9}}{{l}}{{\\textit{{{group}}}}} \\\\\n"
        latex_long += "    \\midrule\n"
    elif prev_judge is not None and r["judge"] != prev_judge:
        latex_long += "    \\addlinespace[1pt]\n"

    prev_group = group
    prev_judge = r["judge"]

    gen_text = str(r["gen"])
    if r["has_refusal"]:
        gen_text += REFUSAL_MARKER

    latex_long += (
        f"    {r['judge']} & {r['prompt']} & {gen_text} & "
        f"{fmt_kappa(r['kappa'], bold=bool(r['best']))} & "
        f"{fmt_bias(r['bias'])} & "
        f"{int(r['fp'])} & {int(r['fn'])} & {int(r['err'])} & "
        f"{r['fp_pct']:.1f} \\\\\n"
    )

latex_long += r"""    \bottomrule
  \end{tabular}
\end{table*}
"""

assert_no_internal_prompt_names_in_text(latex_long)

with open(LONG_OUT, "w") as f:
    f.write(latex_long)

print("\nSaved appendix full table (tab:results_full):")
print(f"  {LONG_OUT}")

print("\n" + "=" * 80)
print("Prompt-ablation tables complete.")
print("=" * 80)

# =============================================================================
# MERGED AUDIT: Verify prompt-ablation table rows against direct recomputation
#
# Purpose:
#   This is an internal reproduction guard for CELL C2.
#   It verifies that audits/prompt_ablation/prompt_ablation_long.csv matches
#   direct recomputation from the locked release master.
#
# Note:
#   This audit checks internal consistency, not correctness of the grouping
#   or display logic: both sides call the same helpers. Structural checks on
#   the emitted LaTeX are done separately below.
#
# This audit is not a separate paper result.
# =============================================================================

print("\n" + "=" * 80)
print("PROMPT-ABLATION INTERNAL AUDIT")
print("=" * 80)

if not RUN_RELEASE_ANALYSIS:
    print("SKIPPED: RUN_RELEASE_ANALYSIS=False.")
else:
    if not LONG_AUDIT_CSV.exists():
        raise FileNotFoundError(
            f"Prompt-ablation long audit CSV not found: {LONG_AUDIT_CSV}"
        )

    stored = pd.read_csv(LONG_AUDIT_CSV)

    expected_n = len(JUDGES) * len(TARGET_MODELS) * len(PROMPT_ORDER)

    if len(stored) != expected_n:
        raise AssertionError(
            f"Expected {expected_n} prompt-ablation rows, found {len(stored)} "
            f"in {LONG_AUDIT_CSV}"
        )

    # -------------------------------------------------------------------------
    # Direct recomputation from release master
    # -------------------------------------------------------------------------

    direct_rows = []

    for judge_name, clean_prefix, gen_override in JUDGES:
        for model_name in TARGET_MODELS:
            df_m = master[master["model"] == model_name].copy()
            gen_label = gen_override.get(model_name, GEN_LABELS[model_name])

            block_rows = []

            for p in PROMPT_ORDER:
                pred_col = f"label_{clean_prefix}_{p}"
                stats = compute_row(df_m, "annotator1_label", pred_col)

                row = {
                    "judge": judge_name,
                    "group": judge_group(clean_prefix),
                    "clean_prefix": clean_prefix,
                    "prompt": p,
                    "model": model_name,
                    "gen": gen_label,
                    **stats,
                }

                block_rows.append(row)

            best_k = max(r["kappa"] for r in block_rows)

            for r in block_rows:
                r["best"] = abs(r["kappa"] - best_k) < 1e-12
                direct_rows.append(r)

    direct = pd.DataFrame(direct_rows)

    # -------------------------------------------------------------------------
    # Normalize dtypes for comparison
    # -------------------------------------------------------------------------

    key_cols = [
        "judge",
        "clean_prefix",
        "prompt",
        "model",
    ]

    text_cols = [
        "group",
        "gen",
    ]

    int_cols = [
        "fp",
        "fn",
        "err",
        "n_valid",
        "n_total",
    ]

    float_cols = [
        "kappa",
        "bias",
        "fp_pct",
    ]

    bool_cols = [
        "has_refusal",
        "best",
    ]

    required_audit_cols = key_cols + text_cols + int_cols + float_cols + bool_cols
    missing_stored = [c for c in required_audit_cols if c not in stored.columns]
    missing_direct = [c for c in required_audit_cols if c not in direct.columns]

    if missing_stored:
        raise KeyError(f"Stored prompt-ablation audit is missing columns: {missing_stored}")

    if missing_direct:
        raise KeyError(f"Direct prompt-ablation recomputation is missing columns: {missing_direct}")

    for df_cmp in [stored, direct]:
        for c in key_cols + text_cols:
            df_cmp[c] = df_cmp[c].astype(str)

        for c in int_cols:
            df_cmp[c] = pd.to_numeric(df_cmp[c], errors="raise").astype(int)

        for c in float_cols:
            df_cmp[c] = pd.to_numeric(df_cmp[c], errors="raise").astype(float)

        for c in bool_cols:
            if df_cmp[c].dtype == bool:
                pass
            else:
                df_cmp[c] = df_cmp[c].astype(str).str.lower().map(
                    {
                        "true": True,
                        "false": False,
                        "1": True,
                        "0": False,
                    }
                )

                if df_cmp[c].isna().any():
                    raise ValueError(f"Could not normalize boolean column: {c}")

    stored = stored.sort_values(key_cols).reset_index(drop=True)
    direct = direct.sort_values(key_cols).reset_index(drop=True)

    # -------------------------------------------------------------------------
    # Key comparison
    # -------------------------------------------------------------------------

    if not stored[key_cols].equals(direct[key_cols]):
        mismatch = pd.concat(
            [
                stored[key_cols].add_prefix("stored_"),
                direct[key_cols].add_prefix("direct_"),
            ],
            axis=1,
        )

        bad = mismatch[
            (mismatch["stored_judge"] != mismatch["direct_judge"])
            | (mismatch["stored_clean_prefix"] != mismatch["direct_clean_prefix"])
            | (mismatch["stored_prompt"] != mismatch["direct_prompt"])
            | (mismatch["stored_model"] != mismatch["direct_model"])
        ]

        display(bad.head(20))

        raise AssertionError(
            "Prompt-ablation audit failed: row keys do not match direct recomputation."
        )

    # -------------------------------------------------------------------------
    # Value comparison
    # -------------------------------------------------------------------------

    audit_errors = []
    diff_rows = []

    for c in text_cols:
        bad_mask = stored[c] != direct[c]

        if bad_mask.any():
            audit_errors.append(f"{c}: {int(bad_mask.sum())} mismatches")

            tmp = stored.loc[bad_mask, key_cols].copy()
            tmp["column"] = c
            tmp["stored"] = stored.loc[bad_mask, c].to_numpy()
            tmp["direct"] = direct.loc[bad_mask, c].to_numpy()
            diff_rows.append(tmp)

    for c in int_cols:
        bad_mask = stored[c].astype(int) != direct[c].astype(int)

        if bad_mask.any():
            audit_errors.append(f"{c}: {int(bad_mask.sum())} mismatches")

            tmp = stored.loc[bad_mask, key_cols].copy()
            tmp["column"] = c
            tmp["stored"] = stored.loc[bad_mask, c].to_numpy()
            tmp["direct"] = direct.loc[bad_mask, c].to_numpy()
            diff_rows.append(tmp)

    for c in bool_cols:
        bad_mask = stored[c].astype(bool) != direct[c].astype(bool)

        if bad_mask.any():
            audit_errors.append(f"{c}: {int(bad_mask.sum())} mismatches")

            tmp = stored.loc[bad_mask, key_cols].copy()
            tmp["column"] = c
            tmp["stored"] = stored.loc[bad_mask, c].to_numpy()
            tmp["direct"] = direct.loc[bad_mask, c].to_numpy()
            diff_rows.append(tmp)

    FLOAT_TOL = 1e-12

    for c in float_cols:
        stored_vals = stored[c].astype(float).to_numpy()
        direct_vals = direct[c].astype(float).to_numpy()

        bad_mask = ~np.isclose(
            stored_vals,
            direct_vals,
            rtol=0.0,
            atol=FLOAT_TOL,
            equal_nan=True,
        )

        if bad_mask.any():
            audit_errors.append(f"{c}: {int(bad_mask.sum())} mismatches")

            tmp = stored.loc[bad_mask, key_cols].copy()
            tmp["column"] = c
            tmp["stored"] = stored.loc[bad_mask, c].to_numpy()
            tmp["direct"] = direct.loc[bad_mask, c].to_numpy()
            tmp["absdiff"] = np.abs(
                stored.loc[bad_mask, c].astype(float).to_numpy()
                - direct.loc[bad_mask, c].astype(float).to_numpy()
            )
            diff_rows.append(tmp)

    # -------------------------------------------------------------------------
    # Save optional diff and raise if needed
    # -------------------------------------------------------------------------

    DIFF_OUT = AUDIT_OUT_DIR / "prompt_ablation_internal_audit_diffs.csv"

    if audit_errors:
        diffs = pd.concat(diff_rows, ignore_index=True) if diff_rows else pd.DataFrame()
        diffs.to_csv(DIFF_OUT, index=False)

        print("\nPrompt-ablation audit errors:")
        for e in audit_errors:
            print("  \u2022", e)

        print("\nSaved diff CSV:")
        print(DIFF_OUT)

        if len(diffs) > 0:
            display(diffs.head(50))

        raise AssertionError(
            "Prompt-ablation internal audit failed. "
            "The LaTeX-table source CSV does not match direct recomputation "
            "from the locked release master."
        )

    # -------------------------------------------------------------------------
    # Additional guardrails
    # -------------------------------------------------------------------------

    if stored["prompt"].isin(["P1", "P2", "P3", "P2B", "P3B"]).any():
        raise AssertionError(
            "Prompt-ablation audit contains internal/uppercase prompt names. "
            "Expected paper-facing prompt names: p1, p2, p3."
        )

    for p in PROMPT_ORDER:
        if stored["prompt"].tolist().count(p) != len(JUDGES) * len(TARGET_MODELS):
            raise AssertionError(f"Unexpected number of {p} rows in prompt-ablation audit.")

    # Grouping guardrail: the API/local split must survive display-name changes.
    expected_local_rows = 3 * len(TARGET_MODELS) * len(PROMPT_ORDER)
    expected_api_rows = 4 * len(TARGET_MODELS) * len(PROMPT_ORDER)

    n_local = int((stored["group"] == "Local judges").sum())
    n_api = int((stored["group"] == "API judges").sum())

    if n_local != expected_local_rows or n_api != expected_api_rows:
        raise AssertionError(
            f"Judge grouping is wrong: expected {expected_api_rows} API rows and "
            f"{expected_local_rows} local rows, found {n_api} and {n_local}. "
            "Check judge_group() and the JUDGES prefixes."
        )

    print("\u2713 PROMPT-ABLATION AUDIT PASSED")
    print(f"  All {len(stored)} rows match direct recomputation from release master.")
    print(f"  Checked columns: {', '.join(text_cols + int_cols + float_cols + bool_cols)}")
    print(f"  Grouping: {n_api} API rows, {n_local} local rows.")

    # -------------------------------------------------------------------------
    # Structural check on the emitted LaTeX
    # -------------------------------------------------------------------------

    # n_header_rows differs by table: tab:results_prompt has a spanning p1/p2/p3 row above
    # the column-name row; tab:results_full has a single column-name row.
    for path, label, n_data_rows, n_header_rows in [
        (WIDE_OUT, "tab:results_prompt", len(JUDGES) * len(TARGET_MODELS), 2),
        (LONG_OUT, "tab:results_full", len(df_rows), 1),
    ]:
        text = path.read_text()

        for heading in ["API judges", "Local judges"]:
            if heading not in text:
                raise AssertionError(f"{label} is missing the '{heading}' group heading.")

        n_rows = text.count("\\\\\n")
        expected_rows = n_data_rows + n_header_rows + 2  # + two group headings

        if n_rows != expected_rows:
            raise AssertionError(
                f"{label}: expected {expected_rows} LaTeX rows "
                f"({n_data_rows} data + {n_header_rows} header + 2 group), "
                f"found {n_rows}."
            )

    print("\u2713 LATEX STRUCTURE CHECK PASSED")

    # -------------------------------------------------------------------------
    # Echo generated LaTeX for visual inspection
    # -------------------------------------------------------------------------

    for path, label in [(WIDE_OUT, "tab:results_prompt"), (LONG_OUT, "tab:results_full")]:
        text = path.read_text()
        n_rows = text.count("\\\\\n")

        print("\n" + "=" * 80)
        print(f"{label}: {path.name}  ({n_rows} rows, {len(text):,} chars)")
        print("=" * 80)
        print(text)

In [ ]:
# =============================================================================
# CELL C3: Generate lexical/NLI baseline LaTeX table from release master CSV
#
# Source of truth:
#   data/human_judge_release_master.csv
#
# Outputs:
#   tables/table_results_lexical_nli.tex
#   audits/lexical_nli/table_results_lexical_nli.csv
#
# Paper label:
#   tab:results_lexical
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# =============================================================================
# Paths
# =============================================================================

# Uses globals from CELL 1:
#   MASTER_PATH
#   TABLE_DIR
#   AUDIT_DIR

OUT_PATH = TABLE_DIR / "table_results_lexical_nli.tex"

AUDIT_OUT_DIR = AUDIT_DIR / "lexical_nli"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_CSV = AUDIT_OUT_DIR / "table_results_lexical_nli.csv"

# Optional convenience copy for Colab download

# =============================================================================
# Config
# =============================================================================

HUMAN_COL = "annotator1_label"

GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

GEN_LABEL = {
    "llama3-8b": "L",
    "gemma-2-9b": "G",
    "mistral-7b": "M",
}

# Change this caption fragment if the actual NLI checkpoint differs.
NLI_MODEL_DESCRIPTION = (
    r"NLI uses a T5-11B TRUE-style entailment model fine-tuned on a mixture "
    r"of NLI datasets (see Appendix C.3)."
)

BASELINE_SPECS = [
    {
        "metric": r"ROUGE-L (oracle-$\tau$)",
        "group": "ROUGE-L",
        "label_col": "rougeL_label_oracle",
        "tau_cols": ("rougeL_tau_oracle",),
        "tau_type": "single",
    },
    {
        "metric": r"ROUGE-L (CV-$\tau$)",
        "group": "ROUGE-L",
        "label_col": "rougeL_label_cv",
        "tau_cols": (
            "rougeL_tau_cv_mean",
            "rougeL_tau_cv_min",
            "rougeL_tau_cv_max",
        ),
        "tau_type": "cv",
    },
    {
        "metric": r"BERTScore (oracle-$\tau$)",
        "group": "BERTScore",
        "label_col": "bertscore_label_oracle",
        "tau_cols": ("bertscore_tau_oracle",),
        "tau_type": "single",
    },
    {
        "metric": r"BERTScore (CV-$\tau$)",
        "group": "BERTScore",
        "label_col": "bertscore_label_cv",
        "tau_cols": (
            "bertscore_tau_cv_mean",
            "bertscore_tau_cv_min",
            "bertscore_tau_cv_max",
        ),
        "tau_type": "cv",
    },
    {
        "metric": r"NLI (T5-11B) strict",
        "group": "NLI",
        "label_col": "nli_label_strict",
        "tau_cols": ("nli_tau_strict",),
        "tau_type": "single",
    },
    {
        "metric": r"NLI (T5-11B) best-$\tau$",
        "group": "NLI",
        "label_col": "nli_label_best_tau",
        "tau_cols": ("nli_tau_best",),
        "tau_type": "single",
    },
]

# =============================================================================
# Helpers
# =============================================================================

def get_unique_tau(df, col, decimals=10):
    """
    Return the unique threshold value in a generator subset.

    Threshold columns should be constant within each generator subset.
    This function makes that assumption explicit.
    """
    vals = (
        pd.to_numeric(df[col], errors="coerce")
        .dropna()
        .round(decimals)
        .unique()
    )

    if len(vals) != 1:
        raise ValueError(f"Expected one unique tau in {col}, found {vals}")

    return float(vals[0])


def format_tau(df, spec):
    """Format tau for LaTeX and audit CSV."""
    tau_cols = spec["tau_cols"]

    if spec["tau_type"] == "single":
        tau = get_unique_tau(df, tau_cols[0])
        return f"{tau:.2f}"

    if spec["tau_type"] == "cv":
        mean_tau = get_unique_tau(df, tau_cols[0])
        min_tau = get_unique_tau(df, tau_cols[1])
        max_tau = get_unique_tau(df, tau_cols[2])
        return f"{mean_tau:.2f} [{min_tau:.2f}, {max_tau:.2f}]"

    raise ValueError(f"Unknown tau_type: {spec['tau_type']}")


def metric_row(df, human_col, pred_col):
    """
    Compute kappa, bias, FP, FN, Err, FP%, n_valid, and n_total.

    Baseline label columns are expected to be complete binary labels.
    """
    if human_col not in df.columns:
        raise KeyError(f"Missing human column: {human_col}")

    if pred_col not in df.columns:
        raise KeyError(f"Missing prediction column: {pred_col}")

    y_true = pd.to_numeric(df[human_col], errors="coerce")
    y_pred = pd.to_numeric(df[pred_col], errors="coerce")

    mask = y_true.isin([0, 1]) & y_pred.isin([0, 1])

    n_total = len(df)
    n_valid = int(mask.sum())

    if n_valid != n_total:
        raise ValueError(
            f"{pred_col} has invalid labels: valid={n_valid}/{n_total}"
        )

    h = y_true.loc[mask].astype(int).to_numpy()
    p = y_pred.loc[mask].astype(int).to_numpy()

    kappa = cohen_kappa_score(h, p, labels=[0, 1])
    bias = p.mean() - h.mean()

    tn, fp, fn, tp = confusion_matrix(h, p, labels=[0, 1]).ravel()

    err = fp + fn
    fp_pct = 100.0 * fp / max(err, 1)

    return {
        "kappa": float(kappa),
        "bias": float(bias),
        "fp": int(fp),
        "fn": int(fn),
        "err": int(err),
        "fp_pct": float(fp_pct),
        "n_valid": int(n_valid),
        "n_total": int(n_total),
    }


def fmt_bias(b):
    """Format signed bias for LaTeX."""
    if abs(b) < 0.0005:
        b = 0.0
    return f"$+${abs(b):.3f}" if b >= 0 else f"$-${abs(b):.3f}"


def latex_table_row(r):
    """Format one baseline row for LaTeX."""
    return (
        f"    {r['metric']} & {r['gen']} & {r['tau']} "
        f"& {r['kappa']:.3f} & {fmt_bias(r['bias'])} "
        f"& {int(r['fp'])} & {int(r['fn'])} & {r['fp_pct']:.1f} \\\\\n"
    )


# =============================================================================
# Load and validate release master
# =============================================================================

master = pd.read_csv(MASTER_PATH)

required_cols = [
    HUMAN_COL,
    "model",
    "rougeL",
    "bertscore_f1",
    "nli_entail_prob",

    "rougeL_label_oracle",
    "rougeL_label_cv",
    "bertscore_label_oracle",
    "bertscore_label_cv",
    "nli_label_strict",
    "nli_label_best_tau",

    "rougeL_tau_oracle",
    "rougeL_tau_cv_mean",
    "rougeL_tau_cv_min",
    "rougeL_tau_cv_max",

    "bertscore_tau_oracle",
    "bertscore_tau_cv_mean",
    "bertscore_tau_cv_min",
    "bertscore_tau_cv_max",

    "nli_tau_strict",
    "nli_tau_best",
]

missing = [c for c in required_cols if c not in master.columns]
if missing:
    raise KeyError(f"Missing required columns from release master: {missing}")

if len(master) != 900:
    raise ValueError(f"Expected 900 release rows, got {len(master)}")

if not master[HUMAN_COL].isin([0, 1]).all():
    bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
    raise ValueError(f"Invalid human labels in {HUMAN_COL}: {bad_n}")

for model_name in GEN_ORDER:
    n = int((master["model"] == model_name).sum())
    if n != 300:
        raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

label_cols = [
    "rougeL_label_oracle",
    "rougeL_label_cv",
    "bertscore_label_oracle",
    "bertscore_label_cv",
    "nli_label_strict",
    "nli_label_best_tau",
]

for c in label_cols:
    n_valid = int(master[c].isin([0, 1]).sum())
    if n_valid != len(master):
        raise ValueError(f"{c} has invalid labels: valid={n_valid}/{len(master)}")

score_cols = [
    "rougeL",
    "bertscore_f1",
    "nli_entail_prob",
]

for c in score_cols:
    numeric = pd.to_numeric(master[c], errors="coerce")
    missing_n = int(numeric.isna().sum())
    if missing_n > 0:
        raise ValueError(f"{c} has missing/non-numeric values: {missing_n}")

print("=" * 80)
print("LEXICAL / NLI BASELINE INPUT")
print("=" * 80)
print("Release master:", MASTER_PATH)
print("Shape:", master.shape)
print("\nRows by generator:")
print(master["model"].value_counts().reindex(GEN_ORDER))

print("\nScore summaries:")
for c in score_cols:
    numeric = pd.to_numeric(master[c], errors="coerce")
    print(
        f"  {c:<16} "
        f"min={numeric.min():.4f} "
        f"mean={numeric.mean():.4f} "
        f"max={numeric.max():.4f}"
    )

# =============================================================================
# Compute audit table
# =============================================================================

audit_rows = []

for spec in BASELINE_SPECS:
    for model_name in GEN_ORDER:
        df_g = master[master["model"] == model_name].copy()
        gen_label = GEN_LABEL[model_name]

        tau_str = format_tau(df_g, spec)
        stats = metric_row(df_g, HUMAN_COL, spec["label_col"])

        audit_rows.append({
            "group": spec["group"],
            "metric": spec["metric"],
            "label_col": spec["label_col"],
            "model": model_name,
            "gen": gen_label,
            "tau": tau_str,
            **stats,
        })

audit = pd.DataFrame(audit_rows)

audit["kappa_3dp"] = audit["kappa"].round(3)
audit["bias_3dp"] = audit["bias"].round(3)
audit["fp_pct_1dp"] = audit["fp_pct"].round(1)

audit.to_csv(AUDIT_CSV, index=False)

print("\n" + "=" * 80)
print("LEXICAL / NLI BASELINE AUDIT TABLE")
print("=" * 80)

display_cols = [
    "group",
    "metric",
    "gen",
    "tau",
    "n_valid",
    "kappa_3dp",
    "bias_3dp",
    "fp",
    "fn",
    "err",
    "fp_pct_1dp",
]

display(audit[display_cols])

print("\nSaved audit CSV:")
print(AUDIT_CSV)

# =============================================================================
# Build LaTeX table from audit table
# =============================================================================

caption = (
    r"Lexical similarity metric and NLI baselines against human labels. "
    r"For ROUGE-L and BERTScore, two threshold protocols are reported. "
    r"Oracle-$\tau$ tunes $\tau$ directly on the evaluation set and therefore "
    r"serves as a diagnostic upper bound, not as a deployable thresholding "
    r"procedure. CV-$\tau$ uses 3-fold cross-validation within each generator: "
    r"for each fold, $\tau$ is selected on the two training folds and applied "
    r"only to the held-out fold. Reported CV-$\tau$ values are computed from "
    r"the concatenated out-of-fold predictions; the reported $\tau$ is the "
    r"mean selected threshold with the per-fold range. Direction is fixed: "
    r"low similarity is treated as evidence for hallucination. "
    + NLI_MODEL_DESCRIPTION + " "
    r"The strict row reports the natural reference-entailment decision at "
    r"$\tau{=}0.5$; the best-$\tau$ row reports an oracle-optimized diagnostic "
    r"threshold. L=Llama-3-8B, G=Gemma-2-9B, M=Mistral-7B. FP\% is false "
    r"positives as a proportion of all errors."
)

latex = r"""\begin{table*}[t]
  \centering
  \caption{""" + caption + r"""}
  \label{tab:results_lexical}
  \scriptsize
  \setlength{\tabcolsep}{5pt}
  \renewcommand{\arraystretch}{0.95}
  \begin{tabular}{lllrrrrr}
    \toprule
    Metric & Gen & $\tau$ & $\kappa$ & Bias & FP & FN & FP\% \\
    \midrule
"""

group_order = ["ROUGE-L", "BERTScore", "NLI"]

for gi, group in enumerate(group_order):
    sub_g = audit[audit["group"] == group].copy()

    if sub_g.empty:
        raise ValueError(f"No rows found for group: {group}")

    if gi > 0:
        latex += r"    \midrule" + "\n"

    for _, r in sub_g.iterrows():
        latex += latex_table_row(r)

latex += r"""    \bottomrule
  \end{tabular}
\end{table*}
"""

with open(OUT_PATH, "w") as f:
    f.write(latex)


print("\nSaved table file:")
print("  ", OUT_PATH)

print("\n" + "=" * 80)
print("Lexical/NLI baseline table complete.")
print("=" * 80)

In [ ]:
# =============================================================================
# CELL C3b: Generate METEOR / BARTScore baseline LaTeX table from release master CSV
#
# Source of truth:
#   data/human_judge_release_master.csv
#
# Reads the master only. No scores or thresholds are recomputed; this cell
# only reads the already-stored label/tau/cv_kappa_oof columns and formats
# them into the paper table, the same way CELL C3 does for the lexical/NLI
# baselines.
#
# Outputs:
#   tables/table_new_metrics.tex
#   audits/new_metrics/new_metric_audit.csv
#
# Paper label:
#   tab:results_new_metrics
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# =============================================================================
# Paths
# =============================================================================

# Uses globals from CELL 1:
#   MASTER_PATH
#   TABLE_DIR
#   AUDIT_DIR

OUT_PATH = TABLE_DIR / "table_new_metrics.tex"

AUDIT_OUT_DIR = AUDIT_DIR / "new_metrics"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_CSV = AUDIT_OUT_DIR / "new_metric_audit.csv"

# =============================================================================
# Config
# =============================================================================

HUMAN_COL = "annotator1_label"

GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

GEN_LABEL = {
    "llama3-8b": "L",
    "gemma-2-9b": "G",
    "mistral-7b": "M",
}

METRIC_SPECS = [
    {
        "display": "METEOR",
        "group": "METEOR",
        "score_col": "meteor",
    },
    {
        "display": r"BARTScore ref$\to$hyp",
        "group": "BARTScore-CNN",
        "score_col": "bartscore_r2h",
    },
    {
        "display": r"BARTScore hyp$\to$ref",
        "group": "BARTScore-CNN",
        "score_col": "bartscore_h2r",
    },
]

for spec in METRIC_SPECS:
    c = spec["score_col"]
    spec.update({
        "label_oracle_col": f"{c}_label_oracle",
        "label_cv_col": f"{c}_label_cv",
        "tau_oracle_col": f"{c}_tau_oracle",
        "tau_cv_mean_col": f"{c}_tau_cv_mean",
        "tau_cv_min_col": f"{c}_tau_cv_min",
        "tau_cv_max_col": f"{c}_tau_cv_max",
        "cv_kappa_oof_col": f"{c}_cv_kappa_oof",
    })

# =============================================================================
# Helpers
# =============================================================================

def get_unique_tau(df, col, decimals=10):
    """
    Return the unique threshold value in a generator subset.

    Threshold columns should be constant within each generator subset.
    This function makes that assumption explicit.
    """
    vals = (
        pd.to_numeric(df[col], errors="coerce")
        .dropna()
        .round(decimals)
        .unique()
    )

    if len(vals) != 1:
        raise ValueError(f"Expected one unique tau in {col}, found {vals}")

    return float(vals[0])


def fmt_signed(x, digits=3):
    """B9's signed-value formatter: math sign, no zeroing near zero."""
    return f"$+${abs(x):.{digits}f}" if x >= 0 else f"$-${abs(x):.{digits}f}"


def fmt_tau(x, fmt="{:.2f}"):
    """B9's threshold formatter: math minus for negatives."""
    text = fmt.format(abs(x))
    return f"$-${text}" if x < 0 else text


def metric_row(df, human_col, pred_col):
    """
    Compute kappa, bias, FP, FN, Err, FP%, n_valid, and n_total.

    Baseline label columns are expected to be complete binary labels.
    """
    if human_col not in df.columns:
        raise KeyError(f"Missing human column: {human_col}")

    if pred_col not in df.columns:
        raise KeyError(f"Missing prediction column: {pred_col}")

    y_true = pd.to_numeric(df[human_col], errors="coerce")
    y_pred = pd.to_numeric(df[pred_col], errors="coerce")

    mask = y_true.isin([0, 1]) & y_pred.isin([0, 1])

    n_total = len(df)
    n_valid = int(mask.sum())

    if n_valid != n_total:
        raise ValueError(
            f"{pred_col} has invalid labels: valid={n_valid}/{n_total}"
        )

    h = y_true.loc[mask].astype(int).to_numpy()
    p = y_pred.loc[mask].astype(int).to_numpy()

    kappa = cohen_kappa_score(h, p, labels=[0, 1])
    bias = p.mean() - h.mean()

    tn, fp, fn, tp = confusion_matrix(h, p, labels=[0, 1]).ravel()

    err = fp + fn
    fp_pct = 100.0 * fp / max(err, 1)

    return {
        "kappa": float(kappa),
        "bias": float(bias),
        "fp": int(fp),
        "fn": int(fn),
        "err": int(err),
        "fp_pct": float(fp_pct),
        "n_valid": int(n_valid),
        "n_total": int(n_total),
        "has_refusal": bool(n_valid < n_total),
    }


# =============================================================================
# Load and validate release master
# =============================================================================

master = pd.read_csv(MASTER_PATH)

required_cols = [HUMAN_COL, "model"]
for spec in METRIC_SPECS:
    required_cols += [
        spec["score_col"],
        spec["label_oracle_col"],
        spec["label_cv_col"],
        spec["tau_oracle_col"],
        spec["tau_cv_mean_col"],
        spec["tau_cv_min_col"],
        spec["tau_cv_max_col"],
        spec["cv_kappa_oof_col"],
    ]

missing = [c for c in required_cols if c not in master.columns]
if missing:
    raise KeyError(f"Missing required columns from release master: {missing}")

if len(master) != 900:
    raise ValueError(f"Expected 900 release rows, got {len(master)}")

if not master[HUMAN_COL].isin([0, 1]).all():
    bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
    raise ValueError(f"Invalid human labels in {HUMAN_COL}: {bad_n}")

for model_name in GEN_ORDER:
    n = int((master["model"] == model_name).sum())
    if n != 300:
        raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

label_cols = [
    c
    for spec in METRIC_SPECS
    for c in (spec["label_oracle_col"], spec["label_cv_col"])
]

for c in label_cols:
    n_valid = int(master[c].isin([0, 1]).sum())
    if n_valid != len(master):
        raise ValueError(f"{c} has invalid labels: valid={n_valid}/{len(master)}")

for spec in METRIC_SPECS:
    numeric = pd.to_numeric(master[spec["score_col"]], errors="coerce")
    missing_n = int(numeric.isna().sum())
    if missing_n > 0:
        raise ValueError(f"{spec['score_col']} has missing/non-numeric values: {missing_n}")

    for model_name in GEN_ORDER:
        df_g = master[master["model"] == model_name]
        get_unique_tau(df_g, spec["tau_oracle_col"])
        get_unique_tau(df_g, spec["tau_cv_mean_col"])
        get_unique_tau(df_g, spec["tau_cv_min_col"])
        get_unique_tau(df_g, spec["tau_cv_max_col"])
        get_unique_tau(df_g, spec["cv_kappa_oof_col"])

print("=" * 80)
print("METEOR / BARTSCORE BASELINE INPUT")
print("=" * 80)
print("Release master:", MASTER_PATH)
print("Shape:", master.shape)
print("\nRows by generator:")
print(master["model"].value_counts().reindex(GEN_ORDER))

# =============================================================================
# Compute audit table
# =============================================================================

audit_rows = []
notes = []

for spec in METRIC_SPECS:
    # Oracle block: all three generators.
    for model_name in GEN_ORDER:
        df_g = master[master["model"] == model_name].copy()
        tau_oracle = get_unique_tau(df_g, spec["tau_oracle_col"])
        stats_oracle = metric_row(df_g, HUMAN_COL, spec["label_oracle_col"])
        audit_rows.append({
            "group": spec["group"],
            "metric": f"{spec['display']} (oracle-$\\tau$)",
            "label_col": spec["label_oracle_col"],
            "model": model_name,
            "gen": GEN_LABEL[model_name],
            "tau": fmt_tau(tau_oracle),
            **stats_oracle,
        })

    # CV block: all three generators.
    for model_name in GEN_ORDER:
        df_g = master[master["model"] == model_name].copy()
        tau_cv_mean = get_unique_tau(df_g, spec["tau_cv_mean_col"])
        tau_cv_min = get_unique_tau(df_g, spec["tau_cv_min_col"])
        tau_cv_max = get_unique_tau(df_g, spec["tau_cv_max_col"])
        stats_cv = metric_row(df_g, HUMAN_COL, spec["label_cv_col"])
        audit_rows.append({
            "group": spec["group"],
            "metric": f"{spec['display']} (CV-$\\tau$)",
            "label_col": spec["label_cv_col"],
            "model": model_name,
            "gen": GEN_LABEL[model_name],
            "tau": (
                f"{fmt_tau(tau_cv_mean)} "
                f"[{fmt_tau(tau_cv_min)}, {fmt_tau(tau_cv_max)}]"
            ),
            **stats_cv,
        })

        # Cross-check: recomputed CV kappa (from the stored label_cv column)
        # against the stored cv_kappa_oof column.
        stored_oof_kappa = get_unique_tau(df_g, spec["cv_kappa_oof_col"])
        recomputed_cv_kappa = stats_cv["kappa"]
        mismatch = abs(stored_oof_kappa - recomputed_cv_kappa)
        if mismatch > 1e-6:
            note = (
                f"NOTE: {spec['score_col']}/{model_name}: stored "
                f"{spec['cv_kappa_oof_col']}={stored_oof_kappa:.6f} differs from "
                f"recomputed CV kappa={recomputed_cv_kappa:.6f} "
                f"(abs diff={mismatch:.2e})"
            )
            notes.append(note)
            print(note)

audit = pd.DataFrame(audit_rows)

if len(audit) != 18:
    raise AssertionError(f"Expected 18 audit rows, got {len(audit)}.")

audit.to_csv(AUDIT_CSV, index=False)

print("\n" + "=" * 80)
print("METEOR / BARTSCORE AUDIT TABLE")
print("=" * 80)

display_cols = [
    "group", "metric", "gen", "tau", "n_valid",
    "kappa", "bias", "fp", "fn", "err", "fp_pct", "has_refusal",
]
display(audit[display_cols])

if notes:
    print(f"\n{len(notes)} NOTE(s) printed above.")
else:
    print("\nNo NOTE: all stored cv_kappa_oof values match recomputed CV kappa.")

print("\nSaved audit CSV:")
print(AUDIT_CSV)

# =============================================================================
# Build LaTeX table from audit table
# =============================================================================

caption = (
    r"METEOR and BARTScore baselines against the same human "
    r"factual-correctness labels, using the multi-reference aggregation and "
    r"threshold protocol of Table~\ref{tab:results_lexical}. BARTScore is an "
    r"unbounded mean token log-likelihood, so its threshold candidates are "
    r"the exact observed score midpoints rather than a fixed grid. "
    r"ref$\to$hyp scores $\log P(\text{answer}\mid\text{reference})$ and "
    r"hyp$\to$ref the reverse. Bias is positive when the automated labeler "
    r"over-labels hallucinations relative to humans. FP\% is false positives "
    r"as a proportion of all errors. L=Llama-3-8B, G=Gemma-2-9B, "
    r"M=Mistral-7B."
)

latex = r"""\begin{table*}[t]
  \centering
  \caption{""" + caption + r"""}
  \label{tab:results_new_metrics}
  \scriptsize
  \setlength{\tabcolsep}{5pt}
  \renewcommand{\arraystretch}{0.95}
  \begin{tabular}{lllrrrrr}
    \toprule
    Metric & Gen & $\tau$ & $\kappa$ & Bias & FP & FN & FP\% \\
    \midrule
"""

for block_idx, spec in enumerate(METRIC_SPECS):
    if block_idx:
        latex += "    \\midrule\n"

    for mode in ["oracle", "cv"]:
        suffix = "(oracle-$\\tau$)" if mode == "oracle" else "(CV-$\\tau$)"
        label_col = spec["label_oracle_col"] if mode == "oracle" else spec["label_cv_col"]

        for model_name in GEN_ORDER:
            r = audit[
                (audit["model"] == model_name)
                & (audit["label_col"] == label_col)
            ].iloc[0]

            latex += (
                f"    {spec['display']} {suffix} & {GEN_LABEL[model_name]} & "
                f"{r['tau']} & {r['kappa']:.3f} & {fmt_signed(r['bias'])} & "
                f"{int(r['fp'])} & {int(r['fn'])} & {r['fp_pct']:.1f} \\\\\n"
            )

latex += r"""    \bottomrule
  \end{tabular}
\end{table*}
"""

with open(OUT_PATH, "w") as f:
    f.write(latex)

print("\nSaved table file:")
print("  ", OUT_PATH)

print("\n" + "=" * 80)
print("METEOR/BARTScore baseline table complete.")
print("=" * 80)


In [ ]:
# =============================================================================
# CELL C4: Pairwise inter-method kappa matrix
#
# Purpose:
#   Reproduce macro-averaged and per-generator pairwise Cohen kappa matrices
#   between automatic label sources.
#
# Source of truth:
#   MASTER_PATH = data/human_judge_release_master.csv
#
# Important:
#   - This matrix compares automatic label sources to each other.
#   - It does NOT include human labels unless explicitly added.
#   - Display labels must match the paper figures exactly; EXPECTED_LABELS
#     below is checked against MATRIX_METHODS on every run.
#
# Outputs:
#   figures/kappa_matrix_avg.pdf                (fig:kappa_matrix)
#   figures/kappa_matrix_all_generators.pdf     (fig:kappa_matrix_per_generator)
#   figures/kappa_matrix_per_generator.pdf      (duplicate of fig:kappa_matrix_per_generator)
#   figures/kappa_matrix_<generator>.pdf
#
#   audits/kappa_matrix/kappa_matrix_avg.csv
#   audits/kappa_matrix/kappa_matrix_<generator>.csv
#   audits/kappa_matrix/kappa_matrix_group_summary.csv
#   audits/kappa_matrix/kappa_matrix_methods.csv
#
# Paper labels:
#   fig:kappa_matrix
#   fig:kappa_matrix_per_generator
# =============================================================================

import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL C4: Pairwise inter-method kappa matrix"

if not RUN_RELEASE_ANALYSIS:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_RELEASE_ANALYSIS=False.")
    print("No kappa matrices were generated.")

else:
    # =========================================================================
    # Paths
    # =========================================================================

    FIG_DIR.mkdir(parents=True, exist_ok=True)

    KAPPA_AUDIT_DIR = AUDIT_DIR / "kappa_matrix"
    KAPPA_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_AVG_CSV = KAPPA_AUDIT_DIR / "kappa_matrix_avg.csv"
    OUT_GROUP_SUMMARY_CSV = KAPPA_AUDIT_DIR / "kappa_matrix_group_summary.csv"
    OUT_METHODS_CSV = KAPPA_AUDIT_DIR / "kappa_matrix_methods.csv"

    # =========================================================================
    # Source definitions
    # =========================================================================
    # Each entry:
    #   (release_master_column, display_label, group)
    #
    # Display labels are paper-facing and must match the axis labels of
    # the paper's kappa-matrix figures exactly. Row order is the figure row order.
    #
    # Group assignment drives only the within/between-group summary CSV; it
    # does not affect the matrices or the figures. The oracle-thresholded NLI
    # variant is grouped with the reference-entailment family because it uses
    # the same model and premise/hypothesis direction, differing only in the
    # decision threshold.
    # =========================================================================

    MATRIX_METHODS = [
        # Lexical similarity baselines
        ("rougeL_label_oracle",        "ROUGE-L",                 "lexical"),
        ("bertscore_label_oracle",     "BERTScore",               "lexical"),

        # Reference-entailment NLI baseline
        ("nli_label_strict",           "NLI (T5-11B) strict",     "faith"),
        ("nli_label_best_tau",         "NLI (T5-11B) best-tau",   "faith"),

        # Faithfulness-style prompts
        ("label_gpt_5_mini_p1",        "GPT-5-mini p1",           "faith"),
        ("label_gpt_5_4_p1",           "GPT-5.4 p1",              "faith"),
        ("label_claude_opus_4_7_p1",   "Opus 4.7 p1",             "faith"),

        # Criterion-aligned API judges
        ("label_gpt_5_mini_p2",        "GPT-5-mini p2",           "factual"),
        ("label_gpt_5_mini_p3",        "GPT-5-mini p3",           "factual"),
        ("label_gpt_5_nano_p2",        "GPT-nano p2",             "factual"),
        ("label_gpt_5_4_p2",           "GPT-5.4 p2",              "factual"),
        ("label_claude_opus_4_7_p2",   "Opus 4.7 p2",             "factual"),

        # Criterion-aligned local judges
        ("label_gemma_2_9b_local_p2",  "Gemma p2 local",          "local"),
        ("label_qwen2_5_7b_local_p2",  "Qwen p2 local",           "local"),
    ]

    # Axis labels of the paper's kappa-matrix figures, in order. Checked on every run so
    # that a rename in MATRIX_METHODS cannot silently diverge from the paper.
    EXPECTED_LABELS = [
        "ROUGE-L",
        "BERTScore",
        "NLI (T5-11B) strict",
        "NLI (T5-11B) best-tau",
        "GPT-5-mini p1",
        "GPT-5.4 p1",
        "Opus 4.7 p1",
        "GPT-5-mini p2",
        "GPT-5-mini p3",
        "GPT-nano p2",
        "GPT-5.4 p2",
        "Opus 4.7 p2",
        "Gemma p2 local",
        "Qwen p2 local",
    ]

    # =========================================================================
    # Helpers
    # =========================================================================

    def safe_filename(name):
        """Make model names safe for filenames."""
        return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name))


    def assert_no_forbidden_prompt_names(labels):
        bad = [
            label for label in labels
            if "p2b" in str(label).lower() or "p3b" in str(label).lower()
        ]

        if bad:
            raise ValueError(f"Forbidden prompt labels found: {bad}")


    def assert_labels_match_paper(labels):
        """Fail loudly if display labels drift away from the paper figures."""
        if list(labels) != EXPECTED_LABELS:
            only_here = [l for l in labels if l not in EXPECTED_LABELS]
            only_paper = [l for l in EXPECTED_LABELS if l not in labels]

            raise ValueError(
                "Matrix display labels do not match the paper figures.\n"
                f"  expected {len(EXPECTED_LABELS)} labels, got {len(labels)}\n"
                f"  not in paper: {only_here}\n"
                f"  missing:      {only_paper}\n"
                "If the paper figure changed, update EXPECTED_LABELS as well."
            )


    def validate_methods(df, methods):
        missing = [col for col, _, _ in methods if col not in df.columns]

        if missing:
            raise KeyError(
                "Missing required method columns from release master:\n"
                + "\n".join(f"  - {c}" for c in missing)
            )

        display_labels = [label for _, label, _ in methods]
        assert_no_forbidden_prompt_names(display_labels)
        assert_labels_match_paper(display_labels)

        duplicate_labels = pd.Series(display_labels).duplicated()
        if duplicate_labels.any():
            dupes = pd.Series(display_labels)[duplicate_labels].tolist()
            raise ValueError(f"Duplicate display labels in MATRIX_METHODS: {dupes}")

        for col, label, _ in methods:
            n_valid = int(pd.to_numeric(df[col], errors="coerce").isin([0, 1]).sum())
            if n_valid != len(df):
                raise ValueError(
                    f"{col} ({label}) has invalid labels: valid={n_valid}/{len(df)}"
                )

        methods_df = pd.DataFrame(
            [
                {
                    "column": col,
                    "display_label": label,
                    "group": group,
                }
                for col, label, group in methods
            ]
        )

        return methods_df


    def compute_pairwise_kappa_matrix(df, methods):
        """
        Compute pairwise Cohen kappa matrix with pairwise deletion.

        Invalid labels:
          - missing
          - values outside {0, 1}

        This keeps the function robust to explicit parse failures encoded as -1,
        although the release master is expected to contain complete binary labels.
        """
        labels = [m[1] for m in methods]
        cols = [m[0] for m in methods]

        n = len(cols)
        K = np.full((n, n), np.nan)

        for i, c1 in enumerate(cols):
            for j, c2 in enumerate(cols):
                x = pd.to_numeric(df[c1], errors="coerce")
                y = pd.to_numeric(df[c2], errors="coerce")

                mask = (
                    x.isin([0, 1])
                    & y.isin([0, 1])
                )

                if int(mask.sum()) < 10:
                    continue

                y1 = x.loc[mask].astype(int).to_numpy()
                y2 = y.loc[mask].astype(int).to_numpy()

                if i == j:
                    K[i, j] = 1.0
                    continue

                try:
                    K[i, j] = float(
                        cohen_kappa_score(
                            y1,
                            y2,
                            labels=[0, 1],
                        )
                    )
                except Exception as e:
                    print(f"Could not compute kappa for {labels[i]} vs {labels[j]}: {e}")

        return pd.DataFrame(K, index=labels, columns=labels)


    def plot_kappa_matrix(kappa_df, ax=None, save_path=None, figsize=(11, 9), title=None):
        """
        Plot pairwise kappa heatmap.

        Only display labels are used, so internal column names never appear
        in the rendered figure.
        """
        assert_no_forbidden_prompt_names(list(kappa_df.index) + list(kappa_df.columns))

        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.figure

        mask = np.isnan(kappa_df.to_numpy())

        sns.heatmap(
            kappa_df,
            ax=ax,
            annot=True,
            fmt=".2f",
            cmap="RdYlGn",
            vmin=0.0,
            vmax=1.0,
            mask=mask,
            linewidths=0.5,
            linecolor="white",
            annot_kws={"size": 8},
            cbar_kws={"label": "Cohen's kappa"},
        )

        if title:
            ax.set_title(title, fontsize=11, fontweight="bold", pad=10)

        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

        fig.tight_layout()

        if save_path is not None:
            save_path = Path(save_path)
            fig.savefig(str(save_path), bbox_inches="tight", dpi=300)
            print(f"  Saved: {save_path}")

        return fig, ax


    def group_summary_from_matrix(kappa_df, methods):
        groups = {
            g: [label for _, label, gg in methods if gg == g]
            for g in sorted(set(g for _, _, g in methods))
        }

        rows = []

        for g1, labels1 in groups.items():
            for g2, labels2 in groups.items():
                vals = []

                for i, l1 in enumerate(labels1):
                    for j, l2 in enumerate(labels2):
                        if l1 == l2:
                            continue

                        if l1 not in kappa_df.index or l2 not in kappa_df.columns:
                            continue

                        # Avoid double-counting within-group symmetric pairs.
                        if g1 == g2 and j <= i:
                            continue

                        v = kappa_df.loc[l1, l2]

                        if np.isfinite(v):
                            vals.append(float(v))

                if vals:
                    rows.append(
                        {
                            "group_1": g1,
                            "group_2": g2,
                            "mean_kappa": float(np.mean(vals)),
                            "min_kappa": float(np.min(vals)),
                            "max_kappa": float(np.max(vals)),
                            "n_pairs": int(len(vals)),
                        }
                    )

        return pd.DataFrame(rows), groups

    # =========================================================================
    # Load and validate release master
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)

    required_core = [
        "model",
    ]

    missing_core = [c for c in required_core if c not in master.columns]

    if missing_core:
        raise KeyError(f"Missing required core columns from release master: {missing_core}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows in release master, got {len(master)}")

    for model_name in TARGET_MODELS:
        n = int((master["model"] == model_name).sum())

        if n != 300:
            raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

    methods_df = validate_methods(master, MATRIX_METHODS)
    methods_df.to_csv(OUT_METHODS_CSV, index=False)

    print("=" * 80)
    print("PAIRWISE INTER-METHOD KAPPA MATRIX")
    print("=" * 80)
    print("Release master:", MASTER_PATH)
    print("Shape:", master.shape)
    print(f"\nResolved methods ({len(MATRIX_METHODS)}), labels match the paper's kappa-matrix figures:")
    display(methods_df)

    print("\nRows by generator:")
    print(master["model"].value_counts().reindex(TARGET_MODELS))

    # =========================================================================
    # 1. Compute per-generator matrices
    # =========================================================================

    print("\n" + "=" * 80)
    print("PER-GENERATOR KAPPA MATRICES")
    print("=" * 80)

    per_gen_matrices = {}

    for model_name in TARGET_MODELS:
        sub = master[master["model"] == model_name].copy()

        K = compute_pairwise_kappa_matrix(sub, MATRIX_METHODS)

        assert_no_forbidden_prompt_names(list(K.index) + list(K.columns))

        per_gen_matrices[model_name] = K

        out_csv = KAPPA_AUDIT_DIR / f"kappa_matrix_{safe_filename(model_name)}.csv"
        K.to_csv(out_csv)

        print(f"\n--- {model_name} ---")
        print(K.round(2).to_string())
        print("Saved CSV:", out_csv)

    # =========================================================================
    # 2. Macro-averaged matrix
    # =========================================================================

    print("\n" + "=" * 80)
    print("MACRO-AVERAGED KAPPA MATRIX")
    print("=" * 80)

    kmat_all = np.stack(
        [
            per_gen_matrices[model_name].to_numpy()
            for model_name in TARGET_MODELS
        ]
    )

    kmat_avg = np.nanmean(kmat_all, axis=0)

    labels = [m[1] for m in MATRIX_METHODS]

    K_avg = pd.DataFrame(kmat_avg, index=labels, columns=labels)

    assert_no_forbidden_prompt_names(list(K_avg.index) + list(K_avg.columns))

    K_avg.to_csv(OUT_AVG_CSV)

    print(K_avg.round(2).to_string())
    print("\nSaved macro-averaged matrix CSV:")
    print(OUT_AVG_CSV)

    fig, ax = plt.subplots(figsize=(11, 9))
    plot_kappa_matrix(
        K_avg,
        ax=ax,
        save_path=FIG_DIR / "kappa_matrix_avg.pdf",
    )
    plt.close(fig)

    # =========================================================================
    # 3. Per-generator panel figure
    # =========================================================================

    print("\n" + "=" * 80)
    print("PER-GENERATOR PANEL FIGURE")
    print("=" * 80)

    fig, axes = plt.subplots(
        1,
        len(TARGET_MODELS),
        figsize=(10 * len(TARGET_MODELS), 9),
    )

    if len(TARGET_MODELS) == 1:
        axes = [axes]

    for ax, model_name in zip(axes, TARGET_MODELS):
        plot_kappa_matrix(
            per_gen_matrices[model_name],
            ax=ax,
            title=str(model_name),
        )

    fig.tight_layout()

    for filename in [
        "kappa_matrix_all_generators.pdf",
        "kappa_matrix_per_generator.pdf",
    ]:
        out_path = FIG_DIR / filename
        fig.savefig(str(out_path), bbox_inches="tight", dpi=300)
        print(f"  Saved: {out_path}")

    plt.close(fig)

    # Individual per-generator figures
    for model_name in TARGET_MODELS:
        fig, ax = plt.subplots(figsize=(11, 9))
        plot_kappa_matrix(
            per_gen_matrices[model_name],
            ax=ax,
            save_path=FIG_DIR / f"kappa_matrix_{safe_filename(model_name)}.pdf",
        )
        plt.close(fig)

    # =========================================================================
    # 4. Within-group vs between-group summary
    # =========================================================================

    print("\n" + "=" * 80)
    print("WITHIN-GROUP VS BETWEEN-GROUP KAPPA SUMMARY")
    print("=" * 80)

    group_summary, groups = group_summary_from_matrix(K_avg, MATRIX_METHODS)
    group_summary.to_csv(OUT_GROUP_SUMMARY_CSV, index=False)

    print("\nGroup definitions:")
    for group_name, members in groups.items():
        print(f"  {group_name:<8}: {members}")

    print("\nMean kappa within and between groups:")
    display(group_summary)

    print("\nSaved group summary CSV:")
    print(OUT_GROUP_SUMMARY_CSV)

    # Values quoted in Appendix D.4: the factual block and the local-to-factual
    # alignment. These must not move when the method list changes.
    def _summary_row(g1, g2):
        m = group_summary[
            (group_summary["group_1"] == g1) & (group_summary["group_2"] == g2)
        ]
        return None if m.empty else m.iloc[0]

    for g1, g2, exp_mean, exp_min, exp_max, where in [
        ("factual", "factual", 0.781, 0.719, 0.879, "D.4 factual block"),
        ("factual", "local", 0.601, 0.531, 0.660, "D.4 local p2 judges"),
    ]:
        r = _summary_row(g1, g2) if _summary_row(g1, g2) is not None else _summary_row(g2, g1)

        if r is None:
            raise AssertionError(f"Group summary is missing the {g1} x {g2} row.")

        for got, exp, what in [
            (r["mean_kappa"], exp_mean, "mean"),
            (r["min_kappa"], exp_min, "min"),
            (r["max_kappa"], exp_max, "max"),
        ]:
            if abs(got - exp) > 0.0006:
                raise AssertionError(
                    f"{where}: {g1} x {g2} {what} kappa is {got:.3f}, "
                    f"paper reports {exp:.3f}."
                )

    print("\u2713 Appendix D.4 group values match the paper.")

    # =========================================================================
    # 5. Final guardrails
    # =========================================================================

    n_methods = len(MATRIX_METHODS)

    for name, K in [("macro-average", K_avg)] + list(per_gen_matrices.items()):
        if K.shape != (n_methods, n_methods):
            raise AssertionError(
                f"{name} matrix has shape {K.shape}, expected "
                f"({n_methods}, {n_methods})."
            )

        assert_labels_match_paper(list(K.index))
        assert_labels_match_paper(list(K.columns))

        if not np.allclose(np.diag(K.to_numpy()), 1.0):
            raise AssertionError(f"{name} matrix diagonal is not all 1.0.")

        if not np.allclose(K.to_numpy(), K.to_numpy().T, equal_nan=True):
            raise AssertionError(f"{name} matrix is not symmetric.")

    expected_figures = [
        FIG_DIR / "kappa_matrix_avg.pdf",
        FIG_DIR / "kappa_matrix_all_generators.pdf",
        FIG_DIR / "kappa_matrix_per_generator.pdf",
    ]

    for model_name in TARGET_MODELS:
        expected_figures.append(
            FIG_DIR / f"kappa_matrix_{safe_filename(model_name)}.pdf"
        )

    expected_csvs = [
        OUT_AVG_CSV,
        OUT_GROUP_SUMMARY_CSV,
        OUT_METHODS_CSV,
    ]

    for model_name in TARGET_MODELS:
        expected_csvs.append(
            KAPPA_AUDIT_DIR / f"kappa_matrix_{safe_filename(model_name)}.csv"
        )

    missing_outputs = [
        str(path)
        for path in expected_figures + expected_csvs
        if not Path(path).exists()
    ]

    if missing_outputs:
        raise FileNotFoundError(
            "Expected kappa-matrix outputs were not created:\n"
            + "\n".join(f"  - {p}" for p in missing_outputs)
        )

    print("\u2713 MATRIX STRUCTURE CHECK PASSED")
    print(f"  {n_methods}x{n_methods}, symmetric, unit diagonal, labels match paper.")

    print("\n" + "=" * 80)
    print("KAPPA MATRIX ANALYSIS COMPLETE")
    print("=" * 80)
    print("Output directory:", FIG_DIR)
    print("Audit directory: ", KAPPA_AUDIT_DIR)
    print("Note: this matrix compares automated label sources to each other, not to human labels.")
    print("=" * 80)

In [ ]:
# =============================================================================
# CELL C5: Bootstrap CIs + McNemar tests for GPT-5-mini prompt comparisons
#
# Purpose:
#   Paired prompt comparisons for the primary ablation judge: bootstrap
#   confidence intervals for kappa and delta-kappa, plus exact two-sided
#   McNemar tests over item-level correctness disagreements.
#
# Source of truth:
#   MASTER_PATH = data/human_judge_release_master.csv
#
# Uses:
#   annotator1_label
#   label_gpt_5_mini_p1 / p2 / p3
#
# Outputs:
#   tables/table_prompt_delta_bootstrap_mcnemar.tex
#   audits/bootstrap_mcnemar/bootstrap_kappa_by_prompt.csv
#   audits/bootstrap_mcnemar/bootstrap_delta_mcnemar.csv
#
# Paper label:
#   tab:prompt_comparisons
# =============================================================================

import math
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from scipy.stats import chi2, binomtest

# =============================================================================
# Paths
# =============================================================================

AUDIT_OUT_DIR = AUDIT_DIR / "bootstrap_mcnemar"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_KAPPA_CSV = AUDIT_OUT_DIR / "bootstrap_kappa_by_prompt.csv"
OUT_DELTA_CSV = AUDIT_OUT_DIR / "bootstrap_delta_mcnemar.csv"

OUT_TEX = TABLE_DIR / "table_prompt_delta_bootstrap_mcnemar.tex"

# =============================================================================
# Config
# =============================================================================

HUMAN_COL = "annotator1_label"

JUDGE_NAME = "GPT-5-mini"
JUDGE_PREFIX = "label_gpt_5_mini"

PROMPTS = ["p1", "p2", "p3"]

CONTRASTS = [
    ("p1", "p2"),
    ("p1", "p3"),
    ("p2", "p3"),
]

if "TARGET_MODELS" in globals():
    GEN_ORDER = list(TARGET_MODELS)
else:
    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

GEN_DISPLAY = {
    "llama3-8b": "L",
    "gemma-2-9b": "G",
    "mistral-7b": "M",
}

N_BOOT = 2000
BOOT_SEED = 42

# =============================================================================
# Helpers
# =============================================================================

def kappa_binary(y_true, y_pred):
    return float(cohen_kappa_score(y_true, y_pred, labels=[0, 1]))


def bootstrap_kappa_ci(y_true, y_pred, n_boot=N_BOOT, seed=BOOT_SEED):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    n = len(y_true)

    vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)

        try:
            k = kappa_binary(y_true[idx], y_pred[idx])
            if np.isfinite(k):
                vals.append(k)
        except Exception:
            pass

    vals = np.asarray(vals)

    if len(vals) == 0:
        return np.nan, np.nan, 0

    lo, hi = np.percentile(vals, [2.5, 97.5])

    return float(lo), float(hi), int(len(vals))


def bootstrap_delta_kappa_ci(y_true, pred_a, pred_b, n_boot=N_BOOT, seed=BOOT_SEED):
    """
    Bootstrap CI for Δκ = κ(pred_b) - κ(pred_a).
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    pred_a = np.asarray(pred_a).astype(int)
    pred_b = np.asarray(pred_b).astype(int)
    n = len(y_true)

    vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)

        try:
            ka = kappa_binary(y_true[idx], pred_a[idx])
            kb = kappa_binary(y_true[idx], pred_b[idx])
            d = kb - ka

            if np.isfinite(d):
                vals.append(d)
        except Exception:
            pass

    vals = np.asarray(vals)

    if len(vals) == 0:
        return np.nan, np.nan, 0

    lo, hi = np.percentile(vals, [2.5, 97.5])

    return float(lo), float(hi), int(len(vals))


def exact_mcnemar_p(b, c):
    """
    Exact two-sided McNemar test over paired correctness disagreements.

    b = prompt A wrong, prompt B correct
    c = prompt A correct, prompt B wrong

    Under H0, b ~ Binomial(b+c, 0.5).
    """
    n = int(b + c)

    if n == 0:
        return 1.0

    return float(binomtest(k=int(min(b, c)), n=n, p=0.5, alternative="two-sided").pvalue)


def chi2_mcnemar_p_continuity(b, c):
    """
    Continuity-corrected chi-square McNemar p-value.
    Included for audit compatibility with the older notebook.
    """
    n = int(b + c)

    if n == 0:
        return 1.0

    stat = (abs(b - c) - 1) ** 2 / n
    return float(1.0 - chi2.cdf(stat, df=1))


def paired_correctness_cells(y_true, pred_a, pred_b):
    """
    Return paired correctness discordance counts.

    a_wrong_b_right:
      prompt A is wrong, prompt B is correct

    a_right_b_wrong:
      prompt A is correct, prompt B is wrong
    """
    y_true = np.asarray(y_true).astype(int)
    pred_a = np.asarray(pred_a).astype(int)
    pred_b = np.asarray(pred_b).astype(int)

    a_correct = pred_a == y_true
    b_correct = pred_b == y_true

    a_wrong_b_right = int(np.sum((~a_correct) & b_correct))
    a_right_b_wrong = int(np.sum(a_correct & (~b_correct)))

    return a_wrong_b_right, a_right_b_wrong


def fmt_p(p):
    if p < 0.0001:
        return r"$<0.0001$"
    return f"{p:.4f}"


def fmt_ci(lo, hi, signed=False):
    if signed:
        return f"[{lo:+.3f}, {hi:+.3f}]"
    return f"[{lo:.3f}, {hi:.3f}]"


# =============================================================================
# Load and validate
# =============================================================================

master = pd.read_csv(MASTER_PATH)

required_cols = [HUMAN_COL, "model"]

for p in PROMPTS:
    required_cols.append(f"{JUDGE_PREFIX}_{p}")

missing = [c for c in required_cols if c not in master.columns]
if missing:
    raise KeyError(f"Missing required columns from release master: {missing}")

if len(master) != 900:
    raise ValueError(f"Expected 900 release rows, got {len(master)}")

if not master[HUMAN_COL].isin([0, 1]).all():
    bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
    raise ValueError(f"Invalid values in {HUMAN_COL}: {bad_n}")

for model_name in GEN_ORDER:
    n = int((master["model"] == model_name).sum())

    if n != 300:
        raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

for p in PROMPTS:
    col = f"{JUDGE_PREFIX}_{p}"
    n_valid = int(master[col].isin([0, 1]).sum())

    if n_valid != len(master):
        raise ValueError(f"{col} has invalid labels: valid={n_valid}/{len(master)}")

print("=" * 80)
print("BOOTSTRAP CIs + McNEMAR TESTS")
print("=" * 80)
print("Release master:", MASTER_PATH)
print("Judge:", JUDGE_NAME)
print("Bootstrap resamples:", N_BOOT)
print("Seed:", BOOT_SEED)

# =============================================================================
# Compute prompt-level bootstrap kappa CIs
# =============================================================================

kappa_rows = []
delta_rows = []

for model_name in GEN_ORDER:
    sub = master[master["model"] == model_name].copy()
    gen = GEN_DISPLAY[model_name]

    y_true_all = pd.to_numeric(sub[HUMAN_COL], errors="coerce")

    pred_all = {
        p: pd.to_numeric(sub[f"{JUDGE_PREFIX}_{p}"], errors="coerce")
        for p in PROMPTS
    }

    valid = y_true_all.isin([0, 1])

    for p in PROMPTS:
        valid &= pred_all[p].isin([0, 1])

    n_valid = int(valid.sum())

    if n_valid != len(sub):
        raise ValueError(
            f"Invalid paired labels for {model_name}: valid={n_valid}/{len(sub)}"
        )

    y_true = y_true_all.loc[valid].astype(int).to_numpy()

    preds = {
        p: pred_all[p].loc[valid].astype(int).to_numpy()
        for p in PROMPTS
    }

    print(f"\n--- {model_name} ({gen}) ---")
    print(f"Paired valid rows: {n_valid}/{len(sub)}")

    # Prompt kappas with bootstrap CIs
    for p in PROMPTS:
        k = kappa_binary(y_true, preds[p])
        lo, hi, n_boot_valid = bootstrap_kappa_ci(
            y_true,
            preds[p],
            n_boot=N_BOOT,
            seed=BOOT_SEED,
        )

        kappa_rows.append({
            "judge": JUDGE_NAME,
            "model": model_name,
            "gen": gen,
            "prompt": p,
            "n_valid": n_valid,
            "kappa": k,
            "ci_low": lo,
            "ci_high": hi,
            "n_boot_valid": n_boot_valid,
        })

        print(f"  {p}: κ={k:.3f} [{lo:.3f}, {hi:.3f}]")

    # Delta-kappa + McNemar
    for p_a, p_b in CONTRASTS:
        ka = kappa_binary(y_true, preds[p_a])
        kb = kappa_binary(y_true, preds[p_b])
        delta = kb - ka

        lo, hi, n_boot_valid = bootstrap_delta_kappa_ci(
            y_true,
            preds[p_a],
            preds[p_b],
            n_boot=N_BOOT,
            seed=BOOT_SEED,
        )

        a_wrong_b_right, a_right_b_wrong = paired_correctness_cells(
            y_true,
            preds[p_a],
            preds[p_b],
        )

        p_exact = exact_mcnemar_p(a_wrong_b_right, a_right_b_wrong)
        p_chi2 = chi2_mcnemar_p_continuity(a_wrong_b_right, a_right_b_wrong)

        sig = "***" if (lo > 0 or hi < 0) else "ns"

        delta_rows.append({
            "judge": JUDGE_NAME,
            "model": model_name,
            "gen": gen,
            "contrast": f"{p_a}$\\to${p_b}",
            "prompt_a": p_a,
            "prompt_b": p_b,
            "n_valid": n_valid,
            "kappa_a": ka,
            "kappa_b": kb,
            "delta_kappa": delta,
            "delta_ci_low": lo,
            "delta_ci_high": hi,
            "delta_boot_valid": n_boot_valid,
            "a_wrong_b_right": a_wrong_b_right,
            "a_right_b_wrong": a_right_b_wrong,
            "discordant": a_wrong_b_right + a_right_b_wrong,
            "mcnemar_p_exact": p_exact,
            "mcnemar_p_chi2_continuity": p_chi2,
            "delta_ci_excludes_zero": bool(lo > 0 or hi < 0),
        })

        print(
            f"  {p_a}->{p_b}: "
            f"Δκ={delta:+.3f} [{lo:+.3f}, {hi:+.3f}] {sig}; "
            f"McNemar exact p={p_exact:.4f} "
            f"(A-wrong/B-right={a_wrong_b_right}, "
            f"A-right/B-wrong={a_right_b_wrong})"
        )

kappa_audit = pd.DataFrame(kappa_rows)
delta_audit = pd.DataFrame(delta_rows)

kappa_audit["kappa_3dp"] = kappa_audit["kappa"].round(3)
kappa_audit["ci_low_3dp"] = kappa_audit["ci_low"].round(3)
kappa_audit["ci_high_3dp"] = kappa_audit["ci_high"].round(3)

delta_audit["kappa_a_3dp"] = delta_audit["kappa_a"].round(3)
delta_audit["kappa_b_3dp"] = delta_audit["kappa_b"].round(3)
delta_audit["delta_kappa_3dp"] = delta_audit["delta_kappa"].round(3)
delta_audit["delta_ci_low_3dp"] = delta_audit["delta_ci_low"].round(3)
delta_audit["delta_ci_high_3dp"] = delta_audit["delta_ci_high"].round(3)

kappa_audit.to_csv(OUT_KAPPA_CSV, index=False)
delta_audit.to_csv(OUT_DELTA_CSV, index=False)

print("\n" + "=" * 80)
print("KAPPA BOOTSTRAP AUDIT")
print("=" * 80)
display(
    kappa_audit[
        [
            "judge",
            "gen",
            "prompt",
            "n_valid",
            "kappa_3dp",
            "ci_low_3dp",
            "ci_high_3dp",
            "n_boot_valid",
        ]
    ]
)

print("\n" + "=" * 80)
print("DELTA-KAPPA + McNEMAR AUDIT")
print("=" * 80)
display(
    delta_audit[
        [
            "judge",
            "gen",
            "contrast",
            "n_valid",
            "kappa_a_3dp",
            "kappa_b_3dp",
            "delta_kappa_3dp",
            "delta_ci_low_3dp",
            "delta_ci_high_3dp",
            "a_wrong_b_right",
            "a_right_b_wrong",
            "discordant",
            "mcnemar_p_exact",
            "mcnemar_p_chi2_continuity",
        ]
    ]
)

print("\nSaved audit CSVs:")
print("  ", OUT_KAPPA_CSV)
print("  ", OUT_DELTA_CSV)

# =============================================================================
# LaTeX table for paper / appendix
# =============================================================================

caption = (
    r"Paired prompt comparisons for GPT-5-mini against the primary human "
    r"labels. $\Delta\kappa$ is computed as $\kappa_b-\kappa_a$ for the "
    r"listed contrast. Brackets give 95\% nonparametric bootstrap intervals "
    r"for $\Delta\kappa$ with 2000 resamples. McNemar $p$ is the exact "
    r"two-sided paired test over correctness disagreements between the two "
    r"prompt variants. L=Llama-3-8B, G=Gemma-2-9B, M=Mistral-7B."
)

latex = r"""\begin{table}[t]
  \centering
  \caption{""" + caption + r"""}
  \label{tab:prompt_comparisons}
  \small
  \setlength{\tabcolsep}{4pt}
  \begin{tabular}{llrrrr}
    \toprule
    Gen & Contrast & $\kappa_a$ & $\kappa_b$ & $\Delta\kappa$ [95\% CI] & McNemar $p$ \\
    \midrule
"""

for _, r in delta_audit.iterrows():
    latex += (
        f"    {r['gen']} & {r['contrast']} "
        f"& {r['kappa_a']:.3f} "
        f"& {r['kappa_b']:.3f} "
        f"& {r['delta_kappa']:+.3f} {fmt_ci(r['delta_ci_low'], r['delta_ci_high'], signed=True)} "
        f"& {fmt_p(r['mcnemar_p_exact'])} \\\\\n"
    )

latex += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

with open(OUT_TEX, "w") as f:
    f.write(latex)

print("\nSaved table file:")
print("  ", OUT_TEX)

# =============================================================================
# Final guardrails
# =============================================================================

for path in [OUT_KAPPA_CSV, OUT_DELTA_CSV, OUT_TEX]:
    if not path.exists():
        raise FileNotFoundError(f"Expected output not created: {path}")

for path in [OUT_KAPPA_CSV, OUT_DELTA_CSV, OUT_TEX]:
    text = path.read_text()
    if "P2B" in text or "P3B" in text:
        raise ValueError(f"Internal prompt name leaked into output: {path}")

print("\n" + "=" * 80)
print("Bootstrap + McNemar analysis complete.")
print("=" * 80)

In [ ]:
# =============================================================================
# CELL C6: Error-direction space from release master CSV
#
# Purpose:
#   Plot macro-averaged false-positive and false-negative rates for each
#   automatic label source, with p1 -> p2 -> p3 trajectories for the
#   multi-prompt judges.
#
# Source of truth:
#   MASTER_PATH = data/human_judge_release_master.csv
#
# Uses:
#   annotator1_label
#   label_*_p1 / p2 / p3 for all seven judges
#   rougeL_label_oracle, bertscore_label_oracle
#   nli_label_strict, nli_label_best_tau
#
# Outputs:
#   figures/error_direction_space.pdf           (fig:error_direction_space)
#   audits/error_direction_space/error_direction_space_points.csv
#   audits/error_direction_space/error_direction_space_trajectories.csv
#
# Paper label:
#   fig:error_direction_space
#
# Note:
#   The figure shows the measured points, the equal-rate diagonal, and the
#   prompt trajectories only. Interpretive region shading and region captions
#   are deliberately not drawn: they are not in the paper figure, and any
#   "low-error" boundary would be an unstated analytical choice.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import confusion_matrix, cohen_kappa_score

# =============================================================================
# Paths
# =============================================================================

# Uses globals from the config cell:
#   MASTER_PATH, FIG_DIR, AUDIT_DIR, TARGET_MODELS

FIG_DIR.mkdir(parents=True, exist_ok=True)

ERR_AUDIT_DIR = AUDIT_DIR / "error_direction_space"
ERR_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FIG_OUT = FIG_DIR / "error_direction_space.pdf"
POINTS_CSV = ERR_AUDIT_DIR / "error_direction_space_points.csv"
TRAJ_CSV = ERR_AUDIT_DIR / "error_direction_space_trajectories.csv"

# =============================================================================
# Config
# =============================================================================

HUMAN_COL = "annotator1_label"

if "TARGET_MODELS" in globals():
    GEN_ORDER = list(TARGET_MODELS)
else:
    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

# -----------------------------------------------------------------------------
# Plot styling
# -----------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
})

MODEL_COLORS = {
    "G5-nano":       "#4ade80",
    "G5-mini":       "#10a37f",
    "GPT-5.4":       "#065f46",
    "Opus 4.7":      "#d4761f",
    "Gemma (local)": "#4285f4",
    "Llama (local)": "#0c2d5c",
    "Qwen (local)":  "#a855f7",
    "NLI (T5-11B)":  "#c0392b",
    "ROUGE-L":       "#6b7280",
    "BERTScore":     "#9ca3af",
}

PROMPT_MARKERS = {
    "p1":       "v",
    "p2":       "o",
    "p3":       "s",
    "strict":   "X",
    "best-tau": "D",
}

FRONTIER_LABELS = {"GPT-5.4", "Opus 4.7"}

# -----------------------------------------------------------------------------
# Judge / baseline definitions
#
# Each entry:
#   display label,
#   ordered list of (criterion display, release-master column),
#   draw prompt trajectory arrows?
# -----------------------------------------------------------------------------

JUDGE_DEFS = [
    (
        "ROUGE-L",
        [("best-tau", "rougeL_label_oracle")],
        False,
    ),
    (
        "BERTScore",
        [("best-tau", "bertscore_label_oracle")],
        False,
    ),
    (
        "NLI (T5-11B)",
        [
            ("strict",   "nli_label_strict"),
            ("best-tau", "nli_label_best_tau"),
        ],
        False,
    ),
    (
        "G5-mini",
        [
            ("p1", "label_gpt_5_mini_p1"),
            ("p2", "label_gpt_5_mini_p2"),
            ("p3", "label_gpt_5_mini_p3"),
        ],
        True,
    ),
    (
        "G5-nano",
        [
            ("p1", "label_gpt_5_nano_p1"),
            ("p2", "label_gpt_5_nano_p2"),
            ("p3", "label_gpt_5_nano_p3"),
        ],
        True,
    ),
    (
        "GPT-5.4",
        [
            ("p1", "label_gpt_5_4_p1"),
            ("p2", "label_gpt_5_4_p2"),
            ("p3", "label_gpt_5_4_p3"),
        ],
        True,
    ),
    (
        "Opus 4.7",
        [
            ("p1", "label_claude_opus_4_7_p1"),
            ("p2", "label_claude_opus_4_7_p2"),
            ("p3", "label_claude_opus_4_7_p3"),
        ],
        True,
    ),
    (
        "Gemma (local)",
        [
            ("p1", "label_gemma_2_9b_local_p1"),
            ("p2", "label_gemma_2_9b_local_p2"),
            ("p3", "label_gemma_2_9b_local_p3"),
        ],
        True,
    ),
    (
        "Qwen (local)",
        [
            ("p1", "label_qwen2_5_7b_local_p1"),
            ("p2", "label_qwen2_5_7b_local_p2"),
            ("p3", "label_qwen2_5_7b_local_p3"),
        ],
        True,
    ),
    (
        "Llama (local)",
        [
            ("p1", "label_llama_3_8b_local_p1"),
            ("p2", "label_llama_3_8b_local_p2"),
            ("p3", "label_llama_3_8b_local_p3"),
        ],
        True,
    ),
]

# =============================================================================
# Helpers
# =============================================================================

def assert_no_internal_prompt_names(text_or_labels):
    """Prevent old internal prompt names from appearing in outputs."""
    if isinstance(text_or_labels, str):
        values = [text_or_labels]
    else:
        values = list(text_or_labels)

    bad = [
        x for x in values
        if "p2b" in str(x).lower() or "p3b" in str(x).lower()
    ]

    if bad:
        raise ValueError(f"Internal prompt names leaked into output: {bad}")


def compute_error_rates(df_all, pred_col, human_col=HUMAN_COL):
    """
    Compute macro-averaged FP-rate, FN-rate, and kappa over generators.

    FP-rate = FP / (FP + TN)
    FN-rate = FN / (FN + TP)
    """
    fp_rates = []
    fn_rates = []
    kappas = []
    n_valids = []
    n_totals = []

    for model_name in GEN_ORDER:
        df = df_all[df_all["model"] == model_name].copy()

        if len(df) != 300:
            raise ValueError(f"Expected 300 rows for {model_name}, got {len(df)}")

        y_true = pd.to_numeric(df[human_col], errors="coerce")
        y_pred = pd.to_numeric(df[pred_col], errors="coerce")

        mask = y_true.isin([0, 1]) & y_pred.isin([0, 1])

        n_total = len(df)
        n_valid = int(mask.sum())

        if n_valid != n_total:
            raise ValueError(
                f"{pred_col} has invalid labels for {model_name}: "
                f"valid={n_valid}/{n_total}"
            )

        yh = y_true.loc[mask].astype(int).to_numpy()
        yp = y_pred.loc[mask].astype(int).to_numpy()

        tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()

        fp_rates.append(fp / max(fp + tn, 1))
        fn_rates.append(fn / max(fn + tp, 1))
        kappas.append(cohen_kappa_score(yh, yp, labels=[0, 1]))

        n_valids.append(n_valid)
        n_totals.append(n_total)

    return {
        "fp_rate": float(np.mean(fp_rates)),
        "fn_rate": float(np.mean(fn_rates)),
        "kappa": float(np.mean(kappas)),
        "n_valid": int(np.sum(n_valids)),
        "n_total": int(np.sum(n_totals)),
    }


# =============================================================================
# Load and validate release master
# =============================================================================

master = pd.read_csv(MASTER_PATH)

print("=" * 80)
print("ERROR-DIRECTION SPACE INPUT")
print("=" * 80)
print("Release master:", MASTER_PATH)
print("Shape:", master.shape)

if len(master) != 900:
    raise ValueError(f"Expected 900 release rows, got {len(master)}")

required_cols = [HUMAN_COL, "model"]

for _, prompt_cols, _ in JUDGE_DEFS:
    for _, col in prompt_cols:
        required_cols.append(col)

missing = [c for c in required_cols if c not in master.columns]
if missing:
    raise KeyError(f"Missing required columns from release master: {missing}")

if not master[HUMAN_COL].isin([0, 1]).all():
    bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
    raise ValueError(f"Invalid human labels in {HUMAN_COL}: {bad_n}")

for model_name in GEN_ORDER:
    n = int((master["model"] == model_name).sum())
    if n != 300:
        raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

label_cols = [
    c for c in required_cols
    if c.startswith("label_")
    or c in [
        "rougeL_label_oracle",
        "bertscore_label_oracle",
        "nli_label_strict",
        "nli_label_best_tau",
    ]
]

for c in label_cols:
    n_valid = int(master[c].isin([0, 1]).sum())
    invalid = len(master) - n_valid

    print(f"  {c:<35} valid={n_valid}/{len(master)} invalid={invalid}")

    if n_valid != len(master):
        raise ValueError(f"{c} has invalid labels: valid={n_valid}/{len(master)}")

display_labels = [label for label, _, _ in JUDGE_DEFS]
assert_no_internal_prompt_names(display_labels)

print("\nRows by generator:")
print(master["model"].value_counts().reindex(GEN_ORDER))

print("\nAll required columns found and valid.")

# =============================================================================
# Collect macro-averaged points
# =============================================================================

points = []

for label, prompt_cols, draw_arrow in JUDGE_DEFS:
    prev_xy = None

    for prompt_name, col in prompt_cols:
        result = compute_error_rates(master, col)

        fp_rate = result["fp_rate"]
        fn_rate = result["fn_rate"]

        points.append({
            "label": label,
            "prompt": prompt_name,
            "column": col,
            "fp_rate": fp_rate,
            "fn_rate": fn_rate,
            "kappa": result["kappa"],
            "n_valid": result["n_valid"],
            "n_total": result["n_total"],
            "draw_arrow": draw_arrow,
            "prev_x": np.nan if prev_xy is None else prev_xy[0],
            "prev_y": np.nan if prev_xy is None else prev_xy[1],
            "is_frontier": label in FRONTIER_LABELS,
        })

        prev_xy = (fp_rate, fn_rate)

df_pts = pd.DataFrame(points)

df_pts["fp_rate_3dp"] = df_pts["fp_rate"].round(3)
df_pts["fn_rate_3dp"] = df_pts["fn_rate"].round(3)
df_pts["kappa_3dp"] = df_pts["kappa"].round(3)

df_pts.to_csv(POINTS_CSV, index=False)

print("\n" + "=" * 80)
print("ERROR-DIRECTION POINTS")
print("=" * 80)

display_cols = [
    "label",
    "prompt",
    "fp_rate_3dp",
    "fn_rate_3dp",
    "kappa_3dp",
    "n_valid",
    "n_total",
]

display(df_pts[display_cols])

print("\nSaved points audit CSV:")
print(POINTS_CSV)

# =============================================================================
# Movement summary: p1 -> p2
# =============================================================================

traj_rows = []

for lab in df_pts["label"].unique():
    sub = df_pts[df_pts["label"] == lab].set_index("prompt")

    if "p1" in sub.index and "p2" in sub.index:
        p1 = sub.loc["p1"]
        p2 = sub.loc["p2"]

        traj_rows.append({
            "label": lab,
            "dFP_p1_p2": float(p2["fp_rate"] - p1["fp_rate"]),
            "dFN_p1_p2": float(p2["fn_rate"] - p1["fn_rate"]),
            "dkappa_p1_p2": float(p2["kappa"] - p1["kappa"]),
            "p1_FP_rate": float(p1["fp_rate"]),
            "p2_FP_rate": float(p2["fp_rate"]),
            "p1_FN_rate": float(p1["fn_rate"]),
            "p2_FN_rate": float(p2["fn_rate"]),
            "p1_kappa": float(p1["kappa"]),
            "p2_kappa": float(p2["kappa"]),
        })

traj_df = pd.DataFrame(traj_rows)

traj_df["dFP_p1_p2_3dp"] = traj_df["dFP_p1_p2"].round(3)
traj_df["dFN_p1_p2_3dp"] = traj_df["dFN_p1_p2"].round(3)
traj_df["dkappa_p1_p2_3dp"] = traj_df["dkappa_p1_p2"].round(3)

traj_df.to_csv(TRAJ_CSV, index=False)

print("\n" + "=" * 80)
print("PROMPT MOVEMENT p1 -> p2")
print("=" * 80)

if len(traj_df) > 0:
    display(
        traj_df[
            [
                "label",
                "dFP_p1_p2_3dp",
                "dFN_p1_p2_3dp",
                "dkappa_p1_p2_3dp",
                "p1_FP_rate",
                "p2_FP_rate",
                "p1_FN_rate",
                "p2_FN_rate",
            ]
        ].sort_values("dFP_p1_p2_3dp")
    )
else:
    print("No p1 -> p2 trajectories found.")

print("\nSaved trajectory audit CSV:")
print(TRAJ_CSV)

# =============================================================================
# Label selection and offsets
# =============================================================================

LABELS_TO_SHOW = {
    ("GPT-5.4",       "p1"),
    ("Opus 4.7",      "p1"),
    ("G5-mini",       "p1"),
    ("NLI (T5-11B)",  "strict"),
    ("NLI (T5-11B)",  "best-tau"),
    ("ROUGE-L",       "best-tau"),
    ("BERTScore",     "best-tau"),
    ("Llama (local)", "p1"),
    ("Gemma (local)", "p1"),
    ("Qwen (local)",  "p1"),
}

LABEL_OFFSETS = {
    ("GPT-5.4",       "p1"):       (0.005,  0.018),
    ("Opus 4.7",      "p1"):       (0.015, -0.045),
    ("G5-mini",       "p1"):       (0.015, -0.035),
    ("NLI (T5-11B)",  "strict"):   (-0.15, 0.070),
    ("NLI (T5-11B)",  "best-tau"): (0.018,  0.025),
    ("ROUGE-L",       "best-tau"): (-0.10,  0.030),
    ("BERTScore",     "best-tau"): (-0.07, 0.040),
    ("Llama (local)", "p1"):       (0.015, -0.035),
    ("Gemma (local)", "p1"):       (0.015,  0.020),
    ("Qwen (local)",  "p1"):       (0.015,  0.020),
}

# =============================================================================
# Plot
# =============================================================================

fig, ax = plt.subplots(figsize=(9, 7))

# Diagonal: equal FP/FN rate
ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.35)

POINT_SIZE = 90
FRONTIER_SIZE = 140

# Arrows
for _, row in df_pts.iterrows():
    if bool(row["draw_arrow"]) and np.isfinite(row["prev_x"]) and np.isfinite(row["prev_y"]):
        x0, y0 = row["prev_x"], row["prev_y"]
        x1, y1 = row["fp_rate"], row["fn_rate"]

        arrow_color = MODEL_COLORS[row["label"]]

        if bool(row["is_frontier"]):
            arrow_lw = 2.0
            arrow_alpha = 1.0
            arrow_style = "->,head_width=0.7,head_length=0.9"
        else:
            arrow_lw = 1.4
            arrow_alpha = 0.75
            arrow_style = "->,head_width=0.3,head_length=0.4"

        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(
                arrowstyle=arrow_style,
                color=arrow_color,
                lw=arrow_lw,
                alpha=arrow_alpha,
                connectionstyle="arc3,rad=0.06",
            ),
        )

# Points
for _, row in df_pts.iterrows():
    color = MODEL_COLORS[row["label"]]
    marker = PROMPT_MARKERS.get(row["prompt"], "o")

    if bool(row["is_frontier"]):
        size = FRONTIER_SIZE
        edge_color = "#222222"
        edge_lw = 1.5
    else:
        size = POINT_SIZE
        edge_color = "white"
        edge_lw = 0.7

    ax.scatter(
        row["fp_rate"],
        row["fn_rate"],
        c=color,
        marker=marker,
        s=size,
        edgecolors=edge_color,
        linewidths=edge_lw,
        zorder=5,
        alpha=0.92,
    )

# Text labels
for _, row in df_pts.iterrows():
    key = (row["label"], row["prompt"])

    if key not in LABELS_TO_SHOW:
        continue

    dx, dy = LABEL_OFFSETS.get(key, (0.013, 0.013))

    if row["prompt"] in ("best-tau", "strict"):
        tag = f" ({row['prompt']})"
    else:
        tag = f" {row['prompt']}"

    fontweight = "bold" if bool(row["is_frontier"]) else "normal"

    ax.annotate(
        f"{row['label']}{tag}",
        xy=(row["fp_rate"], row["fn_rate"]),
        xytext=(row["fp_rate"] + dx, row["fn_rate"] + dy),
        fontsize=7,
        color="#222222",
        ha="left",
        va="bottom",
        fontweight=fontweight,
        arrowprops=dict(
            arrowstyle="-",
            color="#bbbbbb",
            lw=0.5,
            shrinkA=0,
            shrinkB=3,
        ),
    )

# Legends
model_patches = [
    mpatches.Patch(color=MODEL_COLORS["G5-nano"], label="GPT-5-nano"),
    mpatches.Patch(color=MODEL_COLORS["G5-mini"], label="GPT-5-mini"),
    mpatches.Patch(color=MODEL_COLORS["GPT-5.4"], label="GPT-5.4 (frontier)"),
    mpatches.Patch(color=MODEL_COLORS["Opus 4.7"], label="Claude Opus 4.7 (frontier)"),
    mpatches.Patch(color=MODEL_COLORS["Gemma (local)"], label="Gemma-2-9B (local)"),
    mpatches.Patch(color=MODEL_COLORS["Llama (local)"], label="Llama-3-8B (local)"),
    mpatches.Patch(color=MODEL_COLORS["Qwen (local)"], label="Qwen2.5-7B (local)"),
    mpatches.Patch(color=MODEL_COLORS["NLI (T5-11B)"], label="NLI (T5-11B)"),
    mpatches.Patch(color=MODEL_COLORS["ROUGE-L"], label="ROUGE-L"),
    mpatches.Patch(color=MODEL_COLORS["BERTScore"], label="BERTScore"),
]

prompt_handles = [
    plt.scatter([], [], marker="v", c="gray", s=70, label="p1 (faithfulness)"),
    plt.scatter([], [], marker="o", c="gray", s=70, label="p2 (factual)"),
    plt.scatter([], [], marker="s", c="gray", s=70, label="p3 (factual ext.)"),
    plt.scatter([], [], marker="X", c="gray", s=70, label="strict (NLI)"),
    plt.scatter([], [], marker="D", c="gray", s=70, label="best-tau (oracle)"),
]

leg1 = ax.legend(
    handles=model_patches,
    title="Judge / baseline (color)",
    loc="upper left",
    fontsize=7.2,
    title_fontsize=8,
    framealpha=0.92,
)

leg2 = ax.legend(
    handles=prompt_handles,
    title="Criterion (shape)",
    loc="upper right",
    fontsize=7.2,
    title_fontsize=8,
    framealpha=0.92,
)

ax.add_artist(leg1)

# Axes
ax.set_xlabel("FP-rate = FP / (FP + TN)", fontsize=11)
ax.set_ylabel("FN-rate = FN / (FN + TP)", fontsize=11)
ax.set_title("Error-direction space of automatic label sources", fontsize=11)

ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect("equal")
ax.grid(True, alpha=0.22)

plt.tight_layout()

fig.savefig(str(FIG_OUT), dpi=300, bbox_inches="tight")

if not FIG_OUT.exists():
    raise FileNotFoundError(f"Expected figure was not created: {FIG_OUT}")

for csv_path in [POINTS_CSV, TRAJ_CSV]:
    if not csv_path.exists():
        raise FileNotFoundError(f"Expected audit CSV was not created: {csv_path}")

# Guardrail: no internal prompt names in audit CSVs
for csv_path in [POINTS_CSV, TRAJ_CSV]:
    text = csv_path.read_text()
    if "P2B" in text or "P3B" in text:
        raise ValueError(f"Internal prompt name leaked into audit CSV: {csv_path}")

# =============================================================================
# Structural checks
# =============================================================================

expected_points = sum(len(prompt_cols) for _, prompt_cols, _ in JUDGE_DEFS)
expected_trajectories = sum(1 for _, _, draw in JUDGE_DEFS if draw)

if len(df_pts) != expected_points:
    raise AssertionError(
        f"Expected {expected_points} points, found {len(df_pts)}."
    )

if len(traj_df) != expected_trajectories:
    raise AssertionError(
        f"Expected {expected_trajectories} p1->p2 trajectories, found {len(traj_df)}."
    )

for col in ["fp_rate", "fn_rate"]:
    if not df_pts[col].between(0.0, 1.0).all():
        bad = df_pts.loc[~df_pts[col].between(0.0, 1.0), ["label", "prompt", col]]
        raise AssertionError(f"{col} outside [0, 1]:\n{bad.to_string(index=False)}")

# The figure must contain no interpretive region shading or region captions:
# those are not part of the paper figure.
forbidden_texts = {"Low-error region", "FP-dominant", "FN-dominant"}
rendered_texts = {t.get_text() for t in ax.texts}

leaked = {f for f in forbidden_texts if any(f in t for t in rendered_texts)}
if leaked:
    raise AssertionError(
        f"Figure contains interpretive region captions not present in the "
        f"paper figure: {sorted(leaked)}"
    )

n_patches = sum(1 for p in ax.patches if isinstance(p, mpatches.Rectangle))
if n_patches != 0:
    raise AssertionError(
        f"Figure contains {n_patches} rectangle patch(es); the paper figure "
        "has no region shading."
    )

print("\u2713 FIGURE STRUCTURE CHECK PASSED")
print(f"  {len(df_pts)} points, {len(traj_df)} trajectories, no region shading.")

print("\n" + "=" * 80)
print("Error-direction space complete.")
print("=" * 80)
print("Figure saved:")
print("  ", FIG_OUT)
print("Audit CSVs saved:")
print("  ", POINTS_CSV)
print("  ", TRAJ_CSV)
print("=" * 80)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# CELL C7: Per-dataset prompt-ablation breakdown from release master
#
# Purpose:
#   Compute per-dataset Cohen kappa values for each judge, generator, and prompt.
#
# Source of truth:
#   MASTER_PATH = data/human_judge_release_master.csv
#
# Uses:
#   annotator1_label
#   label_<judge>_p1 / label_<judge>_p2 / label_<judge>_p3
#
# Safety:
#   - Reads only from MASTER_PATH.
#   - Does not use temp_results.csv.
#   - Does not use load_results(...), human_label, P2B, or P3B.
#
# Outputs:
#   audits/per_dataset_prompt_ablation/per_dataset_prompt_kappa_long.csv
#   audits/per_dataset_prompt_ablation/per_dataset_prompt_kappa_p1_vs_p2.csv
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL C7: Per-dataset prompt-ablation breakdown"

if not RUN_RELEASE_ANALYSIS:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_RELEASE_ANALYSIS=False.")
    print("No per-dataset prompt-ablation analysis was generated.")

else:
    # =========================================================================
    # Paths
    # =========================================================================

    PER_DATASET_DIR = AUDIT_DIR / "per_dataset_prompt_ablation"
    PER_DATASET_DIR.mkdir(parents=True, exist_ok=True)

    OUT_LONG_CSV = PER_DATASET_DIR / "per_dataset_prompt_kappa_long.csv"
    OUT_P1_P2_CSV = PER_DATASET_DIR / "per_dataset_prompt_kappa_p1_vs_p2.csv"

    # =========================================================================
    # Config
    # =========================================================================

    HUMAN_COL = "annotator1_label"

    PROMPT_ORDER = ["p1", "p2", "p3"]

    GEN_LABELS = {
        "llama3-8b": "L",
        "gemma-2-9b": "G",
        "mistral-7b": "M",
    }

    DATASET_LABELS = {
        "triviaqa": "TriviaQA",
        "hotpotqa": "HotpotQA",
        "truthfulqa": "TruthfulQA",
    }

    JUDGES = [
        ("GPT-5-mini",         "gpt_5_mini",        "api"),
        ("GPT-5-nano",         "gpt_5_nano",        "api"),
        ("GPT-5.4",            "gpt_5_4",           "api"),
        ("Claude Opus 4.7",    "claude_opus_4_7",   "api"),
        ("Gemma-2-9B",         "gemma_2_9b_local",  "local"),
        ("Qwen2.5-7B",         "qwen2_5_7b_local",  "local"),
        ("Llama-3-8B",         "llama_3_8b_local",  "local"),
    ]

    # =========================================================================
    # Helpers
    # =========================================================================

    def safe_kappa(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)

        try:
            k = cohen_kappa_score(y_true, y_pred, labels=[0, 1])
            return float(k) if np.isfinite(k) else np.nan
        except Exception:
            return np.nan


    def compute_metrics_for_subset(df, human_col, pred_col):
        if pred_col not in df.columns:
            raise KeyError(f"Missing prediction column: {pred_col}")

        yh_all = pd.to_numeric(df[human_col], errors="coerce")
        yp_all = pd.to_numeric(df[pred_col], errors="coerce")

        mask = yh_all.isin([0, 1]) & yp_all.isin([0, 1])

        n_total = int(len(df))
        n_valid = int(mask.sum())

        if n_valid < 5:
            return {
                "n_total": n_total,
                "n_valid": n_valid,
                "kappa": np.nan,
                "bias": np.nan,
                "fp": np.nan,
                "fn": np.nan,
                "err": np.nan,
                "fp_pct": np.nan,
                "has_invalid": bool(n_valid < n_total),
            }

        yh = yh_all.loc[mask].astype(int).to_numpy()
        yp = yp_all.loc[mask].astype(int).to_numpy()

        kappa = safe_kappa(yh, yp)
        bias = float(yp.mean() - yh.mean())

        tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()
        err = int(fp + fn)
        fp_pct = 100.0 * fp / max(err, 1)

        return {
            "n_total": n_total,
            "n_valid": n_valid,
            "kappa": float(kappa),
            "bias": bias,
            "fp": int(fp),
            "fn": int(fn),
            "err": err,
            "fp_pct": float(fp_pct),
            "has_invalid": bool(n_valid < n_total),
        }


    def assert_no_internal_prompt_names(df):
        if "prompt" in df.columns:
            bad = df["prompt"].astype(str).isin(["P1", "P2", "P3", "P2B", "P3B"])
            if bad.any():
                raise AssertionError(
                    "Internal/uppercase prompt names found. "
                    "Expected paper-facing prompt names: p1, p2, p3."
                )

    # =========================================================================
    # Load and validate release master
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)

    required_core = [
        "model",
        "dataset",
        HUMAN_COL,
    ]

    missing_core = [c for c in required_core if c not in master.columns]

    if missing_core:
        raise KeyError(f"Missing required core columns: {missing_core}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows in release master, got {len(master)}")

    if not master[HUMAN_COL].isin([0, 1]).all():
        bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
        raise ValueError(f"Invalid {HUMAN_COL} values: {bad_n}")

    if "TARGET_MODELS" not in globals():
        TARGET_MODELS = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    if "TARGET_DATASETS" not in globals():
        TARGET_DATASETS = ["triviaqa", "hotpotqa", "truthfulqa"]

    required_label_cols = [
        f"label_{prefix}_{p}"
        for _, prefix, _ in JUDGES
        for p in PROMPT_ORDER
    ]

    missing_label_cols = [c for c in required_label_cols if c not in master.columns]

    if missing_label_cols:
        raise KeyError(
            "Missing required judge label columns from release master:\n"
            + "\n".join(f"  - {c}" for c in missing_label_cols)
        )

    forbidden_cols = [
        c for c in master.columns
        if "P2B" in c
        or "P3B" in c
        or c.endswith("_P1")
        or c.endswith("_P2")
        or c.endswith("_P3")
    ]

    if forbidden_cols:
        raise ValueError(
            "Old/internal prompt-style columns found in release master: "
            f"{forbidden_cols}"
        )

    print("=" * 80)
    print("PER-DATASET PROMPT-ABLATION BREAKDOWN")
    print("=" * 80)
    print("Release master:", MASTER_PATH)
    print("Shape:", master.shape)
    print("\nRows by dataset:")
    print(master["dataset"].value_counts().reindex(TARGET_DATASETS))
    print("\nRows by generator:")
    print(master["model"].value_counts().reindex(TARGET_MODELS))

    # =========================================================================
    # Compute long-form per-dataset rows
    # =========================================================================

    rows = []

    for judge_display, judge_prefix, judge_family in JUDGES:
        for model_name in TARGET_MODELS:
            for dataset_name in TARGET_DATASETS:
                sub = master[
                    (master["model"] == model_name)
                    & (master["dataset"] == dataset_name)
                ].copy()

                if len(sub) == 0:
                    raise ValueError(
                        f"No rows for model={model_name}, dataset={dataset_name}"
                    )

                for p in PROMPT_ORDER:
                    pred_col = f"label_{judge_prefix}_{p}"

                    metrics = compute_metrics_for_subset(
                        sub,
                        HUMAN_COL,
                        pred_col,
                    )

                    rows.append(
                        {
                            "judge": judge_display,
                            "judge_prefix": judge_prefix,
                            "judge_family": judge_family,
                            "generator": model_name,
                            "gen": GEN_LABELS.get(model_name, model_name),
                            "dataset": dataset_name,
                            "dataset_display": DATASET_LABELS.get(dataset_name, dataset_name),
                            "prompt": p,
                            "label_col": pred_col,
                            **metrics,
                        }
                    )

    df_long = pd.DataFrame(rows)

    judge_order = [j[0] for j in JUDGES]

    df_long["judge"] = pd.Categorical(
        df_long["judge"],
        categories=judge_order,
        ordered=True,
    )
    df_long["generator"] = pd.Categorical(
        df_long["generator"],
        categories=TARGET_MODELS,
        ordered=True,
    )
    df_long["dataset"] = pd.Categorical(
        df_long["dataset"],
        categories=TARGET_DATASETS,
        ordered=True,
    )
    df_long["prompt"] = pd.Categorical(
        df_long["prompt"],
        categories=PROMPT_ORDER,
        ordered=True,
    )

    df_long = df_long.sort_values(
        ["judge", "generator", "dataset", "prompt"]
    ).reset_index(drop=True)

    assert_no_internal_prompt_names(df_long)

    df_long.to_csv(OUT_LONG_CSV, index=False)

    print("\nSaved long per-dataset CSV:")
    print(OUT_LONG_CSV)
    print("Rows:", len(df_long))

    # =========================================================================
    # Compact p1 vs p2 summary
    # =========================================================================

    p1_p2 = df_long[df_long["prompt"].isin(["p1", "p2"])].copy()

    compact_rows = []

    for judge_display, judge_prefix, judge_family in JUDGES:
        for dataset_name in TARGET_DATASETS:
            for model_name in TARGET_MODELS:
                sub = p1_p2[
                    (p1_p2["judge"] == judge_display)
                    & (p1_p2["dataset"] == dataset_name)
                    & (p1_p2["generator"] == model_name)
                ]

                p1_row = sub[sub["prompt"] == "p1"]
                p2_row = sub[sub["prompt"] == "p2"]

                if len(p1_row) != 1 or len(p2_row) != 1:
                    raise ValueError(
                        f"Missing p1/p2 rows for {judge_display}, "
                        f"{dataset_name}, {model_name}"
                    )

                p1_k = float(p1_row.iloc[0]["kappa"])
                p2_k = float(p2_row.iloc[0]["kappa"])

                compact_rows.append(
                    {
                        "judge": judge_display,
                        "judge_prefix": judge_prefix,
                        "judge_family": judge_family,
                        "dataset": dataset_name,
                        "dataset_display": DATASET_LABELS.get(dataset_name, dataset_name),
                        "generator": model_name,
                        "gen": GEN_LABELS.get(model_name, model_name),
                        "p1_kappa": p1_k,
                        "p2_kappa": p2_k,
                        "delta_p2_minus_p1": p2_k - p1_k,
                    }
                )

    df_compact = pd.DataFrame(compact_rows)

    df_compact["judge"] = pd.Categorical(
        df_compact["judge"],
        categories=judge_order,
        ordered=True,
    )
    df_compact["dataset"] = pd.Categorical(
        df_compact["dataset"],
        categories=TARGET_DATASETS,
        ordered=True,
    )
    df_compact["generator"] = pd.Categorical(
        df_compact["generator"],
        categories=TARGET_MODELS,
        ordered=True,
    )

    df_compact = df_compact.sort_values(
        ["judge", "dataset", "generator"]
    ).reset_index(drop=True)

    df_compact.to_csv(OUT_P1_P2_CSV, index=False)

    print("\nSaved compact p1-vs-p2 CSV:")
    print(OUT_P1_P2_CSV)

    # =========================================================================
    # Printed summaries
    # =========================================================================

    print("\n" + "=" * 80)
    print("FULL PER-DATASET κ TABLES")
    print("=" * 80)

    for judge_display, judge_prefix, _ in JUDGES:
        print("\n" + "-" * 80)
        print(judge_display)
        print("-" * 80)

        sub_j = df_long[df_long["judge"] == judge_display].copy()

        for model_name in TARGET_MODELS:
            gen = GEN_LABELS.get(model_name, model_name)

            print(f"\nGenerator: {model_name} ({gen})")

            table = sub_j[sub_j["generator"] == model_name].pivot_table(
                index="dataset_display",
                columns="prompt",
                values="kappa",
                aggfunc="first",
                observed=False,
            )

            table = table.reindex(
                [DATASET_LABELS.get(ds, ds) for ds in TARGET_DATASETS]
            )

            display(table.round(3))

    print("\n" + "=" * 80)
    print("COMPACT p1 vs p2 SUMMARY")
    print("=" * 80)

    for judge_display, _, _ in JUDGES:
        print("\n" + "-" * 80)
        print(judge_display)
        print("-" * 80)

        sub_j = df_compact[df_compact["judge"] == judge_display].copy()

        table = sub_j.pivot_table(
            index="dataset_display",
            columns="gen",
            values=["p1_kappa", "p2_kappa", "delta_p2_minus_p1"],
            aggfunc="first",
            observed=False,
        )

        display(table.round(3))

    # =========================================================================
    # Guardrails
    # =========================================================================

    expected_rows = len(JUDGES) * len(TARGET_MODELS) * len(TARGET_DATASETS) * len(PROMPT_ORDER)

    if len(df_long) != expected_rows:
        raise AssertionError(
            f"Expected {expected_rows} long rows, got {len(df_long)}"
        )

    if df_long["kappa"].isna().any():
        n_nan = int(df_long["kappa"].isna().sum())
        raise AssertionError(
            f"Found NaN kappa values in per-dataset breakdown: {n_nan}"
        )

    if df_long["prompt"].astype(str).isin(["P1", "P2", "P3", "P2B", "P3B"]).any():
        raise AssertionError(
            "Internal prompt names leaked into per-dataset output."
        )

    for path in [OUT_LONG_CSV, OUT_P1_P2_CSV]:
        if not path.exists():
            raise FileNotFoundError(f"Expected output was not created: {path}")

    print("\n" + "=" * 80)
    print("PER-DATASET PROMPT-ABLATION BREAKDOWN COMPLETE")
    print("=" * 80)
    print("Outputs:")
    print("  ", OUT_LONG_CSV)
    print("  ", OUT_P1_P2_CSV)
    print("=" * 80)

In [ ]:
# =============================================================================
# CELL C8: GPT-5-mini per-dataset p1/p2 kappa table
#
# Purpose:
#   Generate the LaTeX table for the per-dataset breakdown of Cohen's kappa
#   between GPT-5-mini judge labels and human labels.
#
# Source of truth:
#   MASTER_PATH = data/human_judge_release_master.csv
#
# Uses:
#   annotator1_label
#   label_gpt_5_mini_p1
#   label_gpt_5_mini_p2
#
# Outputs:
#   tables/table_per_dataset_gpt5mini_kappa.tex
#   audits/per_dataset_gpt5mini/per_dataset_gpt5mini_kappa.csv
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score


# =============================================================================
# Self-check: should this cell run?
# =============================================================================

CELL_NAME = "CELL C8: GPT-5-mini per-dataset p1/p2 kappa table"

if not RUN_RELEASE_ANALYSIS:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_RELEASE_ANALYSIS=False.")
    print("No per-dataset GPT-5-mini table was generated.")

else:
    # =========================================================================
    # Paths
    # =========================================================================

    OUT_DIR = TABLE_DIR
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    AUDIT_OUT_DIR = AUDIT_DIR / "per_dataset_gpt5mini"
    AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_TEX = OUT_DIR / "table_per_dataset_gpt5mini_kappa.tex"
    OUT_CSV = AUDIT_OUT_DIR / "per_dataset_gpt5mini_kappa.csv"

    # =========================================================================
    # Config
    # =========================================================================

    HUMAN_COL = "annotator1_label"

    JUDGE_PREFIX = "gpt_5_mini"

    PROMPTS = ["p1", "p2"]

    DATASET_ORDER = ["hotpotqa", "triviaqa", "truthfulqa"]

    DATASET_DISPLAY = {
        "hotpotqa": "HotpotQA",
        "triviaqa": "TriviaQA",
        "truthfulqa": "TruthfulQA",
    }

    MODEL_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    MODEL_DISPLAY = {
        "llama3-8b": "Llama-3-8B",
        "gemma-2-9b": "Gemma-2-9B",
        "mistral-7b": "Mistral-7B",
    }

    # =========================================================================
    # Helpers
    # =========================================================================

    def safe_kappa(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=int)
        y_pred = np.asarray(y_pred, dtype=int)

        try:
            k = cohen_kappa_score(y_true, y_pred, labels=[0, 1])
            return float(k) if np.isfinite(k) else np.nan
        except Exception:
            return np.nan


    def compute_kappa(df, human_col, pred_col):
        yh_all = pd.to_numeric(df[human_col], errors="coerce")
        yp_all = pd.to_numeric(df[pred_col], errors="coerce")

        mask = yh_all.isin([0, 1]) & yp_all.isin([0, 1])

        if int(mask.sum()) < 5:
            return np.nan, int(mask.sum()), int(len(df))

        yh = yh_all.loc[mask].astype(int).to_numpy()
        yp = yp_all.loc[mask].astype(int).to_numpy()

        return safe_kappa(yh, yp), int(mask.sum()), int(len(df))


    def latex_escape(text):
        text = str(text)
        replacements = {
            "\\": r"\textbackslash{}",
            "&": r"\&",
            "%": r"\%",
            "$": r"\$",
            "#": r"\#",
            "_": r"\_",
            "{": r"\{",
            "}": r"\}",
            "~": r"\textasciitilde{}",
            "^": r"\textasciicircum{}",
        }

        for old, new in replacements.items():
            text = text.replace(old, new)

        return text

    # =========================================================================
    # Load and validate release master
    # =========================================================================

    master = pd.read_csv(MASTER_PATH)

    required_cols = [
        "model",
        "dataset",
        HUMAN_COL,
        "label_gpt_5_mini_p1",
        "label_gpt_5_mini_p2",
    ]

    missing = [c for c in required_cols if c not in master.columns]

    if missing:
        raise KeyError(f"Missing required columns from release master: {missing}")

    if len(master) != 900:
        raise ValueError(f"Expected 900 rows in release master, got {len(master)}")

    if not master[HUMAN_COL].isin([0, 1]).all():
        bad_n = int((~master[HUMAN_COL].isin([0, 1])).sum())
        raise ValueError(f"Invalid {HUMAN_COL} values: {bad_n}")

    for model_name in MODEL_ORDER:
        n = int((master["model"] == model_name).sum())
        if n != 300:
            raise ValueError(f"Expected 300 rows for {model_name}, got {n}")

    print("=" * 80)
    print("GPT-5-mini PER-DATASET p1/p2 KAPPA TABLE")
    print("=" * 80)
    print("Release master:", MASTER_PATH)
    print("Output table:  ", OUT_TEX)
    print("Audit CSV:     ", OUT_CSV)

    # =========================================================================
    # Compute table rows
    # =========================================================================

    rows = []

    for model_name in MODEL_ORDER:
        for dataset_name in DATASET_ORDER:
            sub = master[
                (master["model"] == model_name)
                & (master["dataset"] == dataset_name)
            ].copy()

            if len(sub) == 0:
                raise ValueError(
                    f"No rows found for model={model_name}, dataset={dataset_name}"
                )

            for prompt in PROMPTS:
                pred_col = f"label_{JUDGE_PREFIX}_{prompt}"

                kappa, n_valid, n_total = compute_kappa(
                    sub,
                    HUMAN_COL,
                    pred_col,
                )

                rows.append(
                    {
                        "generator": model_name,
                        "generator_display": MODEL_DISPLAY[model_name],
                        "dataset": dataset_name,
                        "dataset_display": DATASET_DISPLAY[dataset_name],
                        "prompt": prompt,
                        "label_col": pred_col,
                        "kappa": kappa,
                        "n_valid": n_valid,
                        "n_total": n_total,
                    }
                )

    df_table = pd.DataFrame(rows)

    df_table.to_csv(OUT_CSV, index=False)

    print("\nComputed values:")
    display(
        df_table.pivot_table(
            index="generator_display",
            columns=["dataset_display", "prompt"],
            values="kappa",
            aggfunc="first",
            observed=False,
        ).round(3)
    )

    # =========================================================================
    # Generate LaTeX
    # =========================================================================

    latex = r"""\begin{table}[t]
  \centering
  \caption{Per-dataset breakdown of Cohen's $\kappa$ agreement between the GPT-5-mini judge and human labels across HotpotQA, TriviaQA, and TruthfulQA. Results are shown for both the baseline (p1) and criterion-aligned (p2) prompts across three generator models.}
  \label{tab:per_dataset_gpt5mini}
  \small
  \setlength{\tabcolsep}{6pt}
  \renewcommand{\arraystretch}{0.95}
  \begin{tabular}{lrrrrrr}
    \toprule
    & \multicolumn{2}{c}{HotpotQA} & \multicolumn{2}{c}{TriviaQA} & \multicolumn{2}{c}{TruthfulQA} \\
    \cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}
    Generator & p1 & p2 & p1 & p2 & p1 & p2 \\
    \midrule
"""

    for model_name in MODEL_ORDER:
        model_display = MODEL_DISPLAY[model_name]

        cells = []

        for dataset_name in DATASET_ORDER:
            for prompt in PROMPTS:
                val = df_table[
                    (df_table["generator"] == model_name)
                    & (df_table["dataset"] == dataset_name)
                    & (df_table["prompt"] == prompt)
                ]["kappa"]

                if len(val) != 1:
                    raise ValueError(
                        f"Missing value for {model_name}, {dataset_name}, {prompt}"
                    )

                cells.append(f"{float(val.iloc[0]):.3f}")

        latex += (
            f"    {latex_escape(model_display)} "
            f"& {cells[0]} & {cells[1]} "
            f"& {cells[2]} & {cells[3]} "
            f"& {cells[4]} & {cells[5]} \\\\\n"
        )

    latex += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

    if "P2B" in latex or "P3B" in latex:
        raise ValueError("Internal prompt name leaked into LaTeX table.")

    with open(OUT_TEX, "w", encoding="utf-8") as f:
        f.write(latex)

    # =========================================================================
    # Guardrails
    # =========================================================================

    expected_rows = len(MODEL_ORDER) * len(DATASET_ORDER) * len(PROMPTS)

    if len(df_table) != expected_rows:
        raise AssertionError(
            f"Expected {expected_rows} rows, got {len(df_table)}"
        )

    if df_table["kappa"].isna().any():
        bad = df_table[df_table["kappa"].isna()]
        display(bad)
        raise AssertionError("NaN kappa values found in per-dataset GPT-5-mini table.")

    if (df_table["n_valid"] != df_table["n_total"]).any():
        bad = df_table[df_table["n_valid"] != df_table["n_total"]]
        display(bad)
        raise AssertionError("Some per-dataset GPT-5-mini cells have invalid labels.")

    print("\nSaved outputs:")
    print("  ", OUT_TEX)
    print("  ", OUT_CSV)

    print("\n✓ GPT-5-mini per-dataset p1/p2 kappa table complete.")

In [ ]:
# =============================================================================
# CELL C9: Criterion x structure factorial — appendix tables
#
# Purpose:
#   Build the appendix tables for the 2x2 prompt experiment from the stored
#   labels of CELL B8. Makes no API calls.
#
#                     terse          structured
#     faithfulness    p1             p1s
#     factual         p2t            p2
#
#   Four simple contrasts. Two hold structure constant and vary the criterion;
#   two hold the criterion constant and vary structure. Together they separate
#   the effect of the labeling criterion from the effect of prompt elaboration,
#   and expose the interaction: added structure is inert under the factual
#   criterion but harmful under the faithfulness criterion.
#
# Source of truth:
#   audits/factorial_prompt_run/<RUN_ID>/labels.csv
#
# Outputs:
#   tables/table_factorial_agreement.tex      (tab:factorial_agreement)
#   tables/table_factorial_contrasts.tex      (tab:factorial_contrasts)
#   tables/prompt_p1s_verbatim.tex            p1s prompt text for the appendix
#   tables/prompt_p2t_verbatim.tex            p2t prompt text for the appendix
#   tables/prompt_p1_vs_p1s_diff.tex          p1 -> p1s unified diff
#   audits/factorial_prompt_run/<RUN_ID>/agreement_summary.csv
#   audits/factorial_prompt_run/<RUN_ID>/paired_contrasts.csv
#
# Paper labels:
#   tab:factorial_agreement
#   tab:factorial_contrasts
#
# -----------------------------------------------------------------------------
# CHANGES IN THIS REVISION
#
#   1. The p1 -> p1s contrast (structure varied under the faithfulness
#      criterion) completes the fourth cell of the 2x2 design. No new API
#      calls: label_p1 and label_p1s are already in labels.csv.
#
#   2. LaTeX output now matches the manuscript exactly: table* + \scriptsize,
#      "FPR" column header, "Structure: terse" style held-constant strings,
#      hyphenated "Criterion-by-structure". Previously these were hand-edited
#      after generation, so the generated file and the paper had drifted.
#
#   3. Prompt listings are emitted as lstlisting[style=promptstyle] rather than
#      verbatim, matching the five existing prompt listings in the appendix.
#      p2t verbatim is now generated too (it was hand-copied before).
#
#   4. Contrast invariants are asserted, not assumed: full bootstrap sample,
#      label/correctness disagreement identity, marginal consistency, and
#      cross-contrast kappa agreement.
#
#   5. Pre-flight warning when COLUMN_ALIASES resolves to a non-canonical
#      column: a different run than the one reported in the paper would
#      give different pooled kappas.
#
#   Note on RNG: bootstrap_delta_kappa seeds a fresh Generator per call, so
#   contrast order does not affect any interval. Adding the fourth contrast
#   leaves the three published rows bit-identical.
# =============================================================================

import json
import difflib

import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score, confusion_matrix
from IPython.display import display

CELL_NAME = "CELL C9: Factorial prompt experiment tables"

if not RUN_RELEASE_ANALYSIS:
    print("=" * 80)
    print(f"SKIPPED: {CELL_NAME}")
    print("=" * 80)
    print("RUN_RELEASE_ANALYSIS=False.")

else:
    # =========================================================================
    # Paths
    # =========================================================================

    FACTORIAL_RUN_ID = globals().get(
        "FACTORIAL_RUN_ID",
        "gpt5mini_factorial_v3",
    )

    RUN_DIR = AUDIT_DIR / "factorial_prompt_run" / FACTORIAL_RUN_ID
    LABELS_PATH = RUN_DIR / "labels.csv"

    OUT_SUMMARY_CSV = RUN_DIR / "agreement_summary.csv"
    OUT_PAIRS_CSV = RUN_DIR / "paired_contrasts.csv"

    OUT_AGREEMENT_TEX = TABLE_DIR / "table_factorial_agreement.tex"
    OUT_CONTRASTS_TEX = TABLE_DIR / "table_factorial_contrasts.tex"
    OUT_P1S_TEX = TABLE_DIR / "prompt_p1s_verbatim.tex"
    OUT_P2T_TEX = TABLE_DIR / "prompt_p2t_verbatim.tex"
    OUT_DIFF_TEX = TABLE_DIR / "prompt_p1_vs_p1s_diff.tex"

    if not LABELS_PATH.exists():
        raise FileNotFoundError(
            f"Factorial labels not found: {LABELS_PATH}\n"
            "Run CELL B8, or copy the stored labels.csv into this run directory."
        )

    # =========================================================================
    # Config
    # =========================================================================

    HUMAN_COL = "annotator1_label"
    QID_COL = "qid"
    GEN_ORDER = ["llama3-8b", "gemma-2-9b", "mistral-7b"]

    GEN_DISPLAY = {
        "llama3-8b": "L",
        "gemma-2-9b": "G",
        "mistral-7b": "M",
        "ALL": "All",
    }

    # Display name, criterion, structure
    PROMPT_DEFS = [
        ("p1",  "p1",  "Faithfulness", "Terse"),
        ("p1s", "p1s", "Faithfulness", "Structured"),
        ("p2t", "p2t", "Factual",      "Terse"),
        ("p2",  "p2",  "Factual",      "Structured"),
    ]

    # Contrast name, prompt_a, prompt_b, held-constant label for the table.
    #
    # Rows 1-2 vary the criterion with structure held fixed.
    # Rows 3-4 vary the structure with the criterion held fixed.
    #
    # Row 4 (p1 -> p1s) completes the 2x2. Its point estimate is already
    # determined by the agreement table; the bootstrap supplies the interval.
    #
    # The held-constant strings are the manuscript's wording. Do not shorten
    # them back to "Structure held terse": the .tex is pasted into the paper
    # as generated.
    CONTRAST_DEFS = [
        ("Criterion, terse pair",            "p1",  "p2t", "Structure: terse"),
        ("Criterion, structured pair",       "p1s", "p2",  "Structure: structured"),
        ("Structure, factual criterion",     "p2t", "p2",  "Criterion: factual correctness"),
        ("Structure, faithfulness criterion", "p1", "p1s", "Criterion: reference faithfulness"),
    ]

    N_BOOT = 2000
    BOOT_SEED = 42

    # Some stored runs use longer prompt keys. Accept both so that an existing
    # labels.csv does not need to be regenerated.
    COLUMN_ALIASES = {
        "p1":  ["label_p1", "label_gpt_5_mini_p1_original", "label_p1_original"],
        "p1s": ["label_p1s"],
        "p2t": ["label_p2t", "label_gpt_5_mini_p2t", "label_p2t_terse"],
        "p2":  ["label_p2", "label_gpt_5_mini_p2_repaired", "label_p2_repaired"],
    }

    CANONICAL_COLS = {
        "p1": "label_p1",
        "p1s": "label_p1s",
        "p2t": "label_p2t",
        "p2": "label_p2",
    }

    # Guard against loading a different run than the one the manuscript reports.
    # Pooled kappa, rounded to three decimals. Set to None to disable.
    EXPECTED_POOLED_KAPPA = {
        "p1": 0.409,
        "p1s": 0.305,
        "p2t": 0.714,
        "p2": 0.739,
    }

    # =========================================================================
    # Helpers
    # =========================================================================

    def resolve_label_column(df, prompt_key):
        for candidate in COLUMN_ALIASES[prompt_key]:
            if candidate in df.columns:
                return candidate

        raise KeyError(
            f"No label column found for prompt {prompt_key!r}. "
            f"Tried: {COLUMN_ALIASES[prompt_key]}. "
            f"Available: {[c for c in df.columns if c.startswith('label_')]}"
        )


    def as_int_labels(series):
        """Coerce a label column to int, mapping unparseable values to -1."""
        return pd.to_numeric(series, errors="coerce").fillna(-1).to_numpy(dtype=int)


    def metrics(y_true, y_pred):
        """Agreement metrics on rows where both labels are valid."""
        mask = np.isin(y_true, [0, 1]) & np.isin(y_pred, [0, 1])

        yh = np.asarray(y_true)[mask].astype(int)
        yp = np.asarray(y_pred)[mask].astype(int)

        tn, fp, fn, tp = confusion_matrix(yh, yp, labels=[0, 1]).ravel()

        return {
            "n_valid": int(mask.sum()),
            "n_invalid": int((~mask).sum()),
            "kappa": float(cohen_kappa_score(yh, yp, labels=[0, 1])),
            "bias": float(yp.mean() - yh.mean()),
            "fp": int(fp),
            "fn": int(fn),
            "fpr": float(fp / max(fp + tn, 1)),
            "fnr": float(fn / max(fn + tp, 1)),
        }


    def bootstrap_delta_kappa(
        y_true,
        pred_a,
        pred_b,
        qids,
        n_boot=N_BOOT,
        seed=BOOT_SEED,
    ):
        """Question-level bootstrap interval for kappa(b) - kappa(a).

        Questions are resampled with replacement, keeping the three generator
        responses to each question together. Both kappas are recomputed on the
        same resampled questions.
        """
        rng = np.random.default_rng(seed)

        y_true = np.asarray(y_true).astype(int)
        pred_a = np.asarray(pred_a).astype(int)
        pred_b = np.asarray(pred_b).astype(int)
        qids = np.asarray(qids)

        unique_qids = pd.unique(qids)
        groups = [
            np.flatnonzero(qids == qid)
            for qid in unique_qids
        ]

        vals = []

        for _ in range(n_boot):
            sampled_groups = rng.integers(
                0,
                len(groups),
                size=len(groups),
            )
            idx = np.concatenate([
                groups[i]
                for i in sampled_groups
            ])

            ka = cohen_kappa_score(
                y_true[idx],
                pred_a[idx],
                labels=[0, 1],
            )
            kb = cohen_kappa_score(
                y_true[idx],
                pred_b[idx],
                labels=[0, 1],
            )
            delta = kb - ka

            if np.isfinite(delta):
                vals.append(delta)

        if not vals:
            return np.nan, np.nan, 0

        lo, hi = np.percentile(np.asarray(vals), [2.5, 97.5])

        return float(lo), float(hi), len(vals)




    def fmt_signed(x, digits=3):
        return f"$+${abs(x):.{digits}f}" if x >= 0 else f"$-${abs(x):.{digits}f}"


    def fmt_delta(x, digits=3):
        """Delta-kappa formatting.

        Positives keep the plain '+' used by the already published rows, so
        their rendering is unchanged. Negatives use a math minus, because a
        text-mode hyphen is not a minus sign and reads badly next to the
        positive rows.
        """
        return f"+{x:.{digits}f}" if x >= 0 else f"$-${abs(x):.{digits}f}"



    def latex_listing_block(text):
        """Wrap a prompt in lstlisting, matching the appendix prompt listings.

        Content is not escaped, which is correct for lstlisting. Trailing
        whitespace is stripped, but LEADING whitespace must be preserved: the
        diff's context lines begin with a significant space.
        """
        return (
            "\\begin{lstlisting}[style=promptstyle]\n"
            + text.rstrip()
            + "\n\\end{lstlisting}\n"
        )

    # =========================================================================
    # Load
    # =========================================================================

    labels = pd.read_csv(LABELS_PATH)

    if len(labels) != 900:
        raise ValueError(f"Expected 900 rows in labels.csv, found {len(labels)}")

    if HUMAN_COL not in labels.columns:
        raise KeyError(f"Missing {HUMAN_COL} in {LABELS_PATH.name}")

    if QID_COL not in labels.columns:
        raise KeyError(f"Missing {QID_COL} in {LABELS_PATH.name}")

    qid_sizes = labels.groupby(QID_COL).size()

    if len(qid_sizes) != 300 or not (qid_sizes == 3).all():
        raise ValueError(
            "Expected 300 questions with three generator responses each."
        )

    label_cols = {
        key: resolve_label_column(labels, key)
        for key, _, _, _ in PROMPT_DEFS
    }

    print("=" * 80)
    print("CRITERION x STRUCTURE FACTORIAL — APPENDIX TABLES")
    print("=" * 80)
    print("Run ID: ", FACTORIAL_RUN_ID)
    print("Labels: ", LABELS_PATH)
    print("Rows:   ", len(labels))
    print()
    print("Resolved prompt columns:")
    for key, col in label_cols.items():
        marker = "" if col == CANONICAL_COLS[key] else "   <-- NON-CANONICAL"
        print(f"  {key:<4} <- {col}{marker}")

    non_canonical = {
        k: v for k, v in label_cols.items() if v != CANONICAL_COLS[k]
    }

    if non_canonical:
        print()
        print("!" * 80)
        print("WARNING: non-canonical label columns resolved.")
        print("A different run than the one reported in the paper would give different pooled kappas.")
        print("Confirm this is the intended run before pasting the output.")
        print("!" * 80)

    for key, col in label_cols.items():
        n_valid = int(pd.to_numeric(labels[col], errors="coerce").isin([0, 1]).sum())
        if n_valid != len(labels):
            raise ValueError(
                f"{col} has invalid labels: valid={n_valid}/{len(labels)}. "
                "Rerun CELL B8 to fill the failed items."
            )

    print("\n✓ All four prompt columns are complete binary labels.")

    # =========================================================================
    # Agreement summary
    # =========================================================================

    summary_rows = []

    for key, prompt_display, criterion, structure in PROMPT_DEFS:
        col = label_cols[key]

        for generator in GEN_ORDER + ["ALL"]:
            sub = labels if generator == "ALL" else labels[labels["model"] == generator]

            y = sub[HUMAN_COL].to_numpy(dtype=int)
            p = as_int_labels(sub[col])

            m = metrics(y, p)

            summary_rows.append({
                "prompt": prompt_display,
                "criterion": criterion,
                "structure": structure,
                "generator": generator,
                "gen_display": GEN_DISPLAY[generator],
                **m,
            })

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(OUT_SUMMARY_CSV, index=False)

    print("\n" + "=" * 80)
    print("AGREEMENT WITH HUMAN LABELS")
    print("=" * 80)
    display(
        summary[[
            "prompt", "criterion", "structure", "gen_display",
            "kappa", "bias", "fp", "fn", "fpr", "fnr",
        ]].round(3)
    )

    # Guard against silently analysing a different run.
    if EXPECTED_POOLED_KAPPA:
        drift = []

        for key, expected in EXPECTED_POOLED_KAPPA.items():
            got = float(
                summary[
                    (summary["prompt"] == key) & (summary["generator"] == "ALL")
                ]["kappa"].iloc[0]
            )

            if round(got, 3) != expected:
                drift.append((key, expected, round(got, 3)))

        if drift:
            print("\n" + "!" * 80)
            print("WARNING: pooled kappa differs from the manuscript values.")
            for key, expected, got in drift:
                print(f"  {key:<4} expected {expected:.3f}, got {got:.3f}")
            print("Either the wrong run is loaded, or the manuscript needs updating.")
            print("!" * 80)
        else:
            print("\n✓ Pooled kappa matches the manuscript for all four prompts.")

    # =========================================================================
    # Paired contrasts
    # =========================================================================

    y_all = labels[HUMAN_COL].to_numpy(dtype=int)
    qid_all = labels[QID_COL].astype(str).to_numpy()

    pair_rows = []

    for contrast_name, key_a, key_b, held_constant in CONTRAST_DEFS:
        a = as_int_labels(labels[label_cols[key_a]])
        b = as_int_labels(labels[label_cols[key_b]])

        valid = np.isin(y_all, [0, 1]) & np.isin(a, [0, 1]) & np.isin(b, [0, 1])

        yv = y_all[valid]
        av = a[valid]
        bv = b[valid]
        qv = qid_all[valid]

        ka = float(cohen_kappa_score(yv, av, labels=[0, 1]))
        kb = float(cohen_kappa_score(yv, bv, labels=[0, 1]))

        lo, hi, n_boot_valid = bootstrap_delta_kappa(
            yv,
            av,
            bv,
            qv,
        )


        pair_rows.append({
            "contrast": contrast_name,
            "held_constant": held_constant,
            "prompt_a": key_a,
            "prompt_b": key_b,
            "n": int(len(yv)),
            "n_correct_a": int(np.sum(av == yv)),
            "n_correct_b": int(np.sum(bv == yv)),
            "label_differences": int(np.sum(av != bv)),
            "difference_pct": float(100.0 * np.mean(av != bv)),
            "kappa_a": ka,
            "kappa_b": kb,
            "delta_kappa": kb - ka,
            "ci_low": lo,
            "ci_high": hi,
            "n_boot_valid": n_boot_valid,

        })

    pairs = pd.DataFrame(pair_rows)
    pairs.to_csv(OUT_PAIRS_CSV, index=False)

    print("\n" + "=" * 80)
    print("PAIRED CONTRASTS")
    print("=" * 80)
    display(
        pairs[[
            "contrast", "prompt_a", "prompt_b", "difference_pct",
            "kappa_a", "kappa_b", "delta_kappa", "ci_low", "ci_high",

        ]].round(4)
    )

    # =========================================================================
    # Contrast invariants
    #
    # These are identities, not empirical expectations. A failure means a
    # non-binary label leaked through, rows are misaligned, or the bootstrap
    # silently discarded resamples.
    # =========================================================================

    for _, r in pairs.iterrows():
        tag = f"{r['prompt_a']}->{r['prompt_b']}"

        # The caption promises 2000 resamples.
        assert r["n_boot_valid"] == N_BOOT, (
            f"{tag}: only {r['n_boot_valid']}/{N_BOOT} bootstrap resamples were "
            "usable; degenerate resamples were dropped and the caption's "
            "resample count is no longer accurate."
        )


        # Delta must be the difference of the two reported kappas.
        assert abs(r["delta_kappa"] - (r["kappa_b"] - r["kappa_a"])) < 1e-12, (
            f"{tag}: delta_kappa does not equal kappa_b - kappa_a."
        )

    # A prompt appearing in several contrasts must carry the same kappa.
    kappa_by_prompt = {}

    for _, r in pairs.iterrows():
        for side in ["a", "b"]:
            key = r[f"prompt_{side}"]
            val = float(r[f"kappa_{side}"])

            if key in kappa_by_prompt:
                assert abs(kappa_by_prompt[key] - val) < 1e-12, (
                    f"{key}: kappa differs across contrasts "
                    f"({kappa_by_prompt[key]:.6f} vs {val:.6f}). "
                    "The contrasts are not on a common item set."
                )
            else:
                kappa_by_prompt[key] = val

    # The 2x2 must close: the interaction is the same computed either way.
    if {"p1", "p1s", "p2t", "p2"} <= set(kappa_by_prompt):
        by_rows = (
            (kappa_by_prompt["p2"] - kappa_by_prompt["p2t"])
            - (kappa_by_prompt["p1s"] - kappa_by_prompt["p1"])
        )
        by_cols = (
            (kappa_by_prompt["p2"] - kappa_by_prompt["p1s"])
            - (kappa_by_prompt["p2t"] - kappa_by_prompt["p1"])
        )

        assert abs(by_rows - by_cols) < 1e-12, "2x2 interaction is not additive."

        print(f"\n✓ CONTRAST INVARIANTS PASSED")
        print(f"  Interaction (structure x criterion): {by_rows:+.3f}")
    else:
        print("\n✓ CONTRAST INVARIANTS PASSED")

    # =========================================================================
    # tab:factorial_agreement — agreement by prompt and generator
    # =========================================================================

    agreement_caption = (
        r"Criterion-by-structure factorial prompt experiment with GPT-5-mini. "
        r"All four prompts were evaluated on the same 900 items in a single "
        r"interleaved pass. p1 and p2 are the paper prompts; p1s applies the "
        r"reference-faithfulness criterion in p2's structured decision-tree "
        r"format, and p2t applies the factual-correctness criterion in p1's "
        r"terse format. Bias is positive when the automated labeler "
        r"over-labels hallucinations relative to humans. L=Llama-3-8B, "
        r"G=Gemma-2-9B, M=Mistral-7B; All pools the three generator sets."
    )

    latex_agreement = r"""\begin{table*}[t]
  \centering
  \caption{""" + agreement_caption + r"""}
  \label{tab:factorial_agreement}
  \scriptsize
  \setlength{\tabcolsep}{4pt}
  \begin{tabular}{lllrrrrrr}
    \toprule
    Prompt & Criterion & Structure & Gen & $\kappa$ & Bias & FP & FN & FPR \\
    \midrule
"""

    n_agreement_rows = 0
    prev_prompt = None

    for key, prompt_display, criterion, structure in PROMPT_DEFS:
        block = summary[summary["prompt"] == prompt_display]

        if prev_prompt is not None:
            latex_agreement += "    \\addlinespace[2pt]\n"

        prev_prompt = prompt_display

        for generator in GEN_ORDER + ["ALL"]:
            r = block[block["generator"] == generator].iloc[0]

            latex_agreement += (
                f"    {prompt_display} & {criterion} & {structure} & "
                f"{GEN_DISPLAY[generator]} & "
                f"{r['kappa']:.3f} & {fmt_signed(r['bias'])} & "
                f"{int(r['fp'])} & {int(r['fn'])} & {r['fpr']:.3f} \\\\\n"
            )

            n_agreement_rows += 1

    latex_agreement += r"""    \bottomrule
  \end{tabular}
\end{table*}
"""

    with open(OUT_AGREEMENT_TEX, "w", encoding="utf-8") as f:
        f.write(latex_agreement)

    # =========================================================================
    # tab:factorial_contrasts — paired contrasts
    # =========================================================================

    contrasts_caption = (
        r"Paired contrasts from the criterion-by-structure factorial prompt "
        r"experiment, pooled over the three generator sets. The first two rows "
        r"hold prompt structure constant and vary the labeling criterion; the "
        r"last two hold the criterion constant and vary prompt structure. "
        r"$\Delta\kappa=\kappa_b-\kappa_a$. Brackets give 95\% nonparametric "
        r"bootstrap intervals based on 2,000 question-level resamples, keeping "
        r"the three generator responses to each question together. Diff\% is the "
        r"proportion of items receiving different labels under the two prompts."
    )

    latex_contrasts = r"""\begin{table*}[t]
  \centering
  \caption{""" + contrasts_caption + r"""}
  \label{tab:factorial_contrasts}
  \scriptsize
  \setlength{\tabcolsep}{4pt}
  \begin{tabular}{llrrrl}
    \toprule
    Contrast & Held constant & Diff\% & $\kappa_a$ & $\kappa_b$
    & $\Delta\kappa$ [95\% CI] \\
    \midrule
"""

    for _, r in pairs.iterrows():
        latex_contrasts += (
            f"    {r['prompt_a']} $\\to$ {r['prompt_b']} & "
            f"{r['held_constant']}\n"
            f"      & {r['difference_pct']:.1f} & "
            f"{r['kappa_a']:.3f} & {r['kappa_b']:.3f}\n"
            f"      & {fmt_delta(r['delta_kappa'])} "
            f"[{r['ci_low']:+.3f}, {r['ci_high']:+.3f}] \\\\\n"
        )

    latex_contrasts += r"""    \bottomrule
  \end{tabular}
\end{table*}
"""

    with open(OUT_CONTRASTS_TEX, "w", encoding="utf-8") as f:
        f.write(latex_contrasts)

    # =========================================================================
    # Auxiliary prompt listings and the p1 / p1s difference
    #
    # The claim that p1s matches p2 in structural detail while keeping p1's
    # criterion is checkable only if the reader can see what changed. The diff
    # makes it checkable rather than asserted.
    # =========================================================================

    def _lookup_prompt(names):
        for name in names:
            val = globals().get(name, None)
            if isinstance(val, str) and val.strip():
                return val
        return None

    p1_text = _lookup_prompt(["PROMPT_P1", "P1"])
    p1s_text = _lookup_prompt(["PROMPT_P1S"])
    p2t_text = _lookup_prompt(["PROMPT_P2T"])

    if p1s_text is not None:
        with open(OUT_P1S_TEX, "w", encoding="utf-8") as f:
            f.write(latex_listing_block(p1s_text))

    if p2t_text is not None:
        with open(OUT_P2T_TEX, "w", encoding="utf-8") as f:
            f.write(latex_listing_block(p2t_text))

    if p1_text is None or p1s_text is None:
        print(
            "\nNOTE: p1 and/or p1s prompt text not in memory; "
            "skipping the diff file. Run CELL B1 and B8 first."
        )
    else:
        diff_lines = list(
            difflib.unified_diff(
                p1_text.splitlines(),
                p1s_text.splitlines(),
                fromfile="p1 (faithfulness, terse)",
                tofile="p1s (faithfulness, structured)",
                lineterm="",
                n=2,
            )
        )

        with open(OUT_DIFF_TEX, "w", encoding="utf-8") as f:
            f.write(latex_listing_block("\n".join(diff_lines)))

        overlong = [ln for ln in diff_lines if len(ln) > 80]

        print("\n" + "=" * 80)
        print("p1 vs p1s DIFFERENCE")
        print("=" * 80)
        print("\n".join(diff_lines))

        if overlong:
            print(f"\nNOTE: {len(overlong)} diff line(s) exceed 80 characters "
                  "and may overflow the listing box:")
            for ln in overlong:
                print(f"  [{len(ln):3d}] {ln}")

    # =========================================================================
    # Structural checks
    # =========================================================================

    expected_agreement_rows = len(PROMPT_DEFS) * (len(GEN_ORDER) + 1)

    if n_agreement_rows != expected_agreement_rows:
        raise AssertionError(
            f"tab:factorial_agreement: expected {expected_agreement_rows} data rows, "
            f"found {n_agreement_rows}."
        )

    if len(pairs) != len(CONTRAST_DEFS):
        raise AssertionError(
            f"tab:factorial_contrasts: expected {len(CONTRAST_DEFS)} contrasts, found {len(pairs)}."
        )

    print("\n✓ STRUCTURE CHECK PASSED")

    # =========================================================================
    # Values needed for the appendix prose
    # =========================================================================

    print("\n" + "=" * 80)
    print("NUMBERS FOR THE APPENDIX TEXT")
    print("=" * 80)

    for _, r in pairs.iterrows():
        print(
            f"  {r['prompt_a']:>3} -> {r['prompt_b']:<3}  "
            f"dkappa={r['delta_kappa']:+.3f} "
            f"[{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]  "
            f"Diff%={r['difference_pct']:.1f}  "

        )

    fp_by_prompt = {
        key: int(
            summary[
                (summary["prompt"] == key) & (summary["generator"] == "ALL")
            ]["fp"].iloc[0]
        )
        for key, _, _, _ in PROMPT_DEFS
    }

    fn_by_prompt = {
        key: int(
            summary[
                (summary["prompt"] == key) & (summary["generator"] == "ALL")
            ]["fn"].iloc[0]
        )
        for key, _, _, _ in PROMPT_DEFS
    }

    print(
        f"\n  p1 -> p1s pooled: FP {fp_by_prompt['p1']} -> {fp_by_prompt['p1s']}, "
        f"FN {fn_by_prompt['p1']} -> {fn_by_prompt['p1s']}"
    )

    print("\n" + "=" * 80)
    print("tab:factorial_agreement: " + OUT_AGREEMENT_TEX.name)
    print("=" * 80)
    print(latex_agreement)

    print("=" * 80)
    print("tab:factorial_contrasts: " + OUT_CONTRASTS_TEX.name)
    print("=" * 80)
    print(latex_contrasts)
    saved = [OUT_AGREEMENT_TEX, OUT_CONTRASTS_TEX, OUT_SUMMARY_CSV, OUT_PAIRS_CSV]
    if p1s_text is not None:
        saved.append(OUT_P1S_TEX)
    if p2t_text is not None:
        saved.append(OUT_P2T_TEX)
    if p1_text is not None and p1s_text is not None:
        saved.append(OUT_DIFF_TEX)

    print("\nSaved:")
    for path in saved:
        print(" ", path)

In [ ]:
# =============================================================================
# CELL C10: Qualitative FP -> TN examples for the elaboration intervention
#
# Reads the paired results written by CELL B10b from disk and draws a
# reproducible random sample of FP -> TN flips, adding the reference answer
# from the release master. Makes no API calls.
#
# Paper label:
#   tab:elaboration_examples
# =============================================================================

QUALITATIVE_N = 10
QUALITATIVE_SEED = 42
REFERENCE_COL = "ref_str"

matches = sorted(
    (AUDIT_DIR / "elaboration_core_only")
    .glob("*/p1_original_vs_core_b4matched/paired_results.csv")
)
if len(matches) != 1:
    raise FileNotFoundError(
        f"Expected one paired_results.csv, found {len(matches)}. "
        "Run CELL B10b first."
    )
PAIRED_CSV = matches[0]
OUT_DIR = PAIRED_CSV.parent
EXAMPLES_CSV = OUT_DIR / "fp_to_tn_random10.csv"

paired = pd.read_csv(PAIRED_CSV)

flips = paired[
    paired["transition"] == "FP_TO_TN"
].copy()

sample_n = min(
    QUALITATIVE_N,
    len(flips),
)

examples = (
    flips.sample(
        n=sample_n,
        random_state=QUALITATIVE_SEED,
    )
    if sample_n
    else flips
)


# ---------------------------------------------------------------------------
# Add reference answer if paired does not already contain it
# ---------------------------------------------------------------------------

if REFERENCE_COL not in examples.columns:

    reference_lookup = (
        pd.read_csv(MASTER_PATH)[
            [
                "item_id",
                REFERENCE_COL,
            ]
        ]
        .drop_duplicates(subset="item_id")
    )

    examples = examples.merge(
        reference_lookup,
        on="item_id",
        how="left",
        validate="one_to_one",
    )


# Safety check
if examples[REFERENCE_COL].isna().any():
    missing = examples.loc[
        examples[REFERENCE_COL].isna(),
        "item_id",
    ].tolist()

    raise ValueError(
        f"Missing reference answers for: {missing}"
    )


# ---------------------------------------------------------------------------
# Final qualitative table
# ---------------------------------------------------------------------------

examples = examples[
    [
        "item_id",
        "question",
        REFERENCE_COL,
        "original_answer",
        "core_only_answer",
        "p1_original",
        "p1_core",
        "p1_original_confidence",
        "p1_core_confidence",
    ]
]


examples.to_csv(
    EXAMPLES_CSV,
    index=False,
)


print(
    f"\nRandom FP -> TN examples "
    f"(n={sample_n}, seed={QUALITATIVE_SEED}):"
)

display(examples)


print("\nSaved:")
print("Paired:  ", PAIRED_CSV)
print("Examples:", EXAMPLES_CSV)

In [ ]:
# =============================================================================
# CELL C11: ANSWER LENGTH ANALYSIS — FINAL HUMAN-VALIDATED SAMPLE
#
# Reads the paired original/core-only answers written by CELL B10b from
# disk. Makes no API calls.
# =============================================================================

import re
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Locate input
# ---------------------------------------------------------------------------

matches = sorted(
    (AUDIT_DIR / "elaboration_core_only")
    .glob("*/p1_original_vs_core_b4matched/paired_results.csv")
)
if len(matches) != 1:
    raise FileNotFoundError(
        f"Expected one paired_results.csv, found {len(matches)}. "
        "Run CELL B10b first."
    )
PAIRED_CSV = matches[0]
OUT_DIR = PAIRED_CSV.parent

paired = pd.read_csv(PAIRED_CSV)


# ---------------------------------------------------------------------------
# Length functions
# ---------------------------------------------------------------------------

# Same regex-based units used in the extraction validator.
# NOTE: These are NOT model-tokenizer tokens.
def regex_units(text):
    return re.findall(
        r"\w+|[^\w\s]",
        str(text),
        flags=re.UNICODE,
    )


def regex_unit_count(text):
    return len(regex_units(text))


# More intuitive word count for reporting.
# Counts word-like sequences, including contractions/hyphenated forms
# as single words where possible.
def word_count(text):
    return len(
        re.findall(
            r"\b[\w]+(?:['’\-][\w]+)*\b",
            str(text),
            flags=re.UNICODE,
        )
    )


# ---------------------------------------------------------------------------
# Build analysis dataframe
# ---------------------------------------------------------------------------

length_df = paired[
    [
        "item_id",
        "original_answer",
        "core_only_answer",
    ]
].copy()


# ---------------------------------------------------------------------------
# WORD COUNTS
# ---------------------------------------------------------------------------

length_df["original_words"] = (
    length_df["original_answer"]
    .fillna("")
    .apply(word_count)
)

length_df["core_words"] = (
    length_df["core_only_answer"]
    .fillna("")
    .apply(word_count)
)

length_df["words_removed"] = (
    length_df["original_words"]
    - length_df["core_words"]
)

length_df["word_fraction_removed"] = np.where(
    length_df["original_words"] > 0,
    length_df["words_removed"] / length_df["original_words"],
    np.nan,
)

length_df["word_percent_removed"] = (
    100 * length_df["word_fraction_removed"]
)


# ---------------------------------------------------------------------------
# REGEX-BASED UNITS
# Same units as extraction validator; NOT model tokens.
# ---------------------------------------------------------------------------

length_df["original_regex_units"] = (
    length_df["original_answer"]
    .fillna("")
    .apply(regex_unit_count)
)

length_df["core_regex_units"] = (
    length_df["core_only_answer"]
    .fillna("")
    .apply(regex_unit_count)
)

length_df["regex_units_removed"] = (
    length_df["original_regex_units"]
    - length_df["core_regex_units"]
)

length_df["regex_fraction_removed"] = np.where(
    length_df["original_regex_units"] > 0,
    length_df["regex_units_removed"]
    / length_df["original_regex_units"],
    np.nan,
)

length_df["regex_percent_removed"] = (
    100 * length_df["regex_fraction_removed"]
)


# ---------------------------------------------------------------------------
# Summary — WORDS
# ---------------------------------------------------------------------------

n_length = len(length_df)

original_words_total = int(
    length_df["original_words"].sum()
)

core_words_total = int(
    length_df["core_words"].sum()
)

overall_words_removed_pct = (
    100
    * (original_words_total - core_words_total)
    / original_words_total
)


print("\n" + "=" * 70)
print("ANSWER LENGTH — ORIGINAL vs CORE-ONLY")
print("=" * 70)

print(f"N: {n_length}")


print("\nWORD COUNTS")

print("\nOriginal answer:")
print(
    f"  Mean:   {length_df['original_words'].mean():.1f} words"
)
print(
    f"  Median: {length_df['original_words'].median():.1f} words"
)

print("\nCore-only answer:")
print(
    f"  Mean:   {length_df['core_words'].mean():.1f} words"
)
print(
    f"  Median: {length_df['core_words'].median():.1f} words"
)

print("\nAbsolute reduction per answer:")
print(
    f"  Mean:   {length_df['words_removed'].mean():.1f} words"
)
print(
    f"  Median: {length_df['words_removed'].median():.1f} words"
)

print("\nRelative reduction per answer:")
print(
    f"  Mean:   {length_df['word_percent_removed'].mean():.1f}%"
)
print(
    f"  Median: {length_df['word_percent_removed'].median():.1f}%"
)

print("\nTotal word count:")
print(f"  Original: {original_words_total}")
print(f"  Core:     {core_words_total}")
print(
    f"  Overall words removed: {overall_words_removed_pct:.1f}%"
)


# ---------------------------------------------------------------------------
# Summary — REGEX-BASED UNITS
# ---------------------------------------------------------------------------

original_regex_total = int(
    length_df["original_regex_units"].sum()
)

core_regex_total = int(
    length_df["core_regex_units"].sum()
)

overall_regex_removed_pct = (
    100
    * (original_regex_total - core_regex_total)
    / original_regex_total
)


print("\n" + "-" * 70)
print("REGEX-BASED UNITS")
print("(same counting scheme as extraction validator; NOT model tokens)")
print("-" * 70)

print(
    f"Original mean:   "
    f"{length_df['original_regex_units'].mean():.1f}"
)
print(
    f"Original median: "
    f"{length_df['original_regex_units'].median():.1f}"
)

print(
    f"Core mean:       "
    f"{length_df['core_regex_units'].mean():.1f}"
)
print(
    f"Core median:     "
    f"{length_df['core_regex_units'].median():.1f}"
)

print(
    f"Mean reduction:  "
    f"{length_df['regex_percent_removed'].mean():.1f}%"
)
print(
    f"Median reduction:"
    f" {length_df['regex_percent_removed'].median():.1f}%"
)

print(
    f"Overall units removed: "
    f"{overall_regex_removed_pct:.1f}%"
)


# ---------------------------------------------------------------------------
# Save row-level values for reproducibility
# ---------------------------------------------------------------------------

LENGTH_CSV = (
    OUT_DIR / "answer_length_analysis.csv"
)

length_df.to_csv(
    LENGTH_CSV,
    index=False,
)

print("\nSaved:")
print(LENGTH_CSV)